# Kenya Watch — Machine Learning Component
### Report Category Classification for Corruption Reports in Kenya

**Project:** A Machine Learning-Based Web Platform for Anonymous Reporting and Tracking of
Corruption Cases in Kenya
**Student:** Nyangabi Benvictor Mwangi (155499) — BBIT 4C, Strathmore University
**Supervisor:** Ms. Zainabu Muti

---

## Purpose

This notebook reproduces the offline ML pipeline in the project repository (`ml/src/`) so the
complete workflow can be inspected and re-run in one place. It is organised into:

1. Introduction and problem definition
2. Dataset and preprocessing
3. Methodology and model selection
4. Model evaluation
5. Deployment and honest limitations

**To run:** `Runtime ▸ Run all`. No uploads or setup required.

## ⚠️ Honest scope statement (please read first)

The corpus used here is a **synthetic, hand-authored demonstration corpus** of fictional
reports — not real citizen submissions. It exists to exercise and validate the *pipeline*.
Therefore:

- The scores in Section 4 measure performance **on the synthetic corpus only**.
- They are **not** an estimate of accuracy on real Kenyan corruption reports.
- The model produced here is exported to a fingerprinted JSON runtime artifact and is connected
  to the live reviewer workflow as an **experimental suggestion**. It is shown only after a
  reviewer records an independent category assessment. The live app still uses transparent
  keyword rules for submission-time category and urgency triage; the model never blocks the
  workflow or makes a decision.
- Using real reports requires the data-governance process documented in `ml/README.md`.


---
# 1. Introduction and Problem Definition

## 1.1 The problem

Corruption in Kenya is widespread, under-reported and rarely followed up. Citizens who witness or
experience bribery, embezzlement, procurement fraud or abuse of office are deterred by three things:

1. **Fear of retaliation** — reporting means becoming identifiable.
2. **No feedback** — complaints disappear, so trust in the process erodes.
3. **No prioritisation** — oversight bodies are overwhelmed, and serious cases are not triaged fast enough.

Kenya Watch addresses (1) with an anonymity-first architecture: anonymous reporters have **no
account at all**, and their identity is never collected, so there is nothing to leak. It addresses
(2) with a tracking reference + secret access key and a moderated agency-response loop.

This notebook addresses **(3) — triage**.

## 1.2 Where machine learning fits

A human review team cannot read and prioritise every narrative quickly. The platform therefore
uses lightweight text classification to **assist** triage:

- **Suggest a category** for each report (8 statutory categories + `Other`).
- **Suggest a category** to the reviewer. Urgency remains the separate, transparent keyword
  triage used by the live application.

**The ML layer is assistive, never decision-making:** keyword triage runs asynchronously after
submission so it can never block a report, while the experimental model suggestion is requested
only after a reviewer records an independent assessment. The model never sees reporter identity
(which is not collected for anonymous reports), and category labels used for training come from a
**double-blind human review** workflow, not from the reporter's own selection.

## 1.3 Why this matters

Speed of first response is the single biggest driver of citizen trust in a reporting platform. A
reliable triage assist shortens time-to-first-review, surfaces severe cases early, and — because the
system also collects independent human labels — makes the machine layer **measurable and
accountable** rather than a black box.


---
# 2. Dataset and Preprocessing
## 2.1 Dataset description

| Property | Value |
|---|---|
| Source | Combined deterministic **synthetic** seed, colloquial and realistic report-form corpora |
| Size | **2,916 rows** across **729 incident groups** |
| Registers | English, Kiswahili, mixed English–Kiswahili and Sheng |
| Labels | The 8 statutory categories (Anti-Corruption and Economic Crimes Act) + `Other` |
| Incident grouping | Each fictional incident = one `group_id`; all variants of one incident stay in the same fold |
| Real personal data | **None** |

**Schema:** `text, label, group_id, language, county`

- `text` — the report narrative (the only model input besides metadata used for evaluation).
- `label` — independently assignable category (the target).
- `group_id` — the **incident** key. Critical for leakage-free validation (explained in §3.2).
- `language`, `county` — used **only** for aggregate subgroup evaluation, never as features.

**Why synthetic first?** Training on real reports requires institutional approval, de-identification
and reviewed labels. The synthetic corpus validates the pipeline's *mechanics* — grouping, nested
validation, metric computation — before any sensitive data is touched.

## 2.2 The corpus is embedded below

So that this notebook runs anywhere with **no file uploads**, the 2,916-row combined corpus is embedded as a
CSV string. It is byte-identical to `ml/data/synthetic/kenya_watch_synthetic_combined.csv` in the
repository.


In [ ]:
%pip -q install "scikit-learn>=1.5,<2" "joblib>=1.4,<2" pandas matplotlib


In [ ]:
import io, csv, json, statistics
from collections import Counter
from pathlib import Path
import pandas as pd

CATEGORIES = (
    "Bribery", "Embezzlement", "Procurement Fraud", "Abuse of Office",
    "Nepotism/Favoritism", "Extortion", "Fraud/Forgery of Documents",
    "Conflict of Interest", "Other",
)

CSV_DATA = 'text,label,group_id,language,county\nA licensing officer allegedly asked for an unofficial payment before approving a complete small-business permit application.,Bribery,synthetic-bribery-01,English,Nairobi\nThe report states that a licensing officer allegedly asked for an unofficial payment before approving a complete small-business permit application.,Bribery,synthetic-bribery-01,English,Nairobi\nAfisa wa leseni anadaiwa kuomba malipo yasiyo rasmi kabla ya kuidhinisha ombi kamili la kibali cha biashara ndogo.,Bribery,synthetic-bribery-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, afisa wa leseni anadaiwa kuomba malipo yasiyo rasmi kabla ya kuidhinisha ombi kamili la kibali cha biashara ndogo.",Bribery,synthetic-bribery-01,Kiswahili,Nairobi\nA resident says a public clerk requested cash to move a land-record search ahead of other applications.,Bribery,synthetic-bribery-02,English,Mombasa\nThe report states that a resident says a public clerk requested cash to move a land-record search ahead of other applications.,Bribery,synthetic-bribery-02,English,Mombasa\nMkazi anasema karani wa umma aliomba pesa ili kuharakisha upekuzi wa rekodi ya ardhi kuliko maombi mengine.,Bribery,synthetic-bribery-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mkazi anasema karani wa umma aliomba pesa ili kuharakisha upekuzi wa rekodi ya ardhi kuliko maombi mengine.",Bribery,synthetic-bribery-02,Kiswahili,Mombasa\nA county employee allegedly offered to speed up inspection clearance in exchange for a personal payment.,Bribery,synthetic-bribery-03,English,Kisumu\nThe report states that a county employee allegedly offered to speed up inspection clearance in exchange for a personal payment.,Bribery,synthetic-bribery-03,English,Kisumu\nMfanyakazi wa kaunti anadaiwa kutoa ahadi ya kuharakisha kibali cha ukaguzi badala ya malipo binafsi.,Bribery,synthetic-bribery-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mfanyakazi wa kaunti anadaiwa kutoa ahadi ya kuharakisha kibali cha ukaguzi badala ya malipo binafsi.",Bribery,synthetic-bribery-03,Kiswahili,Kisumu\nA public health inspector allegedly requested a gift before issuing a routine food-premises certificate.,Bribery,synthetic-bribery-04,English,Nakuru\nThe report states that a public health inspector allegedly requested a gift before issuing a routine food-premises certificate.,Bribery,synthetic-bribery-04,English,Nakuru\nMkaguzi wa afya ya umma anadaiwa kuomba zawadi kabla ya kutoa cheti cha kawaida cha eneo la chakula.,Bribery,synthetic-bribery-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mkaguzi wa afya ya umma anadaiwa kuomba zawadi kabla ya kutoa cheti cha kawaida cha eneo la chakula.",Bribery,synthetic-bribery-04,Kiswahili,Nakuru\nA service applicant reports that an official asked for money to release a document that was already approved.,Bribery,synthetic-bribery-05,English,Turkana\nThe report states that a service applicant reports that an official asked for money to release a document that was already approved.,Bribery,synthetic-bribery-05,English,Turkana\nMwombaji wa huduma anaripoti kuwa afisa aliomba pesa ili kutoa hati ambayo tayari ilikuwa imeidhinishwa.,Bribery,synthetic-bribery-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mwombaji wa huduma anaripoti kuwa afisa aliomba pesa ili kutoa hati ambayo tayari ilikuwa imeidhinishwa.",Bribery,synthetic-bribery-05,Kiswahili,Turkana\nA public employee allegedly asked a supplier for a personal benefit in return for processing an invoice promptly.,Bribery,synthetic-bribery-06,English,Garissa\nThe report states that a public employee allegedly asked a supplier for a personal benefit in return for processing an invoice promptly.,Bribery,synthetic-bribery-06,English,Garissa\nMfanyakazi wa umma anadaiwa kuomba manufaa binafsi kutoka kwa msambazaji ili kushughulikia ankara haraka.,Bribery,synthetic-bribery-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anadaiwa kuomba manufaa binafsi kutoka kwa msambazaji ili kushughulikia ankara haraka.",Bribery,synthetic-bribery-06,Kiswahili,Garissa\nA resident alleges that an unofficial fee was requested to obtain a place in a public-service queue.,Bribery,synthetic-bribery-07,English,Kilifi\nThe report states that a resident alleges that an unofficial fee was requested to obtain a place in a public-service queue.,Bribery,synthetic-bribery-07,English,Kilifi\nMkazi anadai kuwa ada isiyo rasmi iliombwa ili kupata nafasi kwenye foleni ya huduma ya umma.,Bribery,synthetic-bribery-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mkazi anadai kuwa ada isiyo rasmi iliombwa ili kupata nafasi kwenye foleni ya huduma ya umma.",Bribery,synthetic-bribery-07,Kiswahili,Kilifi\nAn officer allegedly promised to omit a routine penalty after receiving a private payment.,Bribery,synthetic-bribery-08,English,Bungoma\nThe report states that an officer allegedly promised to omit a routine penalty after receiving a private payment.,Bribery,synthetic-bribery-08,English,Bungoma\nAfisa anadaiwa kuahidi kuondoa adhabu ya kawaida baada ya kupokea malipo ya faragha.,Bribery,synthetic-bribery-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, afisa anadaiwa kuahidi kuondoa adhabu ya kawaida baada ya kupokea malipo ya faragha.",Bribery,synthetic-bribery-08,Kiswahili,Bungoma\nA contractor says a public employee requested a personal commission for certifying completed work.,Bribery,synthetic-bribery-09,English,Meru\nThe report states that a contractor says a public employee requested a personal commission for certifying completed work.,Bribery,synthetic-bribery-09,English,Meru\nMkandarasi anasema mfanyakazi wa umma aliomba kamisheni binafsi ili kuthibitisha kazi iliyokamilika.,Bribery,synthetic-bribery-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mkandarasi anasema mfanyakazi wa umma aliomba kamisheni binafsi ili kuthibitisha kazi iliyokamilika.",Bribery,synthetic-bribery-09,Kiswahili,Meru\nA trader reports being asked for an unofficial payment before a market-stall allocation was processed.,Bribery,synthetic-bribery-10,English,Kitui\nThe report states that a trader reports being asked for an unofficial payment before a market-stall allocation was processed.,Bribery,synthetic-bribery-10,English,Kitui\nMfanyabiashara anaripoti kuombwa malipo yasiyo rasmi kabla ya kushughulikiwa kwa mgao wa kibanda sokoni.,Bribery,synthetic-bribery-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, mfanyabiashara anaripoti kuombwa malipo yasiyo rasmi kabla ya kushughulikiwa kwa mgao wa kibanda sokoni.",Bribery,synthetic-bribery-10,Kiswahili,Kitui\nA public clerk allegedly requested money to issue a routine copy of a record already held by the office.,Bribery,synthetic-bribery-11,English,Nairobi\nThe report states that a public clerk allegedly requested money to issue a routine copy of a record already held by the office.,Bribery,synthetic-bribery-11,English,Nairobi\nKarani wa umma anadaiwa kuomba pesa ili kutoa nakala ya kawaida ya rekodi ambayo ofisi tayari inayo.,Bribery,synthetic-bribery-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, karani wa umma anadaiwa kuomba pesa ili kutoa nakala ya kawaida ya rekodi ambayo ofisi tayari inayo.",Bribery,synthetic-bribery-11,Kiswahili,Nairobi\nA supplier reports an official asked for a personal reward before confirming delivery of standard materials.,Bribery,synthetic-bribery-12,English,Mombasa\nThe report states that a supplier reports an official asked for a personal reward before confirming delivery of standard materials.,Bribery,synthetic-bribery-12,English,Mombasa\nMsambazaji anaripoti kuwa afisa aliomba zawadi binafsi kabla ya kuthibitisha uwasilishaji wa vifaa vya kawaida.,Bribery,synthetic-bribery-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, msambazaji anaripoti kuwa afisa aliomba zawadi binafsi kabla ya kuthibitisha uwasilishaji wa vifaa vya kawaida.",Bribery,synthetic-bribery-12,Kiswahili,Mombasa\nA resident says a public officer requested a private payment to schedule an ordinary inspection.,Bribery,synthetic-bribery-13,English,Kisumu\nThe report states that a resident says a public officer requested a private payment to schedule an ordinary inspection.,Bribery,synthetic-bribery-13,English,Kisumu\nMkazi anasema afisa wa umma aliomba malipo ya faragha ili kupanga ukaguzi wa kawaida.,Bribery,synthetic-bribery-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mkazi anasema afisa wa umma aliomba malipo ya faragha ili kupanga ukaguzi wa kawaida.",Bribery,synthetic-bribery-13,Kiswahili,Kisumu\nAn applicant alleges a staff member requested a personal gift before forwarding a complete service form.,Bribery,synthetic-bribery-14,English,Nakuru\nThe report states that an applicant alleges a staff member requested a personal gift before forwarding a complete service form.,Bribery,synthetic-bribery-14,English,Nakuru\nMwombaji anadai kuwa mfanyakazi aliomba zawadi binafsi kabla ya kupeleka fomu kamili ya huduma.,Bribery,synthetic-bribery-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mwombaji anadai kuwa mfanyakazi aliomba zawadi binafsi kabla ya kupeleka fomu kamili ya huduma.",Bribery,synthetic-bribery-14,Kiswahili,Nakuru\nA business owner says a public officer asked for money to perform a duty that should have been routine.,Bribery,synthetic-bribery-15,English,Turkana\nThe report states that a business owner says a public officer asked for money to perform a duty that should have been routine.,Bribery,synthetic-bribery-15,English,Turkana\nMmiliki wa biashara anasema afisa wa umma aliomba pesa ili kutekeleza jukumu la kawaida.,Bribery,synthetic-bribery-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mmiliki wa biashara anasema afisa wa umma aliomba pesa ili kutekeleza jukumu la kawaida.",Bribery,synthetic-bribery-15,Kiswahili,Turkana\nA school committee says money allocated for classroom repairs was withdrawn but the repairs were never done.,Embezzlement,synthetic-embezzlement-01,English,Nairobi\nThe report states that a school committee says money allocated for classroom repairs was withdrawn but the repairs were never done.,Embezzlement,synthetic-embezzlement-01,English,Nairobi\nKamati ya shule inasema pesa za ukarabati wa darasa zilitolewa lakini ukarabati haukufanyika.,Embezzlement,synthetic-embezzlement-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, kamati ya shule inasema pesa za ukarabati wa darasa zilitolewa lakini ukarabati haukufanyika.",Embezzlement,synthetic-embezzlement-01,Kiswahili,Nairobi\nA clinic inventory review allegedly found public medical supplies recorded as issued but missing from storage.,Embezzlement,synthetic-embezzlement-02,English,Mombasa\nThe report states that a clinic inventory review allegedly found public medical supplies recorded as issued but missing from storage.,Embezzlement,synthetic-embezzlement-02,English,Mombasa\nUkaguzi wa stoo ya kliniki unadaiwa kupata vifaa vya afya vya umma vilivyoandikwa kuwa vimetolewa lakini havipo.,Embezzlement,synthetic-embezzlement-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, ukaguzi wa stoo ya kliniki unadaiwa kupata vifaa vya afya vya umma vilivyoandikwa kuwa vimetolewa lakini havipo.",Embezzlement,synthetic-embezzlement-02,Kiswahili,Mombasa\nA community group reports that funds for a water point were transferred to an unrelated private account.,Embezzlement,synthetic-embezzlement-03,English,Kisumu\nThe report states that a community group reports that funds for a water point were transferred to an unrelated private account.,Embezzlement,synthetic-embezzlement-03,English,Kisumu\nKikundi cha jamii kinaripoti kuwa pesa za kituo cha maji zilihamishiwa kwenye akaunti binafsi isiyohusika.,Embezzlement,synthetic-embezzlement-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, kikundi cha jamii kinaripoti kuwa pesa za kituo cha maji zilihamishiwa kwenye akaunti binafsi isiyohusika.",Embezzlement,synthetic-embezzlement-03,Kiswahili,Kisumu\nA public works team says fuel bought for government vehicles was diverted for private use.,Embezzlement,synthetic-embezzlement-04,English,Nakuru\nThe report states that a public works team says fuel bought for government vehicles was diverted for private use.,Embezzlement,synthetic-embezzlement-04,English,Nakuru\nTimu ya kazi za umma inasema mafuta yaliyonunuliwa kwa magari ya serikali yalielekezwa kwa matumizi binafsi.,Embezzlement,synthetic-embezzlement-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, timu ya kazi za umma inasema mafuta yaliyonunuliwa kwa magari ya serikali yalielekezwa kwa matumizi binafsi.",Embezzlement,synthetic-embezzlement-04,Kiswahili,Nakuru\nA project audit allegedly found that building materials paid for from a public budget were removed for private use.,Embezzlement,synthetic-embezzlement-05,English,Turkana\nThe report states that a project audit allegedly found that building materials paid for from a public budget were removed for private use.,Embezzlement,synthetic-embezzlement-05,English,Turkana\nUkaguzi wa mradi unadaiwa kubaini kuwa vifaa vya ujenzi vilivyolipiwa na bajeti ya umma vilichukuliwa kwa matumizi binafsi.,Embezzlement,synthetic-embezzlement-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, ukaguzi wa mradi unadaiwa kubaini kuwa vifaa vya ujenzi vilivyolipiwa na bajeti ya umma vilichukuliwa kwa matumizi binafsi.",Embezzlement,synthetic-embezzlement-05,Kiswahili,Turkana\nA residents\' committee says collected public-service revenue was not deposited into the designated account.,Embezzlement,synthetic-embezzlement-06,English,Garissa\nThe report states that a residents\' committee says collected public-service revenue was not deposited into the designated account.,Embezzlement,synthetic-embezzlement-06,English,Garissa\nKamati ya wakazi inasema mapato ya huduma ya umma yaliyokusanywa hayakuwekwa kwenye akaunti iliyotengwa.,Embezzlement,synthetic-embezzlement-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati ya wakazi inasema mapato ya huduma ya umma yaliyokusanywa hayakuwekwa kwenye akaunti iliyotengwa.",Embezzlement,synthetic-embezzlement-06,Kiswahili,Garissa\n"Funds recorded for a drainage project were allegedly withdrawn, while the site remained untouched.",Embezzlement,synthetic-embezzlement-07,English,Kilifi\n"The report states that funds recorded for a drainage project were allegedly withdrawn, while the site remained untouched.",Embezzlement,synthetic-embezzlement-07,English,Kilifi\nPesa zilizoandikwa kwa mradi wa mifereji ya maji zinadaiwa kutolewa huku eneo la kazi likibaki bila shughuli.,Embezzlement,synthetic-embezzlement-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, pesa zilizoandikwa kwa mradi wa mifereji ya maji zinadaiwa kutolewa huku eneo la kazi likibaki bila shughuli.",Embezzlement,synthetic-embezzlement-07,Kiswahili,Kilifi\nA department report says equipment entrusted to a public office was sold without authorization.,Embezzlement,synthetic-embezzlement-08,English,Bungoma\nThe report states that a department report says equipment entrusted to a public office was sold without authorization.,Embezzlement,synthetic-embezzlement-08,English,Bungoma\nRipoti ya idara inasema vifaa vilivyokabidhiwa kwa ofisi ya umma viliuzwa bila idhini.,Embezzlement,synthetic-embezzlement-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, ripoti ya idara inasema vifaa vilivyokabidhiwa kwa ofisi ya umma viliuzwa bila idhini.",Embezzlement,synthetic-embezzlement-08,Kiswahili,Bungoma\nA local committee alleges that relief supplies received for public distribution were diverted before reaching recipients.,Embezzlement,synthetic-embezzlement-09,English,Meru\nThe report states that a local committee alleges that relief supplies received for public distribution were diverted before reaching recipients.,Embezzlement,synthetic-embezzlement-09,English,Meru\nKamati ya eneo inadai kuwa bidhaa za msaada zilizopokelewa kwa ugawaji wa umma zilielekezwa kwingine kabla ya kuwafikia walengwa.,Embezzlement,synthetic-embezzlement-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, kamati ya eneo inadai kuwa bidhaa za msaada zilizopokelewa kwa ugawaji wa umma zilielekezwa kwingine kabla ya kuwafikia walengwa.",Embezzlement,synthetic-embezzlement-09,Kiswahili,Meru\nAn audit note says a public account was charged for maintenance work that was not carried out.,Embezzlement,synthetic-embezzlement-10,English,Kitui\nThe report states that an audit note says a public account was charged for maintenance work that was not carried out.,Embezzlement,synthetic-embezzlement-10,English,Kitui\nDokezo la ukaguzi linasema akaunti ya umma ilitozwa kwa kazi ya matengenezo ambayo haikufanywa.,Embezzlement,synthetic-embezzlement-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, dokezo la ukaguzi linasema akaunti ya umma ilitozwa kwa kazi ya matengenezo ambayo haikufanywa.",Embezzlement,synthetic-embezzlement-10,Kiswahili,Kitui\nA public office allegedly recorded salary payments for temporary workers who did not perform the listed work.,Embezzlement,synthetic-embezzlement-11,English,Nairobi\nThe report states that a public office allegedly recorded salary payments for temporary workers who did not perform the listed work.,Embezzlement,synthetic-embezzlement-11,English,Nairobi\nOfisi ya umma inadaiwa kurekodi malipo ya wafanyakazi wa muda ambao hawakufanya kazi iliyoandikwa.,Embezzlement,synthetic-embezzlement-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, ofisi ya umma inadaiwa kurekodi malipo ya wafanyakazi wa muda ambao hawakufanya kazi iliyoandikwa.",Embezzlement,synthetic-embezzlement-11,Kiswahili,Nairobi\nA committee reports that construction supplies purchased for a public facility were taken to a private site.,Embezzlement,synthetic-embezzlement-12,English,Mombasa\nThe report states that a committee reports that construction supplies purchased for a public facility were taken to a private site.,Embezzlement,synthetic-embezzlement-12,English,Mombasa\nKamati inaripoti kuwa vifaa vya ujenzi vilivyonunuliwa kwa kituo cha umma vilipelekwa kwenye eneo binafsi.,Embezzlement,synthetic-embezzlement-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati inaripoti kuwa vifaa vya ujenzi vilivyonunuliwa kwa kituo cha umma vilipelekwa kwenye eneo binafsi.",Embezzlement,synthetic-embezzlement-12,Kiswahili,Mombasa\nMoney collected by a public unit allegedly disappeared before being entered into the official ledger.,Embezzlement,synthetic-embezzlement-13,English,Kisumu\nThe report states that money collected by a public unit allegedly disappeared before being entered into the official ledger.,Embezzlement,synthetic-embezzlement-13,English,Kisumu\nPesa zilizokusanywa na kitengo cha umma zinadaiwa kutoweka kabla ya kuingizwa kwenye daftari rasmi.,Embezzlement,synthetic-embezzlement-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, pesa zilizokusanywa na kitengo cha umma zinadaiwa kutoweka kabla ya kuingizwa kwenye daftari rasmi.",Embezzlement,synthetic-embezzlement-13,Kiswahili,Kisumu\nA review found that budgeted funds for a public water repair were used for an undocumented purpose.,Embezzlement,synthetic-embezzlement-14,English,Nakuru\nThe report states that a review found that budgeted funds for a public water repair were used for an undocumented purpose.,Embezzlement,synthetic-embezzlement-14,English,Nakuru\nUkaguzi ulibaini kuwa pesa zilizotengwa kwa ukarabati wa maji ya umma zilitumika kwa kusudi lisiloandikwa.,Embezzlement,synthetic-embezzlement-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, ukaguzi ulibaini kuwa pesa zilizotengwa kwa ukarabati wa maji ya umma zilitumika kwa kusudi lisiloandikwa.",Embezzlement,synthetic-embezzlement-14,Kiswahili,Nakuru\nA storekeeper allegedly removed public tools entrusted to the department and kept them for personal use.,Embezzlement,synthetic-embezzlement-15,English,Turkana\nThe report states that a storekeeper allegedly removed public tools entrusted to the department and kept them for personal use.,Embezzlement,synthetic-embezzlement-15,English,Turkana\nMtunza ghala anadaiwa kutoa zana za umma zilizokabidhiwa idara na kuzitumia binafsi.,Embezzlement,synthetic-embezzlement-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mtunza ghala anadaiwa kutoa zana za umma zilizokabidhiwa idara na kuzitumia binafsi.",Embezzlement,synthetic-embezzlement-15,Kiswahili,Turkana\nSeveral suppliers allege that tender requirements were written to match one bidder\'s unique product.,Procurement Fraud,synthetic-procurement-fraud-01,English,Nairobi\nThe report states that several suppliers allege that tender requirements were written to match one bidder\'s unique product.,Procurement Fraud,synthetic-procurement-fraud-01,English,Nairobi\nWasambazaji kadhaa wanadai kuwa masharti ya zabuni yaliandikwa ili yafanane na bidhaa ya mzabuni mmoja.,Procurement Fraud,synthetic-procurement-fraud-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, wasambazaji kadhaa wanadai kuwa masharti ya zabuni yaliandikwa ili yafanane na bidhaa ya mzabuni mmoja.",Procurement Fraud,synthetic-procurement-fraud-01,Kiswahili,Nairobi\nA bid review allegedly found that competing offers used the same unusual wording and price pattern.,Procurement Fraud,synthetic-procurement-fraud-02,English,Mombasa\nThe report states that a bid review allegedly found that competing offers used the same unusual wording and price pattern.,Procurement Fraud,synthetic-procurement-fraud-02,English,Mombasa\nUkaguzi wa zabuni unadaiwa kubaini kuwa ofa pinzani zilitumia maneno yasiyo ya kawaida na mpangilio wa bei unaofanana.,Procurement Fraud,synthetic-procurement-fraud-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, ukaguzi wa zabuni unadaiwa kubaini kuwa ofa pinzani zilitumia maneno yasiyo ya kawaida na mpangilio wa bei unaofanana.",Procurement Fraud,synthetic-procurement-fraud-02,Kiswahili,Mombasa\nA public tender was allegedly awarded before the advertised deadline for submitting applications.,Procurement Fraud,synthetic-procurement-fraud-03,English,Kisumu\nThe report states that a public tender was allegedly awarded before the advertised deadline for submitting applications.,Procurement Fraud,synthetic-procurement-fraud-03,English,Kisumu\nZabuni ya umma inadaiwa kutolewa kabla ya tarehe ya mwisho iliyotangazwa ya kutuma maombi.,Procurement Fraud,synthetic-procurement-fraud-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, zabuni ya umma inadaiwa kutolewa kabla ya tarehe ya mwisho iliyotangazwa ya kutuma maombi.",Procurement Fraud,synthetic-procurement-fraud-03,Kiswahili,Kisumu\nA supplier says an evaluation panel changed scoring criteria after seeing the submitted bids.,Procurement Fraud,synthetic-procurement-fraud-04,English,Nakuru\nThe report states that a supplier says an evaluation panel changed scoring criteria after seeing the submitted bids.,Procurement Fraud,synthetic-procurement-fraud-04,English,Nakuru\nMsambazaji anasema jopo la tathmini lilibadilisha vigezo vya alama baada ya kuona zabuni zilizowasilishwa.,Procurement Fraud,synthetic-procurement-fraud-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, msambazaji anasema jopo la tathmini lilibadilisha vigezo vya alama baada ya kuona zabuni zilizowasilishwa.",Procurement Fraud,synthetic-procurement-fraud-04,Kiswahili,Nakuru\nA procurement notice allegedly omitted a required qualification in a way that excluded eligible local bidders.,Procurement Fraud,synthetic-procurement-fraud-05,English,Turkana\nThe report states that a procurement notice allegedly omitted a required qualification in a way that excluded eligible local bidders.,Procurement Fraud,synthetic-procurement-fraud-05,English,Turkana\nTangazo la ununuzi linadaiwa kuacha sifa iliyohitajika kwa namna iliyowazuia wazabuni wa eneo waliostahili.,Procurement Fraud,synthetic-procurement-fraud-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, tangazo la ununuzi linadaiwa kuacha sifa iliyohitajika kwa namna iliyowazuia wazabuni wa eneo waliostahili.",Procurement Fraud,synthetic-procurement-fraud-05,Kiswahili,Turkana\nA review alleges that a contract was split into smaller purchases to avoid the required open tender process.,Procurement Fraud,synthetic-procurement-fraud-06,English,Garissa\nThe report states that a review alleges that a contract was split into smaller purchases to avoid the required open tender process.,Procurement Fraud,synthetic-procurement-fraud-06,English,Garissa\nUkaguzi unadai kuwa mkataba uligawanywa kuwa manunuzi madogo ili kuepuka utaratibu wa zabuni ya wazi.,Procurement Fraud,synthetic-procurement-fraud-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, ukaguzi unadai kuwa mkataba uligawanywa kuwa manunuzi madogo ili kuepuka utaratibu wa zabuni ya wazi.",Procurement Fraud,synthetic-procurement-fraud-06,Kiswahili,Garissa\nTwo bidders allegedly coordinated their prices and took turns submitting the lowest offer.,Procurement Fraud,synthetic-procurement-fraud-07,English,Kilifi\nThe report states that two bidders allegedly coordinated their prices and took turns submitting the lowest offer.,Procurement Fraud,synthetic-procurement-fraud-07,English,Kilifi\nWazabuni wawili wanadaiwa kuratibu bei zao na kubadilishana zamu ya kutoa ofa ya chini zaidi.,Procurement Fraud,synthetic-procurement-fraud-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, wazabuni wawili wanadaiwa kuratibu bei zao na kubadilishana zamu ya kutoa ofa ya chini zaidi.",Procurement Fraud,synthetic-procurement-fraud-07,Kiswahili,Kilifi\nA tender committee allegedly accepted a bid after its submission deadline while rejecting earlier compliant bids.,Procurement Fraud,synthetic-procurement-fraud-08,English,Bungoma\nThe report states that a tender committee allegedly accepted a bid after its submission deadline while rejecting earlier compliant bids.,Procurement Fraud,synthetic-procurement-fraud-08,English,Bungoma\nKamati ya zabuni inadaiwa kukubali zabuni baada ya muda wa mwisho huku ikikataa zabuni zilizotii masharti mapema.,Procurement Fraud,synthetic-procurement-fraud-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, kamati ya zabuni inadaiwa kukubali zabuni baada ya muda wa mwisho huku ikikataa zabuni zilizotii masharti mapema.",Procurement Fraud,synthetic-procurement-fraud-08,Kiswahili,Bungoma\nA public contract was allegedly awarded to a supplier that did not meet the published technical requirements.,Procurement Fraud,synthetic-procurement-fraud-09,English,Meru\nThe report states that a public contract was allegedly awarded to a supplier that did not meet the published technical requirements.,Procurement Fraud,synthetic-procurement-fraud-09,English,Meru\nMkataba wa umma unadaiwa kutolewa kwa msambazaji ambaye hakutimiza mahitaji ya kiufundi yaliyochapishwa.,Procurement Fraud,synthetic-procurement-fraud-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mkataba wa umma unadaiwa kutolewa kwa msambazaji ambaye hakutimiza mahitaji ya kiufundi yaliyochapishwa.",Procurement Fraud,synthetic-procurement-fraud-09,Kiswahili,Meru\nA procurement file allegedly shows that the preferred bidder helped draft the specifications used to evaluate competitors.,Procurement Fraud,synthetic-procurement-fraud-10,English,Kitui\nThe report states that a procurement file allegedly shows that the preferred bidder helped draft the specifications used to evaluate competitors.,Procurement Fraud,synthetic-procurement-fraud-10,English,Kitui\nFaili ya ununuzi inadaiwa kuonyesha kuwa mzabuni aliyependekezwa alisaidia kuandika vipimo vilivyotumika kutathmini washindani.,Procurement Fraud,synthetic-procurement-fraud-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, faili ya ununuzi inadaiwa kuonyesha kuwa mzabuni aliyependekezwa alisaidia kuandika vipimo vilivyotumika kutathmini washindani.",Procurement Fraud,synthetic-procurement-fraud-10,Kiswahili,Kitui\nA bid opening record allegedly lists prices that differ from the sealed offers delivered by suppliers.,Procurement Fraud,synthetic-procurement-fraud-11,English,Nairobi\nThe report states that a bid opening record allegedly lists prices that differ from the sealed offers delivered by suppliers.,Procurement Fraud,synthetic-procurement-fraud-11,English,Nairobi\nRekodi ya kufungua zabuni inadaiwa kuonyesha bei zinazotofautiana na ofa zilizofungwa zilizowasilishwa na wasambazaji.,Procurement Fraud,synthetic-procurement-fraud-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, rekodi ya kufungua zabuni inadaiwa kuonyesha bei zinazotofautiana na ofa zilizofungwa zilizowasilishwa na wasambazaji.",Procurement Fraud,synthetic-procurement-fraud-11,Kiswahili,Nairobi\n"A committee reportedly changed the tender scoring after bids were submitted, without issuing a public amendment.",Procurement Fraud,synthetic-procurement-fraud-12,English,Mombasa\n"The report states that a committee reportedly changed the tender scoring after bids were submitted, without issuing a public amendment.",Procurement Fraud,synthetic-procurement-fraud-12,English,Mombasa\nKamati inaripotiwa kubadilisha alama za zabuni baada ya kuwasilishwa bila kutoa marekebisho ya umma.,Procurement Fraud,synthetic-procurement-fraud-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati inaripotiwa kubadilisha alama za zabuni baada ya kuwasilishwa bila kutoa marekebisho ya umma.",Procurement Fraud,synthetic-procurement-fraud-12,Kiswahili,Mombasa\nA contract review alleges that the winning supplier lacked a mandatory document required in the tender notice.,Procurement Fraud,synthetic-procurement-fraud-13,English,Kisumu\nThe report states that a contract review alleges that the winning supplier lacked a mandatory document required in the tender notice.,Procurement Fraud,synthetic-procurement-fraud-13,English,Kisumu\nUkaguzi wa mkataba unadai kuwa msambazaji mshindi hakuwa na hati ya lazima iliyotajwa kwenye tangazo la zabuni.,Procurement Fraud,synthetic-procurement-fraud-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, ukaguzi wa mkataba unadai kuwa msambazaji mshindi hakuwa na hati ya lazima iliyotajwa kwenye tangazo la zabuni.",Procurement Fraud,synthetic-procurement-fraud-13,Kiswahili,Kisumu\nA public entity allegedly gave one bidder advance access to tender information withheld from other participants.,Procurement Fraud,synthetic-procurement-fraud-14,English,Nakuru\nThe report states that a public entity allegedly gave one bidder advance access to tender information withheld from other participants.,Procurement Fraud,synthetic-procurement-fraud-14,English,Nakuru\nTaasisi ya umma inadaiwa kumpa mzabuni mmoja taarifa za zabuni mapema huku ikiwanyima washiriki wengine.,Procurement Fraud,synthetic-procurement-fraud-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, taasisi ya umma inadaiwa kumpa mzabuni mmoja taarifa za zabuni mapema huku ikiwanyima washiriki wengine.",Procurement Fraud,synthetic-procurement-fraud-14,Kiswahili,Nakuru\nAn evaluation panel allegedly discarded a compliant low bid using a reason not found in the published criteria.,Procurement Fraud,synthetic-procurement-fraud-15,English,Turkana\nThe report states that an evaluation panel allegedly discarded a compliant low bid using a reason not found in the published criteria.,Procurement Fraud,synthetic-procurement-fraud-15,English,Turkana\nJopo la tathmini linadaiwa kuondoa zabuni ya chini iliyotii masharti kwa sababu isiyokuwapo kwenye vigezo vilivyochapishwa.,Procurement Fraud,synthetic-procurement-fraud-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, jopo la tathmini linadaiwa kuondoa zabuni ya chini iliyotii masharti kwa sababu isiyokuwapo kwenye vigezo vilivyochapishwa.",Procurement Fraud,synthetic-procurement-fraud-15,Kiswahili,Turkana\nAn officer allegedly used official authority to stop a routine service for a resident after a personal dispute.,Abuse of Office,synthetic-abuse-of-office-01,English,Nairobi\nThe report states that an officer allegedly used official authority to stop a routine service for a resident after a personal dispute.,Abuse of Office,synthetic-abuse-of-office-01,English,Nairobi\nAfisa anadaiwa kutumia mamlaka rasmi kusimamisha huduma ya kawaida kwa mkazi baada ya mzozo binafsi.,Abuse of Office,synthetic-abuse-of-office-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, afisa anadaiwa kutumia mamlaka rasmi kusimamisha huduma ya kawaida kwa mkazi baada ya mzozo binafsi.",Abuse of Office,synthetic-abuse-of-office-01,Kiswahili,Nairobi\nA public manager allegedly ordered staff to use government equipment for a private event.,Abuse of Office,synthetic-abuse-of-office-02,English,Mombasa\nThe report states that a public manager allegedly ordered staff to use government equipment for a private event.,Abuse of Office,synthetic-abuse-of-office-02,English,Mombasa\nMsimamizi wa umma anadaiwa kuwaamuru wafanyakazi kutumia vifaa vya serikali kwa tukio binafsi.,Abuse of Office,synthetic-abuse-of-office-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, msimamizi wa umma anadaiwa kuwaamuru wafanyakazi kutumia vifaa vya serikali kwa tukio binafsi.",Abuse of Office,synthetic-abuse-of-office-02,Kiswahili,Mombasa\nA licensing supervisor allegedly bypassed the normal process to close a competitor\'s lawful business.,Abuse of Office,synthetic-abuse-of-office-03,English,Kisumu\nThe report states that a licensing supervisor allegedly bypassed the normal process to close a competitor\'s lawful business.,Abuse of Office,synthetic-abuse-of-office-03,English,Kisumu\nMsimamizi wa leseni anadaiwa kupuuza utaratibu wa kawaida ili kufunga biashara halali ya mshindani.,Abuse of Office,synthetic-abuse-of-office-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, msimamizi wa leseni anadaiwa kupuuza utaratibu wa kawaida ili kufunga biashara halali ya mshindani.",Abuse of Office,synthetic-abuse-of-office-03,Kiswahili,Kisumu\nA local official allegedly directed public workers to campaign during their paid government hours.,Abuse of Office,synthetic-abuse-of-office-04,English,Nakuru\nThe report states that a local official allegedly directed public workers to campaign during their paid government hours.,Abuse of Office,synthetic-abuse-of-office-04,English,Nakuru\nAfisa wa eneo anadaiwa kuwaelekeza wafanyakazi wa umma kufanya kampeni wakati wa saa za kazi za serikali.,Abuse of Office,synthetic-abuse-of-office-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, afisa wa eneo anadaiwa kuwaelekeza wafanyakazi wa umma kufanya kampeni wakati wa saa za kazi za serikali.",Abuse of Office,synthetic-abuse-of-office-04,Kiswahili,Nakuru\nA department head allegedly used an official vehicle and staff for a personal household task.,Abuse of Office,synthetic-abuse-of-office-05,English,Turkana\nThe report states that a department head allegedly used an official vehicle and staff for a personal household task.,Abuse of Office,synthetic-abuse-of-office-05,English,Turkana\nMkuu wa idara anadaiwa kutumia gari rasmi na wafanyakazi kwa kazi ya nyumbani binafsi.,Abuse of Office,synthetic-abuse-of-office-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mkuu wa idara anadaiwa kutumia gari rasmi na wafanyakazi kwa kazi ya nyumbani binafsi.",Abuse of Office,synthetic-abuse-of-office-05,Kiswahili,Turkana\nA public officer allegedly blocked access to a record without applying the ordinary access rules.,Abuse of Office,synthetic-abuse-of-office-06,English,Garissa\nThe report states that a public officer allegedly blocked access to a record without applying the ordinary access rules.,Abuse of Office,synthetic-abuse-of-office-06,English,Garissa\nAfisa wa umma anadaiwa kuzuia upatikanaji wa rekodi bila kutumia kanuni za kawaida za kutoa taarifa.,Abuse of Office,synthetic-abuse-of-office-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, afisa wa umma anadaiwa kuzuia upatikanaji wa rekodi bila kutumia kanuni za kawaida za kutoa taarifa.",Abuse of Office,synthetic-abuse-of-office-06,Kiswahili,Garissa\nA supervisor allegedly ordered a team to alter a routine inspection schedule to inconvenience one resident.,Abuse of Office,synthetic-abuse-of-office-07,English,Kilifi\nThe report states that a supervisor allegedly ordered a team to alter a routine inspection schedule to inconvenience one resident.,Abuse of Office,synthetic-abuse-of-office-07,English,Kilifi\nMsimamizi anadaiwa kuagiza timu kubadilisha ratiba ya ukaguzi ili kumsumbua mkazi mmoja.,Abuse of Office,synthetic-abuse-of-office-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuagiza timu kubadilisha ratiba ya ukaguzi ili kumsumbua mkazi mmoja.",Abuse of Office,synthetic-abuse-of-office-07,Kiswahili,Kilifi\nA public employee allegedly used an official stamp to approve a personal transaction outside their duties.,Abuse of Office,synthetic-abuse-of-office-08,English,Bungoma\nThe report states that a public employee allegedly used an official stamp to approve a personal transaction outside their duties.,Abuse of Office,synthetic-abuse-of-office-08,English,Bungoma\nMfanyakazi wa umma anadaiwa kutumia muhuri rasmi kuidhinisha shughuli binafsi nje ya majukumu yake.,Abuse of Office,synthetic-abuse-of-office-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anadaiwa kutumia muhuri rasmi kuidhinisha shughuli binafsi nje ya majukumu yake.",Abuse of Office,synthetic-abuse-of-office-08,Kiswahili,Bungoma\nA manager reportedly directed public staff to perform work at a private property during office hours.,Abuse of Office,synthetic-abuse-of-office-09,English,Meru\nThe report states that a manager reportedly directed public staff to perform work at a private property during office hours.,Abuse of Office,synthetic-abuse-of-office-09,English,Meru\nMsimamizi anaripotiwa kuwaelekeza wafanyakazi wa umma kufanya kazi kwenye mali binafsi wakati wa saa za ofisi.,Abuse of Office,synthetic-abuse-of-office-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, msimamizi anaripotiwa kuwaelekeza wafanyakazi wa umma kufanya kazi kwenye mali binafsi wakati wa saa za ofisi.",Abuse of Office,synthetic-abuse-of-office-09,Kiswahili,Meru\nAn official allegedly used their position to cancel a citizen\'s valid application without a recorded reason.,Abuse of Office,synthetic-abuse-of-office-10,English,Kitui\nThe report states that an official allegedly used their position to cancel a citizen\'s valid application without a recorded reason.,Abuse of Office,synthetic-abuse-of-office-10,English,Kitui\nAfisa anadaiwa kutumia nafasi yake kufuta ombi halali la mwananchi bila sababu iliyoandikwa.,Abuse of Office,synthetic-abuse-of-office-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, afisa anadaiwa kutumia nafasi yake kufuta ombi halali la mwananchi bila sababu iliyoandikwa.",Abuse of Office,synthetic-abuse-of-office-10,Kiswahili,Kitui\nA public supervisor allegedly instructed staff to ignore the normal queue for a personal acquaintance.,Abuse of Office,synthetic-abuse-of-office-11,English,Nairobi\nThe report states that a public supervisor allegedly instructed staff to ignore the normal queue for a personal acquaintance.,Abuse of Office,synthetic-abuse-of-office-11,English,Nairobi\nMsimamizi wa umma anadaiwa kuwaagiza wafanyakazi kupuuza foleni ya kawaida kwa ajili ya mtu anayemfahamu.,Abuse of Office,synthetic-abuse-of-office-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, msimamizi wa umma anadaiwa kuwaagiza wafanyakazi kupuuza foleni ya kawaida kwa ajili ya mtu anayemfahamu.",Abuse of Office,synthetic-abuse-of-office-11,Kiswahili,Nairobi\nA public officer reportedly used an office account to promote a personal business.,Abuse of Office,synthetic-abuse-of-office-12,English,Mombasa\nThe report states that a public officer reportedly used an office account to promote a personal business.,Abuse of Office,synthetic-abuse-of-office-12,English,Mombasa\nAfisa wa umma anaripotiwa kutumia akaunti ya ofisi kutangaza biashara yake binafsi.,Abuse of Office,synthetic-abuse-of-office-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, afisa wa umma anaripotiwa kutumia akaunti ya ofisi kutangaza biashara yake binafsi.",Abuse of Office,synthetic-abuse-of-office-12,Kiswahili,Mombasa\nA manager allegedly ordered the removal of a public notice to conceal a lawful community meeting.,Abuse of Office,synthetic-abuse-of-office-13,English,Kisumu\nThe report states that a manager allegedly ordered the removal of a public notice to conceal a lawful community meeting.,Abuse of Office,synthetic-abuse-of-office-13,English,Kisumu\nMsimamizi anadaiwa kuagiza kuondolewa kwa tangazo la umma ili kuficha mkutano halali wa jamii.,Abuse of Office,synthetic-abuse-of-office-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuagiza kuondolewa kwa tangazo la umma ili kuficha mkutano halali wa jamii.",Abuse of Office,synthetic-abuse-of-office-13,Kiswahili,Kisumu\nAn employee allegedly used official authority to deny a routine inspection to a business without cause.,Abuse of Office,synthetic-abuse-of-office-14,English,Nakuru\nThe report states that an employee allegedly used official authority to deny a routine inspection to a business without cause.,Abuse of Office,synthetic-abuse-of-office-14,English,Nakuru\nMfanyakazi anadaiwa kutumia mamlaka rasmi kunyima biashara ukaguzi wa kawaida bila sababu.,Abuse of Office,synthetic-abuse-of-office-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mfanyakazi anadaiwa kutumia mamlaka rasmi kunyima biashara ukaguzi wa kawaida bila sababu.",Abuse of Office,synthetic-abuse-of-office-14,Kiswahili,Nakuru\nA department officer reportedly used public staff and transport to carry personal household goods.,Abuse of Office,synthetic-abuse-of-office-15,English,Turkana\nThe report states that a department officer reportedly used public staff and transport to carry personal household goods.,Abuse of Office,synthetic-abuse-of-office-15,English,Turkana\nAfisa wa idara anaripotiwa kutumia wafanyakazi na usafiri wa umma kubeba bidhaa za nyumbani binafsi.,Abuse of Office,synthetic-abuse-of-office-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, afisa wa idara anaripotiwa kutumia wafanyakazi na usafiri wa umma kubeba bidhaa za nyumbani binafsi.",Abuse of Office,synthetic-abuse-of-office-15,Kiswahili,Turkana\nA qualified applicant alleges that a public job was given to the appointing manager\'s relative without a fair competition.,Nepotism/Favoritism,synthetic-nepotism-favoritism-01,English,Nairobi\nThe report states that a qualified applicant alleges that a public job was given to the appointing manager\'s relative without a fair competition.,Nepotism/Favoritism,synthetic-nepotism-favoritism-01,English,Nairobi\nMwombaji mwenye sifa anadai kuwa kazi ya umma alipewa ndugu wa msimamizi aliyeteua bila ushindani wa haki.,Nepotism/Favoritism,synthetic-nepotism-favoritism-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mwombaji mwenye sifa anadai kuwa kazi ya umma alipewa ndugu wa msimamizi aliyeteua bila ushindani wa haki.",Nepotism/Favoritism,synthetic-nepotism-favoritism-01,Kiswahili,Nairobi\nA school committee says a supply contract was awarded to a company owned by an official\'s close family member.,Nepotism/Favoritism,synthetic-nepotism-favoritism-02,English,Mombasa\nThe report states that a school committee says a supply contract was awarded to a company owned by an official\'s close family member.,Nepotism/Favoritism,synthetic-nepotism-favoritism-02,English,Mombasa\nKamati ya shule inasema mkataba wa vifaa ulitolewa kwa kampuni inayomilikiwa na mwanafamilia wa karibu wa afisa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati ya shule inasema mkataba wa vifaa ulitolewa kwa kampuni inayomilikiwa na mwanafamilia wa karibu wa afisa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-02,Kiswahili,Mombasa\nResidents allege that a public scholarship was reserved for relatives of committee members despite published criteria.,Nepotism/Favoritism,synthetic-nepotism-favoritism-03,English,Kisumu\nThe report states that residents allege that a public scholarship was reserved for relatives of committee members despite published criteria.,Nepotism/Favoritism,synthetic-nepotism-favoritism-03,English,Kisumu\nWakazi wanadai kuwa ufadhili wa masomo wa umma ulihifadhiwa kwa ndugu wa wajumbe licha ya vigezo vilivyochapishwa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, wakazi wanadai kuwa ufadhili wa masomo wa umma ulihifadhiwa kwa ndugu wa wajumbe licha ya vigezo vilivyochapishwa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-03,Kiswahili,Kisumu\nA recruitment shortlist allegedly included an officer\'s cousin who did not meet the stated minimum qualifications.,Nepotism/Favoritism,synthetic-nepotism-favoritism-04,English,Nakuru\nThe report states that a recruitment shortlist allegedly included an officer\'s cousin who did not meet the stated minimum qualifications.,Nepotism/Favoritism,synthetic-nepotism-favoritism-04,English,Nakuru\nOrodha fupi ya ajira inadaiwa kujumuisha binamu wa afisa ambaye hakutimiza sifa za chini zilizotajwa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, orodha fupi ya ajira inadaiwa kujumuisha binamu wa afisa ambaye hakutimiza sifa za chini zilizotajwa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-04,Kiswahili,Nakuru\nSeveral traders say market spaces were allocated to friends of the allocation panel ahead of eligible applicants.,Nepotism/Favoritism,synthetic-nepotism-favoritism-05,English,Turkana\nThe report states that several traders say market spaces were allocated to friends of the allocation panel ahead of eligible applicants.,Nepotism/Favoritism,synthetic-nepotism-favoritism-05,English,Turkana\nWafanyabiashara kadhaa wanasema nafasi za soko zilitolewa kwa marafiki wa jopo kabla ya waombaji waliostahili.,Nepotism/Favoritism,synthetic-nepotism-favoritism-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, wafanyabiashara kadhaa wanasema nafasi za soko zilitolewa kwa marafiki wa jopo kabla ya waombaji waliostahili.",Nepotism/Favoritism,synthetic-nepotism-favoritism-05,Kiswahili,Turkana\nA public committee allegedly selected a relative of one member for a grant while excluding equally eligible groups.,Nepotism/Favoritism,synthetic-nepotism-favoritism-06,English,Garissa\nThe report states that a public committee allegedly selected a relative of one member for a grant while excluding equally eligible groups.,Nepotism/Favoritism,synthetic-nepotism-favoritism-06,English,Garissa\nKamati ya umma inadaiwa kuchagua ndugu wa mjumbe mmoja kwa ruzuku huku ikiondoa vikundi vingine vilivyostahili.,Nepotism/Favoritism,synthetic-nepotism-favoritism-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati ya umma inadaiwa kuchagua ndugu wa mjumbe mmoja kwa ruzuku huku ikiondoa vikundi vingine vilivyostahili.",Nepotism/Favoritism,synthetic-nepotism-favoritism-06,Kiswahili,Garissa\nAn applicant alleges that a vacant public post was filled by an official\'s family member without advertising the role.,Nepotism/Favoritism,synthetic-nepotism-favoritism-07,English,Kilifi\nThe report states that an applicant alleges that a vacant public post was filled by an official\'s family member without advertising the role.,Nepotism/Favoritism,synthetic-nepotism-favoritism-07,English,Kilifi\nMwombaji anadai kuwa nafasi ya umma ilijazwa na mwanafamilia wa afisa bila kutangaza nafasi hiyo.,Nepotism/Favoritism,synthetic-nepotism-favoritism-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mwombaji anadai kuwa nafasi ya umma ilijazwa na mwanafamilia wa afisa bila kutangaza nafasi hiyo.",Nepotism/Favoritism,synthetic-nepotism-favoritism-07,Kiswahili,Kilifi\nA community group says food distribution lists favored the relatives of local selection officials.,Nepotism/Favoritism,synthetic-nepotism-favoritism-08,English,Bungoma\nThe report states that a community group says food distribution lists favored the relatives of local selection officials.,Nepotism/Favoritism,synthetic-nepotism-favoritism-08,English,Bungoma\nKikundi cha jamii kinasema orodha za ugawaji wa chakula ziliwapendelea ndugu wa maafisa wa uteuzi wa eneo.,Nepotism/Favoritism,synthetic-nepotism-favoritism-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, kikundi cha jamii kinasema orodha za ugawaji wa chakula ziliwapendelea ndugu wa maafisa wa uteuzi wa eneo.",Nepotism/Favoritism,synthetic-nepotism-favoritism-08,Kiswahili,Bungoma\nA public board allegedly gave repeated small contracts to a close associate while qualified applicants were overlooked.,Nepotism/Favoritism,synthetic-nepotism-favoritism-09,English,Meru\nThe report states that a public board allegedly gave repeated small contracts to a close associate while qualified applicants were overlooked.,Nepotism/Favoritism,synthetic-nepotism-favoritism-09,English,Meru\nBodi ya umma inadaiwa kumpa mikataba midogo mara kwa mara mshirika wa karibu huku waombaji wenye sifa wakipuuzwa.,Nepotism/Favoritism,synthetic-nepotism-favoritism-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, bodi ya umma inadaiwa kumpa mikataba midogo mara kwa mara mshirika wa karibu huku waombaji wenye sifa wakipuuzwa.",Nepotism/Favoritism,synthetic-nepotism-favoritism-09,Kiswahili,Meru\nA resident reports that a training place was reserved for a committee member\'s relative outside the open selection.,Nepotism/Favoritism,synthetic-nepotism-favoritism-10,English,Kitui\nThe report states that a resident reports that a training place was reserved for a committee member\'s relative outside the open selection.,Nepotism/Favoritism,synthetic-nepotism-favoritism-10,English,Kitui\nMkazi anaripoti kuwa nafasi ya mafunzo ilihifadhiwa kwa ndugu wa mjumbe wa kamati nje ya uteuzi wa wazi.,Nepotism/Favoritism,synthetic-nepotism-favoritism-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, mkazi anaripoti kuwa nafasi ya mafunzo ilihifadhiwa kwa ndugu wa mjumbe wa kamati nje ya uteuzi wa wazi.",Nepotism/Favoritism,synthetic-nepotism-favoritism-10,Kiswahili,Kitui\nAn employment panel allegedly advanced a family member of the chairperson despite a lower assessment score.,Nepotism/Favoritism,synthetic-nepotism-favoritism-11,English,Nairobi\nThe report states that an employment panel allegedly advanced a family member of the chairperson despite a lower assessment score.,Nepotism/Favoritism,synthetic-nepotism-favoritism-11,English,Nairobi\nJopo la ajira linadaiwa kumpandisha mwanafamilia wa mwenyekiti licha ya alama ya tathmini kuwa chini.,Nepotism/Favoritism,synthetic-nepotism-favoritism-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, jopo la ajira linadaiwa kumpandisha mwanafamilia wa mwenyekiti licha ya alama ya tathmini kuwa chini.",Nepotism/Favoritism,synthetic-nepotism-favoritism-11,Kiswahili,Nairobi\nA public project allegedly hired a supplier owned by an official\'s spouse without an open comparison of offers.,Nepotism/Favoritism,synthetic-nepotism-favoritism-12,English,Mombasa\nThe report states that a public project allegedly hired a supplier owned by an official\'s spouse without an open comparison of offers.,Nepotism/Favoritism,synthetic-nepotism-favoritism-12,English,Mombasa\nMradi wa umma unadaiwa kuajiri msambazaji anayemilikiwa na mwenzi wa afisa bila kulinganisha ofa kwa uwazi.,Nepotism/Favoritism,synthetic-nepotism-favoritism-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mradi wa umma unadaiwa kuajiri msambazaji anayemilikiwa na mwenzi wa afisa bila kulinganisha ofa kwa uwazi.",Nepotism/Favoritism,synthetic-nepotism-favoritism-12,Kiswahili,Mombasa\nResidents allege that temporary public jobs were distributed mainly among relatives of the local committee.,Nepotism/Favoritism,synthetic-nepotism-favoritism-13,English,Kisumu\nThe report states that residents allege that temporary public jobs were distributed mainly among relatives of the local committee.,Nepotism/Favoritism,synthetic-nepotism-favoritism-13,English,Kisumu\nWakazi wanadai kuwa kazi za muda za umma ziligawiwa hasa kwa ndugu wa kamati ya eneo.,Nepotism/Favoritism,synthetic-nepotism-favoritism-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, wakazi wanadai kuwa kazi za muda za umma ziligawiwa hasa kwa ndugu wa kamati ya eneo.",Nepotism/Favoritism,synthetic-nepotism-favoritism-13,Kiswahili,Kisumu\nA grant panel reportedly favored an official\'s sibling over applicants who met the same published requirements.,Nepotism/Favoritism,synthetic-nepotism-favoritism-14,English,Nakuru\nThe report states that a grant panel reportedly favored an official\'s sibling over applicants who met the same published requirements.,Nepotism/Favoritism,synthetic-nepotism-favoritism-14,English,Nakuru\nJopo la ruzuku linaripotiwa kumpendelea ndugu wa afisa kuliko waombaji waliotimiza masharti yale yale.,Nepotism/Favoritism,synthetic-nepotism-favoritism-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, jopo la ruzuku linaripotiwa kumpendelea ndugu wa afisa kuliko waombaji waliotimiza masharti yale yale.",Nepotism/Favoritism,synthetic-nepotism-favoritism-14,Kiswahili,Nakuru\nA qualified candidate says a public appointment went to a close friend of the selection officer without a fair process.,Nepotism/Favoritism,synthetic-nepotism-favoritism-15,English,Turkana\nThe report states that a qualified candidate says a public appointment went to a close friend of the selection officer without a fair process.,Nepotism/Favoritism,synthetic-nepotism-favoritism-15,English,Turkana\nMgombea mwenye sifa anasema uteuzi wa umma ulienda kwa rafiki wa karibu wa afisa wa uteuzi bila utaratibu wa haki.,Nepotism/Favoritism,synthetic-nepotism-favoritism-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mgombea mwenye sifa anasema uteuzi wa umma ulienda kwa rafiki wa karibu wa afisa wa uteuzi bila utaratibu wa haki.",Nepotism/Favoritism,synthetic-nepotism-favoritism-15,Kiswahili,Turkana\nA trader says an official threatened to close a lawful stall unless the trader paid an unofficial amount.,Extortion,synthetic-extortion-01,English,Nairobi\nThe report states that a trader says an official threatened to close a lawful stall unless the trader paid an unofficial amount.,Extortion,synthetic-extortion-01,English,Nairobi\nMfanyabiashara anasema afisa alitishia kufunga kibanda halali isipokuwa alipe kiasi kisicho rasmi.,Extortion,synthetic-extortion-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mfanyabiashara anasema afisa alitishia kufunga kibanda halali isipokuwa alipe kiasi kisicho rasmi.",Extortion,synthetic-extortion-01,Kiswahili,Nairobi\nA resident alleges that a clerk threatened to destroy an application unless a personal payment was made.,Extortion,synthetic-extortion-02,English,Mombasa\nThe report states that a resident alleges that a clerk threatened to destroy an application unless a personal payment was made.,Extortion,synthetic-extortion-02,English,Mombasa\nMkazi anadai kuwa karani alitishia kuharibu ombi isipokuwa malipo binafsi yafanywe.,Extortion,synthetic-extortion-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mkazi anadai kuwa karani alitishia kuharibu ombi isipokuwa malipo binafsi yafanywe.",Extortion,synthetic-extortion-02,Kiswahili,Mombasa\nA public inspector allegedly demanded money and threatened to seize equipment if the owner refused.,Extortion,synthetic-extortion-03,English,Kisumu\nThe report states that a public inspector allegedly demanded money and threatened to seize equipment if the owner refused.,Extortion,synthetic-extortion-03,English,Kisumu\nMkaguzi wa umma anadaiwa kudai pesa na kutishia kukamata vifaa ikiwa mmiliki atakataa.,Extortion,synthetic-extortion-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mkaguzi wa umma anadaiwa kudai pesa na kutishia kukamata vifaa ikiwa mmiliki atakataa.",Extortion,synthetic-extortion-03,Kiswahili,Kisumu\nA contractor says a supervisor threatened to withhold an approved certificate unless an unofficial fee was paid.,Extortion,synthetic-extortion-04,English,Nakuru\nThe report states that a contractor says a supervisor threatened to withhold an approved certificate unless an unofficial fee was paid.,Extortion,synthetic-extortion-04,English,Nakuru\nMkandarasi anasema msimamizi alitishia kuzuia cheti kilichoidhinishwa isipokuwa ada isiyo rasmi ilipwe.,Extortion,synthetic-extortion-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mkandarasi anasema msimamizi alitishia kuzuia cheti kilichoidhinishwa isipokuwa ada isiyo rasmi ilipwe.",Extortion,synthetic-extortion-04,Kiswahili,Nakuru\nA service user reports being threatened with removal from a public queue for refusing a demanded payment.,Extortion,synthetic-extortion-05,English,Turkana\nThe report states that a service user reports being threatened with removal from a public queue for refusing a demanded payment.,Extortion,synthetic-extortion-05,English,Turkana\nMtumiaji wa huduma anaripoti kutishiwa kuondolewa kwenye foleni ya umma kwa kukataa malipo yaliyodaiwa.,Extortion,synthetic-extortion-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mtumiaji wa huduma anaripoti kutishiwa kuondolewa kwenye foleni ya umma kwa kukataa malipo yaliyodaiwa.",Extortion,synthetic-extortion-05,Kiswahili,Turkana\nAn officer allegedly said a resident would face a false penalty unless money was handed over privately.,Extortion,synthetic-extortion-06,English,Garissa\nThe report states that an officer allegedly said a resident would face a false penalty unless money was handed over privately.,Extortion,synthetic-extortion-06,English,Garissa\nAfisa anadaiwa kusema mkazi atatozwa adhabu ya uongo isipokuwa pesa zikabidhiwe kwa siri.,Extortion,synthetic-extortion-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, afisa anadaiwa kusema mkazi atatozwa adhabu ya uongo isipokuwa pesa zikabidhiwe kwa siri.",Extortion,synthetic-extortion-06,Kiswahili,Garissa\nA business owner says a public employee threatened to delay renewal unless a demanded personal fee was paid.,Extortion,synthetic-extortion-07,English,Kilifi\nThe report states that a business owner says a public employee threatened to delay renewal unless a demanded personal fee was paid.,Extortion,synthetic-extortion-07,English,Kilifi\nMmiliki wa biashara anasema mfanyakazi wa umma alitishia kuchelewesha upya wa leseni isipokuwa ada binafsi ilipwe.,Extortion,synthetic-extortion-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mmiliki wa biashara anasema mfanyakazi wa umma alitishia kuchelewesha upya wa leseni isipokuwa ada binafsi ilipwe.",Extortion,synthetic-extortion-07,Kiswahili,Kilifi\nA resident alleges that an official threatened to report them unlawfully unless they paid for a routine service.,Extortion,synthetic-extortion-08,English,Bungoma\nThe report states that a resident alleges that an official threatened to report them unlawfully unless they paid for a routine service.,Extortion,synthetic-extortion-08,English,Bungoma\nMkazi anadai kuwa afisa alitishia kumripoti kinyume cha sheria isipokuwa alipe kwa huduma ya kawaida.,Extortion,synthetic-extortion-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, mkazi anadai kuwa afisa alitishia kumripoti kinyume cha sheria isipokuwa alipe kwa huduma ya kawaida.",Extortion,synthetic-extortion-08,Kiswahili,Bungoma\nA vendor says a public officer threatened to confiscate goods unless the vendor paid money outside the official process.,Extortion,synthetic-extortion-09,English,Meru\nThe report states that a vendor says a public officer threatened to confiscate goods unless the vendor paid money outside the official process.,Extortion,synthetic-extortion-09,English,Meru\nMuuzaji anasema afisa wa umma alitishia kuchukua bidhaa isipokuwa alipe pesa nje ya utaratibu rasmi.,Extortion,synthetic-extortion-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, muuzaji anasema afisa wa umma alitishia kuchukua bidhaa isipokuwa alipe pesa nje ya utaratibu rasmi.",Extortion,synthetic-extortion-09,Kiswahili,Meru\nA worker alleges that a supervisor threatened to block a lawful payment unless part of it was returned privately.,Extortion,synthetic-extortion-10,English,Kitui\nThe report states that a worker alleges that a supervisor threatened to block a lawful payment unless part of it was returned privately.,Extortion,synthetic-extortion-10,English,Kitui\nMfanyakazi anadai kuwa msimamizi alitishia kuzuia malipo halali isipokuwa sehemu irudishwe kwa siri.,Extortion,synthetic-extortion-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, mfanyakazi anadai kuwa msimamizi alitishia kuzuia malipo halali isipokuwa sehemu irudishwe kwa siri.",Extortion,synthetic-extortion-10,Kiswahili,Kitui\nA resident says an officer threatened arrest unless an unofficial payment was made.,Extortion,synthetic-extortion-11,English,Nairobi\nThe report states that a resident says an officer threatened arrest unless an unofficial payment was made.,Extortion,synthetic-extortion-11,English,Nairobi\nMkazi anasema afisa alitishia kumkamata isipokuwa malipo yasiyo rasmi yafanywe.,Extortion,synthetic-extortion-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mkazi anasema afisa alitishia kumkamata isipokuwa malipo yasiyo rasmi yafanywe.",Extortion,synthetic-extortion-11,Kiswahili,Nairobi\nA supplier alleges that a public employee threatened to blacklist the business unless a personal demand was met.,Extortion,synthetic-extortion-12,English,Mombasa\nThe report states that a supplier alleges that a public employee threatened to blacklist the business unless a personal demand was met.,Extortion,synthetic-extortion-12,English,Mombasa\nMsambazaji anadai kuwa mfanyakazi wa umma alitishia kuorodhesha biashara kama hairuhusiwi isipokuwa dai binafsi litimizwe.,Extortion,synthetic-extortion-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, msambazaji anadai kuwa mfanyakazi wa umma alitishia kuorodhesha biashara kama hairuhusiwi isipokuwa dai binafsi litimizwe.",Extortion,synthetic-extortion-12,Kiswahili,Mombasa\nA market trader says an inspector threatened to remove their stall unless a private payment was handed over.,Extortion,synthetic-extortion-13,English,Kisumu\nThe report states that a market trader says an inspector threatened to remove their stall unless a private payment was handed over.,Extortion,synthetic-extortion-13,English,Kisumu\nMfanyabiashara wa soko anasema mkaguzi alitishia kuondoa kibanda isipokuwa malipo ya faragha yatolewe.,Extortion,synthetic-extortion-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mfanyabiashara wa soko anasema mkaguzi alitishia kuondoa kibanda isipokuwa malipo ya faragha yatolewe.",Extortion,synthetic-extortion-13,Kiswahili,Kisumu\nAn applicant reports that an official threatened to discard valid papers unless the applicant paid cash.,Extortion,synthetic-extortion-14,English,Nakuru\nThe report states that an applicant reports that an official threatened to discard valid papers unless the applicant paid cash.,Extortion,synthetic-extortion-14,English,Nakuru\nMwombaji anaripoti kuwa afisa alitishia kutupa nyaraka halali isipokuwa mwombaji alipe pesa taslimu.,Extortion,synthetic-extortion-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mwombaji anaripoti kuwa afisa alitishia kutupa nyaraka halali isipokuwa mwombaji alipe pesa taslimu.",Extortion,synthetic-extortion-14,Kiswahili,Nakuru\nA public worker allegedly threatened to cancel a service request unless the user provided a personal benefit.,Extortion,synthetic-extortion-15,English,Turkana\nThe report states that a public worker allegedly threatened to cancel a service request unless the user provided a personal benefit.,Extortion,synthetic-extortion-15,English,Turkana\nMfanyakazi wa umma anadaiwa kutishia kufuta ombi la huduma isipokuwa mtumiaji atoe manufaa binafsi.,Extortion,synthetic-extortion-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anadaiwa kutishia kufuta ombi la huduma isipokuwa mtumiaji atoe manufaa binafsi.",Extortion,synthetic-extortion-15,Kiswahili,Turkana\nA resident alleges that an official receipt was altered to show a different payment amount.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,English,Nairobi\nThe report states that a resident alleges that an official receipt was altered to show a different payment amount.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,English,Nairobi\nMkazi anadai kuwa risiti rasmi ilibadilishwa ili kuonyesha kiasi tofauti cha malipo.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mkazi anadai kuwa risiti rasmi ilibadilishwa ili kuonyesha kiasi tofauti cha malipo.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-01,Kiswahili,Nairobi\nA public project file allegedly contains a completion certificate for work that was not performed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,English,Mombasa\nThe report states that a public project file allegedly contains a completion certificate for work that was not performed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,English,Mombasa\nFaili ya mradi wa umma inadaiwa kuwa na cheti cha kukamilika kwa kazi ambayo haikufanywa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, faili ya mradi wa umma inadaiwa kuwa na cheti cha kukamilika kwa kazi ambayo haikufanywa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-02,Kiswahili,Mombasa\nAn applicant says a signature on a public approval letter was copied without the signatory\'s permission.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,English,Kisumu\nThe report states that an applicant says a signature on a public approval letter was copied without the signatory\'s permission.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,English,Kisumu\nMwombaji anasema sahihi kwenye barua ya idhini ya umma ilinakiliwa bila ruhusa ya aliyeweka sahihi.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mwombaji anasema sahihi kwenye barua ya idhini ya umma ilinakiliwa bila ruhusa ya aliyeweka sahihi.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-03,Kiswahili,Kisumu\nA supplier alleges that a delivery record was changed to claim goods were received when they were not.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,English,Nakuru\nThe report states that a supplier alleges that a delivery record was changed to claim goods were received when they were not.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,English,Nakuru\nMsambazaji anadai kuwa rekodi ya uwasilishaji ilibadilishwa kuonyesha bidhaa zilipokelewa ilhali hazikupokelewa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, msambazaji anadai kuwa rekodi ya uwasilishaji ilibadilishwa kuonyesha bidhaa zilipokelewa ilhali hazikupokelewa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-04,Kiswahili,Nakuru\nA public register reportedly lists a permit number that the issuing office says it never created.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,English,Turkana\nThe report states that a public register reportedly lists a permit number that the issuing office says it never created.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,English,Turkana\nDaftari la umma linaripotiwa kuonyesha nambari ya kibali ambayo ofisi inayotoa vibali inasema haikuunda.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, daftari la umma linaripotiwa kuonyesha nambari ya kibali ambayo ofisi inayotoa vibali inasema haikuunda.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-05,Kiswahili,Turkana\nA committee says a meeting minute was edited to record an approval that members did not give.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,English,Garissa\nThe report states that a committee says a meeting minute was edited to record an approval that members did not give.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,English,Garissa\nKamati inasema kumbukumbu za mkutano zilihaririwa kurekodi idhini ambayo wajumbe hawakutoa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati inasema kumbukumbu za mkutano zilihaririwa kurekodi idhini ambayo wajumbe hawakutoa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-06,Kiswahili,Garissa\nA claimant alleges that a forged letter was used to obtain a public benefit under a false identity.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,English,Kilifi\nThe report states that a claimant alleges that a forged letter was used to obtain a public benefit under a false identity.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,English,Kilifi\nMlipaji anadai kuwa barua ya kughushi ilitumika kupata manufaa ya umma kwa utambulisho wa uongo.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mlipaji anadai kuwa barua ya kughushi ilitumika kupata manufaa ya umma kwa utambulisho wa uongo.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-07,Kiswahili,Kilifi\nAn audit review allegedly found duplicate signatures on separate public payment vouchers.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,English,Bungoma\nThe report states that an audit review allegedly found duplicate signatures on separate public payment vouchers.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,English,Bungoma\nUkaguzi unadaiwa kupata sahihi zilizorudiwa kwenye hati tofauti za malipo ya umma.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, ukaguzi unadaiwa kupata sahihi zilizorudiwa kwenye hati tofauti za malipo ya umma.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-08,Kiswahili,Bungoma\nA resident says a land-record entry was falsified to make an unapproved transfer appear official.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,English,Meru\nThe report states that a resident says a land-record entry was falsified to make an unapproved transfer appear official.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,English,Meru\nMkazi anasema rekodi ya ardhi ilighushiwa ili uhamisho usioidhinishwa uonekane rasmi.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mkazi anasema rekodi ya ardhi ilighushiwa ili uhamisho usioidhinishwa uonekane rasmi.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-09,Kiswahili,Meru\nA public office allegedly issued a certificate bearing a fabricated inspection result.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,English,Kitui\nThe report states that a public office allegedly issued a certificate bearing a fabricated inspection result.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,English,Kitui\nOfisi ya umma inadaiwa kutoa cheti chenye matokeo ya ukaguzi yaliyobuniwa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, ofisi ya umma inadaiwa kutoa cheti chenye matokeo ya ukaguzi yaliyobuniwa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-10,Kiswahili,Kitui\nA payment file reportedly includes a fake invoice submitted as evidence of a public purchase.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,English,Nairobi\nThe report states that a payment file reportedly includes a fake invoice submitted as evidence of a public purchase.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,English,Nairobi\nFaili ya malipo inaripotiwa kuwa na ankara bandia iliyowasilishwa kama ushahidi wa ununuzi wa umma.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, faili ya malipo inaripotiwa kuwa na ankara bandia iliyowasilishwa kama ushahidi wa ununuzi wa umma.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-11,Kiswahili,Nairobi\nA resident alleges that an official altered a date on a public notice after the response deadline had passed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,English,Mombasa\nThe report states that a resident alleges that an official altered a date on a public notice after the response deadline had passed.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,English,Mombasa\nMkazi anadai kuwa afisa alibadilisha tarehe kwenye tangazo la umma baada ya muda wa kujibu kupita.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mkazi anadai kuwa afisa alibadilisha tarehe kwenye tangazo la umma baada ya muda wa kujibu kupita.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-12,Kiswahili,Mombasa\nA public application reportedly carries an approval stamp that the issuing department says is not genuine.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,English,Kisumu\nThe report states that a public application reportedly carries an approval stamp that the issuing department says is not genuine.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,English,Kisumu\nOmbi la umma linaripotiwa kuwa na muhuri wa idhini ambao idara inayotoa inasema si halisi.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, ombi la umma linaripotiwa kuwa na muhuri wa idhini ambao idara inayotoa inasema si halisi.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-13,Kiswahili,Kisumu\nA committee alleges that a record was fabricated to show a public meeting had taken place.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,English,Nakuru\nThe report states that a committee alleges that a record was fabricated to show a public meeting had taken place.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,English,Nakuru\nKamati inadai kuwa rekodi ilibuniwa kuonyesha kuwa mkutano wa umma ulifanyika.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, kamati inadai kuwa rekodi ilibuniwa kuonyesha kuwa mkutano wa umma ulifanyika.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-14,Kiswahili,Nakuru\nAn applicant says an official document was changed to remove a required condition from the approved version.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,English,Turkana\nThe report states that an applicant says an official document was changed to remove a required condition from the approved version.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,English,Turkana\nMwombaji anasema hati rasmi ilibadilishwa kuondoa sharti lililohitajika kutoka toleo lililoidhinishwa.,Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, mwombaji anasema hati rasmi ilibadilishwa kuondoa sharti lililohitajika kutoka toleo lililoidhinishwa.",Fraud/Forgery of Documents,synthetic-fraud-forgery-of-documents-15,Kiswahili,Turkana\nA committee member allegedly voted on a contract that would pay a company in which the member owns shares.,Conflict of Interest,synthetic-conflict-of-interest-01,English,Nairobi\nThe report states that a committee member allegedly voted on a contract that would pay a company in which the member owns shares.,Conflict of Interest,synthetic-conflict-of-interest-01,English,Nairobi\nMjumbe wa kamati anadaiwa kupiga kura kuhusu mkataba ambao ungelipa kampuni ambayo ana hisa.,Conflict of Interest,synthetic-conflict-of-interest-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mjumbe wa kamati anadaiwa kupiga kura kuhusu mkataba ambao ungelipa kampuni ambayo ana hisa.",Conflict of Interest,synthetic-conflict-of-interest-01,Kiswahili,Nairobi\nA public officer reportedly helped select a supplier while privately holding a financial interest in that business.,Conflict of Interest,synthetic-conflict-of-interest-02,English,Mombasa\nThe report states that a public officer reportedly helped select a supplier while privately holding a financial interest in that business.,Conflict of Interest,synthetic-conflict-of-interest-02,English,Mombasa\nAfisa wa umma anaripotiwa kusaidia kuchagua msambazaji huku akiwa na maslahi ya kifedha ya siri katika biashara hiyo.,Conflict of Interest,synthetic-conflict-of-interest-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, afisa wa umma anaripotiwa kusaidia kuchagua msambazaji huku akiwa na maslahi ya kifedha ya siri katika biashara hiyo.",Conflict of Interest,synthetic-conflict-of-interest-02,Kiswahili,Mombasa\nA manager allegedly approved a public lease for land they personally own without disclosing the ownership.,Conflict of Interest,synthetic-conflict-of-interest-03,English,Kisumu\nThe report states that a manager allegedly approved a public lease for land they personally own without disclosing the ownership.,Conflict of Interest,synthetic-conflict-of-interest-03,English,Kisumu\nMsimamizi anadaiwa kuidhinisha upangishaji wa umma wa ardhi anayomiliki bila kufichua umiliki huo.,Conflict of Interest,synthetic-conflict-of-interest-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuidhinisha upangishaji wa umma wa ardhi anayomiliki bila kufichua umiliki huo.",Conflict of Interest,synthetic-conflict-of-interest-03,Kiswahili,Kisumu\nA board member allegedly took part in evaluating an application from a business where they serve as a director.,Conflict of Interest,synthetic-conflict-of-interest-04,English,Nakuru\nThe report states that a board member allegedly took part in evaluating an application from a business where they serve as a director.,Conflict of Interest,synthetic-conflict-of-interest-04,English,Nakuru\nMjumbe wa bodi anadaiwa kushiriki kutathmini ombi la biashara ambako yeye ni mkurugenzi.,Conflict of Interest,synthetic-conflict-of-interest-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mjumbe wa bodi anadaiwa kushiriki kutathmini ombi la biashara ambako yeye ni mkurugenzi.",Conflict of Interest,synthetic-conflict-of-interest-04,Kiswahili,Nakuru\nAn officer allegedly shaped a public decision that would financially benefit their undisclosed private partnership.,Conflict of Interest,synthetic-conflict-of-interest-05,English,Turkana\nThe report states that an officer allegedly shaped a public decision that would financially benefit their undisclosed private partnership.,Conflict of Interest,synthetic-conflict-of-interest-05,English,Turkana\nAfisa anadaiwa kuelekeza uamuzi wa umma ambao ungeleta faida ya kifedha kwa ubia wake binafsi ambao hakufichua.,Conflict of Interest,synthetic-conflict-of-interest-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, afisa anadaiwa kuelekeza uamuzi wa umma ambao ungeleta faida ya kifedha kwa ubia wake binafsi ambao hakufichua.",Conflict of Interest,synthetic-conflict-of-interest-05,Kiswahili,Turkana\nA public employee reportedly evaluated a grant for an organization in which the employee holds a paid position.,Conflict of Interest,synthetic-conflict-of-interest-06,English,Garissa\nThe report states that a public employee reportedly evaluated a grant for an organization in which the employee holds a paid position.,Conflict of Interest,synthetic-conflict-of-interest-06,English,Garissa\nMfanyakazi wa umma anaripotiwa kutathmini ruzuku ya shirika ambako ana nafasi ya kulipwa.,Conflict of Interest,synthetic-conflict-of-interest-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anaripotiwa kutathmini ruzuku ya shirika ambako ana nafasi ya kulipwa.",Conflict of Interest,synthetic-conflict-of-interest-06,Kiswahili,Garissa\nA procurement panel member allegedly reviewed bids from a company in which their spouse has a financial stake.,Conflict of Interest,synthetic-conflict-of-interest-07,English,Kilifi\nThe report states that a procurement panel member allegedly reviewed bids from a company in which their spouse has a financial stake.,Conflict of Interest,synthetic-conflict-of-interest-07,English,Kilifi\nMjumbe wa jopo la ununuzi anadaiwa kutathmini zabuni za kampuni ambako mwenzi wake ana maslahi ya kifedha.,Conflict of Interest,synthetic-conflict-of-interest-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, mjumbe wa jopo la ununuzi anadaiwa kutathmini zabuni za kampuni ambako mwenzi wake ana maslahi ya kifedha.",Conflict of Interest,synthetic-conflict-of-interest-07,Kiswahili,Kilifi\nA manager allegedly approved use of public property by a business they privately co-own.,Conflict of Interest,synthetic-conflict-of-interest-08,English,Bungoma\nThe report states that a manager allegedly approved use of public property by a business they privately co-own.,Conflict of Interest,synthetic-conflict-of-interest-08,English,Bungoma\nMsimamizi anadaiwa kuidhinisha matumizi ya mali ya umma na biashara anayomiliki kwa pamoja kwa siri.,Conflict of Interest,synthetic-conflict-of-interest-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, msimamizi anadaiwa kuidhinisha matumizi ya mali ya umma na biashara anayomiliki kwa pamoja kwa siri.",Conflict of Interest,synthetic-conflict-of-interest-08,Kiswahili,Bungoma\nA public board member reportedly influenced a permit decision involving their own private investment.,Conflict of Interest,synthetic-conflict-of-interest-09,English,Meru\nThe report states that a public board member reportedly influenced a permit decision involving their own private investment.,Conflict of Interest,synthetic-conflict-of-interest-09,English,Meru\nMjumbe wa bodi ya umma anaripotiwa kushawishi uamuzi wa kibali unaohusu uwekezaji wake binafsi.,Conflict of Interest,synthetic-conflict-of-interest-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, mjumbe wa bodi ya umma anaripotiwa kushawishi uamuzi wa kibali unaohusu uwekezaji wake binafsi.",Conflict of Interest,synthetic-conflict-of-interest-09,Kiswahili,Meru\nAn officer allegedly participated in setting a public contract price while owning part of the bidding company.,Conflict of Interest,synthetic-conflict-of-interest-10,English,Kitui\nThe report states that an officer allegedly participated in setting a public contract price while owning part of the bidding company.,Conflict of Interest,synthetic-conflict-of-interest-10,English,Kitui\nAfisa anadaiwa kushiriki kupanga bei ya mkataba wa umma huku akimiliki sehemu ya kampuni iliyotoa zabuni.,Conflict of Interest,synthetic-conflict-of-interest-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, afisa anadaiwa kushiriki kupanga bei ya mkataba wa umma huku akimiliki sehemu ya kampuni iliyotoa zabuni.",Conflict of Interest,synthetic-conflict-of-interest-10,Kiswahili,Kitui\nA public employee reportedly approved a business application that would increase the value of their private property.,Conflict of Interest,synthetic-conflict-of-interest-11,English,Nairobi\nThe report states that a public employee reportedly approved a business application that would increase the value of their private property.,Conflict of Interest,synthetic-conflict-of-interest-11,English,Nairobi\nMfanyakazi wa umma anaripotiwa kuidhinisha ombi la biashara ambalo lingeongeza thamani ya mali yake binafsi.,Conflict of Interest,synthetic-conflict-of-interest-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, mfanyakazi wa umma anaripotiwa kuidhinisha ombi la biashara ambalo lingeongeza thamani ya mali yake binafsi.",Conflict of Interest,synthetic-conflict-of-interest-11,Kiswahili,Nairobi\nA committee member allegedly selected a consultant whose firm pays the member for separate private work.,Conflict of Interest,synthetic-conflict-of-interest-12,English,Mombasa\nThe report states that a committee member allegedly selected a consultant whose firm pays the member for separate private work.,Conflict of Interest,synthetic-conflict-of-interest-12,English,Mombasa\nMjumbe wa kamati anadaiwa kuchagua mshauri ambaye kampuni yake humlipa kwa kazi tofauti ya binafsi.,Conflict of Interest,synthetic-conflict-of-interest-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, mjumbe wa kamati anadaiwa kuchagua mshauri ambaye kampuni yake humlipa kwa kazi tofauti ya binafsi.",Conflict of Interest,synthetic-conflict-of-interest-12,Kiswahili,Mombasa\nAn officer reportedly took part in a public allocation decision affecting a company they privately advise for a fee.,Conflict of Interest,synthetic-conflict-of-interest-13,English,Kisumu\nThe report states that an officer reportedly took part in a public allocation decision affecting a company they privately advise for a fee.,Conflict of Interest,synthetic-conflict-of-interest-13,English,Kisumu\nAfisa anaripotiwa kushiriki uamuzi wa mgao wa umma unaoathiri kampuni anayoshauri kwa ada binafsi.,Conflict of Interest,synthetic-conflict-of-interest-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, afisa anaripotiwa kushiriki uamuzi wa mgao wa umma unaoathiri kampuni anayoshauri kwa ada binafsi.",Conflict of Interest,synthetic-conflict-of-interest-13,Kiswahili,Kisumu\nA board member allegedly voted on selling public equipment to a business in which they have a financial interest.,Conflict of Interest,synthetic-conflict-of-interest-14,English,Nakuru\nThe report states that a board member allegedly voted on selling public equipment to a business in which they have a financial interest.,Conflict of Interest,synthetic-conflict-of-interest-14,English,Nakuru\nMjumbe wa bodi anadaiwa kupiga kura kuhusu kuuza vifaa vya umma kwa biashara ambako ana maslahi ya kifedha.,Conflict of Interest,synthetic-conflict-of-interest-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, mjumbe wa bodi anadaiwa kupiga kura kuhusu kuuza vifaa vya umma kwa biashara ambako ana maslahi ya kifedha.",Conflict of Interest,synthetic-conflict-of-interest-14,Kiswahili,Nakuru\nA public manager allegedly influenced a land-use decision that would benefit an undisclosed personal investment.,Conflict of Interest,synthetic-conflict-of-interest-15,English,Turkana\nThe report states that a public manager allegedly influenced a land-use decision that would benefit an undisclosed personal investment.,Conflict of Interest,synthetic-conflict-of-interest-15,English,Turkana\nMsimamizi wa umma anadaiwa kushawishi uamuzi wa matumizi ya ardhi ambao ungenufaisha uwekezaji binafsi ambao hakufichua.,Conflict of Interest,synthetic-conflict-of-interest-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, msimamizi wa umma anadaiwa kushawishi uamuzi wa matumizi ya ardhi ambao ungenufaisha uwekezaji binafsi ambao hakufichua.",Conflict of Interest,synthetic-conflict-of-interest-15,Kiswahili,Turkana\n"A residents\' group says a public office repeatedly failed to publish meeting records required for community oversight, without alleging a payment or personal benefit.",Other,synthetic-other-01,English,Nairobi\n"The report states that a residents\' group says a public office repeatedly failed to publish meeting records required for community oversight, without alleging a payment or personal benefit.",Other,synthetic-other-01,English,Nairobi\n"Kikundi cha wakazi kinasema ofisi ya umma ilishindwa mara kwa mara kuchapisha kumbukumbu za mikutano zinazohitajika kwa ufuatiliaji wa jamii, bila kudai malipo au manufaa binafsi.",Other,synthetic-other-01,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, kikundi cha wakazi kinasema ofisi ya umma ilishindwa mara kwa mara kuchapisha kumbukumbu za mikutano zinazohitajika kwa ufuatiliaji wa jamii, bila kudai malipo au manufaa binafsi.",Other,synthetic-other-01,Kiswahili,Nairobi\n"A community committee alleges that a public department ignored required notice and consultation steps for a local plan, with no specific payment, threat, or favored person described.",Other,synthetic-other-02,English,Mombasa\n"The report states that a community committee alleges that a public department ignored required notice and consultation steps for a local plan, with no specific payment, threat, or favored person described.",Other,synthetic-other-02,English,Mombasa\n"Kamati ya jamii inadai kuwa idara ya umma ilipuuza hatua zinazohitajika za notisi na mashauriano kwa mpango wa eneo, bila kueleza malipo, tishio au mtu aliyependelewa.",Other,synthetic-other-02,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati ya jamii inadai kuwa idara ya umma ilipuuza hatua zinazohitajika za notisi na mashauriano kwa mpango wa eneo, bila kueleza malipo, tishio au mtu aliyependelewa.",Other,synthetic-other-02,Kiswahili,Mombasa\n"A resident says an office did not provide a public spending explanation after repeated formal requests; the report describes no diversion, false record, or private gain.",Other,synthetic-other-03,English,Kisumu\n"The report states that a resident says an office did not provide a public spending explanation after repeated formal requests; the report describes no diversion, false record, or private gain.",Other,synthetic-other-03,English,Kisumu\n"Mkazi anasema ofisi haikutoa maelezo ya matumizi ya umma baada ya maombi rasmi kurudiwa; ripoti haielezi ubadhirifu, rekodi ya uongo au faida binafsi.",Other,synthetic-other-03,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, mkazi anasema ofisi haikutoa maelezo ya matumizi ya umma baada ya maombi rasmi kurudiwa; ripoti haielezi ubadhirifu, rekodi ya uongo au faida binafsi.",Other,synthetic-other-03,Kiswahili,Kisumu\n"A community group alleges that an oversight meeting was cancelled without explanation and minutes were not made available, without identifying a bribe or altered document.",Other,synthetic-other-04,English,Nakuru\n"The report states that a community group alleges that an oversight meeting was cancelled without explanation and minutes were not made available, without identifying a bribe or altered document.",Other,synthetic-other-04,English,Nakuru\n"Kikundi cha jamii kinadai kuwa mkutano wa uangalizi ulifutwa bila maelezo na kumbukumbu hazikupatikana, bila kutaja hongo au hati iliyobadilishwa.",Other,synthetic-other-04,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, kikundi cha jamii kinadai kuwa mkutano wa uangalizi ulifutwa bila maelezo na kumbukumbu hazikupatikana, bila kutaja hongo au hati iliyobadilishwa.",Other,synthetic-other-04,Kiswahili,Nakuru\n"A public notice about a local works plan was not displayed as required, according to residents; no contract manipulation or payment demand is alleged.",Other,synthetic-other-05,English,Turkana\n"The report states that a public notice about a local works plan was not displayed as required, according to residents; no contract manipulation or payment demand is alleged.",Other,synthetic-other-05,English,Turkana\n"Tangazo la umma kuhusu mpango wa kazi za eneo halikuwekwa kama ilivyohitajika, kulingana na wakazi; hakuna udanganyifu wa mkataba au ombi la malipo linalodaiwa.",Other,synthetic-other-05,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, tangazo la umma kuhusu mpango wa kazi za eneo halikuwekwa kama ilivyohitajika, kulingana na wakazi; hakuna udanganyifu wa mkataba au ombi la malipo linalodaiwa.",Other,synthetic-other-05,Kiswahili,Turkana\n"A committee reports that required attendance and decision records were not maintained for routine public meetings, without alleging forged entries.",Other,synthetic-other-06,English,Garissa\n"The report states that a committee reports that required attendance and decision records were not maintained for routine public meetings, without alleging forged entries.",Other,synthetic-other-06,English,Garissa\n"Kamati inaripoti kuwa rekodi zinazohitajika za mahudhurio na maamuzi hazikutunzwa kwa mikutano ya kawaida ya umma, bila kudai maingizo ya kughushi.",Other,synthetic-other-06,Kiswahili,Garissa\n"Kwa mujibu wa taarifa, kamati inaripoti kuwa rekodi zinazohitajika za mahudhurio na maamuzi hazikutunzwa kwa mikutano ya kawaida ya umma, bila kudai maingizo ya kughushi.",Other,synthetic-other-06,Kiswahili,Garissa\nResidents allege that a public feedback process was not held before a service plan was adopted; no personal beneficiary is identified.,Other,synthetic-other-07,English,Kilifi\nThe report states that residents allege that a public feedback process was not held before a service plan was adopted; no personal beneficiary is identified.,Other,synthetic-other-07,English,Kilifi\nWakazi wanadai kuwa mchakato wa maoni ya umma haukufanyika kabla ya mpango wa huduma kupitishwa; hakuna mnufaika binafsi aliyetajwa.,Other,synthetic-other-07,Kiswahili,Kilifi\n"Kwa mujibu wa taarifa, wakazi wanadai kuwa mchakato wa maoni ya umma haukufanyika kabla ya mpango wa huduma kupitishwa; hakuna mnufaika binafsi aliyetajwa.",Other,synthetic-other-07,Kiswahili,Kilifi\n"A civic group says a public office did not respond to a required audit follow-up, without describing missing funds or a personal payment.",Other,synthetic-other-08,English,Bungoma\n"The report states that a civic group says a public office did not respond to a required audit follow-up, without describing missing funds or a personal payment.",Other,synthetic-other-08,English,Bungoma\n"Kikundi cha kiraia kinasema ofisi ya umma haikujibu ufuatiliaji wa ukaguzi uliotakiwa, bila kueleza pesa zilizopotea au malipo binafsi.",Other,synthetic-other-08,Kiswahili,Bungoma\n"Kwa mujibu wa taarifa, kikundi cha kiraia kinasema ofisi ya umma haikujibu ufuatiliaji wa ukaguzi uliotakiwa, bila kueleza pesa zilizopotea au malipo binafsi.",Other,synthetic-other-08,Kiswahili,Bungoma\nA community alleges that a public service schedule was not posted in the required place and residents could not verify the announced timetable.,Other,synthetic-other-09,English,Meru\nThe report states that a community alleges that a public service schedule was not posted in the required place and residents could not verify the announced timetable.,Other,synthetic-other-09,English,Meru\nJamii inadai kuwa ratiba ya huduma ya umma haikuwekwa mahali palipotakiwa na wakazi hawakuweza kuthibitisha muda uliotangazwa.,Other,synthetic-other-09,Kiswahili,Meru\n"Kwa mujibu wa taarifa, jamii inadai kuwa ratiba ya huduma ya umma haikuwekwa mahali palipotakiwa na wakazi hawakuweza kuthibitisha muda uliotangazwa.",Other,synthetic-other-09,Kiswahili,Meru\n"A residents\' association says a public department did not provide a required explanation of how it selected service priorities, with no allegation of a private interest.",Other,synthetic-other-10,English,Kitui\n"The report states that a residents\' association says a public department did not provide a required explanation of how it selected service priorities, with no allegation of a private interest.",Other,synthetic-other-10,English,Kitui\n"Chama cha wakazi kinasema idara ya umma haikutoa maelezo yaliyotakiwa kuhusu jinsi ilivyochagua vipaumbele vya huduma, bila madai ya maslahi binafsi.",Other,synthetic-other-10,Kiswahili,Kitui\n"Kwa mujibu wa taarifa, chama cha wakazi kinasema idara ya umma haikutoa maelezo yaliyotakiwa kuhusu jinsi ilivyochagua vipaumbele vya huduma, bila madai ya maslahi binafsi.",Other,synthetic-other-10,Kiswahili,Kitui\n"A community group alleges that a public consultation notice omitted the date and location, preventing residents from taking part; no bribe or threat is described.",Other,synthetic-other-11,English,Nairobi\n"The report states that a community group alleges that a public consultation notice omitted the date and location, preventing residents from taking part; no bribe or threat is described.",Other,synthetic-other-11,English,Nairobi\n"Kikundi cha jamii kinadai kuwa notisi ya mashauriano ya umma iliacha tarehe na mahali, na kuwazuia wakazi kushiriki; hakuna hongo au tishio lililoelezwa.",Other,synthetic-other-11,Kiswahili,Nairobi\n"Kwa mujibu wa taarifa, kikundi cha jamii kinadai kuwa notisi ya mashauriano ya umma iliacha tarehe na mahali, na kuwazuia wakazi kushiriki; hakuna hongo au tishio lililoelezwa.",Other,synthetic-other-11,Kiswahili,Nairobi\n"A local committee says quarterly public-service statistics were not released as required, without alleging that figures were falsified.",Other,synthetic-other-12,English,Mombasa\n"The report states that a local committee says quarterly public-service statistics were not released as required, without alleging that figures were falsified.",Other,synthetic-other-12,English,Mombasa\n"Kamati ya eneo inasema takwimu za huduma za umma za robo mwaka hazikutolewa kama ilivyotakiwa, bila kudai kuwa takwimu zilighushiwa.",Other,synthetic-other-12,Kiswahili,Mombasa\n"Kwa mujibu wa taarifa, kamati ya eneo inasema takwimu za huduma za umma za robo mwaka hazikutolewa kama ilivyotakiwa, bila kudai kuwa takwimu zilighushiwa.",Other,synthetic-other-12,Kiswahili,Mombasa\n"Residents report that a public office did not keep a required register of service requests, with no specific official benefit alleged.",Other,synthetic-other-13,English,Kisumu\n"The report states that residents report that a public office did not keep a required register of service requests, with no specific official benefit alleged.",Other,synthetic-other-13,English,Kisumu\n"Wakazi wanaripoti kuwa ofisi ya umma haikutunza daftari lililohitajika la maombi ya huduma, bila kudai manufaa maalum ya afisa.",Other,synthetic-other-13,Kiswahili,Kisumu\n"Kwa mujibu wa taarifa, wakazi wanaripoti kuwa ofisi ya umma haikutunza daftari lililohitajika la maombi ya huduma, bila kudai manufaa maalum ya afisa.",Other,synthetic-other-13,Kiswahili,Kisumu\nA civic group says the public office failed to disclose the criteria used to set service priorities; the report alleges no tender or payment scheme.,Other,synthetic-other-14,English,Nakuru\nThe report states that a civic group says the public office failed to disclose the criteria used to set service priorities; the report alleges no tender or payment scheme.,Other,synthetic-other-14,English,Nakuru\nKikundi cha kiraia kinasema ofisi ya umma haikufichua vigezo vya kupanga vipaumbele vya huduma; ripoti haina madai ya zabuni au mpango wa malipo.,Other,synthetic-other-14,Kiswahili,Nakuru\n"Kwa mujibu wa taarifa, kikundi cha kiraia kinasema ofisi ya umma haikufichua vigezo vya kupanga vipaumbele vya huduma; ripoti haina madai ya zabuni au mpango wa malipo.",Other,synthetic-other-14,Kiswahili,Nakuru\n"A community alleges that a required public notice about a service change was not issued, without describing a threat, altered record, or personal gain.",Other,synthetic-other-15,English,Turkana\n"The report states that a community alleges that a required public notice about a service change was not issued, without describing a threat, altered record, or personal gain.",Other,synthetic-other-15,English,Turkana\n"Jamii inadai kuwa notisi ya umma iliyohitajika kuhusu mabadiliko ya huduma haikutolewa, bila kueleza tishio, rekodi iliyobadilishwa au faida binafsi.",Other,synthetic-other-15,Kiswahili,Turkana\n"Kwa mujibu wa taarifa, jamii inadai kuwa notisi ya umma iliyohitajika kuhusu mabadiliko ya huduma haikutolewa, bila kueleza tishio, rekodi iliyobadilishwa au faida binafsi.",Other,synthetic-other-15,Kiswahili,Turkana\nAsked for 500 before he would stamp my papers.,Bribery,colloquial-bribery-01,English,Mombasa\nthe clerk wanted something small to sign my form,Bribery,colloquial-bribery-01,English,Kisumu\nAlinidai kitu kidogo ndio apige stempu.,Bribery,colloquial-bribery-01,Sheng,Nakuru\nmsee akaniambia nilete hongo ya 500,Bribery,colloquial-bribery-01,Sheng,Meru\ndey askd me 4 500 2 fix ma permit,Bribery,colloquial-bribery-02,English,Kisumu\nofficer said pay me n ill approve it quick,Bribery,colloquial-bribery-02,English,Nakuru\nkanjo alinidai 500 nimpitishe haraka,Bribery,colloquial-bribery-02,Sheng,Meru\naliniambia nitoe chai ya stempu,Bribery,colloquial-bribery-02,Sheng,Kitui\nMy brother says the lands clerk wanted money to open the file.,Bribery,colloquial-bribery-03,English,Nakuru\nhe was asked for a bribe at the counter,Bribery,colloquial-bribery-03,English,Meru\nJamaa yangu alidaiwa rushwa kwa ofisi ya lands.,Bribery,colloquial-bribery-03,Sheng,Kitui\nkarani alitaka pesa aonee faili,Bribery,colloquial-bribery-03,Sheng,Kilifi\nWhy do I have to pay extra for a service I already paid for?,Bribery,colloquial-bribery-04,English,Meru\nthey want payment in hand to serve us,Bribery,colloquial-bribery-04,English,Kitui\nKwa nini nilipe zaidi kwa huduma nimeshalipa?,Bribery,colloquial-bribery-04,Sheng,Kilifi\nwanataka malipo ya mkono kwa huduma,Bribery,colloquial-bribery-04,Sheng,Bungoma\nInspector from health came and demanded a gift for the certificate.,Bribery,colloquial-bribery-05,English,Kitui\nhe asked for something for himself before signing,Bribery,colloquial-bribery-05,English,Kilifi\nMkaguzi wa afya alidai zawadi kwa cheti.,Bribery,colloquial-bribery-05,Sheng,Bungoma\nalitaka kitu chake kabla ya kutia sahihi,Bribery,colloquial-bribery-05,Sheng,Nairobi\nThe guy at the window said no money no service.,Bribery,colloquial-bribery-06,English,Kilifi\nhe wanted a kickback to process my invoice,Bribery,colloquial-bribery-06,English,Bungoma\nMsee wa dirisha akasema hakuna pesa hakuna huduma.,Bribery,colloquial-bribery-06,Sheng,Nairobi\nalitaka kitu kidogo kushughulikia ankara,Bribery,colloquial-bribery-06,Sheng,Mombasa\nThe CDF money for the borehole is gone and nothing was built.,Embezzlement,colloquial-embezzlement-01,English,Mombasa\nthe funds disappeared and the project stopped,Embezzlement,colloquial-embezzlement-01,English,Kisumu\nMullah za CDF zimepotea na hakuna kisima.,Embezzlement,colloquial-embezzlement-01,Sheng,Nakuru\ndoo za mradi zilipotea ghafla,Embezzlement,colloquial-embezzlement-01,Sheng,Meru\nmoney 4 the shcool lunch program is missing,Embezzlement,colloquial-embezzlement-02,English,Kisumu\nthe bursary cash was withdrawn n never seen,Embezzlement,colloquial-embezzlement-02,English,Nakuru\npesa za chakula cha shule zimepotea,Embezzlement,colloquial-embezzlement-02,Sheng,Meru\nfedha za bursary zilichukuliwa hazikuonekana,Embezzlement,colloquial-embezzlement-02,Sheng,Kitui\nSays the clerk diverted the health budget to his own account.,Embezzlement,colloquial-embezzlement-03,English,Nakuru\nthe funds were misappropriated quietly,Embezzlement,colloquial-embezzlement-03,English,Meru\nAnasema karani alichota fedha za afya.,Embezzlement,colloquial-embezzlement-03,Sheng,Kitui\nfedha za umma zilitumika vibaya,Embezzlement,colloquial-embezzlement-03,Sheng,Kilifi\nWhere did the county money for the road go?,Embezzlement,colloquial-embezzlement-04,English,Meru\nthe budget for the dispensary vanished,Embezzlement,colloquial-embezzlement-04,English,Kitui\nFedha za barabara zilienda wapi?,Embezzlement,colloquial-embezzlement-04,Sheng,Kilifi\nbajeti ya zahanati ilipotea,Embezzlement,colloquial-embezzlement-04,Sheng,Bungoma\nHe took all the money and left the project half done.,Embezzlement,colloquial-embezzlement-05,English,Kitui\nthe cash meant for supplies was never used for supplies,Embezzlement,colloquial-embezzlement-05,English,Kilifi\nAlichukua doo zote akaacha mradi nusu.,Embezzlement,colloquial-embezzlement-05,Sheng,Bungoma\npesa ya vifaa haikununua vifaa,Embezzlement,colloquial-embezzlement-05,Sheng,Nairobi\nWater project money missing from the account,Embezzlement,colloquial-embezzlement-06,English,Kilifi\nthey ate the funds for the youth program,Embezzlement,colloquial-embezzlement-06,English,Bungoma\nPesa ya mradi wa maji haipo kwa akaunti,Embezzlement,colloquial-embezzlement-06,Sheng,Nairobi\nwalikula fedha za vijana,Embezzlement,colloquial-embezzlement-06,Sheng,Mombasa\nThe tender for the road was already decided before it opened.,Procurement Fraud,colloquial-procurement-fraud-01,English,Mombasa\nthe bid was rigged for one company,Procurement Fraud,colloquial-procurement-fraud-01,English,Kisumu\nZabuni ya barabara ilikuwa imeamuliwa kabla kufunguliwa.,Procurement Fraud,colloquial-procurement-fraud-01,Sheng,Nakuru\nzabuni ilipangwa kwa kampuni moja,Procurement Fraud,colloquial-procurement-fraud-01,Sheng,Meru\ncontract was inflated n the co is owned by a relative,Procurement Fraud,colloquial-procurement-fraud-02,English,Kisumu\nghost supplier invoiced for nothing,Procurement Fraud,colloquial-procurement-fraud-02,English,Nakuru\nbei ya mkataba ilipandishwa na kampuni ni ya ndugu,Procurement Fraud,colloquial-procurement-fraud-02,Sheng,Meru\nmsambazaji wa hewa alitoa ankara,Procurement Fraud,colloquial-procurement-fraud-02,Sheng,Kitui\nSays the evaluation committee read only one bid.,Procurement Fraud,colloquial-procurement-fraud-03,English,Nakuru\nthe award went to the company that did not quote lowest,Procurement Fraud,colloquial-procurement-fraud-03,English,Meru\nAnasema kamati ilisoma zabuni moja tu.,Procurement Fraud,colloquial-procurement-fraud-03,Sheng,Kitui\ntuzo ilienda kwa kampuni isiyo na bei ya chini,Procurement Fraud,colloquial-procurement-fraud-03,Sheng,Kilifi\nWhy did the hospital pay for 40 beds and receive 12?,Procurement Fraud,colloquial-procurement-fraud-04,English,Meru\nthe same supplier wins every tender,Procurement Fraud,colloquial-procurement-fraud-04,English,Kitui\nKwa nini hospitali ililipa vitanda 40 ikapokea 12?,Procurement Fraud,colloquial-procurement-fraud-04,Sheng,Kilifi\nmsambazaji mmoja hushinda zabuni zote,Procurement Fraud,colloquial-procurement-fraud-04,Sheng,Bungoma\nThe tender documents were changed after closing.,Procurement Fraud,colloquial-procurement-fraud-05,English,Kitui\nthe contract price doubled after signing,Procurement Fraud,colloquial-procurement-fraud-05,English,Kilifi\nNyaraka za zabuni zilibadilishwa baada ya kufungwa.,Procurement Fraud,colloquial-procurement-fraud-05,Sheng,Bungoma\nbei ya mkataba iliongezwa mara mbili,Procurement Fraud,colloquial-procurement-fraud-05,Sheng,Nairobi\ncompany registered last month won a 200m tender,Procurement Fraud,colloquial-procurement-fraud-06,English,Kilifi\nthey split the contract to avoid the limit,Procurement Fraud,colloquial-procurement-fraud-06,English,Bungoma\nkampuni iliyosajiliwa mwezi juzi ikashinda zabuni ya milioni,Procurement Fraud,colloquial-procurement-fraud-06,Sheng,Nairobi\nwaligawanya mkataba kuepuka kikomo,Procurement Fraud,colloquial-procurement-fraud-06,Sheng,Mombasa\nHe used his position to have my licence cancelled.,Abuse of Office,colloquial-abuse-of-office-01,English,Mombasa\nthe officer misused his office against me,Abuse of Office,colloquial-abuse-of-office-01,English,Kisumu\nAlitumia ofisi yake kufuta leseni yangu.,Abuse of Office,colloquial-abuse-of-office-01,Sheng,Nakuru\nalitumia cheo vibaya dhidi yangu,Abuse of Office,colloquial-abuse-of-office-01,Sheng,Meru\nchief used his power 2 close my shop,Abuse of Office,colloquial-abuse-of-office-02,English,Kisumu\nhe abused his position to punish me,Abuse of Office,colloquial-abuse-of-office-02,English,Nakuru\nchief alitumia mamlaka kufunga duka langu,Abuse of Office,colloquial-abuse-of-office-02,Sheng,Meru\nalitumia ofisi kuniadhibu,Abuse of Office,colloquial-abuse-of-office-02,Sheng,Kitui\nSays the director ordered staff to serve his friends first.,Abuse of Office,colloquial-abuse-of-office-03,English,Nakuru\nthe boss used the office for personal orders,Abuse of Office,colloquial-abuse-of-office-03,English,Meru\nAnasema mkurugenzi aliamuru wafuasi wahudumiwe kwanza.,Abuse of Office,colloquial-abuse-of-office-03,Sheng,Kitui\nbosi alitumia ofisi kwa maagizo binafsi,Abuse of Office,colloquial-abuse-of-office-03,Sheng,Kilifi\nWhy does the officer decide who gets served and who waits?,Abuse of Office,colloquial-abuse-of-office-04,English,Meru\nhe acts as if the office belongs to him,Abuse of Office,colloquial-abuse-of-office-04,English,Kitui\nKwa nini afisa anaamua nani ahudumiwe?,Abuse of Office,colloquial-abuse-of-office-04,Sheng,Kilifi\nanafanya kama ofisi ni yake,Abuse of Office,colloquial-abuse-of-office-04,Sheng,Bungoma\nThe officer refused to release my documents without reason.,Abuse of Office,colloquial-abuse-of-office-05,English,Kitui\nhe used his power to delay my case,Abuse of Office,colloquial-abuse-of-office-05,English,Kilifi\nAfisa alikataa kutoa nyaraka bila sababu.,Abuse of Office,colloquial-abuse-of-office-05,Sheng,Bungoma\nalitumia mamlaka kuchelewesha kesi yangu,Abuse of Office,colloquial-abuse-of-office-05,Sheng,Nairobi\nmanager blocked my transfer after I complained,Abuse of Office,colloquial-abuse-of-office-06,English,Kilifi\nhe misused the position to silence us,Abuse of Office,colloquial-abuse-of-office-06,English,Bungoma\nmeneja alizuia uhamisho wangu baada ya malalamiko,Abuse of Office,colloquial-abuse-of-office-06,Sheng,Nairobi\nalitumia cheo kutunyamazisha,Abuse of Office,colloquial-abuse-of-office-06,Sheng,Mombasa\n"The job went to the manager\'s cousin, not to those who applied.",Nepotism/Favoritism,colloquial-nepotism-favoritism-01,English,Mombasa\nthey hired a relative who never applied,Nepotism/Favoritism,colloquial-nepotism-favoritism-01,English,Kisumu\n"Kazi ilienda kwa ndugu wa meneja, si kwa walioomba.",Nepotism/Favoritism,colloquial-nepotism-favoritism-01,Sheng,Nakuru\nwaliajiri ndugu ambaye hakuomba,Nepotism/Favoritism,colloquial-nepotism-favoritism-01,Sheng,Meru\nhis daughter got the post w no interview,Nepotism/Favoritism,colloquial-nepotism-favoritism-02,English,Kisumu\nthey only hire ppl from their village,Nepotism/Favoritism,colloquial-nepotism-favoritism-02,English,Nakuru\nmtoto wake akapata kazi bila usaili,Nepotism/Favoritism,colloquial-nepotism-favoritism-02,Sheng,Meru\nwanaajiri ndugu tu,Nepotism/Favoritism,colloquial-nepotism-favoritism-02,Sheng,Kitui\nSays the promotion list is all names from one family.,Nepotism/Favoritism,colloquial-nepotism-favoritism-03,English,Nakuru\nthe contract was given to a family friend,Nepotism/Favoritism,colloquial-nepotism-favoritism-03,English,Meru\nAnasema orodha ya kupandishwa ni ndugu tu.,Nepotism/Favoritism,colloquial-nepotism-favoritism-03,Sheng,Kitui\nmkataba ulipewa rafiki wa familia,Nepotism/Favoritism,colloquial-nepotism-favoritism-03,Sheng,Kilifi\nWhy was I skipped when her nephew was hired?,Nepotism/Favoritism,colloquial-nepotism-favoritism-04,English,Meru\nthe vacancy was never advertised,Nepotism/Favoritism,colloquial-nepotism-favoritism-04,English,Kitui\nKwa nini nilipitwa huku mpwa wake akaajiriwa?,Nepotism/Favoritism,colloquial-nepotism-favoritism-04,Sheng,Kilifi\nnafasi haikutangazwa,Nepotism/Favoritism,colloquial-nepotism-favoritism-04,Sheng,Bungoma\nEvery posting in this office goes to the same clan.,Nepotism/Favoritism,colloquial-nepotism-favoritism-05,English,Kitui\nthe shortlist was already known,Nepotism/Favoritism,colloquial-nepotism-favoritism-05,English,Kilifi\nNafasi zote ofisi hii zinenda kwa kabila moja.,Nepotism/Favoritism,colloquial-nepotism-favoritism-05,Sheng,Bungoma\norodha ya mwisho ilijulikana,Nepotism/Favoritism,colloquial-nepotism-favoritism-05,Sheng,Nairobi\nmy sister was denied the job but the chief\'s son got it,Nepotism/Favoritism,colloquial-nepotism-favoritism-06,English,Kilifi\nfavouritism in the recruitment,Nepotism/Favoritism,colloquial-nepotism-favoritism-06,English,Bungoma\ndada yangu alinyimwa kazi lakini mtoto wa chief akapata,Nepotism/Favoritism,colloquial-nepotism-favoritism-06,Sheng,Nairobi\nupendeleo kwa ndugu kwenye kuajiri,Nepotism/Favoritism,colloquial-nepotism-favoritism-06,Sheng,Mombasa\nHe said if I refuse he will have me arrested.,Extortion,colloquial-extortion-01,English,Mombasa\nthe officer threatened me until I paid,Extortion,colloquial-extortion-01,English,Kisumu\nAlisema nikikataa atanifunga.,Extortion,colloquial-extortion-01,Sheng,Nakuru\nafisa alinitishia hadi nililipa,Extortion,colloquial-extortion-01,Sheng,Meru\nkarao said pay or ill sleep in cells,Extortion,colloquial-extortion-02,English,Kisumu\nthey threatened to cut me if i didnt pay,Extortion,colloquial-extortion-02,English,Nakuru\nkarao akasema lipe au ulale ndani,Extortion,colloquial-extortion-02,Sheng,Meru\nwalinitishia kunichoma kisu nikikataa,Extortion,colloquial-extortion-02,Sheng,Kitui\nSays the guard demanded money to let the lorry pass.,Extortion,colloquial-extortion-03,English,Nakuru\nhe forced my brother to pay at the gate,Extortion,colloquial-extortion-03,English,Meru\nAnasema mlinzi alidai pesa lori ipite.,Extortion,colloquial-extortion-03,Sheng,Kitui\nalimlazimisha ndugu yangu kulipa langoni,Extortion,colloquial-extortion-03,Sheng,Kilifi\nWhy must I pay to be released at the roadblock?,Extortion,colloquial-extortion-04,English,Meru\nhe would not let me go without payment,Extortion,colloquial-extortion-04,English,Kitui\nKwa nini nilipie nitolewe kwenye kizuizi?,Extortion,colloquial-extortion-04,Sheng,Kilifi\nhakuachia bila malipo,Extortion,colloquial-extortion-04,Sheng,Bungoma\nThe man at the gate threatened my staff until we paid.,Extortion,colloquial-extortion-05,English,Kitui\nhe demanded and warned us of consequences,Extortion,colloquial-extortion-05,English,Kilifi\nMtu wa lango alitishia wafanyakazi hadi tulilipa.,Extortion,colloquial-extortion-05,Sheng,Bungoma\nalidai na kututishia madhara,Extortion,colloquial-extortion-05,Sheng,Nairobi\naskari held my id n said pay 200 or else,Extortion,colloquial-extortion-06,English,Kilifi\nforced to pay under threat,Extortion,colloquial-extortion-06,English,Bungoma\naskari akashika kitambulisho akasema lipe 200 au la,Extortion,colloquial-extortion-06,Sheng,Nairobi\nkulazimishwa kulipa kwa vitisho,Extortion,colloquial-extortion-06,Sheng,Mombasa\nThe receipt they gave me has a different name on it.,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-01,English,Mombasa\nthe certificate is fake,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-01,English,Kisumu\nRisiti waliyonipa ina jina lingine.,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-01,Sheng,Nakuru\ncheti ni bandia,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-01,Sheng,Meru\nsignature on my form was forged,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-02,English,Kisumu\nthe land title is fake n was printed elsewhere,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-02,English,Nakuru\nsahihi kwenye fomu yangu ilighushiwa,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-02,Sheng,Meru\nhati ya ardhi ni bandia,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-02,Sheng,Kitui\nSays the payslip submitted was not the real one.,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-03,English,Nakuru\nthe academic paper is a forgery,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-03,English,Meru\nAnasema payslip iliyowasilishwa si ya kweli.,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-03,Sheng,Kitui\nkaratasi ya masomo ni ya kughushi,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-03,Sheng,Kilifi\nWhy does the record show a payment I never made?,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-04,English,Meru\nthe invoice number does not exist in the system,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-04,English,Kitui\nKwa nini rekodi inaonyesha malipo nisiyofanya?,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-04,Sheng,Kilifi\nnamba ya ankara haipo kwenye mfumo,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-04,Sheng,Bungoma\nThe clearance letter carries a stamp that is not theirs.,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-05,English,Kitui\nthey falsified the delivery note,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-05,English,Kilifi\nBarua ya kibali ina muhuri si wao.,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-05,Sheng,Bungoma\nwalighushi hati ya uwasilishaji,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-05,Sheng,Nairobi\nmy degree cert was changed,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-06,English,Kilifi\nthey edited the register to add a name,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-06,English,Bungoma\ncheti changu cha shahada kilibadilishwa,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-06,Sheng,Nairobi\nwalibadilisha rejista kuongeza jina,Fraud/Forgery of Documents,colloquial-fraud-forgery-of-documents-06,Sheng,Mombasa\nThe officer who approved the tender owns the company.,Conflict of Interest,colloquial-conflict-of-interest-01,English,Mombasa\nhe had a personal interest in the award,Conflict of Interest,colloquial-conflict-of-interest-01,English,Kisumu\nAfisa aliyeidhinisha zabuni anamiliki kampuni.,Conflict of Interest,colloquial-conflict-of-interest-01,Sheng,Nakuru\nalikuwa na maslahi binafsi kwenye tuzo,Conflict of Interest,colloquial-conflict-of-interest-01,Sheng,Meru\ndirector sits on the board n also owns the supplier,Conflict of Interest,colloquial-conflict-of-interest-02,English,Kisumu\nhe never declared his shares,Conflict of Interest,colloquial-conflict-of-interest-02,English,Nakuru\nmkurugenzi ni mjumbe na pia anamiliki msambazaji,Conflict of Interest,colloquial-conflict-of-interest-02,Sheng,Meru\nhakutoa tamko la hisa zake,Conflict of Interest,colloquial-conflict-of-interest-02,Sheng,Kitui\nSays the committee chair has a stake in the winning bid.,Conflict of Interest,colloquial-conflict-of-interest-03,English,Nakuru\nan undisclosed interest affected the decision,Conflict of Interest,colloquial-conflict-of-interest-03,English,Meru\nAnasema mwenyekiti ana hisa kwa zabuni iliyoshinda.,Conflict of Interest,colloquial-conflict-of-interest-03,Sheng,Kitui\nmaslahi yasiyotangazwa yaliathiri uamuzi,Conflict of Interest,colloquial-conflict-of-interest-03,Sheng,Kilifi\nWhy is his own brother working for the company he supervises?,Conflict of Interest,colloquial-conflict-of-interest-04,English,Meru\nhe approved a payment to himself,Conflict of Interest,colloquial-conflict-of-interest-04,English,Kitui\nKwa nini ndugu yake anafanya kazi kwa kampuni anayoisimamia?,Conflict of Interest,colloquial-conflict-of-interest-04,Sheng,Kilifi\naliidhinisha malipo kwake mwenyewe,Conflict of Interest,colloquial-conflict-of-interest-04,Sheng,Bungoma\nThe officer signed for land next to his own plot.,Conflict of Interest,colloquial-conflict-of-interest-05,English,Kitui\na financial interest was never disclosed,Conflict of Interest,colloquial-conflict-of-interest-05,English,Kilifi\nAfisa alitia sahihi ardhi jirani na yake.,Conflict of Interest,colloquial-conflict-of-interest-05,Sheng,Bungoma\nmaslahi ya kifedha hayakutangazwa,Conflict of Interest,colloquial-conflict-of-interest-05,Sheng,Nairobi\nmanager approved his wifes company,Conflict of Interest,colloquial-conflict-of-interest-06,English,Kilifi\nfamily interest in the contract,Conflict of Interest,colloquial-conflict-of-interest-06,English,Bungoma\nmeneja aliidhinisha kampuni ya mkewe,Conflict of Interest,colloquial-conflict-of-interest-06,Sheng,Nairobi\nmaslahi ya familia kwenye mkataba,Conflict of Interest,colloquial-conflict-of-interest-06,Sheng,Mombasa\nThe office sells public forms at three times the printed price.,Other,colloquial-other-01,English,Mombasa\nthey charge for a service that should be free,Other,colloquial-other-01,English,Kisumu\nOfisi inauza fomu kwa bei mara tatu.,Other,colloquial-other-01,Sheng,Nakuru\nwanatoza huduma ambayo ni bure,Other,colloquial-other-01,Sheng,Meru\nstaff keep our files n refuse to return them,Other,colloquial-other-02,English,Kisumu\nthe queue is moved only for those who know someone,Other,colloquial-other-02,English,Nakuru\nwafanyakazi wanashikilia faili zetu,Other,colloquial-other-02,Sheng,Meru\nfoleni inasogea kwa wanaojuana tu,Other,colloquial-other-02,Sheng,Kitui\nSays the office closed early for three weeks with no notice.,Other,colloquial-other-03,English,Nakuru\nnobody answers the phone or the counter,Other,colloquial-other-03,English,Meru\nAnasema ofisi ilifungwa wiki tatu bila taarifa.,Other,colloquial-other-03,Sheng,Kitui\nhakuna anayejibu simu wala dirisha,Other,colloquial-other-03,Sheng,Kilifi\nWhy are we sent from one office to another for the same paper?,Other,colloquial-other-04,English,Meru\nnobody will say who is responsible,Other,colloquial-other-04,English,Kitui\nKwa nini tunapelekwa ofisi moja hadi nyingine?,Other,colloquial-other-04,Sheng,Kilifi\nhakuna anayesema nani mhusika,Other,colloquial-other-04,Sheng,Bungoma\nThe public notice was changed after people complained.,Other,colloquial-other-05,English,Kitui\nthe rules keep changing without explanation,Other,colloquial-other-05,English,Kilifi\nTaarifa ya umma ilibadilishwa baada ya malalamiko.,Other,colloquial-other-05,Sheng,Bungoma\nsheria zinabadilika bila maelezo,Other,colloquial-other-05,Sheng,Nairobi\nwater point closed for months no reason,Other,colloquial-other-06,English,Kilifi\npromised road never built no explanation,Other,colloquial-other-06,English,Bungoma\nkituo cha maji kimefungwa miezi hakuna sababu,Other,colloquial-other-06,Sheng,Nairobi\nbarabara iliyoahidiwa haikujengwa,Other,colloquial-other-06,Sheng,Mombasa\n"Juzi 17 February 2026 nikaingia pale office ya Imenti Central, Meru, for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Bribery,realistic-bribery-54,Sheng,Meru\n"Nili-check document ya ilihaririwa pale Nakuru East, Nakuru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-60,Sheng,Nakuru\n"For tender ICT-09 in Garissa Township, Garissa, the bid evaluation team used scoring criteria that were not published in the notice. The file concerns computer and network systems and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Procurement Fraud,realistic-procurement-fraud-29,English,Garissa\n"For tender ICT-09 in Kapseret, Eldoret, the bid evaluation team used scoring criteria that were not published in the notice. The file concerns computer and network systems and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Procurement Fraud,realistic-procurement-fraud-05,English,Eldoret\n"Nilienda pale office ya Embakasi East, Nairobi, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 1,000, na screenshot na receipt bado. Case/rekodi KW-051.",Bribery,realistic-bribery-51,Mixed,Nairobi\n"Nilienda pale office ya Bungoma Central, Bungoma, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 1,000, na screenshot na receipt bado. Case/rekodi KW-027.",Bribery,realistic-bribery-27,Mixed,Bungoma\n"Kuna doo za pesa ya mradi wa vijana zilipotea pale Embakasi East, Nairobi after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Embezzlement,realistic-embezzlement-14,Sheng,Nairobi\n"Pale Kapseret, Eldoret on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no action taken, na screenshot na receipt bado. Case/rekodi KW-013.",Other,realistic-other-13,Mixed,Eldoret\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Embakasi East, Nairobi, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Bribery,realistic-bribery-01,Kiswahili,Nairobi\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-014.",Extortion,realistic-extortion-14,Mixed,Eldoret\n"Tarehe 22 April 2026, nilishughulika na simu ya msaada ya umma katika Imenti Central, Meru. baada ya simu nyingi bila nambari ya kufuatilia. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Other,realistic-other-48,Kiswahili,Meru\n"Kuna money misuse kwa fedha za kisima pale Likoni, Mombasa, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 300, na screenshot na receipt bado. Case/rekodi KW-001.",Embezzlement,realistic-embezzlement-01,Mixed,Mombasa\n"Tarehe 3 January 2026 katika Bungoma Central, Bungoma, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Abuse of Office,realistic-abuse-of-office-26,Kiswahili,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Likoni, Mombasa, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Embezzlement,realistic-embezzlement-01,Kiswahili,Mombasa\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Likoni, Mombasa, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-034.",Procurement Fraud,realistic-procurement-fraud-34,Mixed,Mombasa\n"Kwa tender ya usambazaji wa mafuta pale Likoni, Mombasa, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-024.",Procurement Fraud,realistic-procurement-fraud-24,Mixed,Mombasa\n"On 28 July 2026 in Garissa Township, Garissa, the county officer changed a queue order to serve a personal acquaintance first while others were told to wait. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Abuse of Office,realistic-abuse-of-office-32,English,Garissa\n"On 14 June 2026 in Nakuru East, Nakuru, the police officer held my identification card and demanded KSh 2,500, about KSh 1,000. They threatening arrest unless I paid in cash. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Extortion,realistic-extortion-57,English,Nakuru\n"Kwa ukodishaji wa magari katika Likoni, Mombasa, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Tukio linahusu ukodishaji wa magari yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Procurement Fraud,realistic-procurement-fraud-54,Kiswahili,Mombasa\n"I am reporting irregular use of KSh 20,000 in Kilifi North, Kilifi, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 1,000 was withdrawn but the borehole was never completed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Embezzlement,realistic-embezzlement-33,English,Kilifi\n"Nimekagua hati kuhusu ilibadilishwa katika Kapseret, Eldoret, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-27,Kiswahili,Eldoret\n"Pale Kapseret, Eldoret, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-051.",Nepotism/Favoritism,realistic-nepotism-favoritism-51,Mixed,Eldoret\n"Juzi 11 August 2026 nikaingia pale office ya Machakos Central, Machakos, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Bribery,realistic-bribery-36,Sheng,Machakos\n"Kuna money misuse kwa chakula na blanketi pale Bungoma Central, Bungoma, after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 2,500, nimeweka details ya time na office number. Case/rekodi KW-040.",Embezzlement,realistic-embezzlement-40,Mixed,Bungoma\n"For contract SUP-77 in Likoni, Mombasa, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-004.",Procurement Fraud,realistic-procurement-fraud-04,English,Mombasa\n"Pale Nakuru East, Nakuru on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Nepotism/Favoritism,realistic-nepotism-favoritism-34,Sheng,Nakuru\n"Tarehe 28 July 2026 katika Embakasi East, Nairobi, afisa wa kaunti wengine wakaambiwa wangojee. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Abuse of Office,realistic-abuse-of-office-40,Kiswahili,Nairobi\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Kilifi North, Kilifi. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Conflict of Interest,realistic-conflict-of-interest-51,English,Kilifi\n"On 11 August 2026 pale Machakos Central, Machakos, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-005.",Abuse of Office,realistic-abuse-of-office-05,Sheng,Machakos\n"Kuna money misuse kwa chakula na blanketi pale Kapseret, Eldoret, after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 500, nimeweka details ya time na office number. Case/rekodi KW-032.",Embezzlement,realistic-embezzlement-32,Mixed,Eldoret\n"On 17 February 2026 in Likoni, Mombasa, the public officer threatened to blacklist my supplier account and demanded KSh 5,000, about KSh 300. They unless I paid from my personal funds. I kept the receipt and the message thread. Case note KW-013.",Extortion,realistic-extortion-13,English,Mombasa\n"Pale Bungoma Central, Bungoma on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-041.",Other,realistic-other-41,Mixed,Bungoma\n"Nili-check document ya iliandika bidhaa pale Nakuru East, Nakuru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-50,Sheng,Nakuru\n"Tarehe 6 May 2026 katika Kisumu Central, Kisumu, mkaguzi akadai malipo ya faragha mara moja, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Extortion,realistic-extortion-60,Kiswahili,Kisumu\n"Pale Nakuru East, Nakuru on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Other,realistic-other-46,Sheng,Nakuru\n"Nimekagua hati kuhusu ilibadilishwa katika Embakasi East, Nairobi, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-19,Kiswahili,Nairobi\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Kapseret, Eldoret. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Conflict of Interest,realistic-conflict-of-interest-20,Kiswahili,Eldoret\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Kilifi North, Kilifi. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Conflict of Interest,realistic-conflict-of-interest-11,Kiswahili,Kilifi\n"On 17 February 2026, I went to the public office in Nakuru East, Nakuru, for release of a passport. The the immigration officer requested money to place the application ahead of the queue; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-022.",Bribery,realistic-bribery-22,English,Nakuru\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Kisumu Central, Kisumu, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Bribery,realistic-bribery-25,Kiswahili,Kisumu\n"Katika Kilifi North, Kilifi, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Nepotism/Favoritism,realistic-nepotism-favoritism-42,Kiswahili,Kilifi\n"On 3 January 2026 in Embakasi East, Nairobi, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I kept the receipt and the message thread. Case note KW-010.",Abuse of Office,realistic-abuse-of-office-10,English,Nairobi\n"On 22 April 2026 in Likoni, Mombasa, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Abuse of Office,realistic-abuse-of-office-57,English,Mombasa\n"On 11 August 2026 pale Likoni, Mombasa, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-037.",Abuse of Office,realistic-abuse-of-office-37,Sheng,Mombasa\n"Nimekagua hati kuhusu iliandika bidhaa katika Likoni, Mombasa, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-26,Kiswahili,Mombasa\n"I am reporting irregular use of KSh 5,000 in Kilifi North, Kilifi, after 14 June 2026. Records show that the public works foreman handled road-repair funds, and the funds worth KSh 300 were spent while the damaged section remained unfinished. I kept the receipt and the message thread. Case note KW-013.",Embezzlement,realistic-embezzlement-13,English,Kilifi\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Nakuru East, Nakuru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-058.",Procurement Fraud,realistic-procurement-fraud-58,Mixed,Nakuru\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Garissa Township, Garissa, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Bribery,realistic-bribery-33,Kiswahili,Garissa\n"On 22 April 2026, I dealt with the public help line in Likoni, Mombasa. The office did not answer for repeated calls and provided no follow-up reference. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Other,realistic-other-32,English,Mombasa\n"On 14 June 2026 in Nakuru East, Nakuru, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Abuse of Office,realistic-abuse-of-office-51,English,Nakuru\n"On 3 January 2026 pale Kapseret, Eldoret, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-024.",Extortion,realistic-extortion-24,Mixed,Eldoret\n"On 17 February 2026 pale Nakuru East, Nakuru, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-003.",Conflict of Interest,realistic-conflict-of-interest-03,Mixed,Nakuru\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Garissa Township, Garissa. kutofanya kazi kwa miezi kadhaa bila maelezo. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Other,realistic-other-27,Kiswahili,Garissa\n"Katika Bungoma Central, Bungoma, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Nepotism/Favoritism,realistic-nepotism-favoritism-59,Kiswahili,Bungoma\n"Pale Kisumu Central, Kisumu on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-017.",Nepotism/Favoritism,realistic-nepotism-favoritism-17,Sheng,Kisumu\n"On 6 May 2026 pale Embakasi East, Nairobi, afisa wa rekodi akatishia kuondoa rekodi ya ombi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Abuse of Office,realistic-abuse-of-office-30,Sheng,Nairobi\n"Tarehe 17 February 2026, nilishughulika na dirisha la hospitali katika Nakuru East, Nakuru. kwa nini foleni ilisitishwa bila kutoa njia ya kulalamika. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Other,realistic-other-46,Kiswahili,Nakuru\n"On 28 July 2026, I went to the public office in Bungoma Central, Bungoma, for printing an identification document. The the Huduma Centre clerk asked for a personal payment to speed up the ordinary service; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Bribery,realistic-bribery-47,English,Bungoma\n"On 3 January 2026 in Bungoma Central, Bungoma, the licensing inspector demanded cash and demanded KSh 800, about KSh 500. They and said he would report a compliance breach if I refused. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Extortion,realistic-extortion-32,English,Bungoma\n"On 6 May 2026 in Kapseret, Eldoret, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 2,500. They and demanded an immediate private payment. I kept the receipt and the message thread. Case note KW-004.",Extortion,realistic-extortion-04,English,Eldoret\n"On 28 July 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-044.",Conflict of Interest,realistic-conflict-of-interest-44,Mixed,Garissa\n"Kuna money misuse kwa pesa za dawa pale Bungoma Central, Bungoma, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 500, nimeweka details ya time na office number. Case/rekodi KW-020.",Embezzlement,realistic-embezzlement-20,Mixed,Bungoma\n"Kwa tender ya huduma za ulinzi pale Garissa Township, Garissa, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-039.",Procurement Fraud,realistic-procurement-fraud-39,Mixed,Garissa\n"For contract FUEL-33 in Imenti Central, Meru, the evaluation secretary altered the comparison sheet after bids were submitted. The file concerns fuel supply and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-040.",Procurement Fraud,realistic-procurement-fraud-40,English,Meru\n"Katika Kisumu Central, Kisumu, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Nepotism/Favoritism,realistic-nepotism-favoritism-57,Kiswahili,Kisumu\n"On 6 May 2026, I went to the public office in Garissa Township, Garissa, for permission to continue trading. The the county askari offered to overlook the stall issue for a private payment; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-013.",Bribery,realistic-bribery-13,English,Garissa\n"Kuna money misuse kwa pesa za dawa pale Embakasi East, Nairobi, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 500, nimeweka details ya time na office number. Case/rekodi KW-044.",Embezzlement,realistic-embezzlement-44,Mixed,Nairobi\n"Nilienda pale office ya Kisumu Central, Kisumu, on 6 May 2026 for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha, 1,000, na screenshot na receipt bado. Case/rekodi KW-045.",Bribery,realistic-bribery-45,Mixed,Kisumu\n"On 9 March 2026 in Kapseret, Eldoret, the county askari took the goods and demanded KSh 1,000, about KSh 2,500. They said they would be returned only after a payment. I kept the receipt and the message thread. Case note KW-034.",Extortion,realistic-extortion-34,English,Eldoret\n"Tarehe 17 February 2026, nilishughulika na dirisha la hospitali katika Likoni, Mombasa. kwa nini foleni ilisitishwa bila kutoa njia ya kulalamika. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Other,realistic-other-22,Kiswahili,Mombasa\n"Pale Garissa Township, Garissa on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-035.",Nepotism/Favoritism,realistic-nepotism-favoritism-35,Sheng,Garissa\n"I am reporting irregular use of KSh 2,500 in Machakos Central, Machakos, after 11 August 2026. Records show that the county market cashier handled weekly market fee collections, and the funds worth KSh 1,000 were missing from the ledger and bank deposit. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Embezzlement,realistic-embezzlement-39,English,Machakos\n"Katika Bungoma Central, Bungoma, tarehe 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Nepotism/Favoritism,realistic-nepotism-favoritism-29,Kiswahili,Bungoma\n"On 17 February 2026 in Nakuru East, Nakuru, the school principal directed a teacher to use school equipment for a private function during school time. This was not a routine decision. I kept the receipt and the message thread. Case note KW-031.",Abuse of Office,realistic-abuse-of-office-31,English,Nakuru\n"Nili-check document ya ilikuwa na sahihi iliyorudishwa pale Kapseret, Eldoret, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, nimehifadhi screenshot na receipt. Ref KW-007.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-07,Sheng,Eldoret\n"On 17 February 2026 pale Likoni, Mombasa, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-019.",Conflict of Interest,realistic-conflict-of-interest-19,Sheng,Mombasa\n"Nilikagua document for ilitumia muhuri wa kughushi pale Likoni, Mombasa, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeweka details ya time na office number. Case/rekodi KW-056.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-56,Mixed,Mombasa\n"Nilienda pale office ya Kilifi North, Kilifi, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 2,500, nimeweka details ya time na office number. Case/rekodi KW-010.",Bribery,realistic-bribery-10,Mixed,Kilifi\n"Tarehe 6 May 2026, nilienda ofisi ya umma katika Kisumu Central, Kisumu, kwa ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Bribery,realistic-bribery-45,Kiswahili,Kisumu\n"Nilikagua document for ilonyesha anwani ya uongo pale Nakuru East, Nakuru, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeweka details ya time na office number. Case/rekodi KW-030.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-30,Mixed,Nakuru\n"Tender ya mifumo ya kompyuta na mtandao pale Bungoma Central, Bungoma ilikuwa imepangwa. timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa, around 300, nimehifadhi screenshot na receipt. Ref KW-013.",Procurement Fraud,realistic-procurement-fraud-13,Sheng,Bungoma\n"Katika Kisumu Central, Kisumu, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Nepotism/Favoritism,realistic-nepotism-favoritism-17,Kiswahili,Kisumu\n"Nilikagua document for ilikuwa na sahihi iliyorudishwa pale Embakasi East, Nairobi, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, na screenshot na receipt bado. Case/rekodi KW-039.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-39,Mixed,Nairobi\n"On 3 January 2026, I dealt with the public office in Bungoma Central, Bungoma. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-001.",Other,realistic-other-01,English,Bungoma\n"Nilienda pale office ya Kilifi North, Kilifi, on 22 April 2026 for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada, 2,500, nimeweka details ya time na office number. Case/rekodi KW-040.",Bribery,realistic-bribery-40,Mixed,Kilifi\n"On 28 July 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Conflict of Interest,realistic-conflict-of-interest-04,Sheng,Garissa\n"On 9 March 2026 pale Embakasi East, Nairobi, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-020.",Abuse of Office,realistic-abuse-of-office-20,Mixed,Nairobi\n"Pale Kapseret, Eldoret, on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-021.",Nepotism/Favoritism,realistic-nepotism-favoritism-21,Mixed,Eldoret\n"I am reporting irregular use of KSh 5,000 in Garissa Township, Garissa, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 500 entered receipts that could not be matched to payments. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Embezzlement,realistic-embezzlement-26,English,Garissa\n"On 17 February 2026 in Machakos Central, Machakos, the public officer threatened to blacklist my supplier account and demanded KSh 5,000, about KSh 1,000. They unless I paid from my personal funds. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Extortion,realistic-extortion-21,English,Machakos\n"Kuna money misuse kwa pesa za dawa pale Embakasi East, Nairobi, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 2,500, nimeweka details ya time na office number. Case/rekodi KW-004.",Embezzlement,realistic-embezzlement-04,Mixed,Nairobi\n"Katika Embakasi East, Nairobi, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Nepotism/Favoritism,realistic-nepotism-favoritism-03,Kiswahili,Nairobi\n"Kuna doo za fedha za chakula zilipotea pale Kilifi North, Kilifi after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 1000, nimehifadhi screenshot na receipt. Ref KW-003.",Embezzlement,realistic-embezzlement-03,Sheng,Kilifi\n"On 14 June 2026 pale Likoni, Mombasa, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-033.",Extortion,realistic-extortion-33,Sheng,Mombasa\n"On 11 August 2026 pale Likoni, Mombasa, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-003.",Extortion,realistic-extortion-03,Sheng,Mombasa\n"Kwa ukodishaji wa magari katika Imenti Central, Meru, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Tukio linahusu ukodishaji wa magari yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Procurement Fraud,realistic-procurement-fraud-30,Kiswahili,Meru\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Kisumu Central, Kisumu, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-041.",Procurement Fraud,realistic-procurement-fraud-41,Mixed,Kisumu\n"Nilienda pale office ya Likoni, Mombasa, on 17 February 2026 for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni, 500, nimeweka details ya time na office number. Case/rekodi KW-038.",Bribery,realistic-bribery-38,Mixed,Mombasa\n"Katika Kilifi North, Kilifi, tarehe 14 June 2026, bodi ya CDF bila kura ya wazi. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Nepotism/Favoritism,realistic-nepotism-favoritism-22,Kiswahili,Kilifi\n"Nimekagua hati kuhusu ilihaririwa katika Kilifi North, Kilifi, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-28,Kiswahili,Kilifi\n"In Likoni, Mombasa, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Nepotism/Favoritism,realistic-nepotism-favoritism-60,English,Mombasa\n"Tarehe 14 June 2026, mwenyekiti wa jopo la ruzuku alishiriki uamuzi kuhusu alikuwa mshirika katika Machakos Central, Machakos. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Conflict of Interest,realistic-conflict-of-interest-47,Kiswahili,Machakos\n"Nilikagua document for iliandika bidhaa pale Nakuru East, Nakuru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-050.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-50,Mixed,Nakuru\n"On 11 August 2026 in Imenti Central, Meru, the department director used an official vehicle for a private family trip and signed the fuel claim through the office. This was not a routine decision. I kept the receipt and the message thread. Case note KW-013.",Abuse of Office,realistic-abuse-of-office-13,English,Meru\n"In Embakasi East, Nairobi, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-043.",Nepotism/Favoritism,realistic-nepotism-favoritism-43,English,Nairobi\n"On 11 August 2026 pale Kilifi North, Kilifi, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-001.",Conflict of Interest,realistic-conflict-of-interest-01,Sheng,Kilifi\n"Nili-check document ya ilikuwa na sahihi iliyorudishwa pale Bungoma Central, Bungoma, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, nimehifadhi screenshot na receipt. Ref KW-015.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-15,Sheng,Bungoma\n"On 3 January 2026, I went to the public office in Bungoma Central, Bungoma, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Bribery,realistic-bribery-17,English,Bungoma\n"On 28 July 2026, I dealt with the county portal in Kapseret, Eldoret. The office removed a public notice without a published reason after residents requested services. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Other,realistic-other-23,English,Eldoret\n"On 3 January 2026, I went to the public office in Kapseret, Eldoret, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Bribery,realistic-bribery-09,English,Eldoret\n"Pale Likoni, Mombasa on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Nepotism/Favoritism,realistic-nepotism-favoritism-60,Sheng,Mombasa\n"On 3 January 2026 pale Garissa Township, Garissa, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-042.",Abuse of Office,realistic-abuse-of-office-42,Mixed,Garissa\n"On 11 August 2026 pale Machakos Central, Machakos, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-045.",Abuse of Office,realistic-abuse-of-office-45,Mixed,Machakos\n"On 22 April 2026 pale Nakuru East, Nakuru, afisa wa fedha wa kaunti joined the decision on alikuwa mmiliki binafsi; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-053.",Conflict of Interest,realistic-conflict-of-interest-53,Mixed,Nakuru\n"On 6 May 2026 pale Kapseret, Eldoret, afisa wa rekodi akatishia kuondoa rekodi ya ombi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Abuse of Office,realistic-abuse-of-office-38,Sheng,Eldoret\n"On 28 July 2026 pale Kapseret, Eldoret, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Conflict of Interest,realistic-conflict-of-interest-60,Sheng,Eldoret\n"On 17 February 2026 pale Kilifi North, Kilifi, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-011.",Conflict of Interest,realistic-conflict-of-interest-11,Mixed,Kilifi\n"Kwa samani za ofisi katika Likoni, Mombasa, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Procurement Fraud,realistic-procurement-fraud-04,Kiswahili,Mombasa\n"For contract FUEL-33 in Likoni, Mombasa, the evaluation secretary altered the comparison sheet after bids were submitted. The file concerns fuel supply and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Procurement Fraud,realistic-procurement-fraud-24,English,Mombasa\n"Kuna money misuse kwa pesa ya mradi wa vijana pale Garissa Township, Garissa, after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 20,000, nimeweka details ya time na office number. Case/rekodi KW-006.",Embezzlement,realistic-embezzlement-06,Mixed,Garissa\n"On 3 January 2026 in Kisumu Central, Kisumu, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I kept the receipt and the message thread. Case note KW-034.",Abuse of Office,realistic-abuse-of-office-34,English,Kisumu\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Kilifi North, Kilifi. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-001.",Conflict of Interest,realistic-conflict-of-interest-01,English,Kilifi\n"Tarehe 22 April 2026 katika Nakuru East, Nakuru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Extortion,realistic-extortion-47,Kiswahili,Nakuru\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Nakuru East, Nakuru, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Embezzlement,realistic-embezzlement-25,Kiswahili,Nakuru\n"On 9 March 2026 pale Garissa Township, Garissa, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Abuse of Office,realistic-abuse-of-office-52,Sheng,Garissa\n"On 11 August 2026, I dealt with the public counter in Machakos Central, Machakos. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Other,realistic-other-20,English,Machakos\n"Tender ya ukodishaji wa magari pale Likoni, Mombasa ilikuwa imepangwa. kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Procurement Fraud,realistic-procurement-fraud-14,Sheng,Mombasa\n"On 14 June 2026 pale Nakuru East, Nakuru, mwenyekiti wa jopo la ruzuku aliingia decision ya alikuwa mshirika; was a partner, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-023.",Conflict of Interest,realistic-conflict-of-interest-23,Sheng,Nakuru\n"Nimekagua hati kuhusu iliandika bidhaa katika Nakuru East, Nakuru, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-10,Kiswahili,Nakuru\n"Pale Garissa Township, Garissa on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-057.",Other,realistic-other-57,Mixed,Garissa\n"Kuna doo za fedha za wanafunzi zilipotea pale Garissa Township, Garissa after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Embezzlement,realistic-embezzlement-26,Sheng,Garissa\n"On 6 May 2026 pale Kapseret, Eldoret, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-050.",Conflict of Interest,realistic-conflict-of-interest-50,Mixed,Eldoret\n"On 6 May 2026, I went to the public office in Kapseret, Eldoret, for permission to continue trading. The the county askari offered to overlook the stall issue for a private payment; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Bribery,realistic-bribery-29,English,Eldoret\n"On 9 March 2026 pale Bungoma Central, Bungoma, askari wa kaunti akasema zingerudishwa baada ya malipo, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Extortion,realistic-extortion-02,Sheng,Bungoma\n"Kwa zabuni ya vifaa vya hospitali katika Kilifi North, Kilifi, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Procurement Fraud,realistic-procurement-fraud-26,Kiswahili,Kilifi\n"Katika Garissa Township, Garissa, tarehe 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Nepotism/Favoritism,realistic-nepotism-favoritism-55,Kiswahili,Garissa\n"I am reporting irregular use of KSh 20,000 in Garissa Township, Garissa, after 9 March 2026. Records show that the youth fund administrator handled approved youth project money, and the funds worth KSh 2,500 was transferred to a personal account. I kept the receipt and the message thread. Case note KW-046.",Embezzlement,realistic-embezzlement-46,English,Garissa\n"I checked a delivery invoice relating to the procurement file in Nakuru East, Nakuru, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-010.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-10,English,Nakuru\n"Tender ya mifumo ya kompyuta na mtandao pale Kapseret, Eldoret ilikuwa imepangwa. timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa, around 1000, nimehifadhi screenshot na receipt. Ref KW-045.",Procurement Fraud,realistic-procurement-fraud-45,Sheng,Eldoret\n"On 28 July 2026 pale Garissa Township, Garissa, afisa wa kaunti wengine wakaambiwa wangojee; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-032.",Abuse of Office,realistic-abuse-of-office-32,Mixed,Garissa\n"I checked a payment voucher relating to the hospital account in Bungoma Central, Bungoma, on 6 May 2026. The document carried a duplicate signature for a purchase that has no supplier evidence. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-055.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-55,English,Bungoma\n"Nilienda pale office ya Embakasi East, Nairobi, on 6 May 2026 for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha, 1,000, na screenshot na receipt bado. Case/rekodi KW-021.",Bribery,realistic-bribery-21,Mixed,Nairobi\n"Pale Kisumu Central, Kisumu on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-047.",Nepotism/Favoritism,realistic-nepotism-favoritism-47,Sheng,Kisumu\n"On 3 January 2026 pale Embakasi East, Nairobi, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-016.",Extortion,realistic-extortion-16,Mixed,Nairobi\n"Ninawasilisha tuhuma za matumizi mabaya ya mapato ya ada za soko katika Imenti Central, Meru, tangu 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Embezzlement,realistic-embezzlement-07,Kiswahili,Meru\n"On 22 April 2026 pale Likoni, Mombasa, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-017.",Abuse of Office,realistic-abuse-of-office-17,Mixed,Mombasa\n"Kwa huduma za ulinzi katika Embakasi East, Nairobi, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Tukio linahusu huduma za ulinzi yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Procurement Fraud,realistic-procurement-fraud-07,Kiswahili,Nairobi\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Embakasi East, Nairobi. kutofanya kazi kwa miezi kadhaa bila maelezo. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Other,realistic-other-35,Kiswahili,Nairobi\n"Kuna money misuse kwa fedha za wanafunzi pale Bungoma Central, Bungoma, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 500, nimeweka details ya time na office number. Case/rekodi KW-050.",Embezzlement,realistic-embezzlement-50,Mixed,Bungoma\n"Tarehe 3 January 2026 katika Garissa Township, Garissa, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Extortion,realistic-extortion-08,Kiswahili,Garissa\n"Kwa tender ya samani za ofisi pale Nakuru East, Nakuru, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-028.",Procurement Fraud,realistic-procurement-fraud-28,Mixed,Nakuru\n"Nimekagua hati kuhusu kilitiwa sahihi katika Garissa Township, Garissa, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-41,Kiswahili,Garissa\n"For contract FUEL-33 in Nakuru East, Nakuru, the evaluation secretary altered the comparison sheet after bids were submitted. The file concerns fuel supply and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Procurement Fraud,realistic-procurement-fraud-48,English,Nakuru\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Likoni, Mombasa, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Bribery,realistic-bribery-28,Kiswahili,Mombasa\n"Tarehe 22 April 2026 katika Machakos Central, Machakos, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Abuse of Office,realistic-abuse-of-office-25,Kiswahili,Machakos\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Extortion,realistic-extortion-14,Sheng,Eldoret\n"Kwa tender ya usambazaji wa mafuta pale Nakuru East, Nakuru, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-008.",Procurement Fraud,realistic-procurement-fraud-08,Mixed,Nakuru\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Likoni, Mombasa. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Conflict of Interest,realistic-conflict-of-interest-59,Kiswahili,Mombasa\n"Kwa tender ya usambazaji wa mafuta pale Kilifi North, Kilifi, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-056.",Procurement Fraud,realistic-procurement-fraud-56,Mixed,Kilifi\n"Nilikagua document for ilitumia muhuri wa kughushi pale Imenti Central, Meru, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeweka details ya time na office number. Case/rekodi KW-032.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-32,Mixed,Meru\n"I checked a meeting minute relating to the committee record in Nakuru East, Nakuru, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-60,English,Nakuru\n"On 14 June 2026 pale Kilifi North, Kilifi, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-059.",Abuse of Office,realistic-abuse-of-office-59,Sheng,Kilifi\n"Nilikagua document for kilitiwa sahihi pale Kisumu Central, Kisumu, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-033.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-33,Mixed,Kisumu\n"Juzi 6 May 2026 nikaingia pale office ya Bungoma Central, Bungoma, for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha 300, nimehifadhi screenshot na receipt. Ref KW-037.",Bribery,realistic-bribery-37,Sheng,Bungoma\n"On 14 June 2026 pale Kilifi North, Kilifi, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-059.",Abuse of Office,realistic-abuse-of-office-59,Mixed,Kilifi\n"On 22 April 2026, the county finance officer took part in a decision about a lease decision in Likoni, Mombasa. The officer personally owned the land being leased to the public entity. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Conflict of Interest,realistic-conflict-of-interest-29,English,Mombasa\n"Kuna doo za pesa za dawa zilipotea pale Garissa Township, Garissa after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Embezzlement,realistic-embezzlement-36,Sheng,Garissa\n"On 9 March 2026 pale Kisumu Central, Kisumu, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-044.",Abuse of Office,realistic-abuse-of-office-44,Mixed,Kisumu\n"On 3 January 2026 in Garissa Township, Garissa, the licensing inspector demanded cash and demanded KSh 800, about KSh 20,000. They and said he would report a compliance breach if I refused. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Extortion,realistic-extortion-48,English,Garissa\n"On 6 May 2026 pale Bungoma Central, Bungoma, afisa wa rekodi akatishia kuondoa rekodi ya ombi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-046.",Abuse of Office,realistic-abuse-of-office-46,Mixed,Bungoma\n"Tarehe 3 January 2026 katika Kapseret, Eldoret, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Abuse of Office,realistic-abuse-of-office-18,Kiswahili,Eldoret\n"In Imenti Central, Meru, on 14 June 2026, the CDF board funded the chairman\'s project ahead of equally eligible projects without an open vote. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-046.",Nepotism/Favoritism,realistic-nepotism-favoritism-46,English,Meru\n"Tarehe 28 July 2026 katika Garissa Township, Garissa, afisa wa kaunti wengine wakaambiwa wangojee. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Abuse of Office,realistic-abuse-of-office-32,Kiswahili,Garissa\n"I am reporting irregular use of KSh 20,000 in Imenti Central, Meru, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 1,000 were used to buy items not delivered to the school. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Embezzlement,realistic-embezzlement-27,English,Meru\n"On 28 July 2026, I dealt with the county portal in Garissa Township, Garissa. The office removed a public notice without a published reason after residents requested services. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Other,realistic-other-47,English,Garissa\n"Kuna doo za fedha za chakula zilipotea pale Likoni, Mombasa after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 5000, nimehifadhi screenshot na receipt. Ref KW-011.",Embezzlement,realistic-embezzlement-11,Sheng,Mombasa\n"Nilikagua document for ilihaririwa pale Imenti Central, Meru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-052.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-52,Mixed,Meru\n"On 14 June 2026, I went to the public office in Machakos Central, Machakos, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Bribery,realistic-bribery-26,English,Machakos\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Nakuru East, Nakuru. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Conflict of Interest,realistic-conflict-of-interest-03,English,Nakuru\n"On 6 May 2026 pale Kisumu Central, Kisumu, mkaguzi akadai malipo ya faragha mara moja, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-060.",Extortion,realistic-extortion-60,Mixed,Kisumu\n"On 14 June 2026, I dealt with the service desk in Machakos Central, Machakos. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Other,realistic-other-50,English,Machakos\n"Tender ya samani za ofisi pale Likoni, Mombasa ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Procurement Fraud,realistic-procurement-fraud-04,Sheng,Mombasa\n"On 6 May 2026, the project director took part in a decision about a construction contract in Embakasi East, Nairobi. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Conflict of Interest,realistic-conflict-of-interest-02,English,Nairobi\n"Tarehe 22 April 2026, nilienda ofisi ya umma katika Likoni, Mombasa, kwa abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Bribery,realistic-bribery-08,Kiswahili,Mombasa\n"Nili-check document ya ilitumia muhuri wa kughushi pale Likoni, Mombasa, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-16,Sheng,Mombasa\n"On 28 July 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Conflict of Interest,realistic-conflict-of-interest-44,Sheng,Garissa\n"Tarehe 28 July 2026 katika Kapseret, Eldoret, afisa wa kaunti wengine wakaambiwa wangojee. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Abuse of Office,realistic-abuse-of-office-48,Kiswahili,Eldoret\n"On 6 May 2026 pale Kapseret, Eldoret, mkaguzi akadai malipo ya faragha mara moja, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-004.",Extortion,realistic-extortion-04,Mixed,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Imenti Central, Meru, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Embezzlement,realistic-embezzlement-27,Kiswahili,Meru\n"Nilikagua document for kilikuwa na jina lililobadilishwa pale Garissa Township, Garissa, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, na screenshot na receipt bado. Case/rekodi KW-021.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-21,Mixed,Garissa\n"Nimekagua hati kuhusu ilibadilishwa katika Kisumu Central, Kisumu, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-03,Kiswahili,Kisumu\n"Tarehe 3 January 2026 katika Kapseret, Eldoret, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Abuse of Office,realistic-abuse-of-office-58,Kiswahili,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Likoni, Mombasa, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Embezzlement,realistic-embezzlement-51,Kiswahili,Mombasa\n"Juzi 22 April 2026 nikaingia pale office ya Likoni, Mombasa, for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada 500, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Bribery,realistic-bribery-08,Sheng,Mombasa\n"Juzi 28 July 2026 nikaingia pale office ya Kisumu Central, Kisumu, for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida 300, nimehifadhi screenshot na receipt. Ref KW-055.",Bribery,realistic-bribery-55,Sheng,Kisumu\n"On 3 January 2026, I dealt with the public office in Kapseret, Eldoret. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Other,realistic-other-33,English,Eldoret\n"Tender ya zabuni ya ukarabati wa barabara pale Garissa Township, Garissa ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 300, nimehifadhi screenshot na receipt. Ref KW-049.",Procurement Fraud,realistic-procurement-fraud-49,Sheng,Garissa\n"On 11 August 2026 pale Imenti Central, Meru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-013.",Abuse of Office,realistic-abuse-of-office-13,Mixed,Meru\n"Kuna doo za pesa ya mradi wa vijana zilipotea pale Kisumu Central, Kisumu after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Embezzlement,realistic-embezzlement-38,Sheng,Kisumu\n"On 14 June 2026, I dealt with the service desk in Kilifi North, Kilifi. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-034.",Other,realistic-other-34,English,Kilifi\n"Tarehe 14 June 2026 katika Machakos Central, Machakos, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Extortion,realistic-extortion-41,Kiswahili,Machakos\n"Kuna money misuse kwa fedha za wanafunzi pale Kapseret, Eldoret, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 20,000, nimeweka details ya time na office number. Case/rekodi KW-042.",Embezzlement,realistic-embezzlement-42,Mixed,Eldoret\n"In Garissa Township, Garissa, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Nepotism/Favoritism,realistic-nepotism-favoritism-35,English,Garissa\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa ya mradi wa vijana katika Embakasi East, Nairobi, tangu 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Embezzlement,realistic-embezzlement-14,Kiswahili,Nairobi\n"On 9 March 2026 pale Bungoma Central, Bungoma, afisa wa uteuzi wa umma aliingia decision ya alikuwa mwenzi; was the spouse, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Conflict of Interest,realistic-conflict-of-interest-08,Sheng,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Machakos Central, Machakos, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Embezzlement,realistic-embezzlement-09,Kiswahili,Machakos\n"On 14 June 2026 pale Machakos Central, Machakos, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 300, na screenshot na receipt bado. Case/rekodi KW-001.",Extortion,realistic-extortion-01,Mixed,Machakos\n"Juzi 28 July 2026 nikaingia pale office ya Bungoma Central, Bungoma, for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida 300, nimehifadhi screenshot na receipt. Ref KW-007.",Bribery,realistic-bribery-07,Sheng,Bungoma\n"On 22 April 2026 pale Nakuru East, Nakuru, afisa wa fedha wa kaunti joined the decision on alikuwa mmiliki binafsi; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-013.",Conflict of Interest,realistic-conflict-of-interest-13,Mixed,Nakuru\n"Tarehe 9 March 2026 katika Kisumu Central, Kisumu, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Extortion,realistic-extortion-10,Kiswahili,Kisumu\n"I checked an academic certificate relating to the appointment file in Embakasi East, Nairobi, on 9 March 2026. The document contained a changed name and a stamp that the issuing school says is not genuine. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-29,English,Nairobi\n"On 22 April 2026, I went to the public office in Nakuru East, Nakuru, for an approved police abstract. The the police abstract clerk said the document would not be issued without an extra amount; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Bribery,realistic-bribery-32,English,Nakuru\n"Kwa zabuni ya ujenzi wa madarasa katika Garissa Township, Garissa, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Procurement Fraud,realistic-procurement-fraud-19,Kiswahili,Garissa\n"On 14 June 2026 pale Kilifi North, Kilifi, mwenyekiti wa jopo la ruzuku aliingia decision ya alikuwa mshirika; was a partner, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-031.",Conflict of Interest,realistic-conflict-of-interest-31,Sheng,Kilifi\n"For tender SEC-55 in Embakasi East, Nairobi, the procuring entity gave one supplier confidential bid information. The file concerns security services and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-007.",Procurement Fraud,realistic-procurement-fraud-07,English,Nairobi\n"Tender ya huduma za ulinzi pale Garissa Township, Garissa ilikuwa imepangwa. taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri, around 1000, nimehifadhi screenshot na receipt. Ref KW-039.",Procurement Fraud,realistic-procurement-fraud-39,Sheng,Garissa\n"On 6 May 2026 pale Garissa Township, Garissa, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-034.",Conflict of Interest,realistic-conflict-of-interest-34,Mixed,Garissa\n"On 3 January 2026 in Embakasi East, Nairobi, the licensing inspector demanded cash and demanded KSh 800, about KSh 500. They and said he would report a compliance breach if I refused. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Extortion,realistic-extortion-56,English,Nairobi\n"Kwa tender ya huduma za ulinzi pale Embakasi East, Nairobi, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-047.",Procurement Fraud,realistic-procurement-fraud-47,Mixed,Nairobi\n"I am reporting irregular use of KSh 20,000 in Embakasi East, Nairobi, after 9 March 2026. Records show that the youth fund administrator handled approved youth project money, and the funds worth KSh 20,000 was transferred to a personal account. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Embezzlement,realistic-embezzlement-54,English,Nairobi\n"Tender ya zabuni ya vifaa vya hospitali pale Nakuru East, Nakuru ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Procurement Fraud,realistic-procurement-fraud-58,Sheng,Nakuru\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Machakos Central, Machakos. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Other,realistic-other-10,Kiswahili,Machakos\n"Juzi 14 June 2026 nikaingia pale office ya Likoni, Mombasa, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Bribery,realistic-bribery-58,Sheng,Mombasa\n"On 9 March 2026 pale Kisumu Central, Kisumu, afisa wa uteuzi wa umma aliingia decision ya alikuwa mwenzi; was the spouse, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Conflict of Interest,realistic-conflict-of-interest-16,Sheng,Kisumu\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Embakasi East, Nairobi, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Bribery,realistic-bribery-41,Kiswahili,Nairobi\n"Pale Machakos Central, Machakos on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Nepotism/Favoritism,realistic-nepotism-favoritism-08,Sheng,Machakos\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Embakasi East, Nairobi, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Bribery,realistic-bribery-11,Kiswahili,Nairobi\n"On 14 June 2026 pale Likoni, Mombasa, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-027.",Abuse of Office,realistic-abuse-of-office-27,Sheng,Mombasa\n"Tarehe 6 May 2026, nilienda ofisi ya umma katika Garissa Township, Garissa, kwa ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Bribery,realistic-bribery-13,Kiswahili,Garissa\n"Pale Kisumu Central, Kisumu on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-029.",Other,realistic-other-29,Sheng,Kisumu\n"Nilikagua document for ilihaririwa pale Nakuru East, Nakuru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-020.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-20,Mixed,Nakuru\n"On 22 April 2026, the county finance officer took part in a decision about a lease decision in Kilifi North, Kilifi. The officer personally owned the land being leased to the public entity. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Conflict of Interest,realistic-conflict-of-interest-21,English,Kilifi\n"Pale Kapseret, Eldoret on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no action taken, na screenshot na receipt bado. Case/rekodi KW-053.",Other,realistic-other-53,Mixed,Eldoret\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-054.",Extortion,realistic-extortion-54,Mixed,Eldoret\n"On 11 August 2026 pale Likoni, Mombasa, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-003.",Extortion,realistic-extortion-03,Mixed,Mombasa\n"On 6 May 2026 pale Embakasi East, Nairobi, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Conflict of Interest,realistic-conflict-of-interest-42,Sheng,Nairobi\n"Nili-check document ya ilibadilishwa pale Embakasi East, Nairobi, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-019.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-19,Sheng,Nairobi\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Kapseret, Eldoret. kutofanya kazi kwa miezi kadhaa bila maelezo. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Other,realistic-other-03,Kiswahili,Eldoret\n"On 22 April 2026, I went to the public office in Machakos Central, Machakos, for an approved police abstract. The the police abstract clerk said the document would not be issued without an extra amount; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-016.",Bribery,realistic-bribery-16,English,Machakos\n"Tarehe 22 April 2026, nilishughulika na simu ya msaada ya umma katika Nakuru East, Nakuru. baada ya simu nyingi bila nambari ya kufuatilia. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Other,realistic-other-16,Kiswahili,Nakuru\n"On 17 February 2026 pale Machakos Central, Machakos, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-021.",Extortion,realistic-extortion-21,Mixed,Machakos\n"Nimekagua hati kuhusu ilonyesha anwani ya uongo katika Imenti Central, Meru, tarehe 11 August 2026. leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-22,Kiswahili,Meru\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa kaunti wengine wakaambiwa wangojee; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Abuse of Office,realistic-abuse-of-office-08,Sheng,Eldoret\n"On 11 August 2026, I went to the public office in Imenti Central, Meru, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Bribery,realistic-bribery-44,English,Meru\n"Pale Kilifi North, Kilifi on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Other,realistic-other-54,Sheng,Kilifi\n"Katika Likoni, Mombasa, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Nepotism/Favoritism,realistic-nepotism-favoritism-50,Kiswahili,Mombasa\n"Juzi 14 June 2026 nikaingia pale office ya Nakuru East, Nakuru, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Bribery,realistic-bribery-42,Sheng,Nakuru\n"On 14 June 2026 pale Likoni, Mombasa, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-033.",Extortion,realistic-extortion-33,Mixed,Mombasa\n"On 14 June 2026 pale Kilifi North, Kilifi, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-025.",Extortion,realistic-extortion-25,Sheng,Kilifi\n"Nilikagua document for iliandika bidhaa pale Imenti Central, Meru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-002.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-02,Mixed,Meru\n"Tarehe 17 February 2026, nilienda ofisi ya umma katika Likoni, Mombasa, kwa utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Bribery,realistic-bribery-38,Kiswahili,Mombasa\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Kapseret, Eldoret, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Bribery,realistic-bribery-09,Kiswahili,Eldoret\n"On 14 June 2026 pale Machakos Central, Machakos, mwenyekiti wa jopo la ruzuku joined the decision on alikuwa mshirika; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-047.",Conflict of Interest,realistic-conflict-of-interest-47,Mixed,Machakos\n"Kwa tender ya usambazaji wa mafuta pale Nakuru East, Nakuru, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-048.",Procurement Fraud,realistic-procurement-fraud-48,Mixed,Nakuru\n"Tarehe 28 July 2026, nilienda ofisi ya umma katika Kisumu Central, Kisumu, kwa uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Bribery,realistic-bribery-15,Kiswahili,Kisumu\n"On 3 January 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya shule aliingia decision ya aliwahi kuajiri; employed, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Conflict of Interest,realistic-conflict-of-interest-54,Sheng,Garissa\n"On 28 July 2026 pale Embakasi East, Nairobi, afisa wa kaunti wengine wakaambiwa wangojee; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-040.",Abuse of Office,realistic-abuse-of-office-40,Mixed,Nairobi\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Likoni, Mombasa, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Bribery,realistic-bribery-58,Kiswahili,Mombasa\n"On 3 January 2026, I went to the public office in Embakasi East, Nairobi, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-001.",Bribery,realistic-bribery-01,English,Nairobi\n"Tender ya zabuni ya ukarabati wa barabara pale Kisumu Central, Kisumu ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 300, nimehifadhi screenshot na receipt. Ref KW-001.",Procurement Fraud,realistic-procurement-fraud-01,Sheng,Kisumu\n"Nilikagua document for kilitiwa sahihi pale Embakasi East, Nairobi, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-049.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-49,Mixed,Nairobi\n"On 6 May 2026 pale Embakasi East, Nairobi, afisa wa rekodi akatishia kuondoa rekodi ya ombi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-030.",Abuse of Office,realistic-abuse-of-office-30,Mixed,Nairobi\n"Pale Kilifi North, Kilifi on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Other,realistic-other-24,Sheng,Kilifi\n"On 28 July 2026 pale Kisumu Central, Kisumu, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Conflict of Interest,realistic-conflict-of-interest-36,Sheng,Kisumu\n"Tarehe 14 June 2026, mwenyekiti wa jopo la ruzuku alishiriki uamuzi kuhusu alikuwa mshirika katika Nakuru East, Nakuru. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Conflict of Interest,realistic-conflict-of-interest-23,Kiswahili,Nakuru\n"I am reporting irregular use of KSh 20,000 in Likoni, Mombasa, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 5,000 was withdrawn but the borehole was never completed. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Embezzlement,realistic-embezzlement-41,English,Mombasa\n"In Garissa Township, Garissa, on 3 January 2026, the appointment board promoted an officer\'s spouse to a senior post despite a lower interview score. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Nepotism/Favoritism,realistic-nepotism-favoritism-45,English,Garissa\n"Katika Kisumu Central, Kisumu, tarehe 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Nepotism/Favoritism,realistic-nepotism-favoritism-47,Kiswahili,Kisumu\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Likoni, Mombasa. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-019.",Conflict of Interest,realistic-conflict-of-interest-19,English,Mombasa\n"Tarehe 22 April 2026, nilienda ofisi ya umma katika Nakuru East, Nakuru, kwa abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Bribery,realistic-bribery-32,Kiswahili,Nakuru\n"Juzi 3 January 2026 nikaingia pale office ya Bungoma Central, Bungoma, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 5000, nimehifadhi screenshot na receipt. Ref KW-017.",Bribery,realistic-bribery-17,Sheng,Bungoma\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Embakasi East, Nairobi, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-017.",Procurement Fraud,realistic-procurement-fraud-17,Mixed,Nairobi\n"I checked a payment voucher relating to the hospital account in Bungoma Central, Bungoma, on 6 May 2026. The document carried a duplicate signature for a purchase that has no supplier evidence. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-15,English,Bungoma\n"On 3 January 2026 in Kapseret, Eldoret, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I kept the receipt and the message thread. Case note KW-058.",Abuse of Office,realistic-abuse-of-office-58,English,Eldoret\n"On 11 August 2026, I went to the public office in Machakos Central, Machakos, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Bribery,realistic-bribery-36,English,Machakos\n"Pale Embakasi East, Nairobi on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-043.",Nepotism/Favoritism,realistic-nepotism-favoritism-43,Sheng,Nairobi\n"On 28 July 2026 pale Bungoma Central, Bungoma, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-022.",Extortion,realistic-extortion-22,Mixed,Bungoma\n"Katika Kilifi North, Kilifi, tarehe 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Nepotism/Favoritism,realistic-nepotism-favoritism-32,Kiswahili,Kilifi\n"Kwa zabuni ya ukarabati wa barabara katika Embakasi East, Nairobi, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Procurement Fraud,realistic-procurement-fraud-57,Kiswahili,Nairobi\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Imenti Central, Meru. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Other,realistic-other-58,Kiswahili,Meru\n"I am reporting irregular use of KSh 5,000 in Nakuru East, Nakuru, after 14 June 2026. Records show that the public works foreman handled road-repair funds, and the funds worth KSh 5,000 were spent while the damaged section remained unfinished. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Embezzlement,realistic-embezzlement-05,English,Nakuru\n"Kuna money misuse kwa fedha za ukarabati wa barabara pale Machakos Central, Machakos, after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 5,000, na screenshot na receipt bado. Case/rekodi KW-029.",Embezzlement,realistic-embezzlement-29,Mixed,Machakos\n"On 9 March 2026, the public appointment officer took part in a decision about a job assessment in Kisumu Central, Kisumu. The officer was the spouse of one shortlisted candidate. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-016.",Conflict of Interest,realistic-conflict-of-interest-16,English,Kisumu\n"Tender ya zabuni ya ukarabati wa barabara pale Embakasi East, Nairobi ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 5000, nimehifadhi screenshot na receipt. Ref KW-017.",Procurement Fraud,realistic-procurement-fraud-17,Sheng,Nairobi\n"Kwa tender ya ukodishaji wa magari pale Likoni, Mombasa, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-054.",Procurement Fraud,realistic-procurement-fraud-54,Mixed,Mombasa\n"On 9 March 2026 pale Bungoma Central, Bungoma, afisa wa uteuzi wa umma joined the decision on alikuwa mwenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-008.",Conflict of Interest,realistic-conflict-of-interest-08,Mixed,Bungoma\n"On 17 February 2026 pale Nakuru East, Nakuru, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-037.",Extortion,realistic-extortion-37,Sheng,Nakuru\n"Kwa ukodishaji wa magari katika Machakos Central, Machakos, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Tukio linahusu ukodishaji wa magari yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Procurement Fraud,realistic-procurement-fraud-22,Kiswahili,Machakos\n"On 17 February 2026, I went to the public office in Machakos Central, Machakos, for release of a passport. The the immigration officer requested money to place the application ahead of the queue; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-046.",Bribery,realistic-bribery-46,English,Machakos\n"Tarehe 22 April 2026, afisa wa fedha wa kaunti alishiriki uamuzi kuhusu alikuwa mmiliki binafsi katika Likoni, Mombasa. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Conflict of Interest,realistic-conflict-of-interest-29,Kiswahili,Mombasa\n"For tender KW-026 in Kisumu Central, Kisumu, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-001.",Procurement Fraud,realistic-procurement-fraud-01,English,Kisumu\n"Tarehe 17 February 2026 katika Likoni, Mombasa, mkuu wa shule wakati wa shule. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Abuse of Office,realistic-abuse-of-office-07,Kiswahili,Mombasa\n"Nili-check document ya kilitiwa sahihi pale Embakasi East, Nairobi, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-009.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-09,Sheng,Nairobi\n"On 9 March 2026 pale Bungoma Central, Bungoma, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Abuse of Office,realistic-abuse-of-office-36,Sheng,Bungoma\n"Kuna money misuse kwa fedha za wanafunzi pale Kisumu Central, Kisumu, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 2,500, nimeweka details ya time na office number. Case/rekodi KW-058.",Embezzlement,realistic-embezzlement-58,Mixed,Kisumu\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Bungoma Central, Bungoma. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Conflict of Interest,realistic-conflict-of-interest-18,Kiswahili,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Kisumu Central, Kisumu, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Embezzlement,realistic-embezzlement-18,Kiswahili,Kisumu\n"Tarehe 14 June 2026, mwenyekiti wa jopo la ruzuku alishiriki uamuzi kuhusu alikuwa mshirika katika Likoni, Mombasa. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Conflict of Interest,realistic-conflict-of-interest-39,Kiswahili,Mombasa\n"On 11 August 2026, I dealt with the public counter in Imenti Central, Meru. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-028.",Other,realistic-other-28,English,Meru\n"Kuna money misuse kwa mapato ya ada za soko pale Imenti Central, Meru, after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 300, na screenshot na receipt bado. Case/rekodi KW-007.",Embezzlement,realistic-embezzlement-07,Mixed,Meru\n"For tender ICT-09 in Embakasi East, Nairobi, the bid evaluation team used scoring criteria that were not published in the notice. The file concerns computer and network systems and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-037.",Procurement Fraud,realistic-procurement-fraud-37,English,Nairobi\n"On 11 August 2026 pale Nakuru East, Nakuru, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-033.",Conflict of Interest,realistic-conflict-of-interest-33,Mixed,Nakuru\n"On 3 January 2026 pale Bungoma Central, Bungoma, mjumbe wa bodi ya shule joined the decision on aliwahi kuajiri; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-038.",Conflict of Interest,realistic-conflict-of-interest-38,Mixed,Bungoma\n"Tender ya huduma za ulinzi pale Kapseret, Eldoret ilikuwa imepangwa. taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri, around 300, nimehifadhi screenshot na receipt. Ref KW-055.",Procurement Fraud,realistic-procurement-fraud-55,Sheng,Eldoret\n"For tender KW-026 in Embakasi East, Nairobi, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Procurement Fraud,realistic-procurement-fraud-57,English,Nairobi\n"Kuna money misuse kwa fedha za wanafunzi pale Garissa Township, Garissa, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 500, nimeweka details ya time na office number. Case/rekodi KW-026.",Embezzlement,realistic-embezzlement-26,Mixed,Garissa\n"Juzi 28 July 2026 nikaingia pale office ya Garissa Township, Garissa, for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida 5000, nimehifadhi screenshot na receipt. Ref KW-023.",Bribery,realistic-bribery-23,Sheng,Garissa\n"I am reporting irregular use of KSh 20,000 in Bungoma Central, Bungoma, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 20,000 paid a supplier for drugs that are absent from the store. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Embezzlement,realistic-embezzlement-60,English,Bungoma\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Kapseret, Eldoret. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Conflict of Interest,realistic-conflict-of-interest-60,Kiswahili,Eldoret\n"Tender ya zabuni ya ujenzi wa madarasa pale Embakasi East, Nairobi ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 1000, nimehifadhi screenshot na receipt. Ref KW-027.",Procurement Fraud,realistic-procurement-fraud-27,Sheng,Nairobi\n"Pale Garissa Township, Garissa, on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-015.",Nepotism/Favoritism,realistic-nepotism-favoritism-15,Mixed,Garissa\n"On 3 January 2026 pale Bungoma Central, Bungoma, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-026.",Abuse of Office,realistic-abuse-of-office-26,Mixed,Bungoma\n"Nilikagua document for ilikuwa na sahihi iliyorudishwa pale Bungoma Central, Bungoma, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, na screenshot na receipt bado. Case/rekodi KW-015.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-15,Mixed,Bungoma\n"Pale Likoni, Mombasa, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-060.",Nepotism/Favoritism,realistic-nepotism-favoritism-60,Mixed,Mombasa\n"Tender ya usambazaji wa mafuta pale Likoni, Mombasa ilikuwa imepangwa. katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Procurement Fraud,realistic-procurement-fraud-24,Sheng,Mombasa\n"Tarehe 28 July 2026 katika Kapseret, Eldoret, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Extortion,realistic-extortion-14,Kiswahili,Eldoret\n"On 22 April 2026, I dealt with the public help line in Imenti Central, Meru. The office did not answer for repeated calls and provided no follow-up reference. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Other,realistic-other-48,English,Meru\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Kapseret, Eldoret. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Conflict of Interest,realistic-conflict-of-interest-10,Kiswahili,Eldoret\n"Kwa tender ya huduma za ulinzi pale Kapseret, Eldoret, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-015.",Procurement Fraud,realistic-procurement-fraud-15,Mixed,Eldoret\n"On 11 August 2026 pale Nakuru East, Nakuru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-021.",Abuse of Office,realistic-abuse-of-office-21,Mixed,Nakuru\n"Tarehe 14 June 2026 katika Imenti Central, Meru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Abuse of Office,realistic-abuse-of-office-43,Kiswahili,Meru\n"Pale Imenti Central, Meru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no action taken, nimeweka details ya time na office number. Case/rekodi KW-008.",Other,realistic-other-08,Mixed,Meru\n"For tender KW-026 in Garissa Township, Garissa, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-049.",Procurement Fraud,realistic-procurement-fraud-49,English,Garissa\n"On 11 August 2026 pale Machakos Central, Machakos, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-051.",Extortion,realistic-extortion-51,Mixed,Machakos\n"Nili-check document ya ilitumia muhuri wa kughushi pale Machakos Central, Machakos, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-24,Sheng,Machakos\n"Pale Likoni, Mombasa, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-020.",Nepotism/Favoritism,realistic-nepotism-favoritism-20,Mixed,Mombasa\n"Nili-check document ya kilitiwa sahihi pale Embakasi East, Nairobi, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-049.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-49,Sheng,Nairobi\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Garissa Township, Garissa, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-059.",Procurement Fraud,realistic-procurement-fraud-59,Mixed,Garissa\n"Nilikagua document for ilihaririwa pale Nakuru East, Nakuru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-060.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-60,Mixed,Nakuru\n"On 11 August 2026 pale Imenti Central, Meru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-053.",Abuse of Office,realistic-abuse-of-office-53,Mixed,Meru\n"Pale Kisumu Central, Kisumu on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-059.",Other,realistic-other-59,Mixed,Kisumu\n"Tender ya huduma za ulinzi pale Kapseret, Eldoret ilikuwa imepangwa. taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri, around 1000, nimehifadhi screenshot na receipt. Ref KW-015.",Procurement Fraud,realistic-procurement-fraud-15,Sheng,Eldoret\n"On 22 April 2026 in Nakuru East, Nakuru, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I kept the receipt and the message thread. Case note KW-001.",Abuse of Office,realistic-abuse-of-office-01,English,Nakuru\n"Juzi 14 June 2026 nikaingia pale office ya Nakuru East, Nakuru, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 500, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Bribery,realistic-bribery-02,Sheng,Nakuru\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Garissa Township, Garissa. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Conflict of Interest,realistic-conflict-of-interest-44,Kiswahili,Garissa\n"Nili-check document ya ilitumia muhuri wa kughushi pale Nakuru East, Nakuru, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-40,Sheng,Nakuru\n"Nimekagua hati kuhusu ilihaririwa katika Machakos Central, Machakos, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-04,Kiswahili,Machakos\n"On 22 April 2026, I went to the public office in Machakos Central, Machakos, for an approved police abstract. The the police abstract clerk said the document would not be issued without an extra amount; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Bribery,realistic-bribery-56,English,Machakos\n"Nimekagua hati kuhusu ilonyesha anwani ya uongo katika Machakos Central, Machakos, tarehe 11 August 2026. leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-14,Kiswahili,Machakos\n"Kwa zabuni ya ukarabati wa barabara katika Kisumu Central, Kisumu, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Procurement Fraud,realistic-procurement-fraud-01,Kiswahili,Kisumu\n"Tarehe 28 July 2026, nilienda ofisi ya umma katika Bungoma Central, Bungoma, kwa uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Bribery,realistic-bribery-07,Kiswahili,Bungoma\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Nakuru East, Nakuru, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Bribery,realistic-bribery-42,Kiswahili,Nakuru\n"Pale Imenti Central, Meru on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Other,realistic-other-38,Sheng,Meru\n"On 9 March 2026, I dealt with the water department in Kapseret, Eldoret. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Other,realistic-other-03,English,Eldoret\n"Tender ya samani za ofisi pale Machakos Central, Machakos ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Procurement Fraud,realistic-procurement-fraud-52,Sheng,Machakos\n"On 14 June 2026 pale Machakos Central, Machakos, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-035.",Abuse of Office,realistic-abuse-of-office-35,Sheng,Machakos\n"Kuna money misuse kwa mapato ya ada za soko pale Kilifi North, Kilifi, after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 5,000, na screenshot na receipt bado. Case/rekodi KW-023.",Embezzlement,realistic-embezzlement-23,Mixed,Kilifi\n"Pale Imenti Central, Meru on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Other,realistic-other-28,Sheng,Meru\n"In Kilifi North, Kilifi, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Nepotism/Favoritism,realistic-nepotism-favoritism-42,English,Kilifi\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkaguzi akadai malipo ya faragha mara moja, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-052.",Extortion,realistic-extortion-52,Mixed,Bungoma\n"Tarehe 22 April 2026, nilishughulika na simu ya msaada ya umma katika Likoni, Mombasa. baada ya simu nyingi bila nambari ya kufuatilia. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Other,realistic-other-32,Kiswahili,Mombasa\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Kisumu Central, Kisumu. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Conflict of Interest,realistic-conflict-of-interest-36,English,Kisumu\n"On 17 February 2026 pale Kilifi North, Kilifi, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-045.",Extortion,realistic-extortion-45,Sheng,Kilifi\n"On 14 June 2026 pale Imenti Central, Meru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-003.",Abuse of Office,realistic-abuse-of-office-03,Sheng,Meru\n"I checked a land transfer record relating to the county registry in Garissa Township, Garissa, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-11,English,Garissa\n"I am reporting irregular use of KSh 20,000 in Bungoma Central, Bungoma, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 500 paid a supplier for drugs that are absent from the store. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Embezzlement,realistic-embezzlement-20,English,Bungoma\n"Kwa zabuni ya vifaa vya hospitali katika Likoni, Mombasa, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Procurement Fraud,realistic-procurement-fraud-34,Kiswahili,Mombasa\n"Tarehe 9 March 2026 katika Embakasi East, Nairobi, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Abuse of Office,realistic-abuse-of-office-60,Kiswahili,Nairobi\n"I checked an academic certificate relating to the appointment file in Bungoma Central, Bungoma, on 9 March 2026. The document contained a changed name and a stamp that the issuing school says is not genuine. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-05,English,Bungoma\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Garissa Township, Garissa, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Bribery,realistic-bribery-03,Kiswahili,Garissa\n"Nilienda pale office ya Bungoma Central, Bungoma, on 28 July 2026 for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida, 5,000, na screenshot na receipt bado. Case/rekodi KW-047.",Bribery,realistic-bribery-47,Mixed,Bungoma\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Likoni, Mombasa. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Conflict of Interest,realistic-conflict-of-interest-59,English,Mombasa\n"On 22 April 2026, the county finance officer took part in a decision about a lease decision in Nakuru East, Nakuru. The officer personally owned the land being leased to the public entity. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-013.",Conflict of Interest,realistic-conflict-of-interest-13,English,Nakuru\n"Kuna doo za fedha za ukarabati wa barabara zilipotea pale Imenti Central, Meru after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 300, nimehifadhi screenshot na receipt. Ref KW-037.",Embezzlement,realistic-embezzlement-37,Sheng,Meru\n"On 9 March 2026 pale Bungoma Central, Bungoma, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-042.",Extortion,realistic-extortion-42,Mixed,Bungoma\n"On 28 July 2026, I dealt with the county portal in Embakasi East, Nairobi. The office removed a public notice without a published reason after residents requested services. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-055.",Other,realistic-other-55,English,Nairobi\n"Pale Kilifi North, Kilifi on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Nepotism/Favoritism,realistic-nepotism-favoritism-42,Sheng,Kilifi\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Embakasi East, Nairobi. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Conflict of Interest,realistic-conflict-of-interest-52,Kiswahili,Nairobi\n"Pale Imenti Central, Meru on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-018.",Other,realistic-other-18,Mixed,Meru\n"In Nakuru East, Nakuru, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Nepotism/Favoritism,realistic-nepotism-favoritism-44,English,Nakuru\n"Nili-check document ya iliandika bidhaa pale Likoni, Mombasa, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-26,Sheng,Mombasa\n"Pale Kilifi North, Kilifi on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no action taken, nimeweka details ya time na office number. Case/rekodi KW-014.",Other,realistic-other-14,Mixed,Kilifi\n"Kwa tender ya ukodishaji wa magari pale Kilifi North, Kilifi, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-006.",Procurement Fraud,realistic-procurement-fraud-06,Mixed,Kilifi\n"Nilienda pale office ya Imenti Central, Meru, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 2,500, nimeweka details ya time na office number. Case/rekodi KW-034.",Bribery,realistic-bribery-34,Mixed,Meru\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-001.",Abuse of Office,realistic-abuse-of-office-01,Sheng,Nakuru\n"Tarehe 6 May 2026 katika Kisumu Central, Kisumu, afisa wa rekodi akatishia kuondoa rekodi ya ombi. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Abuse of Office,realistic-abuse-of-office-54,Kiswahili,Kisumu\n"Nimekagua hati kuhusu ilonyesha anwani ya uongo katika Kilifi North, Kilifi, tarehe 11 August 2026. leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-38,Kiswahili,Kilifi\n"Nili-check document ya kilikuwa na jina lililobadilishwa pale Bungoma Central, Bungoma, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, nimehifadhi screenshot na receipt. Ref KW-005.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-05,Sheng,Bungoma\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Machakos Central, Machakos, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Bribery,realistic-bribery-26,Kiswahili,Machakos\n"Tarehe 11 August 2026 katika Kilifi North, Kilifi, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Abuse of Office,realistic-abuse-of-office-29,Kiswahili,Kilifi\n"Nilienda pale office ya Machakos Central, Machakos, on 17 February 2026 for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni, 20,000, nimeweka details ya time na office number. Case/rekodi KW-006.",Bribery,realistic-bribery-06,Mixed,Machakos\n"Kwa mifumo ya kompyuta na mtandao katika Bungoma Central, Bungoma, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Tukio linahusu mifumo ya kompyuta na mtandao yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Procurement Fraud,realistic-procurement-fraud-53,Kiswahili,Bungoma\n"On 9 March 2026, the public appointment officer took part in a decision about a job assessment in Kisumu Central, Kisumu. The officer was the spouse of one shortlisted candidate. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Conflict of Interest,realistic-conflict-of-interest-56,English,Kisumu\n"On 14 June 2026 in Imenti Central, Meru, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Abuse of Office,realistic-abuse-of-office-03,English,Meru\n"Juzi 3 January 2026 nikaingia pale office ya Embakasi East, Nairobi, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 5000, nimehifadhi screenshot na receipt. Ref KW-041.",Bribery,realistic-bribery-41,Sheng,Nairobi\n"Tender ya usambazaji wa mafuta pale Kilifi North, Kilifi ilikuwa imepangwa. katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Procurement Fraud,realistic-procurement-fraud-16,Sheng,Kilifi\n"Kwa zabuni ya ukarabati wa barabara katika Embakasi East, Nairobi, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Procurement Fraud,realistic-procurement-fraud-17,Kiswahili,Nairobi\n"Tarehe 9 March 2026, afisa wa uteuzi wa umma alishiriki uamuzi kuhusu alikuwa mwenzi katika Kisumu Central, Kisumu. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Conflict of Interest,realistic-conflict-of-interest-56,Kiswahili,Kisumu\n"Tarehe 22 April 2026, afisa wa fedha wa kaunti alishiriki uamuzi kuhusu alikuwa mmiliki binafsi katika Nakuru East, Nakuru. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Conflict of Interest,realistic-conflict-of-interest-13,Kiswahili,Nakuru\n"In Nakuru East, Nakuru, on 11 August 2026, the procurement committee gave repeated contracts to the chairperson\'s close associate while qualified suppliers were not invited. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Nepotism/Favoritism,realistic-nepotism-favoritism-24,English,Nakuru\n"Pale Nakuru East, Nakuru on 14 June 2026, bodi ya CDF bila kura ya wazi; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Nepotism/Favoritism,realistic-nepotism-favoritism-14,Sheng,Nakuru\n"Kuna doo za chakula na blanketi zilipotea pale Garissa Township, Garissa after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Embezzlement,realistic-embezzlement-16,Sheng,Garissa\n"Pale Bungoma Central, Bungoma on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-041.",Other,realistic-other-41,Sheng,Bungoma\n"Pale Kisumu Central, Kisumu on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-027.",Nepotism/Favoritism,realistic-nepotism-favoritism-27,Sheng,Kisumu\n"Katika Kapseret, Eldoret, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Nepotism/Favoritism,realistic-nepotism-favoritism-11,Kiswahili,Eldoret\n"Tarehe 9 March 2026 katika Garissa Township, Garissa, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Abuse of Office,realistic-abuse-of-office-12,Kiswahili,Garissa\n"Nimekagua hati kuhusu kilitiwa sahihi katika Kapseret, Eldoret, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-57,Kiswahili,Eldoret\n"Kuna doo za fedha za chakula zilipotea pale Machakos Central, Machakos after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 5000, nimehifadhi screenshot na receipt. Ref KW-059.",Embezzlement,realistic-embezzlement-59,Sheng,Machakos\n"Juzi 3 January 2026 nikaingia pale office ya Kisumu Central, Kisumu, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 300, nimehifadhi screenshot na receipt. Ref KW-025.",Bribery,realistic-bribery-25,Sheng,Kisumu\n"On 14 June 2026 pale Likoni, Mombasa, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-027.",Abuse of Office,realistic-abuse-of-office-27,Mixed,Mombasa\n"Kuna money misuse kwa fedha za wanafunzi pale Kisumu Central, Kisumu, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 20,000, nimeweka details ya time na office number. Case/rekodi KW-018.",Embezzlement,realistic-embezzlement-18,Mixed,Kisumu\n"On 11 August 2026 in Imenti Central, Meru, the department director used an official vehicle for a private family trip and signed the fuel claim through the office. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Abuse of Office,realistic-abuse-of-office-53,English,Meru\n"Nimekagua hati kuhusu iliandika bidhaa katika Imenti Central, Meru, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-02,Kiswahili,Meru\n"Kuna doo za fedha za ukarabati wa barabara zilipotea pale Likoni, Mombasa after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 1000, nimehifadhi screenshot na receipt. Ref KW-021.",Embezzlement,realistic-embezzlement-21,Sheng,Mombasa\n"I checked a land transfer record relating to the county registry in Embakasi East, Nairobi, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-59,English,Nairobi\n"Pale Nakuru East, Nakuru on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no action taken, nimeweka details ya time na office number. Case/rekodi KW-046.",Other,realistic-other-46,Mixed,Nakuru\n"On 6 May 2026, I dealt with the roads office in Embakasi East, Nairobi. The office promised resurfacing for our street but no work began before the deadline. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Other,realistic-other-45,English,Nairobi\n"Pale Garissa Township, Garissa on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-047.",Other,realistic-other-47,Sheng,Garissa\n"I am reporting irregular use of KSh 20,000 in Likoni, Mombasa, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 1,000 were used to buy items not delivered to the school. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Embezzlement,realistic-embezzlement-51,English,Mombasa\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Bungoma Central, Bungoma, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-033.",Procurement Fraud,realistic-procurement-fraud-33,Mixed,Bungoma\n"On 11 August 2026, I dealt with the public counter in Machakos Central, Machakos. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Other,realistic-other-60,English,Machakos\n"On 9 March 2026 pale Garissa Township, Garissa, askari wa kaunti akasema zingerudishwa baada ya malipo, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Extortion,realistic-extortion-58,Sheng,Garissa\n"Pale Nakuru East, Nakuru on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-026.",Other,realistic-other-26,Mixed,Nakuru\n"I checked a payment voucher relating to the hospital account in Embakasi East, Nairobi, on 6 May 2026. The document carried a duplicate signature for a purchase that has no supplier evidence. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-39,English,Nairobi\n"Juzi 14 June 2026 nikaingia pale office ya Kilifi North, Kilifi, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Bribery,realistic-bribery-10,Sheng,Kilifi\n"On 17 February 2026, I went to the public office in Imenti Central, Meru, for release of a passport. The the immigration officer requested money to place the application ahead of the queue; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Bribery,realistic-bribery-14,English,Meru\n"Kwa usambazaji wa mafuta katika Kilifi North, Kilifi, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Tukio linahusu usambazaji wa mafuta yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Procurement Fraud,realistic-procurement-fraud-16,Kiswahili,Kilifi\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za ukarabati wa barabara katika Imenti Central, Meru, tangu 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Embezzlement,realistic-embezzlement-37,Kiswahili,Meru\n"Nimekagua hati kuhusu ilihaririwa katika Imenti Central, Meru, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-52,Kiswahili,Meru\n"Katika Bungoma Central, Bungoma, tarehe 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Nepotism/Favoritism,realistic-nepotism-favoritism-39,Kiswahili,Bungoma\n"On 6 May 2026 pale Garissa Township, Garissa, mkaguzi akadai malipo ya faragha mara moja, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Extortion,realistic-extortion-28,Sheng,Garissa\n"Pale Garissa Township, Garissa, on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-045.",Nepotism/Favoritism,realistic-nepotism-favoritism-45,Mixed,Garissa\n"Pale Embakasi East, Nairobi on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no action taken, na screenshot na receipt bado. Case/rekodi KW-045.",Other,realistic-other-45,Mixed,Nairobi\n"On 28 July 2026, I dealt with the county portal in Kisumu Central, Kisumu. The office removed a public notice without a published reason after residents requested services. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Other,realistic-other-39,English,Kisumu\n"Kuna doo za fedha za ukarabati wa barabara zilipotea pale Nakuru East, Nakuru after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 5000, nimehifadhi screenshot na receipt. Ref KW-005.",Embezzlement,realistic-embezzlement-05,Sheng,Nakuru\n"On 9 March 2026 in Embakasi East, Nairobi, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Abuse of Office,realistic-abuse-of-office-60,English,Nairobi\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Embakasi East, Nairobi. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Conflict of Interest,realistic-conflict-of-interest-12,Kiswahili,Nairobi\n"Pale Embakasi East, Nairobi, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-043.",Nepotism/Favoritism,realistic-nepotism-favoritism-43,Mixed,Nairobi\n"I am reporting irregular use of KSh 20,000 in Kilifi North, Kilifi, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 1,000 were used to buy items not delivered to the school. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Embezzlement,realistic-embezzlement-03,English,Kilifi\n"On 28 July 2026 in Embakasi East, Nairobi, the station officer refused to release my brother and demanded KSh 2,000, about KSh 20,000. They while warning that the case file could be damaged. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Extortion,realistic-extortion-06,English,Nairobi\n"Nilienda pale office ya Garissa Township, Garissa, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 1,000, na screenshot na receipt bado. Case/rekodi KW-033.",Bribery,realistic-bribery-33,Mixed,Garissa\n"I am reporting irregular use of KSh 2,500 in Imenti Central, Meru, after 11 August 2026. Records show that the county market cashier handled weekly market fee collections, and the funds worth KSh 300 were missing from the ledger and bank deposit. I kept the receipt and the message thread. Case note KW-007.",Embezzlement,realistic-embezzlement-07,English,Meru\n"Nilienda pale office ya Kapseret, Eldoret, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 300, na screenshot na receipt bado. Case/rekodi KW-019.",Bribery,realistic-bribery-19,Mixed,Eldoret\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Nakuru East, Nakuru. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Conflict of Interest,realistic-conflict-of-interest-03,Kiswahili,Nakuru\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Bungoma Central, Bungoma. bila taarifa wala haikutangaza wakati huduma ingerejea. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Other,realistic-other-41,Kiswahili,Bungoma\n"For tender VEH-441 in Nakuru East, Nakuru, the contract committee accepted a bid after the deadline while rejecting timely bidders. The file concerns vehicle leasing and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Procurement Fraud,realistic-procurement-fraud-38,English,Nakuru\n"Juzi 9 March 2026 nikaingia pale office ya Embakasi East, Nairobi, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 5000, nimehifadhi screenshot na receipt. Ref KW-011.",Bribery,realistic-bribery-11,Sheng,Nairobi\n"On 22 April 2026 pale Nakuru East, Nakuru, afisa wa fedha wa kaunti aliingia decision ya alikuwa mmiliki binafsi; personally owned, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-013.",Conflict of Interest,realistic-conflict-of-interest-13,Sheng,Nakuru\n"Kuna money misuse kwa pesa za dawa pale Kisumu Central, Kisumu, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 2,500, nimeweka details ya time na office number. Case/rekodi KW-028.",Embezzlement,realistic-embezzlement-28,Mixed,Kisumu\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Likoni, Mombasa. kwa fomu ya kawaida bila risiti rasmi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Other,realistic-other-52,Kiswahili,Mombasa\n"Tarehe 9 March 2026 katika Kisumu Central, Kisumu, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Abuse of Office,realistic-abuse-of-office-04,Kiswahili,Kisumu\n"Pale Kisumu Central, Kisumu on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-007.",Nepotism/Favoritism,realistic-nepotism-favoritism-07,Sheng,Kisumu\n"Pale Nakuru East, Nakuru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Other,realistic-other-56,Sheng,Nakuru\n"For tender SCH-204 in Kisumu Central, Kisumu, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Procurement Fraud,realistic-procurement-fraud-51,English,Kisumu\n"On 3 January 2026 pale Kapseret, Eldoret, mjumbe wa bodi ya shule joined the decision on aliwahi kuajiri; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-030.",Conflict of Interest,realistic-conflict-of-interest-30,Mixed,Eldoret\n"On 22 April 2026 pale Imenti Central, Meru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-039.",Extortion,realistic-extortion-39,Sheng,Meru\n"Pale Imenti Central, Meru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no action taken, nimeweka details ya time na office number. Case/rekodi KW-048.",Other,realistic-other-48,Mixed,Meru\n"Tarehe 22 April 2026, nilishughulika na simu ya msaada ya umma katika Kilifi North, Kilifi. baada ya simu nyingi bila nambari ya kufuatilia. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Other,realistic-other-24,Kiswahili,Kilifi\n"In Kapseret, Eldoret, on 9 March 2026, the public internship programme selected a local official\'s child for all available placements even though the position was not advertised. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-031.",Nepotism/Favoritism,realistic-nepotism-favoritism-31,English,Eldoret\n"Pale Kisumu Central, Kisumu on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-049.",Other,realistic-other-49,Mixed,Kisumu\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Kapseret, Eldoret, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Bribery,realistic-bribery-19,Kiswahili,Eldoret\n"Tarehe 28 July 2026, nilishughulika na tovuti ya kaunti katika Embakasi East, Nairobi. bila sababu iliyochapishwa baada ya wakazi kuomba huduma. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Other,realistic-other-55,Kiswahili,Nairobi\n"Tender ya samani za ofisi pale Likoni, Mombasa ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Procurement Fraud,realistic-procurement-fraud-44,Sheng,Mombasa\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa ya mradi wa vijana katika Garissa Township, Garissa, tangu 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Embezzlement,realistic-embezzlement-06,Kiswahili,Garissa\n"I checked a business permit relating to the revenue office in Likoni, Mombasa, on 11 August 2026. The document showed a false address and a permit number not in the official register. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-06,English,Mombasa\n"Tarehe 14 June 2026 katika Nakuru East, Nakuru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Abuse of Office,realistic-abuse-of-office-51,Kiswahili,Nakuru\n"Juzi 11 August 2026 nikaingia pale office ya Nakuru East, Nakuru, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Bribery,realistic-bribery-52,Sheng,Nakuru\n"Katika Imenti Central, Meru, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Nepotism/Favoritism,realistic-nepotism-favoritism-36,Kiswahili,Meru\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Machakos Central, Machakos. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Conflict of Interest,realistic-conflict-of-interest-57,Kiswahili,Machakos\n"I am reporting irregular use of KSh 20,000 in Kisumu Central, Kisumu, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 2,500 paid a supplier for drugs that are absent from the store. I kept the receipt and the message thread. Case note KW-028.",Embezzlement,realistic-embezzlement-28,English,Kisumu\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Bungoma Central, Bungoma, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Embezzlement,realistic-embezzlement-10,Kiswahili,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Imenti Central, Meru, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Embezzlement,realistic-embezzlement-57,Kiswahili,Meru\n"In Kilifi North, Kilifi, on 14 June 2026, the CDF board funded the chairman\'s project ahead of equally eligible projects without an open vote. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-022.",Nepotism/Favoritism,realistic-nepotism-favoritism-22,English,Kilifi\n"I am reporting irregular use of KSh 20,000 in Garissa Township, Garissa, after 9 March 2026. Records show that the youth fund administrator handled approved youth project money, and the funds worth KSh 20,000 was transferred to a personal account. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Embezzlement,realistic-embezzlement-06,English,Garissa\n"In Machakos Central, Machakos, on 11 August 2026, the procurement committee gave repeated contracts to the chairperson\'s close associate while qualified suppliers were not invited. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Nepotism/Favoritism,realistic-nepotism-favoritism-48,English,Machakos\n"On 11 August 2026, I dealt with the public counter in Nakuru East, Nakuru. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Other,realistic-other-36,English,Nakuru\n"Tender ya samani za ofisi pale Kilifi North, Kilifi ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Procurement Fraud,realistic-procurement-fraud-36,Sheng,Kilifi\n"On 3 January 2026 pale Kisumu Central, Kisumu, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Abuse of Office,realistic-abuse-of-office-34,Sheng,Kisumu\n"Tender ya ukodishaji wa magari pale Likoni, Mombasa ilikuwa imepangwa. kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Procurement Fraud,realistic-procurement-fraud-54,Sheng,Mombasa\n"Pale Likoni, Mombasa on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Other,realistic-other-32,Sheng,Mombasa\n"Kuna money misuse kwa mapato ya ada za soko pale Nakuru East, Nakuru, after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 1,000, na screenshot na receipt bado. Case/rekodi KW-015.",Embezzlement,realistic-embezzlement-15,Mixed,Nakuru\n"Pale Kapseret, Eldoret on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-051.",Nepotism/Favoritism,realistic-nepotism-favoritism-51,Sheng,Eldoret\n"Tarehe 22 April 2026, nilienda ofisi ya umma katika Imenti Central, Meru, kwa abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Bribery,realistic-bribery-24,Kiswahili,Meru\n"Pale Embakasi East, Nairobi on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-033.",Nepotism/Favoritism,realistic-nepotism-favoritism-33,Sheng,Nairobi\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Kapseret, Eldoret, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-025.",Procurement Fraud,realistic-procurement-fraud-25,Mixed,Eldoret\n"On 6 May 2026 pale Embakasi East, Nairobi, mkaguzi akadai malipo ya faragha mara moja, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Extortion,realistic-extortion-36,Sheng,Nairobi\n"On 17 February 2026 pale Machakos Central, Machakos, mkuu wa shule wakati wa shule; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-055.",Abuse of Office,realistic-abuse-of-office-55,Sheng,Machakos\n"Pale Garissa Township, Garissa on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-037.",Other,realistic-other-37,Sheng,Garissa\n"On 6 May 2026 pale Kapseret, Eldoret, afisa wa rekodi akatishia kuondoa rekodi ya ombi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-038.",Abuse of Office,realistic-abuse-of-office-38,Mixed,Eldoret\n"Kuna doo za pesa ya mradi wa vijana zilipotea pale Garissa Township, Garissa after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Embezzlement,realistic-embezzlement-46,Sheng,Garissa\n"Nilienda pale office ya Kilifi North, Kilifi, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 500, nimeweka details ya time na office number. Case/rekodi KW-020.",Bribery,realistic-bribery-20,Mixed,Kilifi\n"I am reporting irregular use of KSh 5,000 in Bungoma Central, Bungoma, after 6 May 2026. Records show that the relief distribution committee handled food and blankets, and the funds worth KSh 2,500 were recorded as delivered but never reached beneficiaries. I kept the receipt and the message thread. Case note KW-040.",Embezzlement,realistic-embezzlement-40,English,Bungoma\n"Pale Garissa Township, Garissa on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-027.",Other,realistic-other-27,Mixed,Garissa\n"Pale Imenti Central, Meru on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Nepotism/Favoritism,realistic-nepotism-favoritism-16,Sheng,Meru\n"Tarehe 28 July 2026, nilishughulika na tovuti ya kaunti katika Garissa Township, Garissa. bila sababu iliyochapishwa baada ya wakazi kuomba huduma. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Other,realistic-other-47,Kiswahili,Garissa\n"In Kilifi North, Kilifi, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Nepotism/Favoritism,realistic-nepotism-favoritism-12,English,Kilifi\n"In Kisumu Central, Kisumu, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Nepotism/Favoritism,realistic-nepotism-favoritism-57,English,Kisumu\n"Pale Nakuru East, Nakuru, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-044.",Nepotism/Favoritism,realistic-nepotism-favoritism-44,Mixed,Nakuru\n"On 11 August 2026 in Imenti Central, Meru, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 300. They threatened to make a false inspection fault. I kept the receipt and the message thread. Case note KW-019.",Extortion,realistic-extortion-19,English,Meru\n"On 17 February 2026 pale Nakuru East, Nakuru, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-043.",Conflict of Interest,realistic-conflict-of-interest-43,Sheng,Nakuru\n"I checked a land transfer record relating to the county registry in Kisumu Central, Kisumu, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-03,English,Kisumu\n"Juzi 6 May 2026 nikaingia pale office ya Embakasi East, Nairobi, for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha 1000, nimehifadhi screenshot na receipt. Ref KW-021.",Bribery,realistic-bribery-21,Sheng,Nairobi\n"Pale Bungoma Central, Bungoma on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-031.",Other,realistic-other-31,Sheng,Bungoma\n"Tender ya zabuni ya ujenzi wa madarasa pale Kisumu Central, Kisumu ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 5000, nimehifadhi screenshot na receipt. Ref KW-011.",Procurement Fraud,realistic-procurement-fraud-11,Sheng,Kisumu\n"On 17 February 2026 pale Kilifi North, Kilifi, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-005.",Extortion,realistic-extortion-05,Sheng,Kilifi\n"Kuna doo za fedha za kisima zilipotea pale Likoni, Mombasa after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 300, nimehifadhi screenshot na receipt. Ref KW-001.",Embezzlement,realistic-embezzlement-01,Sheng,Mombasa\n"Tarehe 17 February 2026 katika Kilifi North, Kilifi, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Extortion,realistic-extortion-45,Kiswahili,Kilifi\n"On 17 February 2026 pale Kilifi North, Kilifi, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-051.",Conflict of Interest,realistic-conflict-of-interest-51,Mixed,Kilifi\n"Tarehe 11 August 2026 katika Nakuru East, Nakuru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Extortion,realistic-extortion-27,Kiswahili,Nakuru\n"On 14 June 2026 pale Nakuru East, Nakuru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-011.",Abuse of Office,realistic-abuse-of-office-11,Mixed,Nakuru\n"For tender KW-026 in Bungoma Central, Bungoma, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Procurement Fraud,realistic-procurement-fraud-33,English,Bungoma\n"Tarehe 14 June 2026 katika Machakos Central, Machakos, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Abuse of Office,realistic-abuse-of-office-35,Kiswahili,Machakos\n"Pale Garissa Township, Garissa on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-017.",Other,realistic-other-17,Mixed,Garissa\n"On 28 July 2026 pale Kisumu Central, Kisumu, afisa wa kaunti wengine wakaambiwa wangojee; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Abuse of Office,realistic-abuse-of-office-24,Sheng,Kisumu\n"Pale Nakuru East, Nakuru on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Other,realistic-other-06,Sheng,Nakuru\n"On 22 April 2026 in Kilifi North, Kilifi, the gate security supervisor blocked my lorry and demanded KSh 1,500, about KSh 1,000. They threatening to call a taxing inspection. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Extortion,realistic-extortion-15,English,Kilifi\n"On 9 March 2026 in Kapseret, Eldoret, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I kept the receipt and the message thread. Case note KW-028.",Abuse of Office,realistic-abuse-of-office-28,English,Eldoret\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Likoni, Mombasa. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-049.",Conflict of Interest,realistic-conflict-of-interest-49,English,Mombasa\n"Nilienda pale office ya Embakasi East, Nairobi, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 300, na screenshot na receipt bado. Case/rekodi KW-001.",Bribery,realistic-bribery-01,Mixed,Nairobi\n"Tarehe 14 June 2026, mwenyekiti wa jopo la ruzuku alishiriki uamuzi kuhusu alikuwa mshirika katika Kilifi North, Kilifi. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Conflict of Interest,realistic-conflict-of-interest-31,Kiswahili,Kilifi\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Kapseret, Eldoret, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Embezzlement,realistic-embezzlement-52,Kiswahili,Eldoret\n"On 22 April 2026 pale Likoni, Mombasa, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-023.",Extortion,realistic-extortion-23,Sheng,Mombasa\n"On 11 August 2026 pale Kilifi North, Kilifi, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-029.",Abuse of Office,realistic-abuse-of-office-29,Sheng,Kilifi\n"Katika Nakuru East, Nakuru, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Nepotism/Favoritism,realistic-nepotism-favoritism-44,Kiswahili,Nakuru\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Bungoma Central, Bungoma. kutofanya kazi kwa miezi kadhaa bila maelezo. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Other,realistic-other-51,Kiswahili,Bungoma\n"On 22 April 2026 pale Imenti Central, Meru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-033.",Abuse of Office,realistic-abuse-of-office-33,Mixed,Meru\n"On 6 May 2026 in Bungoma Central, Bungoma, the records officer refused access to my own file despite an approved request and threatened to remove the request record. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Abuse of Office,realistic-abuse-of-office-06,English,Bungoma\n"For tender KW-026 in Kisumu Central, Kisumu, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Procurement Fraud,realistic-procurement-fraud-41,English,Kisumu\n"Tarehe 22 April 2026, afisa wa fedha wa kaunti alishiriki uamuzi kuhusu alikuwa mmiliki binafsi katika Machakos Central, Machakos. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Conflict of Interest,realistic-conflict-of-interest-37,Kiswahili,Machakos\n"In Kilifi North, Kilifi, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-052.",Nepotism/Favoritism,realistic-nepotism-favoritism-52,English,Kilifi\n"Kuna doo za fedha za kisima zilipotea pale Machakos Central, Machakos after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 1000, nimehifadhi screenshot na receipt. Ref KW-009.",Embezzlement,realistic-embezzlement-09,Sheng,Machakos\n"Juzi 6 May 2026 nikaingia pale office ya Garissa Township, Garissa, for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha 5000, nimehifadhi screenshot na receipt. Ref KW-053.",Bribery,realistic-bribery-53,Sheng,Garissa\n"Pale Likoni, Mombasa on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Other,realistic-other-52,Sheng,Mombasa\n"On 22 April 2026 pale Machakos Central, Machakos, afisa wa fedha wa kaunti joined the decision on alikuwa mmiliki binafsi; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-037.",Conflict of Interest,realistic-conflict-of-interest-37,Mixed,Machakos\n"On 17 February 2026, I dealt with the hospital desk in Likoni, Mombasa. The office refused to explain why the queue was suspended and gave no complaint channel. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-022.",Other,realistic-other-22,English,Mombasa\n"I am reporting irregular use of KSh 20,000 in Nakuru East, Nakuru, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 5,000 were used to buy items not delivered to the school. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Embezzlement,realistic-embezzlement-35,English,Nakuru\n"For contract SUP-77 in Imenti Central, Meru, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Procurement Fraud,realistic-procurement-fraud-20,English,Meru\n"Tarehe 9 March 2026 katika Bungoma Central, Bungoma, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Abuse of Office,realistic-abuse-of-office-36,Kiswahili,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa ya mradi wa vijana katika Kapseret, Eldoret, tangu 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Embezzlement,realistic-embezzlement-22,Kiswahili,Eldoret\n"Kuna money misuse kwa fedha za kisima pale Likoni, Mombasa, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 5,000, na screenshot na receipt bado. Case/rekodi KW-041.",Embezzlement,realistic-embezzlement-41,Mixed,Mombasa\n"Pale Kisumu Central, Kisumu on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no action taken, na screenshot na receipt bado. Case/rekodi KW-039.",Other,realistic-other-39,Mixed,Kisumu\n"Kuna doo za fedha za chakula zilipotea pale Kilifi North, Kilifi after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 300, nimehifadhi screenshot na receipt. Ref KW-043.",Embezzlement,realistic-embezzlement-43,Sheng,Kilifi\n"Juzi 11 August 2026 nikaingia pale office ya Likoni, Mombasa, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Bribery,realistic-bribery-28,Sheng,Mombasa\n"Tarehe 3 January 2026 katika Kapseret, Eldoret, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Extortion,realistic-extortion-24,Kiswahili,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya mapato ya ada za soko katika Nakuru East, Nakuru, tangu 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Embezzlement,realistic-embezzlement-55,Kiswahili,Nakuru\n"On 14 June 2026, I went to the public office in Nakuru East, Nakuru, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Bribery,realistic-bribery-02,English,Nakuru\n"Kuna doo za fedha za wanafunzi zilipotea pale Embakasi East, Nairobi after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Embezzlement,realistic-embezzlement-34,Sheng,Nairobi\n"Nili-check document ya ilonyesha anwani ya uongo pale Likoni, Mombasa, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-46,Sheng,Mombasa\n"I checked a police clearance letter relating to the public application in Likoni, Mombasa, on 17 February 2026. The document used a fabricated stamp to make an unapproved clearance appear valid. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-56,English,Mombasa\n"On 11 August 2026 in Machakos Central, Machakos, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 5,000. They threatened to make a false inspection fault. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Extortion,realistic-extortion-11,English,Machakos\n"On 3 January 2026, I went to the public office in Bungoma Central, Bungoma, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Bribery,realistic-bribery-57,English,Bungoma\n"On 14 June 2026 in Likoni, Mombasa, the police officer held my identification card and demanded KSh 2,500, about KSh 1,000. They threatening arrest unless I paid in cash. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Extortion,realistic-extortion-33,English,Mombasa\n"Nimekagua hati kuhusu kilikuwa na jina lililobadilishwa katika Kapseret, Eldoret, tarehe 9 March 2026. cheti cha elimu na muhuri ambao shule inasema si halisi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-37,Kiswahili,Eldoret\n"On 11 August 2026 pale Machakos Central, Machakos, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-011.",Extortion,realistic-extortion-11,Sheng,Machakos\n"Tarehe 17 February 2026, nilienda ofisi ya umma katika Imenti Central, Meru, kwa utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Bribery,realistic-bribery-54,Kiswahili,Meru\n"On 9 March 2026 in Garissa Township, Garissa, the county askari took the goods and demanded KSh 1,000, about KSh 2,500. They said they would be returned only after a payment. I kept the receipt and the message thread. Case note KW-058.",Extortion,realistic-extortion-58,English,Garissa\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkaguzi akadai malipo ya faragha mara moja, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Extortion,realistic-extortion-52,Sheng,Bungoma\n"Nimekagua hati kuhusu kilitiwa sahihi katika Kapseret, Eldoret, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-17,Kiswahili,Eldoret\n"Kwa tender ya ukodishaji wa magari pale Nakuru East, Nakuru, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-038.",Procurement Fraud,realistic-procurement-fraud-38,Mixed,Nakuru\n"Nilikagua document for ilihaririwa pale Machakos Central, Machakos, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-044.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-44,Mixed,Machakos\n"Tarehe 17 February 2026 katika Machakos Central, Machakos, mkuu wa shule wakati wa shule. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Abuse of Office,realistic-abuse-of-office-15,Kiswahili,Machakos\n"On 17 February 2026 in Likoni, Mombasa, the school principal directed a teacher to use school equipment for a private function during school time. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Abuse of Office,realistic-abuse-of-office-47,English,Mombasa\n"On 9 March 2026 pale Garissa Township, Garissa, askari wa kaunti akasema zingerudishwa baada ya malipo, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Extortion,realistic-extortion-18,Sheng,Garissa\n"Nili-check document ya ilitumia muhuri wa kughushi pale Kilifi North, Kilifi, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-08,Sheng,Kilifi\n"In Machakos Central, Machakos, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Nepotism/Favoritism,realistic-nepotism-favoritism-18,English,Machakos\n"On 11 August 2026 pale Kilifi North, Kilifi, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-029.",Abuse of Office,realistic-abuse-of-office-29,Mixed,Kilifi\n"On 9 March 2026 in Embakasi East, Nairobi, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Abuse of Office,realistic-abuse-of-office-20,English,Nairobi\n"On 3 January 2026 in Kisumu Central, Kisumu, the licensing inspector demanded cash and demanded KSh 800, about KSh 2,500. They and said he would report a compliance breach if I refused. I kept the receipt and the message thread. Case note KW-040.",Extortion,realistic-extortion-40,English,Kisumu\n"In Nakuru East, Nakuru, on 14 June 2026, the CDF board funded the chairman\'s project ahead of equally eligible projects without an open vote. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Nepotism/Favoritism,realistic-nepotism-favoritism-54,English,Nakuru\n"On 6 May 2026 pale Kisumu Central, Kisumu, afisa wa rekodi akatishia kuondoa rekodi ya ombi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-014.",Abuse of Office,realistic-abuse-of-office-14,Mixed,Kisumu\n"On 28 July 2026 pale Bungoma Central, Bungoma, afisa wa kaunti wengine wakaambiwa wangojee; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Abuse of Office,realistic-abuse-of-office-16,Sheng,Bungoma\n"On 17 February 2026 pale Imenti Central, Meru, mkuu wa shule wakati wa shule; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-023.",Abuse of Office,realistic-abuse-of-office-23,Mixed,Meru\n"In Bungoma Central, Bungoma, on 9 March 2026, the public internship programme selected a local official\'s child for all available placements even though the position was not advertised. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Nepotism/Favoritism,realistic-nepotism-favoritism-39,English,Bungoma\n"Kwa tender ya samani za ofisi pale Imenti Central, Meru, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-060.",Procurement Fraud,realistic-procurement-fraud-60,Mixed,Meru\n"Pale Nakuru East, Nakuru on 14 June 2026, bodi ya CDF bila kura ya wazi; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Nepotism/Favoritism,realistic-nepotism-favoritism-54,Sheng,Nakuru\n"Tarehe 14 June 2026 katika Imenti Central, Meru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Extortion,realistic-extortion-49,Kiswahili,Meru\n"Nili-check document ya kilitiwa sahihi pale Garissa Township, Garissa, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-001.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-01,Sheng,Garissa\n"Katika Likoni, Mombasa, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Nepotism/Favoritism,realistic-nepotism-favoritism-60,Kiswahili,Mombasa\n"Katika Imenti Central, Meru, tarehe 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Nepotism/Favoritism,realistic-nepotism-favoritism-56,Kiswahili,Meru\n"Katika Machakos Central, Machakos, tarehe 14 June 2026, bodi ya CDF bila kura ya wazi. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Nepotism/Favoritism,realistic-nepotism-favoritism-38,Kiswahili,Machakos\n"Nili-check document ya kilitiwa sahihi pale Bungoma Central, Bungoma, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-025.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-25,Sheng,Bungoma\n"On 3 January 2026 pale Embakasi East, Nairobi, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-010.",Abuse of Office,realistic-abuse-of-office-10,Mixed,Nairobi\n"Kuna money misuse kwa fedha za chakula pale Kilifi North, Kilifi, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 1,000, na screenshot na receipt bado. Case/rekodi KW-003.",Embezzlement,realistic-embezzlement-03,Mixed,Kilifi\n"I am reporting irregular use of KSh 20,000 in Imenti Central, Meru, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 1,000 was withdrawn but the borehole was never completed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Embezzlement,realistic-embezzlement-57,English,Meru\n"On 3 January 2026 pale Bungoma Central, Bungoma, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Extortion,realistic-extortion-32,Sheng,Bungoma\n"Nimekagua hati kuhusu ilonyesha anwani ya uongo katika Likoni, Mombasa, tarehe 11 August 2026. leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-46,Kiswahili,Mombasa\n"On 9 March 2026 pale Garissa Township, Garissa, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-052.",Abuse of Office,realistic-abuse-of-office-52,Mixed,Garissa\n"Tarehe 28 July 2026, nilishughulika na tovuti ya kaunti katika Kapseret, Eldoret. bila sababu iliyochapishwa baada ya wakazi kuomba huduma. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Other,realistic-other-23,Kiswahili,Eldoret\n"Nilienda pale office ya Imenti Central, Meru, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 500, nimeweka details ya time na office number. Case/rekodi KW-044.",Bribery,realistic-bribery-44,Mixed,Meru\n"On 3 January 2026 pale Kisumu Central, Kisumu, mjumbe wa bodi ya shule joined the decision on aliwahi kuajiri; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-006.",Conflict of Interest,realistic-conflict-of-interest-06,Mixed,Kisumu\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Imenti Central, Meru. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Conflict of Interest,realistic-conflict-of-interest-35,Kiswahili,Meru\n"Kwa tender ya huduma za ulinzi pale Kisumu Central, Kisumu, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-031.",Procurement Fraud,realistic-procurement-fraud-31,Mixed,Kisumu\n"On 3 January 2026 in Garissa Township, Garissa, the licensing inspector demanded cash and demanded KSh 800, about KSh 500. They and said he would report a compliance breach if I refused. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Extortion,realistic-extortion-08,English,Garissa\n"On 3 January 2026 in Embakasi East, Nairobi, the licensing inspector demanded cash and demanded KSh 800, about KSh 2,500. They and said he would report a compliance breach if I refused. I kept the receipt and the message thread. Case note KW-016.",Extortion,realistic-extortion-16,English,Nairobi\n"On 3 January 2026 in Embakasi East, Nairobi, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Abuse of Office,realistic-abuse-of-office-50,English,Nairobi\n"Tender ya zabuni ya vifaa vya hospitali pale Machakos Central, Machakos ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Procurement Fraud,realistic-procurement-fraud-42,Sheng,Machakos\n"On 9 March 2026 pale Garissa Township, Garissa, afisa wa uteuzi wa umma joined the decision on alikuwa mwenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-024.",Conflict of Interest,realistic-conflict-of-interest-24,Mixed,Garissa\n"Kwa zabuni ya ujenzi wa madarasa katika Kisumu Central, Kisumu, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Procurement Fraud,realistic-procurement-fraud-11,Kiswahili,Kisumu\n"Kwa tender ya mifumo ya kompyuta na mtandao pale Embakasi East, Nairobi, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-037.",Procurement Fraud,realistic-procurement-fraud-37,Mixed,Nairobi\n"On 3 January 2026 pale Embakasi East, Nairobi, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-056.",Extortion,realistic-extortion-56,Mixed,Nairobi\n"On 28 July 2026 pale Garissa Township, Garissa, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-038.",Extortion,realistic-extortion-38,Mixed,Garissa\n"Tender ya mifumo ya kompyuta na mtandao pale Embakasi East, Nairobi ilikuwa imepangwa. timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa, around 300, nimehifadhi screenshot na receipt. Ref KW-037.",Procurement Fraud,realistic-procurement-fraud-37,Sheng,Nairobi\n"In Kisumu Central, Kisumu, on 9 March 2026, the public internship programme selected a local official\'s child for all available placements even though the position was not advertised. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Nepotism/Favoritism,realistic-nepotism-favoritism-47,English,Kisumu\n"On 11 August 2026 in Imenti Central, Meru, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 5,000. They threatened to make a false inspection fault. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Extortion,realistic-extortion-59,English,Meru\n"On 28 July 2026 pale Embakasi East, Nairobi, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-046.",Extortion,realistic-extortion-46,Mixed,Nairobi\n"Pale Kisumu Central, Kisumu on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-059.",Other,realistic-other-59,Sheng,Kisumu\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Kilifi North, Kilifi, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-026.",Procurement Fraud,realistic-procurement-fraud-26,Mixed,Kilifi\n"Pale Machakos Central, Machakos on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no action taken, nimeweka details ya time na office number. Case/rekodi KW-030.",Other,realistic-other-30,Mixed,Machakos\n"Pale Embakasi East, Nairobi, on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-023.",Nepotism/Favoritism,realistic-nepotism-favoritism-23,Mixed,Nairobi\n"On 22 April 2026 in Kilifi North, Kilifi, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I kept the receipt and the message thread. Case note KW-049.",Abuse of Office,realistic-abuse-of-office-49,English,Kilifi\n"Pale Bungoma Central, Bungoma on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-011.",Other,realistic-other-11,Sheng,Bungoma\n"Tender ya zabuni ya ujenzi wa madarasa pale Bungoma Central, Bungoma ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 1000, nimehifadhi screenshot na receipt. Ref KW-003.",Procurement Fraud,realistic-procurement-fraud-03,Sheng,Bungoma\n"Juzi 6 May 2026 nikaingia pale office ya Kapseret, Eldoret, for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha 5000, nimehifadhi screenshot na receipt. Ref KW-029.",Bribery,realistic-bribery-29,Sheng,Eldoret\n"On 17 February 2026 in Imenti Central, Meru, the public officer threatened to blacklist my supplier account and demanded KSh 5,000, about KSh 5,000. They unless I paid from my personal funds. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Extortion,realistic-extortion-29,English,Meru\n"Tarehe 17 February 2026 katika Likoni, Mombasa, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Extortion,realistic-extortion-13,Kiswahili,Mombasa\n"On 9 March 2026, the public appointment officer took part in a decision about a job assessment in Bungoma Central, Bungoma. The officer was the spouse of one shortlisted candidate. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Conflict of Interest,realistic-conflict-of-interest-48,English,Bungoma\n"Pale Bungoma Central, Bungoma on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-021.",Other,realistic-other-21,Sheng,Bungoma\n"Juzi 22 April 2026 nikaingia pale office ya Nakuru East, Nakuru, for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada 500, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Bribery,realistic-bribery-32,Sheng,Nakuru\n"Nimekagua hati kuhusu ilitumia muhuri wa kughushi katika Likoni, Mombasa, tarehe 17 February 2026. barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-56,Kiswahili,Mombasa\n"Nilikagua document for iliandika bidhaa pale Nakuru East, Nakuru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-010.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-10,Mixed,Nakuru\n"In Machakos Central, Machakos, on 14 June 2026, the CDF board funded the chairman\'s project ahead of equally eligible projects without an open vote. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Nepotism/Favoritism,realistic-nepotism-favoritism-38,English,Machakos\n"On 17 February 2026 pale Kilifi North, Kilifi, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-005.",Extortion,realistic-extortion-05,Mixed,Kilifi\n"Ninawasilisha tuhuma za matumizi mabaya ya chakula na blanketi katika Garissa Township, Garissa, tangu 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Embezzlement,realistic-embezzlement-56,Kiswahili,Garissa\n"Tender ya zabuni ya ujenzi wa madarasa pale Garissa Township, Garissa ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 300, nimehifadhi screenshot na receipt. Ref KW-019.",Procurement Fraud,realistic-procurement-fraud-19,Sheng,Garissa\n"On 17 February 2026 pale Imenti Central, Meru, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-029.",Extortion,realistic-extortion-29,Sheng,Meru\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Machakos Central, Machakos. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Conflict of Interest,realistic-conflict-of-interest-57,English,Machakos\n"Katika Garissa Township, Garissa, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Nepotism/Favoritism,realistic-nepotism-favoritism-25,Kiswahili,Garissa\n"On 6 May 2026 in Bungoma Central, Bungoma, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 2,500. They and demanded an immediate private payment. I kept the receipt and the message thread. Case note KW-052.",Extortion,realistic-extortion-52,English,Bungoma\n"For tender SCH-204 in Embakasi East, Nairobi, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Procurement Fraud,realistic-procurement-fraud-27,English,Nairobi\n"On 3 January 2026 pale Kapseret, Eldoret, mjumbe wa bodi ya shule aliingia decision ya aliwahi kuajiri; employed, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Conflict of Interest,realistic-conflict-of-interest-30,Sheng,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya mapato ya ada za soko katika Kilifi North, Kilifi, tangu 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Embezzlement,realistic-embezzlement-23,Kiswahili,Kilifi\n"On 6 May 2026, the project director took part in a decision about a construction contract in Bungoma Central, Bungoma. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-058.",Conflict of Interest,realistic-conflict-of-interest-58,English,Bungoma\n"Tarehe 9 March 2026, afisa wa uteuzi wa umma alishiriki uamuzi kuhusu alikuwa mwenzi katika Garissa Township, Garissa. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Conflict of Interest,realistic-conflict-of-interest-24,Kiswahili,Garissa\n"Tender ya zabuni ya vifaa vya hospitali pale Kilifi North, Kilifi ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Procurement Fraud,realistic-procurement-fraud-26,Sheng,Kilifi\n"On 28 July 2026, I went to the public office in Embakasi East, Nairobi, for printing an identification document. The the Huduma Centre clerk asked for a personal payment to speed up the ordinary service; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-031.",Bribery,realistic-bribery-31,English,Nairobi\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Kisumu Central, Kisumu. bila taarifa wala haikutangaza wakati huduma ingerejea. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Other,realistic-other-09,Kiswahili,Kisumu\n"On 17 February 2026, I went to the public office in Imenti Central, Meru, for release of a passport. The the immigration officer requested money to place the application ahead of the queue; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Bribery,realistic-bribery-54,English,Meru\n"Pale Likoni, Mombasa on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Other,realistic-other-02,Sheng,Mombasa\n"Tarehe 14 June 2026 katika Nakuru East, Nakuru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Extortion,realistic-extortion-57,Kiswahili,Nakuru\n"Pale Likoni, Mombasa on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-042.",Other,realistic-other-42,Mixed,Mombasa\n"In Nakuru East, Nakuru, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-034.",Nepotism/Favoritism,realistic-nepotism-favoritism-34,English,Nakuru\n"Tarehe 6 May 2026 katika Bungoma Central, Bungoma, afisa wa rekodi akatishia kuondoa rekodi ya ombi. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Abuse of Office,realistic-abuse-of-office-46,Kiswahili,Bungoma\n"In Kapseret, Eldoret, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-001.",Nepotism/Favoritism,realistic-nepotism-favoritism-01,English,Eldoret\n"Katika Nakuru East, Nakuru, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Nepotism/Favoritism,realistic-nepotism-favoritism-04,Kiswahili,Nakuru\n"Pale Machakos Central, Machakos on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-050.",Other,realistic-other-50,Mixed,Machakos\n"Tender ya zabuni ya vifaa vya hospitali pale Imenti Central, Meru ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Procurement Fraud,realistic-procurement-fraud-10,Sheng,Meru\n"On 22 April 2026 in Machakos Central, Machakos, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I kept the receipt and the message thread. Case note KW-025.",Abuse of Office,realistic-abuse-of-office-25,English,Machakos\n"Juzi 17 February 2026 nikaingia pale office ya Nakuru East, Nakuru, for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Bribery,realistic-bribery-22,Sheng,Nakuru\n"Tarehe 22 April 2026, nilienda ofisi ya umma katika Kilifi North, Kilifi, kwa abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Bribery,realistic-bribery-40,Kiswahili,Kilifi\n"I am reporting irregular use of KSh 5,000 in Kapseret, Eldoret, after 6 May 2026. Records show that the relief distribution committee handled food and blankets, and the funds worth KSh 500 were recorded as delivered but never reached beneficiaries. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Embezzlement,realistic-embezzlement-32,English,Eldoret\n"Kwa tender ya mifumo ya kompyuta na mtandao pale Kapseret, Eldoret, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-045.",Procurement Fraud,realistic-procurement-fraud-45,Mixed,Eldoret\n"Tarehe 3 January 2026, mjumbe wa bodi ya shule alishiriki uamuzi kuhusu aliwahi kuajiri katika Kisumu Central, Kisumu. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Conflict of Interest,realistic-conflict-of-interest-06,Kiswahili,Kisumu\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Nakuru East, Nakuru. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-043.",Conflict of Interest,realistic-conflict-of-interest-43,English,Nakuru\n"Tarehe 9 March 2026 katika Bungoma Central, Bungoma, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Extortion,realistic-extortion-02,Kiswahili,Bungoma\n"On 11 August 2026 pale Machakos Central, Machakos, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-045.",Abuse of Office,realistic-abuse-of-office-45,Sheng,Machakos\n"On 28 July 2026 in Kapseret, Eldoret, the county officer changed a queue order to serve a personal acquaintance first while others were told to wait. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Abuse of Office,realistic-abuse-of-office-48,English,Eldoret\n"Katika Imenti Central, Meru, tarehe 14 June 2026, bodi ya CDF bila kura ya wazi. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Nepotism/Favoritism,realistic-nepotism-favoritism-06,Kiswahili,Meru\n"On 3 January 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya shule joined the decision on aliwahi kuajiri; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-054.",Conflict of Interest,realistic-conflict-of-interest-54,Mixed,Garissa\n"Nilikagua document for iliandika bidhaa pale Machakos Central, Machakos, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-034.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-34,Mixed,Machakos\n"Nimekagua hati kuhusu iliandika bidhaa katika Machakos Central, Machakos, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-34,Kiswahili,Machakos\n"For tender ICT-09 in Bungoma Central, Bungoma, the bid evaluation team used scoring criteria that were not published in the notice. The file concerns computer and network systems and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-013.",Procurement Fraud,realistic-procurement-fraud-13,English,Bungoma\n"On 14 June 2026, I went to the public office in Kilifi North, Kilifi, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-010.",Bribery,realistic-bribery-10,English,Kilifi\n"Nilikagua document for ilibadilishwa pale Kapseret, Eldoret, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-027.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-27,Mixed,Eldoret\n"On 11 August 2026 in Nakuru East, Nakuru, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 1,000. They threatened to make a false inspection fault. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Extortion,realistic-extortion-27,English,Nakuru\n"Kwa tender ya mifumo ya kompyuta na mtandao pale Bungoma Central, Bungoma, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-013.",Procurement Fraud,realistic-procurement-fraud-13,Mixed,Bungoma\n"Juzi 6 May 2026 nikaingia pale office ya Garissa Township, Garissa, for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha 300, nimehifadhi screenshot na receipt. Ref KW-013.",Bribery,realistic-bribery-13,Sheng,Garissa\n"On 9 March 2026 pale Bungoma Central, Bungoma, afisa wa uteuzi wa umma joined the decision on alikuwa mwenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-048.",Conflict of Interest,realistic-conflict-of-interest-48,Mixed,Bungoma\n"On 28 July 2026 in Embakasi East, Nairobi, the station officer refused to release my brother and demanded KSh 2,000, about KSh 2,500. They while warning that the case file could be damaged. I kept the receipt and the message thread. Case note KW-046.",Extortion,realistic-extortion-46,English,Nairobi\n"Kwa huduma za ulinzi katika Kisumu Central, Kisumu, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Tukio linahusu huduma za ulinzi yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Procurement Fraud,realistic-procurement-fraud-31,Kiswahili,Kisumu\n"On 9 March 2026 pale Embakasi East, Nairobi, askari wa kaunti akasema zingerudishwa baada ya malipo, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Extortion,realistic-extortion-26,Sheng,Nairobi\n"Juzi 11 August 2026 nikaingia pale office ya Kilifi North, Kilifi, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Bribery,realistic-bribery-20,Sheng,Kilifi\n"Tarehe 14 June 2026, mwenyekiti wa jopo la ruzuku alishiriki uamuzi kuhusu alikuwa mshirika katika Imenti Central, Meru. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Conflict of Interest,realistic-conflict-of-interest-15,Kiswahili,Meru\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Machakos Central, Machakos. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Conflict of Interest,realistic-conflict-of-interest-17,Kiswahili,Machakos\n"Tarehe 3 January 2026 katika Kisumu Central, Kisumu, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Extortion,realistic-extortion-40,Kiswahili,Kisumu\n"In Kisumu Central, Kisumu, on 3 January 2026, the appointment board promoted an officer\'s spouse to a senior post despite a lower interview score. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-037.",Nepotism/Favoritism,realistic-nepotism-favoritism-37,English,Kisumu\n"Nili-check document ya ilikuwa na sahihi iliyorudishwa pale Kapseret, Eldoret, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, nimehifadhi screenshot na receipt. Ref KW-047.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-47,Sheng,Eldoret\n"Kuna money misuse kwa fedha za ukarabati wa barabara pale Imenti Central, Meru, after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 300, na screenshot na receipt bado. Case/rekodi KW-037.",Embezzlement,realistic-embezzlement-37,Mixed,Meru\n"Tarehe 14 June 2026 katika Nakuru East, Nakuru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Extortion,realistic-extortion-17,Kiswahili,Nakuru\n"Tender ya mifumo ya kompyuta na mtandao pale Kapseret, Eldoret ilikuwa imepangwa. timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa, around 5000, nimehifadhi screenshot na receipt. Ref KW-005.",Procurement Fraud,realistic-procurement-fraud-05,Sheng,Eldoret\n"On 22 April 2026 pale Likoni, Mombasa, afisa wa fedha wa kaunti joined the decision on alikuwa mmiliki binafsi; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-029.",Conflict of Interest,realistic-conflict-of-interest-29,Mixed,Mombasa\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Imenti Central, Meru. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Other,realistic-other-18,Kiswahili,Meru\n"Tender ya samani za ofisi pale Machakos Central, Machakos ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Procurement Fraud,realistic-procurement-fraud-12,Sheng,Machakos\n"On 3 January 2026 pale Kapseret, Eldoret, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Abuse of Office,realistic-abuse-of-office-58,Sheng,Eldoret\n"On 22 April 2026 pale Imenti Central, Meru, afisa wa fedha wa kaunti joined the decision on alikuwa mmiliki binafsi; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-045.",Conflict of Interest,realistic-conflict-of-interest-45,Mixed,Meru\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Kilifi North, Kilifi. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Other,realistic-other-34,Kiswahili,Kilifi\n"On 6 May 2026 in Kisumu Central, Kisumu, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 20,000. They and demanded an immediate private payment. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Extortion,realistic-extortion-60,English,Kisumu\n"On 14 June 2026, I dealt with the service desk in Imenti Central, Meru. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-058.",Other,realistic-other-58,English,Meru\n"I am reporting irregular use of KSh 5,000 in Embakasi East, Nairobi, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 2,500 entered receipts that could not be matched to payments. I kept the receipt and the message thread. Case note KW-034.",Embezzlement,realistic-embezzlement-34,English,Nairobi\n"I am reporting irregular use of KSh 5,000 in Imenti Central, Meru, after 14 June 2026. Records show that the public works foreman handled road-repair funds, and the funds worth KSh 300 were spent while the damaged section remained unfinished. I kept the receipt and the message thread. Case note KW-037.",Embezzlement,realistic-embezzlement-37,English,Meru\n"On 17 February 2026 pale Kilifi North, Kilifi, mkuu wa shule wakati wa shule; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-039.",Abuse of Office,realistic-abuse-of-office-39,Sheng,Kilifi\n"On 6 May 2026 pale Embakasi East, Nairobi, mkaguzi akadai malipo ya faragha mara moja, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-036.",Extortion,realistic-extortion-36,Mixed,Nairobi\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Garissa Township, Garissa. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Conflict of Interest,realistic-conflict-of-interest-04,Kiswahili,Garissa\n"Tarehe 17 February 2026 katika Nakuru East, Nakuru, mkuu wa shule wakati wa shule. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Abuse of Office,realistic-abuse-of-office-31,Kiswahili,Nakuru\n"Nilikagua document for ilibadilishwa pale Garissa Township, Garissa, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-051.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-51,Mixed,Garissa\n"On 28 July 2026 pale Embakasi East, Nairobi, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Conflict of Interest,realistic-conflict-of-interest-52,Sheng,Nairobi\n"Katika Kisumu Central, Kisumu, tarehe 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Nepotism/Favoritism,realistic-nepotism-favoritism-07,Kiswahili,Kisumu\n"On 14 June 2026, I went to the public office in Likoni, Mombasa, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Bribery,realistic-bribery-18,English,Mombasa\n"Katika Kapseret, Eldoret, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Nepotism/Favoritism,realistic-nepotism-favoritism-41,Kiswahili,Eldoret\n"Tarehe 14 June 2026, mwenyekiti wa jopo la ruzuku alishiriki uamuzi kuhusu alikuwa mshirika katika Imenti Central, Meru. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Conflict of Interest,realistic-conflict-of-interest-55,Kiswahili,Meru\n"For tender VEH-441 in Machakos Central, Machakos, the contract committee accepted a bid after the deadline while rejecting timely bidders. The file concerns vehicle leasing and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-022.",Procurement Fraud,realistic-procurement-fraud-22,English,Machakos\n"On 14 June 2026 pale Machakos Central, Machakos, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-035.",Abuse of Office,realistic-abuse-of-office-35,Mixed,Machakos\n"On 14 June 2026 in Nakuru East, Nakuru, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Abuse of Office,realistic-abuse-of-office-11,English,Nakuru\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Machakos Central, Machakos. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Other,realistic-other-50,Kiswahili,Machakos\n"Kwa zabuni ya vifaa vya hospitali katika Nakuru East, Nakuru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Procurement Fraud,realistic-procurement-fraud-18,Kiswahili,Nakuru\n"Nilienda pale office ya Kisumu Central, Kisumu, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 5,000, na screenshot na receipt bado. Case/rekodi KW-035.",Bribery,realistic-bribery-35,Mixed,Kisumu\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Likoni, Mombasa. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Conflict of Interest,realistic-conflict-of-interest-19,Kiswahili,Mombasa\n"Kuna doo za pesa ya mradi wa vijana zilipotea pale Bungoma Central, Bungoma after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Embezzlement,realistic-embezzlement-30,Sheng,Bungoma\n"Pale Bungoma Central, Bungoma, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-059.",Nepotism/Favoritism,realistic-nepotism-favoritism-59,Mixed,Bungoma\n"In Nakuru East, Nakuru, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-004.",Nepotism/Favoritism,realistic-nepotism-favoritism-04,English,Nakuru\n"Ninawasilisha tuhuma za matumizi mabaya ya chakula na blanketi katika Garissa Township, Garissa, tangu 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Embezzlement,realistic-embezzlement-16,Kiswahili,Garissa\n"On 9 March 2026 pale Kisumu Central, Kisumu, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Abuse of Office,realistic-abuse-of-office-44,Sheng,Kisumu\n"On 9 March 2026 pale Kisumu Central, Kisumu, afisa wa uteuzi wa umma aliingia decision ya alikuwa mwenzi; was the spouse, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Conflict of Interest,realistic-conflict-of-interest-56,Sheng,Kisumu\n"Tarehe 3 January 2026 katika Kisumu Central, Kisumu, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Abuse of Office,realistic-abuse-of-office-34,Kiswahili,Kisumu\n"On 17 February 2026 in Likoni, Mombasa, the public officer threatened to blacklist my supplier account and demanded KSh 5,000, about KSh 5,000. They unless I paid from my personal funds. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Extortion,realistic-extortion-53,English,Mombasa\n"Nimekagua hati kuhusu kilitiwa sahihi katika Bungoma Central, Bungoma, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-25,Kiswahili,Bungoma\n"Tarehe 17 February 2026, nilienda ofisi ya umma katika Machakos Central, Machakos, kwa utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Bribery,realistic-bribery-46,Kiswahili,Machakos\n"In Likoni, Mombasa, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Nepotism/Favoritism,realistic-nepotism-favoritism-20,English,Mombasa\n"Kuna doo za fedha za kisima zilipotea pale Likoni, Mombasa after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 5000, nimehifadhi screenshot na receipt. Ref KW-041.",Embezzlement,realistic-embezzlement-41,Sheng,Mombasa\n"Nili-check document ya iliandika bidhaa pale Imenti Central, Meru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-02,Sheng,Meru\n"I checked a completion certificate relating to the public works file in Garissa Township, Garissa, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-41,English,Garissa\n"Pale Nakuru East, Nakuru, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-034.",Nepotism/Favoritism,realistic-nepotism-favoritism-34,Mixed,Nakuru\n"Pale Imenti Central, Meru on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-028.",Other,realistic-other-28,Mixed,Meru\n"Juzi 22 April 2026 nikaingia pale office ya Machakos Central, Machakos, for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Bribery,realistic-bribery-16,Sheng,Machakos\n"Katika Bungoma Central, Bungoma, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Nepotism/Favoritism,realistic-nepotism-favoritism-19,Kiswahili,Bungoma\n"On 11 August 2026 pale Imenti Central, Meru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-013.",Abuse of Office,realistic-abuse-of-office-13,Sheng,Meru\n"Kuna doo za fedha za wanafunzi zilipotea pale Kapseret, Eldoret after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Embezzlement,realistic-embezzlement-02,Sheng,Eldoret\n"Kuna money misuse kwa mapato ya ada za soko pale Likoni, Mombasa, after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 300, na screenshot na receipt bado. Case/rekodi KW-031.",Embezzlement,realistic-embezzlement-31,Mixed,Mombasa\n"I checked an academic certificate relating to the appointment file in Kisumu Central, Kisumu, on 9 March 2026. The document contained a changed name and a stamp that the issuing school says is not genuine. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-013.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-13,English,Kisumu\n"I checked an academic certificate relating to the appointment file in Garissa Township, Garissa, on 9 March 2026. The document contained a changed name and a stamp that the issuing school says is not genuine. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-21,English,Garissa\n"On 9 March 2026 in Kisumu Central, Kisumu, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Abuse of Office,realistic-abuse-of-office-44,English,Kisumu\n"On 9 March 2026 in Kisumu Central, Kisumu, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I kept the receipt and the message thread. Case note KW-004.",Abuse of Office,realistic-abuse-of-office-04,English,Kisumu\n"Nimekagua hati kuhusu kilitiwa sahihi katika Garissa Township, Garissa, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-01,Kiswahili,Garissa\n"On 17 February 2026 pale Kilifi North, Kilifi, mkuu wa shule wakati wa shule; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-039.",Abuse of Office,realistic-abuse-of-office-39,Mixed,Kilifi\n"For tender SEC-55 in Garissa Township, Garissa, the procuring entity gave one supplier confidential bid information. The file concerns security services and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Procurement Fraud,realistic-procurement-fraud-39,English,Garissa\n"On 6 May 2026 pale Kisumu Central, Kisumu, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-026.",Conflict of Interest,realistic-conflict-of-interest-26,Mixed,Kisumu\n"Pale Embakasi East, Nairobi on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no action taken, na screenshot na receipt bado. Case/rekodi KW-055.",Other,realistic-other-55,Mixed,Nairobi\n"Pale Embakasi East, Nairobi on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-025.",Other,realistic-other-25,Mixed,Nairobi\n"On 3 January 2026, I went to the public office in Kisumu Central, Kisumu, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-025.",Bribery,realistic-bribery-25,English,Kisumu\n"Kuna doo za pesa ya mradi wa vijana zilipotea pale Kapseret, Eldoret after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Embezzlement,realistic-embezzlement-22,Sheng,Eldoret\n"Tarehe 14 June 2026 katika Likoni, Mombasa, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Abuse of Office,realistic-abuse-of-office-27,Kiswahili,Mombasa\n"On 11 August 2026 pale Imenti Central, Meru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-019.",Extortion,realistic-extortion-19,Sheng,Meru\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-007.",Extortion,realistic-extortion-07,Sheng,Nakuru\n"Nilienda pale office ya Kisumu Central, Kisumu, on 28 July 2026 for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida, 300, na screenshot na receipt bado. Case/rekodi KW-055.",Bribery,realistic-bribery-55,Mixed,Kisumu\n"Kuna doo za pesa za dawa zilipotea pale Bungoma Central, Bungoma after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Embezzlement,realistic-embezzlement-20,Sheng,Bungoma\n"On 9 March 2026 pale Embakasi East, Nairobi, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-026.",Extortion,realistic-extortion-26,Mixed,Nairobi\n"Pale Kapseret, Eldoret, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-001.",Nepotism/Favoritism,realistic-nepotism-favoritism-01,Mixed,Eldoret\n"On 17 February 2026 pale Machakos Central, Machakos, mkuu wa shule wakati wa shule; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-055.",Abuse of Office,realistic-abuse-of-office-55,Mixed,Machakos\n"On 14 June 2026, the grant panel chair took part in a decision about a community grant in Imenti Central, Meru. The officer was a partner in the organisation applying for funds. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Conflict of Interest,realistic-conflict-of-interest-15,English,Meru\n"Pale Kilifi North, Kilifi, on 14 June 2026, bodi ya CDF bila kura ya wazi, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-022.",Nepotism/Favoritism,realistic-nepotism-favoritism-22,Mixed,Kilifi\n"I am reporting irregular use of KSh 5,000 in Bungoma Central, Bungoma, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 500 entered receipts that could not be matched to payments. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Embezzlement,realistic-embezzlement-50,English,Bungoma\n"I checked a completion certificate relating to the public works file in Embakasi East, Nairobi, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-09,English,Nairobi\n"Tarehe 6 May 2026 katika Kisumu Central, Kisumu, mkaguzi akadai malipo ya faragha mara moja, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Extortion,realistic-extortion-20,Kiswahili,Kisumu\n"On 3 January 2026, the school board member took part in a decision about a tender for uniforms in Kisumu Central, Kisumu. The officer employed the bidder as a private consultant. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-046.",Conflict of Interest,realistic-conflict-of-interest-46,English,Kisumu\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa ya mradi wa vijana katika Kisumu Central, Kisumu, tangu 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Embezzlement,realistic-embezzlement-38,Kiswahili,Kisumu\n"I checked a completion certificate relating to the public works file in Garissa Township, Garissa, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-001.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-01,English,Garissa\n"Juzi 28 July 2026 nikaingia pale office ya Kapseret, Eldoret, for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida 1000, nimehifadhi screenshot na receipt. Ref KW-039.",Bribery,realistic-bribery-39,Sheng,Eldoret\n"On 3 January 2026 pale Kapseret, Eldoret, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-018.",Abuse of Office,realistic-abuse-of-office-18,Mixed,Eldoret\n"On 22 April 2026 in Likoni, Mombasa, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Abuse of Office,realistic-abuse-of-office-17,English,Mombasa\n"Tarehe 11 August 2026 katika Likoni, Mombasa, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Extortion,realistic-extortion-03,Kiswahili,Mombasa\n"On 6 May 2026 pale Kapseret, Eldoret, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Conflict of Interest,realistic-conflict-of-interest-10,Sheng,Eldoret\n"Kwa mifumo ya kompyuta na mtandao katika Kapseret, Eldoret, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Tukio linahusu mifumo ya kompyuta na mtandao yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Procurement Fraud,realistic-procurement-fraud-45,Kiswahili,Eldoret\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Kisumu Central, Kisumu. bila taarifa wala haikutangaza wakati huduma ingerejea. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Other,realistic-other-49,Kiswahili,Kisumu\n"Kuna doo za pesa za dawa zilipotea pale Kapseret, Eldoret after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Embezzlement,realistic-embezzlement-52,Sheng,Eldoret\n"Pale Likoni, Mombasa on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Other,realistic-other-42,Sheng,Mombasa\n"Pale Garissa Township, Garissa on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no action taken, na screenshot na receipt bado. Case/rekodi KW-047.",Other,realistic-other-47,Mixed,Garissa\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za ukarabati wa barabara katika Kilifi North, Kilifi, tangu 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Embezzlement,realistic-embezzlement-53,Kiswahili,Kilifi\n"Kwa tender ya huduma za ulinzi pale Kapseret, Eldoret, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-055.",Procurement Fraud,realistic-procurement-fraud-55,Mixed,Eldoret\n"Tarehe 6 May 2026 katika Garissa Township, Garissa, mkaguzi akadai malipo ya faragha mara moja, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Extortion,realistic-extortion-28,Kiswahili,Garissa\n"On 28 July 2026 pale Kapseret, Eldoret, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-020.",Conflict of Interest,realistic-conflict-of-interest-20,Mixed,Eldoret\n"Pale Machakos Central, Machakos on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Nepotism/Favoritism,realistic-nepotism-favoritism-18,Sheng,Machakos\n"Juzi 22 April 2026 nikaingia pale office ya Kilifi North, Kilifi, for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Bribery,realistic-bribery-40,Sheng,Kilifi\n"Nilikagua document for ilonyesha anwani ya uongo pale Kilifi North, Kilifi, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeweka details ya time na office number. Case/rekodi KW-038.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-38,Mixed,Kilifi\n"Kuna doo za chakula na blanketi zilipotea pale Embakasi East, Nairobi after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Embezzlement,realistic-embezzlement-24,Sheng,Nairobi\n"On 9 March 2026, the public appointment officer took part in a decision about a job assessment in Bungoma Central, Bungoma. The officer was the spouse of one shortlisted candidate. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Conflict of Interest,realistic-conflict-of-interest-08,English,Bungoma\n"Pale Imenti Central, Meru on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Other,realistic-other-58,Sheng,Meru\n"On 14 June 2026, I went to the public office in Imenti Central, Meru, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-034.",Bribery,realistic-bribery-34,English,Meru\n"Tarehe 17 February 2026 katika Nakuru East, Nakuru, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Extortion,realistic-extortion-37,Kiswahili,Nakuru\n"Kuna money misuse kwa fedha za chakula pale Kilifi North, Kilifi, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 300, na screenshot na receipt bado. Case/rekodi KW-043.",Embezzlement,realistic-embezzlement-43,Mixed,Kilifi\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Likoni, Mombasa. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Conflict of Interest,realistic-conflict-of-interest-49,Kiswahili,Mombasa\n"On 11 August 2026 in Likoni, Mombasa, the department director used an official vehicle for a private family trip and signed the fuel claim through the office. This was not a routine decision. I kept the receipt and the message thread. Case note KW-037.",Abuse of Office,realistic-abuse-of-office-37,English,Mombasa\n"Tender ya zabuni ya ukarabati wa barabara pale Kapseret, Eldoret ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 300, nimehifadhi screenshot na receipt. Ref KW-025.",Procurement Fraud,realistic-procurement-fraud-25,Sheng,Eldoret\n"Tarehe 22 April 2026, afisa wa fedha wa kaunti alishiriki uamuzi kuhusu alikuwa mmiliki binafsi katika Imenti Central, Meru. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Conflict of Interest,realistic-conflict-of-interest-05,Kiswahili,Meru\n"On 3 January 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya shule joined the decision on aliwahi kuajiri; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-014.",Conflict of Interest,realistic-conflict-of-interest-14,Mixed,Garissa\n"On 17 February 2026 in Likoni, Mombasa, the school principal directed a teacher to use school equipment for a private function during school time. This was not a routine decision. I kept the receipt and the message thread. Case note KW-007.",Abuse of Office,realistic-abuse-of-office-07,English,Mombasa\n"Tarehe 9 March 2026 katika Kapseret, Eldoret, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Extortion,realistic-extortion-34,Kiswahili,Eldoret\n"Kwa zabuni ya ujenzi wa madarasa katika Kapseret, Eldoret, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Procurement Fraud,realistic-procurement-fraud-35,Kiswahili,Eldoret\n"Kwa samani za ofisi katika Likoni, Mombasa, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Procurement Fraud,realistic-procurement-fraud-44,Kiswahili,Mombasa\n"Kuna money misuse kwa fedha za chakula pale Likoni, Mombasa, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 1,000, na screenshot na receipt bado. Case/rekodi KW-051.",Embezzlement,realistic-embezzlement-51,Mixed,Mombasa\n"Pale Imenti Central, Meru on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Other,realistic-other-18,Sheng,Meru\n"On 3 January 2026 pale Garissa Township, Garissa, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Extortion,realistic-extortion-48,Sheng,Garissa\n"On 9 March 2026 pale Kisumu Central, Kisumu, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-050.",Extortion,realistic-extortion-50,Mixed,Kisumu\n"On 17 February 2026 pale Nakuru East, Nakuru, mkuu wa shule wakati wa shule; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-031.",Abuse of Office,realistic-abuse-of-office-31,Sheng,Nakuru\n"Kuna doo za fedha za ukarabati wa barabara zilipotea pale Nakuru East, Nakuru after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 1000, nimehifadhi screenshot na receipt. Ref KW-045.",Embezzlement,realistic-embezzlement-45,Sheng,Nakuru\n"Juzi 3 January 2026 nikaingia pale office ya Kapseret, Eldoret, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 300, nimehifadhi screenshot na receipt. Ref KW-049.",Bribery,realistic-bribery-49,Sheng,Eldoret\n"Kwa ukodishaji wa magari katika Likoni, Mombasa, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Tukio linahusu ukodishaji wa magari yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Procurement Fraud,realistic-procurement-fraud-14,Kiswahili,Mombasa\n"On 9 March 2026 pale Garissa Township, Garissa, afisa wa uteuzi wa umma aliingia decision ya alikuwa mwenzi; was the spouse, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Conflict of Interest,realistic-conflict-of-interest-24,Sheng,Garissa\n"Katika Garissa Township, Garissa, tarehe 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Nepotism/Favoritism,realistic-nepotism-favoritism-45,Kiswahili,Garissa\n"Juzi 28 July 2026 nikaingia pale office ya Embakasi East, Nairobi, for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida 300, nimehifadhi screenshot na receipt. Ref KW-031.",Bribery,realistic-bribery-31,Sheng,Nairobi\n"Tarehe 6 May 2026, nilishughulika na ofisi ya barabara katika Embakasi East, Nairobi. kwenye barabara yetu lakini hakuna kazi iliyoanza kabla ya muda. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Other,realistic-other-05,Kiswahili,Nairobi\n"On 14 June 2026 pale Likoni, Mombasa, mwenyekiti wa jopo la ruzuku joined the decision on alikuwa mshirika; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-039.",Conflict of Interest,realistic-conflict-of-interest-39,Mixed,Mombasa\n"On 14 June 2026 pale Imenti Central, Meru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-043.",Abuse of Office,realistic-abuse-of-office-43,Sheng,Meru\n"Tarehe 6 May 2026, nilienda ofisi ya umma katika Kisumu Central, Kisumu, kwa ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Bribery,realistic-bribery-05,Kiswahili,Kisumu\n"Tarehe 14 June 2026 katika Nakuru East, Nakuru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Abuse of Office,realistic-abuse-of-office-11,Kiswahili,Nakuru\n"On 28 July 2026 pale Bungoma Central, Bungoma, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Conflict of Interest,realistic-conflict-of-interest-28,Sheng,Bungoma\n"Nilikagua document for ilihaririwa pale Likoni, Mombasa, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-036.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-36,Mixed,Mombasa\n"On 22 April 2026 in Kilifi North, Kilifi, the gate security supervisor blocked my lorry and demanded KSh 1,500, about KSh 300. They threatening to call a taxing inspection. I kept the receipt and the message thread. Case note KW-055.",Extortion,realistic-extortion-55,English,Kilifi\n"Pale Garissa Township, Garissa, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-025.",Nepotism/Favoritism,realistic-nepotism-favoritism-25,Mixed,Garissa\n"Pale Kilifi North, Kilifi, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-052.",Nepotism/Favoritism,realistic-nepotism-favoritism-52,Mixed,Kilifi\n"On 17 February 2026, I went to the public office in Kilifi North, Kilifi, for release of a passport. The the immigration officer requested money to place the application ahead of the queue; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Bribery,realistic-bribery-30,English,Kilifi\n"Kuna doo za pesa ya mradi wa vijana zilipotea pale Garissa Township, Garissa after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Embezzlement,realistic-embezzlement-06,Sheng,Garissa\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Kisumu Central, Kisumu. kutofanya kazi kwa miezi kadhaa bila maelezo. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Other,realistic-other-59,Kiswahili,Kisumu\n"Nili-check document ya ilonyesha anwani ya uongo pale Machakos Central, Machakos, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-14,Sheng,Machakos\n"Tarehe 17 February 2026 katika Likoni, Mombasa, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Extortion,realistic-extortion-53,Kiswahili,Mombasa\n"Nilikagua document for iliandika bidhaa pale Imenti Central, Meru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-042.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-42,Mixed,Meru\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za ukarabati wa barabara katika Nakuru East, Nakuru, tangu 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Embezzlement,realistic-embezzlement-45,Kiswahili,Nakuru\n"Tender ya huduma za ulinzi pale Embakasi East, Nairobi ilikuwa imepangwa. taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri, around 300, nimehifadhi screenshot na receipt. Ref KW-007.",Procurement Fraud,realistic-procurement-fraud-07,Sheng,Nairobi\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Machakos Central, Machakos, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-042.",Procurement Fraud,realistic-procurement-fraud-42,Mixed,Machakos\n"Pale Bungoma Central, Bungoma on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-001.",Other,realistic-other-01,Mixed,Bungoma\n"Tarehe 28 July 2026 katika Embakasi East, Nairobi, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Extortion,realistic-extortion-46,Kiswahili,Nairobi\n"Nimekagua hati kuhusu ilikuwa na sahihi iliyorudishwa katika Kapseret, Eldoret, tarehe 6 May 2026. vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-07,Kiswahili,Eldoret\n"Kwa samani za ofisi katika Nakuru East, Nakuru, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Procurement Fraud,realistic-procurement-fraud-28,Kiswahili,Nakuru\n"Pale Likoni, Mombasa on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Other,realistic-other-22,Sheng,Mombasa\n"On 6 May 2026, I dealt with the roads office in Embakasi East, Nairobi. The office promised resurfacing for our street but no work began before the deadline. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Other,realistic-other-05,English,Nairobi\n"Pale Imenti Central, Meru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Other,realistic-other-08,Sheng,Meru\n"Nilienda pale office ya Kapseret, Eldoret, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 5,000, na screenshot na receipt bado. Case/rekodi KW-059.",Bribery,realistic-bribery-59,Mixed,Eldoret\n"On 14 June 2026, I dealt with the service desk in Nakuru East, Nakuru. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Other,realistic-other-26,English,Nakuru\n"Tarehe 28 July 2026, nilienda ofisi ya umma katika Kisumu Central, Kisumu, kwa uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Bribery,realistic-bribery-55,Kiswahili,Kisumu\n"On 28 July 2026, I went to the public office in Kapseret, Eldoret, for printing an identification document. The the Huduma Centre clerk asked for a personal payment to speed up the ordinary service; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Bribery,realistic-bribery-39,English,Eldoret\n"Nimekagua hati kuhusu ilihaririwa katika Likoni, Mombasa, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-36,Kiswahili,Mombasa\n"Nilienda pale office ya Likoni, Mombasa, on 22 April 2026 for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada, 500, nimeweka details ya time na office number. Case/rekodi KW-008.",Bribery,realistic-bribery-08,Mixed,Mombasa\n"In Machakos Central, Machakos, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-028.",Nepotism/Favoritism,realistic-nepotism-favoritism-28,English,Machakos\n"Tarehe 6 May 2026 katika Embakasi East, Nairobi, afisa wa rekodi akatishia kuondoa rekodi ya ombi. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Abuse of Office,realistic-abuse-of-office-30,Kiswahili,Nairobi\n"On 9 March 2026, I went to the public office in Embakasi East, Nairobi, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Bribery,realistic-bribery-11,English,Nairobi\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Garissa Township, Garissa, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Embezzlement,realistic-embezzlement-36,Kiswahili,Garissa\n"Tender ya ukodishaji wa magari pale Nakuru East, Nakuru ilikuwa imepangwa. kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Procurement Fraud,realistic-procurement-fraud-38,Sheng,Nakuru\n"On 17 February 2026 pale Imenti Central, Meru, mkuu wa shule wakati wa shule; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-023.",Abuse of Office,realistic-abuse-of-office-23,Sheng,Meru\n"On 11 August 2026 pale Likoni, Mombasa, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-009.",Conflict of Interest,realistic-conflict-of-interest-09,Sheng,Mombasa\n"Ninawasilisha tuhuma za matumizi mabaya ya mapato ya ada za soko katika Imenti Central, Meru, tangu 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Embezzlement,realistic-embezzlement-47,Kiswahili,Meru\n"Nili-check document ya kilikuwa na jina lililobadilishwa pale Bungoma Central, Bungoma, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, nimehifadhi screenshot na receipt. Ref KW-045.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-45,Sheng,Bungoma\n"In Kapseret, Eldoret, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Nepotism/Favoritism,realistic-nepotism-favoritism-11,English,Eldoret\n"Pale Likoni, Mombasa on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-002.",Other,realistic-other-02,Mixed,Mombasa\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Garissa Township, Garissa, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-019.",Procurement Fraud,realistic-procurement-fraud-19,Mixed,Garissa\n"Tarehe 6 May 2026 katika Kapseret, Eldoret, mkaguzi akadai malipo ya faragha mara moja, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Extortion,realistic-extortion-44,Kiswahili,Eldoret\n"For tender HOS-118 in Likoni, Mombasa, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-034.",Procurement Fraud,realistic-procurement-fraud-34,English,Mombasa\n"I checked a payment voucher relating to the hospital account in Kisumu Central, Kisumu, on 6 May 2026. The document carried a duplicate signature for a purchase that has no supplier evidence. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-23,English,Kisumu\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Bungoma Central, Bungoma, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Embezzlement,realistic-embezzlement-50,Kiswahili,Bungoma\n"Tarehe 14 June 2026 katika Kilifi North, Kilifi, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Abuse of Office,realistic-abuse-of-office-59,Kiswahili,Kilifi\n"On 22 April 2026 pale Likoni, Mombasa, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-057.",Abuse of Office,realistic-abuse-of-office-57,Mixed,Mombasa\n"On 9 March 2026 pale Garissa Township, Garissa, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-058.",Extortion,realistic-extortion-58,Mixed,Garissa\n"Pale Embakasi East, Nairobi on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-013.",Nepotism/Favoritism,realistic-nepotism-favoritism-13,Sheng,Nairobi\n"I am reporting irregular use of KSh 5,000 in Garissa Township, Garissa, after 6 May 2026. Records show that the relief distribution committee handled food and blankets, and the funds worth KSh 2,500 were recorded as delivered but never reached beneficiaries. I kept the receipt and the message thread. Case note KW-016.",Embezzlement,realistic-embezzlement-16,English,Garissa\n"In Bungoma Central, Bungoma, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Nepotism/Favoritism,realistic-nepotism-favoritism-59,English,Bungoma\n"Pale Kisumu Central, Kisumu, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-057.",Nepotism/Favoritism,realistic-nepotism-favoritism-57,Mixed,Kisumu\n"Kwa tender ya samani za ofisi pale Machakos Central, Machakos, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-052.",Procurement Fraud,realistic-procurement-fraud-52,Mixed,Machakos\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Likoni, Mombasa. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Other,realistic-other-42,Kiswahili,Mombasa\n"Katika Kilifi North, Kilifi, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Nepotism/Favoritism,realistic-nepotism-favoritism-52,Kiswahili,Kilifi\n"Pale Machakos Central, Machakos on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Nepotism/Favoritism,realistic-nepotism-favoritism-58,Sheng,Machakos\n"On 14 June 2026 in Imenti Central, Meru, the police officer held my identification card and demanded KSh 2,500, about KSh 1,000. They threatening arrest unless I paid in cash. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Extortion,realistic-extortion-09,English,Meru\n"Kwa zabuni ya ukarabati wa barabara katika Kapseret, Eldoret, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Procurement Fraud,realistic-procurement-fraud-25,Kiswahili,Eldoret\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Machakos Central, Machakos. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Conflict of Interest,realistic-conflict-of-interest-17,English,Machakos\n"Nilienda pale office ya Kisumu Central, Kisumu, on 6 May 2026 for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha, 5,000, na screenshot na receipt bado. Case/rekodi KW-005.",Bribery,realistic-bribery-05,Mixed,Kisumu\n"Kuna money misuse kwa fedha za chakula pale Nakuru East, Nakuru, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 5,000, na screenshot na receipt bado. Case/rekodi KW-035.",Embezzlement,realistic-embezzlement-35,Mixed,Nakuru\n"On 3 January 2026 pale Embakasi East, Nairobi, mjumbe wa bodi ya shule aliingia decision ya aliwahi kuajiri; employed, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Conflict of Interest,realistic-conflict-of-interest-22,Sheng,Nairobi\n"Nilienda pale office ya Machakos Central, Machakos, on 22 April 2026 for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada, 500, nimeweka details ya time na office number. Case/rekodi KW-056.",Bribery,realistic-bribery-56,Mixed,Machakos\n"Kuna money misuse kwa fedha za kisima pale Imenti Central, Meru, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 1,000, na screenshot na receipt bado. Case/rekodi KW-057.",Embezzlement,realistic-embezzlement-57,Mixed,Meru\n"Nilikagua document for ilonyesha anwani ya uongo pale Machakos Central, Machakos, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeweka details ya time na office number. Case/rekodi KW-054.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-54,Mixed,Machakos\n"Tarehe 22 April 2026 katika Nakuru East, Nakuru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Abuse of Office,realistic-abuse-of-office-41,Kiswahili,Nakuru\n"Tarehe 6 May 2026 katika Kisumu Central, Kisumu, afisa wa rekodi akatishia kuondoa rekodi ya ombi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Abuse of Office,realistic-abuse-of-office-14,Kiswahili,Kisumu\n"Nili-check document ya ilihaririwa pale Likoni, Mombasa, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-36,Sheng,Mombasa\n"Katika Likoni, Mombasa, tarehe 14 June 2026, bodi ya CDF bila kura ya wazi. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Nepotism/Favoritism,realistic-nepotism-favoritism-30,Kiswahili,Mombasa\n"Tarehe 3 January 2026 katika Garissa Township, Garissa, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Extortion,realistic-extortion-48,Kiswahili,Garissa\n"For contract FUEL-33 in Kilifi North, Kilifi, the evaluation secretary altered the comparison sheet after bids were submitted. The file concerns fuel supply and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-016.",Procurement Fraud,realistic-procurement-fraud-16,English,Kilifi\n"On 22 April 2026, I dealt with the public help line in Kilifi North, Kilifi. The office did not answer for repeated calls and provided no follow-up reference. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Other,realistic-other-24,English,Kilifi\n"Pale Embakasi East, Nairobi on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-045.",Other,realistic-other-45,Sheng,Nairobi\n"I checked a business permit relating to the revenue office in Imenti Central, Meru, on 11 August 2026. The document showed a false address and a permit number not in the official register. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-022.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-22,English,Meru\n"Nilikagua document for ilibadilishwa pale Kisumu Central, Kisumu, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-043.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-43,Mixed,Kisumu\n"Pale Kilifi North, Kilifi on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-044.",Other,realistic-other-44,Mixed,Kilifi\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Nakuru East, Nakuru, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Bribery,realistic-bribery-02,Kiswahili,Nakuru\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Imenti Central, Meru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-050.",Procurement Fraud,realistic-procurement-fraud-50,Mixed,Meru\n"I checked a land transfer record relating to the county registry in Embakasi East, Nairobi, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-019.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-19,English,Nairobi\n"On 22 April 2026 in Nakuru East, Nakuru, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Abuse of Office,realistic-abuse-of-office-41,English,Nakuru\n"I checked a land transfer record relating to the county registry in Garissa Township, Garissa, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-51,English,Garissa\n"I checked a payment voucher relating to the hospital account in Kapseret, Eldoret, on 6 May 2026. The document carried a duplicate signature for a purchase that has no supplier evidence. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-007.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-07,English,Eldoret\n"Katika Kisumu Central, Kisumu, tarehe 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Nepotism/Favoritism,realistic-nepotism-favoritism-37,Kiswahili,Kisumu\n"On 22 April 2026 in Kilifi North, Kilifi, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Abuse of Office,realistic-abuse-of-office-09,English,Kilifi\n"Juzi 3 January 2026 nikaingia pale office ya Embakasi East, Nairobi, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 300, nimehifadhi screenshot na receipt. Ref KW-001.",Bribery,realistic-bribery-01,Sheng,Nairobi\n"On 28 July 2026 pale Embakasi East, Nairobi, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-052.",Conflict of Interest,realistic-conflict-of-interest-52,Mixed,Nairobi\n"Tender ya zabuni ya ujenzi wa madarasa pale Kisumu Central, Kisumu ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 1000, nimehifadhi screenshot na receipt. Ref KW-051.",Procurement Fraud,realistic-procurement-fraud-51,Sheng,Kisumu\n"Tarehe 3 January 2026 katika Bungoma Central, Bungoma, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Extortion,realistic-extortion-32,Kiswahili,Bungoma\n"I am reporting irregular use of KSh 20,000 in Nakuru East, Nakuru, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 300 was withdrawn but the borehole was never completed. I kept the receipt and the message thread. Case note KW-025.",Embezzlement,realistic-embezzlement-25,English,Nakuru\n"Pale Kapseret, Eldoret, on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-031.",Nepotism/Favoritism,realistic-nepotism-favoritism-31,Mixed,Eldoret\n"Pale Embakasi East, Nairobi, on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-013.",Nepotism/Favoritism,realistic-nepotism-favoritism-13,Mixed,Nairobi\n"On 6 May 2026, the project director took part in a decision about a construction contract in Garissa Township, Garissa. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-034.",Conflict of Interest,realistic-conflict-of-interest-34,English,Garissa\n"I am reporting irregular use of KSh 5,000 in Garissa Township, Garissa, after 6 May 2026. Records show that the relief distribution committee handled food and blankets, and the funds worth KSh 500 were recorded as delivered but never reached beneficiaries. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Embezzlement,realistic-embezzlement-56,English,Garissa\n"Kwa samani za ofisi katika Machakos Central, Machakos, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Procurement Fraud,realistic-procurement-fraud-52,Kiswahili,Machakos\n"Nili-check document ya kilitiwa sahihi pale Garissa Township, Garissa, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-041.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-41,Sheng,Garissa\n"Nilikagua document for kilitiwa sahihi pale Kapseret, Eldoret, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-017.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-17,Mixed,Eldoret\n"Pale Nakuru East, Nakuru on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Nepotism/Favoritism,realistic-nepotism-favoritism-24,Sheng,Nakuru\n"On 14 June 2026, I dealt with the service desk in Imenti Central, Meru. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Other,realistic-other-18,English,Meru\n"Nilikagua document for kilitiwa sahihi pale Embakasi East, Nairobi, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-009.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-09,Mixed,Nairobi\n"On 6 May 2026 pale Kapseret, Eldoret, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Conflict of Interest,realistic-conflict-of-interest-50,Sheng,Eldoret\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-047.",Extortion,realistic-extortion-47,Sheng,Nakuru\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Bungoma Central, Bungoma, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-003.",Procurement Fraud,realistic-procurement-fraud-03,Mixed,Bungoma\n"Kuna money misuse kwa fedha za kisima pale Nakuru East, Nakuru, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 300, na screenshot na receipt bado. Case/rekodi KW-025.",Embezzlement,realistic-embezzlement-25,Mixed,Nakuru\n"On 14 June 2026, the grant panel chair took part in a decision about a community grant in Imenti Central, Meru. The officer was a partner in the organisation applying for funds. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-055.",Conflict of Interest,realistic-conflict-of-interest-55,English,Meru\n"Kwa tender ya huduma za ulinzi pale Embakasi East, Nairobi, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-007.",Procurement Fraud,realistic-procurement-fraud-07,Mixed,Nairobi\n"Pale Imenti Central, Meru on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no action taken, nimeweka details ya time na office number. Case/rekodi KW-038.",Other,realistic-other-38,Mixed,Meru\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-018.",Conflict of Interest,realistic-conflict-of-interest-18,Mixed,Bungoma\n"Nili-check document ya kilikuwa na jina lililobadilishwa pale Embakasi East, Nairobi, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, nimehifadhi screenshot na receipt. Ref KW-029.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-29,Sheng,Nairobi\n"On 6 May 2026 pale Bungoma Central, Bungoma, afisa wa rekodi akatishia kuondoa rekodi ya ombi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Abuse of Office,realistic-abuse-of-office-06,Sheng,Bungoma\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Garissa Township, Garissa, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Bribery,realistic-bribery-43,Kiswahili,Garissa\n"I checked a completion certificate relating to the public works file in Kisumu Central, Kisumu, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-33,English,Kisumu\n"Pale Embakasi East, Nairobi on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-053.",Nepotism/Favoritism,realistic-nepotism-favoritism-53,Sheng,Nairobi\n"Pale Kapseret, Eldoret on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-043.",Other,realistic-other-43,Sheng,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Kilifi North, Kilifi, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Embezzlement,realistic-embezzlement-33,Kiswahili,Kilifi\n"On 11 August 2026 pale Kilifi North, Kilifi, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-035.",Extortion,realistic-extortion-35,Sheng,Kilifi\n"Kuna money misuse kwa fedha za wanafunzi pale Embakasi East, Nairobi, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 2,500, nimeweka details ya time na office number. Case/rekodi KW-034.",Embezzlement,realistic-embezzlement-34,Mixed,Nairobi\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Nakuru East, Nakuru. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Conflict of Interest,realistic-conflict-of-interest-33,Kiswahili,Nakuru\n"Pale Garissa Township, Garissa on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no action taken, na screenshot na receipt bado. Case/rekodi KW-007.",Other,realistic-other-07,Mixed,Garissa\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Nakuru East, Nakuru. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Conflict of Interest,realistic-conflict-of-interest-43,Kiswahili,Nakuru\n"On 17 February 2026 pale Likoni, Mombasa, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-013.",Extortion,realistic-extortion-13,Sheng,Mombasa\n"Juzi 17 February 2026 nikaingia pale office ya Imenti Central, Meru, for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni 500, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Bribery,realistic-bribery-14,Sheng,Meru\n"Nili-check document ya ilitumia muhuri wa kughushi pale Likoni, Mombasa, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-56,Sheng,Mombasa\n"Tarehe 6 May 2026, nilienda ofisi ya umma katika Embakasi East, Nairobi, kwa ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Bribery,realistic-bribery-21,Kiswahili,Nairobi\n"Pale Likoni, Mombasa on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Nepotism/Favoritism,realistic-nepotism-favoritism-20,Sheng,Mombasa\n"On 17 February 2026, I dealt with the hospital desk in Kilifi North, Kilifi. The office refused to explain why the queue was suspended and gave no complaint channel. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Other,realistic-other-54,English,Kilifi\n"On 14 June 2026, the grant panel chair took part in a decision about a community grant in Machakos Central, Machakos. The officer was a partner in the organisation applying for funds. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Conflict of Interest,realistic-conflict-of-interest-47,English,Machakos\n"On 9 March 2026 in Kisumu Central, Kisumu, the county askari took the goods and demanded KSh 1,000, about KSh 2,500. They said they would be returned only after a payment. I kept the receipt and the message thread. Case note KW-010.",Extortion,realistic-extortion-10,English,Kisumu\n"Tarehe 14 June 2026, mwenyekiti wa jopo la ruzuku alishiriki uamuzi kuhusu alikuwa mshirika katika Machakos Central, Machakos. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Conflict of Interest,realistic-conflict-of-interest-07,Kiswahili,Machakos\n"Pale Machakos Central, Machakos, on 14 June 2026, bodi ya CDF bila kura ya wazi, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-038.",Nepotism/Favoritism,realistic-nepotism-favoritism-38,Mixed,Machakos\n"Tarehe 22 April 2026, afisa wa fedha wa kaunti alishiriki uamuzi kuhusu alikuwa mmiliki binafsi katika Imenti Central, Meru. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Conflict of Interest,realistic-conflict-of-interest-45,Kiswahili,Meru\n"On 14 June 2026 in Machakos Central, Machakos, the police officer held my identification card and demanded KSh 2,500, about KSh 300. They threatening arrest unless I paid in cash. I kept the receipt and the message thread. Case note KW-001.",Extortion,realistic-extortion-01,English,Machakos\n"In Likoni, Mombasa, on 11 August 2026, the procurement committee gave repeated contracts to the chairperson\'s close associate while qualified suppliers were not invited. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-040.",Nepotism/Favoritism,realistic-nepotism-favoritism-40,English,Mombasa\n"Pale Embakasi East, Nairobi on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-025.",Other,realistic-other-25,Sheng,Nairobi\n"In Nakuru East, Nakuru, on 14 June 2026, the CDF board funded the chairman\'s project ahead of equally eligible projects without an open vote. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Nepotism/Favoritism,realistic-nepotism-favoritism-14,English,Nakuru\n"On 22 April 2026 in Machakos Central, Machakos, the gate security supervisor blocked my lorry and demanded KSh 1,500, about KSh 300. They threatening to call a taxing inspection. I kept the receipt and the message thread. Case note KW-031.",Extortion,realistic-extortion-31,English,Machakos\n"Tarehe 6 May 2026, nilishughulika na ofisi ya barabara katika Embakasi East, Nairobi. kwenye barabara yetu lakini hakuna kazi iliyoanza kabla ya muda. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Other,realistic-other-45,Kiswahili,Nairobi\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Imenti Central, Meru, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Embezzlement,realistic-embezzlement-17,Kiswahili,Meru\n"Tender ya usambazaji wa mafuta pale Kilifi North, Kilifi ilikuwa imepangwa. katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Procurement Fraud,realistic-procurement-fraud-56,Sheng,Kilifi\n"On 17 February 2026 in Kilifi North, Kilifi, the public officer threatened to blacklist my supplier account and demanded KSh 5,000, about KSh 5,000. They unless I paid from my personal funds. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Extortion,realistic-extortion-05,English,Kilifi\n"On 17 February 2026 pale Kilifi North, Kilifi, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-051.",Conflict of Interest,realistic-conflict-of-interest-51,Sheng,Kilifi\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-047.",Extortion,realistic-extortion-47,Mixed,Nakuru\n"I checked a police clearance letter relating to the public application in Kilifi North, Kilifi, on 17 February 2026. The document used a fabricated stamp to make an unapproved clearance appear valid. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-08,English,Kilifi\n"On 14 June 2026 pale Machakos Central, Machakos, mwenyekiti wa jopo la ruzuku joined the decision on alikuwa mshirika; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-007.",Conflict of Interest,realistic-conflict-of-interest-07,Mixed,Machakos\n"On 14 June 2026 pale Nakuru East, Nakuru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-017.",Extortion,realistic-extortion-17,Sheng,Nakuru\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za ukarabati wa barabara katika Nakuru East, Nakuru, tangu 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Embezzlement,realistic-embezzlement-05,Kiswahili,Nakuru\n"Ninawasilisha tuhuma za matumizi mabaya ya chakula na blanketi katika Bungoma Central, Bungoma, tangu 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Embezzlement,realistic-embezzlement-40,Kiswahili,Bungoma\n"Kuna doo za fedha za kisima zilipotea pale Machakos Central, Machakos after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 300, nimehifadhi screenshot na receipt. Ref KW-049.",Embezzlement,realistic-embezzlement-49,Sheng,Machakos\n"On 22 April 2026 pale Imenti Central, Meru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-033.",Abuse of Office,realistic-abuse-of-office-33,Sheng,Meru\n"Nili-check document ya kilitiwa sahihi pale Kapseret, Eldoret, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-057.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-57,Sheng,Eldoret\n"On 28 July 2026 in Kisumu Central, Kisumu, the county officer changed a queue order to serve a personal acquaintance first while others were told to wait. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Abuse of Office,realistic-abuse-of-office-24,English,Kisumu\n"Pale Bungoma Central, Bungoma, on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-039.",Nepotism/Favoritism,realistic-nepotism-favoritism-39,Mixed,Bungoma\n"Nili-check document ya kilikuwa na jina lililobadilishwa pale Kisumu Central, Kisumu, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, nimehifadhi screenshot na receipt. Ref KW-013.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-13,Sheng,Kisumu\n"For tender SEC-55 in Kapseret, Eldoret, the procuring entity gave one supplier confidential bid information. The file concerns security services and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Procurement Fraud,realistic-procurement-fraud-15,English,Eldoret\n"Nilikagua document for ilonyesha anwani ya uongo pale Likoni, Mombasa, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeweka details ya time na office number. Case/rekodi KW-006.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-06,Mixed,Mombasa\n"On 14 June 2026 pale Kilifi North, Kilifi, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-019.",Abuse of Office,realistic-abuse-of-office-19,Mixed,Kilifi\n"Tarehe 22 April 2026 katika Nakuru East, Nakuru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Abuse of Office,realistic-abuse-of-office-01,Kiswahili,Nakuru\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Kilifi North, Kilifi, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Bribery,realistic-bribery-50,Kiswahili,Kilifi\n"I am reporting irregular use of KSh 20,000 in Machakos Central, Machakos, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 300 were used to buy items not delivered to the school. I kept the receipt and the message thread. Case note KW-019.",Embezzlement,realistic-embezzlement-19,English,Machakos\n"Pale Kilifi North, Kilifi on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Nepotism/Favoritism,realistic-nepotism-favoritism-12,Sheng,Kilifi\n"On 6 May 2026 pale Kisumu Central, Kisumu, mkaguzi akadai malipo ya faragha mara moja, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Extortion,realistic-extortion-60,Sheng,Kisumu\n"On 6 May 2026, the project director took part in a decision about a construction contract in Kapseret, Eldoret. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-010.",Conflict of Interest,realistic-conflict-of-interest-10,English,Eldoret\n"Nili-check document ya iliandika bidhaa pale Imenti Central, Meru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-42,Sheng,Meru\n"Katika Bungoma Central, Bungoma, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Nepotism/Favoritism,realistic-nepotism-favoritism-09,Kiswahili,Bungoma\n"On 9 March 2026, the public appointment officer took part in a decision about a job assessment in Embakasi East, Nairobi. The officer was the spouse of one shortlisted candidate. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Conflict of Interest,realistic-conflict-of-interest-32,English,Nairobi\n"Pale Likoni, Mombasa on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Nepotism/Favoritism,realistic-nepotism-favoritism-50,Sheng,Mombasa\n"Tender ya huduma za ulinzi pale Embakasi East, Nairobi ilikuwa imepangwa. taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri, around 5000, nimehifadhi screenshot na receipt. Ref KW-047.",Procurement Fraud,realistic-procurement-fraud-47,Sheng,Nairobi\n"Tarehe 17 February 2026, nilienda ofisi ya umma katika Machakos Central, Machakos, kwa utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Bribery,realistic-bribery-06,Kiswahili,Machakos\n"On 22 April 2026 pale Kilifi North, Kilifi, afisa wa fedha wa kaunti aliingia decision ya alikuwa mmiliki binafsi; personally owned, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-021.",Conflict of Interest,realistic-conflict-of-interest-21,Sheng,Kilifi\n"Kwa huduma za ulinzi katika Embakasi East, Nairobi, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Tukio linahusu huduma za ulinzi yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Procurement Fraud,realistic-procurement-fraud-47,Kiswahili,Nairobi\n"On 9 March 2026 pale Kapseret, Eldoret, afisa wa uteuzi wa umma aliingia decision ya alikuwa mwenzi; was the spouse, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Conflict of Interest,realistic-conflict-of-interest-40,Sheng,Eldoret\n"Nimekagua hati kuhusu ilikuwa na sahihi iliyorudishwa katika Garissa Township, Garissa, tarehe 6 May 2026. vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-31,Kiswahili,Garissa\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Kilifi North, Kilifi. kwa fomu ya kawaida bila risiti rasmi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Other,realistic-other-44,Kiswahili,Kilifi\n"In Imenti Central, Meru, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Nepotism/Favoritism,realistic-nepotism-favoritism-26,English,Meru\n"Tarehe 11 August 2026 katika Machakos Central, Machakos, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Extortion,realistic-extortion-51,Kiswahili,Machakos\n"I checked a payment voucher relating to the hospital account in Kapseret, Eldoret, on 6 May 2026. The document carried a duplicate signature for a purchase that has no supplier evidence. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-47,English,Eldoret\n"On 28 July 2026 in Kapseret, Eldoret, the county officer changed a queue order to serve a personal acquaintance first while others were told to wait. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Abuse of Office,realistic-abuse-of-office-08,English,Eldoret\n"I am reporting irregular use of KSh 2,500 in Imenti Central, Meru, after 11 August 2026. Records show that the county market cashier handled weekly market fee collections, and the funds worth KSh 5,000 were missing from the ledger and bank deposit. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Embezzlement,realistic-embezzlement-47,English,Meru\n"Pale Nakuru East, Nakuru on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no action taken, nimeweka details ya time na office number. Case/rekodi KW-006.",Other,realistic-other-06,Mixed,Nakuru\n"Pale Kapseret, Eldoret on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-033.",Other,realistic-other-33,Mixed,Eldoret\n"On 9 March 2026 in Bungoma Central, Bungoma, the county askari took the goods and demanded KSh 1,000, about KSh 20,000. They said they would be returned only after a payment. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Extortion,realistic-extortion-42,English,Bungoma\n"On 22 April 2026 pale Machakos Central, Machakos, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-031.",Extortion,realistic-extortion-31,Sheng,Machakos\n"I checked a meeting minute relating to the committee record in Imenti Central, Meru, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-12,English,Meru\n"Tarehe 3 January 2026 katika Garissa Township, Garissa, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Abuse of Office,realistic-abuse-of-office-42,Kiswahili,Garissa\n"Juzi 9 March 2026 nikaingia pale office ya Embakasi East, Nairobi, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 1000, nimehifadhi screenshot na receipt. Ref KW-051.",Bribery,realistic-bribery-51,Sheng,Nairobi\n"Tarehe 3 January 2026, mjumbe wa bodi ya shule alishiriki uamuzi kuhusu aliwahi kuajiri katika Kapseret, Eldoret. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Conflict of Interest,realistic-conflict-of-interest-30,Kiswahili,Eldoret\n"For tender SCH-204 in Bungoma Central, Bungoma, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-043.",Procurement Fraud,realistic-procurement-fraud-43,English,Bungoma\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Kapseret, Eldoret. kutofanya kazi kwa miezi kadhaa bila maelezo. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Other,realistic-other-43,Kiswahili,Eldoret\n"Juzi 22 April 2026 nikaingia pale office ya Machakos Central, Machakos, for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada 500, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Bribery,realistic-bribery-56,Sheng,Machakos\n"On 28 July 2026 in Kapseret, Eldoret, the station officer refused to release my brother and demanded KSh 2,000, about KSh 20,000. They while warning that the case file could be damaged. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Extortion,realistic-extortion-54,English,Eldoret\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-058.",Conflict of Interest,realistic-conflict-of-interest-58,Mixed,Bungoma\n"Nili-check document ya iliandika bidhaa pale Nakuru East, Nakuru, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-10,Sheng,Nakuru\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-055.",Extortion,realistic-extortion-55,Sheng,Kilifi\n"I checked an academic certificate relating to the appointment file in Bungoma Central, Bungoma, on 9 March 2026. The document contained a changed name and a stamp that the issuing school says is not genuine. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-45,English,Bungoma\n"Juzi 22 April 2026 nikaingia pale office ya Imenti Central, Meru, for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Bribery,realistic-bribery-24,Sheng,Meru\n"Nili-check document ya ilitumia muhuri wa kughushi pale Kilifi North, Kilifi, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-48,Sheng,Kilifi\n"Kwa mifumo ya kompyuta na mtandao katika Bungoma Central, Bungoma, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Tukio linahusu mifumo ya kompyuta na mtandao yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Procurement Fraud,realistic-procurement-fraud-13,Kiswahili,Bungoma\n"Nili-check document ya ilihaririwa pale Kilifi North, Kilifi, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-28,Sheng,Kilifi\n"Nimekagua hati kuhusu ilibadilishwa katika Garissa Township, Garissa, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-51,Kiswahili,Garissa\n"Nili-check document ya ilibadilishwa pale Bungoma Central, Bungoma, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-035.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-35,Sheng,Bungoma\n"On 9 March 2026 pale Garissa Township, Garissa, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-018.",Extortion,realistic-extortion-18,Mixed,Garissa\n"Tarehe 22 April 2026 katika Kilifi North, Kilifi, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Extortion,realistic-extortion-55,Kiswahili,Kilifi\n"Juzi 22 April 2026 nikaingia pale office ya Likoni, Mombasa, for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Bribery,realistic-bribery-48,Sheng,Mombasa\n"Tarehe 28 July 2026, nilishughulika na tovuti ya kaunti katika Bungoma Central, Bungoma. bila sababu iliyochapishwa baada ya wakazi kuomba huduma. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Other,realistic-other-31,Kiswahili,Bungoma\n"On 28 July 2026 pale Garissa Township, Garissa, afisa wa kaunti wengine wakaambiwa wangojee; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Abuse of Office,realistic-abuse-of-office-32,Sheng,Garissa\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Imenti Central, Meru, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Bribery,realistic-bribery-04,Kiswahili,Meru\n"On 28 July 2026 pale Bungoma Central, Bungoma, afisa wa kaunti wengine wakaambiwa wangojee; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-016.",Abuse of Office,realistic-abuse-of-office-16,Mixed,Bungoma\n"Kuna doo za mapato ya ada za soko zilipotea pale Imenti Central, Meru after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 5000, nimehifadhi screenshot na receipt. Ref KW-047.",Embezzlement,realistic-embezzlement-47,Sheng,Meru\n"On 11 August 2026 in Kilifi North, Kilifi, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 5,000. They threatened to make a false inspection fault. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Extortion,realistic-extortion-35,English,Kilifi\n"On 28 July 2026 pale Embakasi East, Nairobi, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Extortion,realistic-extortion-46,Sheng,Nairobi\n"On 9 March 2026 pale Kisumu Central, Kisumu, askari wa kaunti akasema zingerudishwa baada ya malipo, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Extortion,realistic-extortion-50,Sheng,Kisumu\n"On 3 January 2026 in Kapseret, Eldoret, the licensing inspector demanded cash and demanded KSh 800, about KSh 20,000. They and said he would report a compliance breach if I refused. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Extortion,realistic-extortion-24,English,Eldoret\n"On 9 March 2026, I dealt with the water department in Kapseret, Eldoret. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-043.",Other,realistic-other-43,English,Eldoret\n"Kuna doo za fedha za kisima zilipotea pale Imenti Central, Meru after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 5000, nimehifadhi screenshot na receipt. Ref KW-017.",Embezzlement,realistic-embezzlement-17,Sheng,Meru\n"Nimekagua hati kuhusu ilitumia muhuri wa kughushi katika Kilifi North, Kilifi, tarehe 17 February 2026. barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-08,Kiswahili,Kilifi\n"Pale Bungoma Central, Bungoma, on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-029.",Nepotism/Favoritism,realistic-nepotism-favoritism-29,Mixed,Bungoma\n"Tarehe 9 March 2026 katika Garissa Township, Garissa, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Extortion,realistic-extortion-18,Kiswahili,Garissa\n"Nimekagua hati kuhusu kilikuwa na jina lililobadilishwa katika Kisumu Central, Kisumu, tarehe 9 March 2026. cheti cha elimu na muhuri ambao shule inasema si halisi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-53,Kiswahili,Kisumu\n"On 17 February 2026 pale Likoni, Mombasa, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, walidai 300, na screenshot na receipt bado. Case/rekodi KW-013.",Extortion,realistic-extortion-13,Mixed,Mombasa\n"Nilienda pale office ya Kapseret, Eldoret, on 28 July 2026 for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida, 1,000, na screenshot na receipt bado. Case/rekodi KW-039.",Bribery,realistic-bribery-39,Mixed,Eldoret\n"Nilikagua document for iliandika bidhaa pale Kilifi North, Kilifi, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-018.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-18,Mixed,Kilifi\n"On 28 July 2026 in Kapseret, Eldoret, the station officer refused to release my brother and demanded KSh 2,000, about KSh 500. They while warning that the case file could be damaged. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Extortion,realistic-extortion-14,English,Eldoret\n"On 9 March 2026 in Kisumu Central, Kisumu, the county askari took the goods and demanded KSh 1,000, about KSh 500. They said they would be returned only after a payment. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Extortion,realistic-extortion-50,English,Kisumu\n"Pale Nakuru East, Nakuru, on 14 June 2026, bodi ya CDF bila kura ya wazi, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-054.",Nepotism/Favoritism,realistic-nepotism-favoritism-54,Mixed,Nakuru\n"Nili-check document ya ilihaririwa pale Imenti Central, Meru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-12,Sheng,Meru\n"On 11 August 2026 pale Likoni, Mombasa, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-049.",Conflict of Interest,realistic-conflict-of-interest-49,Mixed,Mombasa\n"Nili-check document ya ilihaririwa pale Nakuru East, Nakuru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-20,Sheng,Nakuru\n"On 11 August 2026 in Likoni, Mombasa, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 300. They threatened to make a false inspection fault. I kept the receipt and the message thread. Case note KW-043.",Extortion,realistic-extortion-43,English,Mombasa\n"In Garissa Township, Garissa, on 9 March 2026, the public internship programme selected a local official\'s child for all available placements even though the position was not advertised. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-055.",Nepotism/Favoritism,realistic-nepotism-favoritism-55,English,Garissa\n"In Embakasi East, Nairobi, on 3 January 2026, the appointment board promoted an officer\'s spouse to a senior post despite a lower interview score. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-013.",Nepotism/Favoritism,realistic-nepotism-favoritism-13,English,Nairobi\n"Pale Garissa Township, Garissa on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-027.",Other,realistic-other-27,Sheng,Garissa\n"On 9 March 2026 pale Bungoma Central, Bungoma, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-002.",Extortion,realistic-extortion-02,Mixed,Bungoma\n"Pale Nakuru East, Nakuru, on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-024.",Nepotism/Favoritism,realistic-nepotism-favoritism-24,Mixed,Nakuru\n"Juzi 9 March 2026 nikaingia pale office ya Garissa Township, Garissa, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 1000, nimehifadhi screenshot na receipt. Ref KW-003.",Bribery,realistic-bribery-03,Sheng,Garissa\n"Kwa zabuni ya vifaa vya hospitali katika Imenti Central, Meru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Procurement Fraud,realistic-procurement-fraud-50,Kiswahili,Meru\n"On 17 February 2026 pale Likoni, Mombasa, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-059.",Conflict of Interest,realistic-conflict-of-interest-59,Mixed,Mombasa\n"Nilienda pale office ya Machakos Central, Machakos, on 22 April 2026 for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada, 2,500, nimeweka details ya time na office number. Case/rekodi KW-016.",Bribery,realistic-bribery-16,Mixed,Machakos\n"On 17 February 2026 in Kilifi North, Kilifi, the school principal directed a teacher to use school equipment for a private function during school time. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Abuse of Office,realistic-abuse-of-office-39,English,Kilifi\n"Tarehe 9 March 2026 katika Embakasi East, Nairobi, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Abuse of Office,realistic-abuse-of-office-20,Kiswahili,Nairobi\n"I checked a meeting minute relating to the committee record in Machakos Central, Machakos, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-004.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-04,English,Machakos\n"Tender ya zabuni ya ukarabati wa barabara pale Garissa Township, Garissa ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 1000, nimehifadhi screenshot na receipt. Ref KW-009.",Procurement Fraud,realistic-procurement-fraud-09,Sheng,Garissa\n"On 3 January 2026 in Garissa Township, Garissa, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Abuse of Office,realistic-abuse-of-office-42,English,Garissa\n"I am reporting irregular use of KSh 5,000 in Kapseret, Eldoret, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 20,000 entered receipts that could not be matched to payments. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Embezzlement,realistic-embezzlement-42,English,Eldoret\n"Pale Bungoma Central, Bungoma on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-051.",Other,realistic-other-51,Mixed,Bungoma\n"Katika Kilifi North, Kilifi, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Nepotism/Favoritism,realistic-nepotism-favoritism-12,Kiswahili,Kilifi\n"On 9 March 2026, I went to the public office in Kapseret, Eldoret, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-019.",Bribery,realistic-bribery-19,English,Eldoret\n"In Imenti Central, Meru, on 11 August 2026, the procurement committee gave repeated contracts to the chairperson\'s close associate while qualified suppliers were not invited. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Nepotism/Favoritism,realistic-nepotism-favoritism-56,English,Meru\n"On 28 July 2026 in Kisumu Central, Kisumu, the station officer refused to release my brother and demanded KSh 2,000, about KSh 20,000. They while warning that the case file could be damaged. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Extortion,realistic-extortion-30,English,Kisumu\n"Pale Kilifi North, Kilifi on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Other,realistic-other-04,Sheng,Kilifi\n"Pale Machakos Central, Machakos on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Other,realistic-other-10,Sheng,Machakos\n"Tarehe 9 March 2026, afisa wa uteuzi wa umma alishiriki uamuzi kuhusu alikuwa mwenzi katika Bungoma Central, Bungoma. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Conflict of Interest,realistic-conflict-of-interest-48,Kiswahili,Bungoma\n"On 3 January 2026 pale Kapseret, Eldoret, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-024.",Extortion,realistic-extortion-24,Sheng,Eldoret\n"In Embakasi East, Nairobi, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Nepotism/Favoritism,realistic-nepotism-favoritism-03,English,Nairobi\n"Pale Imenti Central, Meru on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-058.",Other,realistic-other-58,Mixed,Meru\n"On 28 July 2026 in Bungoma Central, Bungoma, the county officer changed a queue order to serve a personal acquaintance first while others were told to wait. This was not a routine decision. I kept the receipt and the message thread. Case note KW-016.",Abuse of Office,realistic-abuse-of-office-16,English,Bungoma\n"On 14 June 2026 pale Nakuru East, Nakuru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-051.",Abuse of Office,realistic-abuse-of-office-51,Mixed,Nakuru\n"Kwa huduma za ulinzi katika Kapseret, Eldoret, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Tukio linahusu huduma za ulinzi yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Procurement Fraud,realistic-procurement-fraud-55,Kiswahili,Eldoret\n"Nimekagua hati kuhusu ilihaririwa katika Nakuru East, Nakuru, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-20,Kiswahili,Nakuru\n"On 9 March 2026, I dealt with the water department in Kisumu Central, Kisumu. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-019.",Other,realistic-other-19,English,Kisumu\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Imenti Central, Meru. kwa fomu ya kawaida bila risiti rasmi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Other,realistic-other-28,Kiswahili,Meru\n"Nilikagua document for ilonyesha anwani ya uongo pale Imenti Central, Meru, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeweka details ya time na office number. Case/rekodi KW-022.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-22,Mixed,Meru\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Imenti Central, Meru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-010.",Procurement Fraud,realistic-procurement-fraud-10,Mixed,Meru\n"On 9 March 2026, I dealt with the water department in Kisumu Central, Kisumu. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Other,realistic-other-59,English,Kisumu\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-001.",Abuse of Office,realistic-abuse-of-office-01,Mixed,Nakuru\n"Nimekagua hati kuhusu ilikuwa na sahihi iliyorudishwa katika Bungoma Central, Bungoma, tarehe 6 May 2026. vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-15,Kiswahili,Bungoma\n"In Embakasi East, Nairobi, on 3 January 2026, the appointment board promoted an officer\'s spouse to a senior post despite a lower interview score. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Nepotism/Favoritism,realistic-nepotism-favoritism-53,English,Nairobi\n"Pale Embakasi East, Nairobi on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-055.",Other,realistic-other-55,Sheng,Nairobi\n"Pale Kilifi North, Kilifi, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-012.",Nepotism/Favoritism,realistic-nepotism-favoritism-12,Mixed,Kilifi\n"Pale Likoni, Mombasa on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no action taken, nimeweka details ya time na office number. Case/rekodi KW-022.",Other,realistic-other-22,Mixed,Mombasa\n"On 6 May 2026, the project director took part in a decision about a construction contract in Embakasi East, Nairobi. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Conflict of Interest,realistic-conflict-of-interest-42,English,Nairobi\n"Pale Embakasi East, Nairobi on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no action taken, na screenshot na receipt bado. Case/rekodi KW-015.",Other,realistic-other-15,Mixed,Nairobi\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Kapseret, Eldoret, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Bribery,realistic-bribery-49,Kiswahili,Eldoret\n"On 3 January 2026, the school board member took part in a decision about a tender for uniforms in Bungoma Central, Bungoma. The officer employed the bidder as a private consultant. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Conflict of Interest,realistic-conflict-of-interest-38,English,Bungoma\n"Nimekagua hati kuhusu ilibadilishwa katika Garissa Township, Garissa, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-11,Kiswahili,Garissa\n"For tender SEC-55 in Embakasi East, Nairobi, the procuring entity gave one supplier confidential bid information. The file concerns security services and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Procurement Fraud,realistic-procurement-fraud-47,English,Nairobi\n"Pale Embakasi East, Nairobi on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-005.",Other,realistic-other-05,Sheng,Nairobi\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Nakuru East, Nakuru, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Embezzlement,realistic-embezzlement-35,Kiswahili,Nakuru\n"Pale Kisumu Central, Kisumu on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-019.",Other,realistic-other-19,Sheng,Kisumu\n"Pale Bungoma Central, Bungoma on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-011.",Other,realistic-other-11,Mixed,Bungoma\n"Nili-check document ya ilibadilishwa pale Kapseret, Eldoret, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-027.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-27,Sheng,Eldoret\n"Nilikagua document for ilibadilishwa pale Bungoma Central, Bungoma, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-035.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-35,Mixed,Bungoma\n"On 28 July 2026 pale Bungoma Central, Bungoma, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Extortion,realistic-extortion-22,Sheng,Bungoma\n"On 9 March 2026 pale Kisumu Central, Kisumu, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-004.",Abuse of Office,realistic-abuse-of-office-04,Mixed,Kisumu\n"On 22 April 2026 pale Likoni, Mombasa, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-023.",Extortion,realistic-extortion-23,Mixed,Mombasa\n"Pale Nakuru East, Nakuru on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Other,realistic-other-36,Sheng,Nakuru\n"On 14 June 2026 pale Machakos Central, Machakos, mwenyekiti wa jopo la ruzuku aliingia decision ya alikuwa mshirika; was a partner, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-007.",Conflict of Interest,realistic-conflict-of-interest-07,Sheng,Machakos\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Likoni, Mombasa. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Conflict of Interest,realistic-conflict-of-interest-09,English,Mombasa\n"Tarehe 22 April 2026, afisa wa fedha wa kaunti alishiriki uamuzi kuhusu alikuwa mmiliki binafsi katika Kilifi North, Kilifi. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Conflict of Interest,realistic-conflict-of-interest-21,Kiswahili,Kilifi\n"Nilikagua document for kilikuwa na jina lililobadilishwa pale Kapseret, Eldoret, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, na screenshot na receipt bado. Case/rekodi KW-037.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-37,Mixed,Eldoret\n"Tarehe 17 February 2026, nilishughulika na dirisha la hospitali katika Nakuru East, Nakuru. kwa nini foleni ilisitishwa bila kutoa njia ya kulalamika. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Other,realistic-other-06,Kiswahili,Nakuru\n"Katika Machakos Central, Machakos, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Nepotism/Favoritism,realistic-nepotism-favoritism-58,Kiswahili,Machakos\n"On 9 March 2026 pale Bungoma Central, Bungoma, askari wa kaunti akasema zingerudishwa baada ya malipo, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Extortion,realistic-extortion-42,Sheng,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za ukarabati wa barabara katika Likoni, Mombasa, tangu 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Embezzlement,realistic-embezzlement-21,Kiswahili,Mombasa\n"On 11 August 2026 in Machakos Central, Machakos, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 1,000. They threatened to make a false inspection fault. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Extortion,realistic-extortion-51,English,Machakos\n"In Kisumu Central, Kisumu, on 9 March 2026, the public internship programme selected a local official\'s child for all available placements even though the position was not advertised. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-007.",Nepotism/Favoritism,realistic-nepotism-favoritism-07,English,Kisumu\n"On 6 May 2026, I went to the public office in Kisumu Central, Kisumu, for permission to continue trading. The the county askari offered to overlook the stall issue for a private payment; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Bribery,realistic-bribery-45,English,Kisumu\n"Pale Kisumu Central, Kisumu on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-039.",Other,realistic-other-39,Sheng,Kisumu\n"Kwa zabuni ya ukarabati wa barabara katika Garissa Township, Garissa, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Procurement Fraud,realistic-procurement-fraud-09,Kiswahili,Garissa\n"On 3 January 2026, the school board member took part in a decision about a tender for uniforms in Embakasi East, Nairobi. The officer employed the bidder as a private consultant. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-022.",Conflict of Interest,realistic-conflict-of-interest-22,English,Nairobi\n"Pale Bungoma Central, Bungoma, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-049.",Nepotism/Favoritism,realistic-nepotism-favoritism-49,Mixed,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Likoni, Mombasa, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Embezzlement,realistic-embezzlement-11,Kiswahili,Mombasa\n"On 3 January 2026 pale Kisumu Central, Kisumu, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Extortion,realistic-extortion-40,Sheng,Kisumu\n"Pale Garissa Township, Garissa on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-007.",Other,realistic-other-07,Sheng,Garissa\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Kisumu Central, Kisumu. kutofanya kazi kwa miezi kadhaa bila maelezo. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Other,realistic-other-19,Kiswahili,Kisumu\n"On 14 June 2026 pale Kilifi North, Kilifi, mwenyekiti wa jopo la ruzuku joined the decision on alikuwa mshirika; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-031.",Conflict of Interest,realistic-conflict-of-interest-31,Mixed,Kilifi\n"Tarehe 28 July 2026, nilienda ofisi ya umma katika Bungoma Central, Bungoma, kwa uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Bribery,realistic-bribery-47,Kiswahili,Bungoma\n"On 3 January 2026 pale Embakasi East, Nairobi, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Extortion,realistic-extortion-16,Sheng,Nairobi\n"On 14 June 2026, I dealt with the service desk in Likoni, Mombasa. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Other,realistic-other-42,English,Mombasa\n"On 3 January 2026, the school board member took part in a decision about a tender for uniforms in Kisumu Central, Kisumu. The officer employed the bidder as a private consultant. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Conflict of Interest,realistic-conflict-of-interest-06,English,Kisumu\n"I am reporting irregular use of KSh 2,500 in Kilifi North, Kilifi, after 11 August 2026. Records show that the county market cashier handled weekly market fee collections, and the funds worth KSh 5,000 were missing from the ledger and bank deposit. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Embezzlement,realistic-embezzlement-23,English,Kilifi\n"On 9 March 2026, I went to the public office in Bungoma Central, Bungoma, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Bribery,realistic-bribery-27,English,Bungoma\n"Katika Kisumu Central, Kisumu, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Nepotism/Favoritism,realistic-nepotism-favoritism-27,Kiswahili,Kisumu\n"Kuna doo za fedha za wanafunzi zilipotea pale Kisumu Central, Kisumu after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Embezzlement,realistic-embezzlement-58,Sheng,Kisumu\n"On 6 May 2026, I went to the public office in Embakasi East, Nairobi, for permission to continue trading. The the county askari offered to overlook the stall issue for a private payment; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Bribery,realistic-bribery-21,English,Nairobi\n"Nili-check document ya iliandika bidhaa pale Kilifi North, Kilifi, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-58,Sheng,Kilifi\n"On 3 January 2026 in Kapseret, Eldoret, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Abuse of Office,realistic-abuse-of-office-18,English,Eldoret\n"Tarehe 22 April 2026 katika Likoni, Mombasa, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Abuse of Office,realistic-abuse-of-office-17,Kiswahili,Mombasa\n"On 9 March 2026, I dealt with the water department in Bungoma Central, Bungoma. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Other,realistic-other-51,English,Bungoma\n"Pale Kilifi North, Kilifi on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Nepotism/Favoritism,realistic-nepotism-favoritism-02,Sheng,Kilifi\n"On 6 May 2026 pale Kisumu Central, Kisumu, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Conflict of Interest,realistic-conflict-of-interest-26,Sheng,Kisumu\n"Nilikagua document for ilikuwa na sahihi iliyorudishwa pale Kapseret, Eldoret, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, na screenshot na receipt bado. Case/rekodi KW-047.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-47,Mixed,Eldoret\n"Pale Machakos Central, Machakos on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-010.",Other,realistic-other-10,Mixed,Machakos\n"Tarehe 22 April 2026, nilienda ofisi ya umma katika Machakos Central, Machakos, kwa abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Bribery,realistic-bribery-16,Kiswahili,Machakos\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Nakuru East, Nakuru. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Other,realistic-other-26,Kiswahili,Nakuru\n"On 17 February 2026 in Nakuru East, Nakuru, the public officer threatened to blacklist my supplier account and demanded KSh 5,000, about KSh 300. They unless I paid from my personal funds. I kept the receipt and the message thread. Case note KW-037.",Extortion,realistic-extortion-37,English,Nakuru\n"I am reporting irregular use of KSh 20,000 in Machakos Central, Machakos, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 300 was withdrawn but the borehole was never completed. I kept the receipt and the message thread. Case note KW-049.",Embezzlement,realistic-embezzlement-49,English,Machakos\n"Tarehe 9 March 2026, afisa wa uteuzi wa umma alishiriki uamuzi kuhusu alikuwa mwenzi katika Kisumu Central, Kisumu. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Conflict of Interest,realistic-conflict-of-interest-16,Kiswahili,Kisumu\n"Pale Garissa Township, Garissa on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-005.",Nepotism/Favoritism,realistic-nepotism-favoritism-05,Sheng,Garissa\n"In Embakasi East, Nairobi, on 9 March 2026, the public internship programme selected a local official\'s child for all available placements even though the position was not advertised. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Nepotism/Favoritism,realistic-nepotism-favoritism-23,English,Nairobi\n"Tender ya ukodishaji wa magari pale Machakos Central, Machakos ilikuwa imepangwa. kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Procurement Fraud,realistic-procurement-fraud-22,Sheng,Machakos\n"Nimekagua hati kuhusu iliandika bidhaa katika Kilifi North, Kilifi, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-18,Kiswahili,Kilifi\n"I checked a delivery invoice relating to the procurement file in Nakuru East, Nakuru, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-50,English,Nakuru\n"I checked a delivery invoice relating to the procurement file in Likoni, Mombasa, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-26,English,Mombasa\n"On 11 August 2026 pale Machakos Central, Machakos, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-057.",Conflict of Interest,realistic-conflict-of-interest-57,Sheng,Machakos\n"Kwa zabuni ya ukarabati wa barabara katika Bungoma Central, Bungoma, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Procurement Fraud,realistic-procurement-fraud-33,Kiswahili,Bungoma\n"Tarehe 28 July 2026 katika Kisumu Central, Kisumu, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Extortion,realistic-extortion-30,Kiswahili,Kisumu\n"Nilienda pale office ya Nakuru East, Nakuru, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 20,000, nimeweka details ya time na office number. Case/rekodi KW-012.",Bribery,realistic-bribery-12,Mixed,Nakuru\n"On 3 January 2026 in Garissa Township, Garissa, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Abuse of Office,realistic-abuse-of-office-02,English,Garissa\n"On 6 May 2026 pale Kisumu Central, Kisumu, afisa wa rekodi akatishia kuondoa rekodi ya ombi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Abuse of Office,realistic-abuse-of-office-14,Sheng,Kisumu\n"Kuna doo za pesa za dawa zilipotea pale Kisumu Central, Kisumu after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Embezzlement,realistic-embezzlement-28,Sheng,Kisumu\n"On 28 July 2026, I went to the public office in Bungoma Central, Bungoma, for printing an identification document. The the Huduma Centre clerk asked for a personal payment to speed up the ordinary service; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-007.",Bribery,realistic-bribery-07,English,Bungoma\n"Pale Imenti Central, Meru on 14 June 2026, bodi ya CDF bila kura ya wazi; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Nepotism/Favoritism,realistic-nepotism-favoritism-06,Sheng,Meru\n"On 6 May 2026 pale Embakasi East, Nairobi, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-042.",Conflict of Interest,realistic-conflict-of-interest-42,Mixed,Nairobi\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa kaunti wengine wakaambiwa wangojee; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-048.",Abuse of Office,realistic-abuse-of-office-48,Mixed,Eldoret\n"On 22 April 2026 pale Machakos Central, Machakos, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, walidai 300, na screenshot na receipt bado. Case/rekodi KW-031.",Extortion,realistic-extortion-31,Mixed,Machakos\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Bungoma Central, Bungoma. bila taarifa wala haikutangaza wakati huduma ingerejea. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Other,realistic-other-01,Kiswahili,Bungoma\n"Juzi 17 February 2026 nikaingia pale office ya Kilifi North, Kilifi, for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Bribery,realistic-bribery-30,Sheng,Kilifi\n"On 14 June 2026 pale Nakuru East, Nakuru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-017.",Extortion,realistic-extortion-17,Mixed,Nakuru\n"On 3 January 2026 pale Garissa Township, Garissa, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Abuse of Office,realistic-abuse-of-office-42,Sheng,Garissa\n"On 28 July 2026 pale Kisumu Central, Kisumu, afisa wa kaunti wengine wakaambiwa wangojee; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-024.",Abuse of Office,realistic-abuse-of-office-24,Mixed,Kisumu\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Kisumu Central, Kisumu, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Embezzlement,realistic-embezzlement-58,Kiswahili,Kisumu\n"Ninawasilisha tuhuma za matumizi mabaya ya chakula na blanketi katika Kapseret, Eldoret, tangu 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Embezzlement,realistic-embezzlement-32,Kiswahili,Eldoret\n"On 3 January 2026 pale Bungoma Central, Bungoma, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Abuse of Office,realistic-abuse-of-office-26,Sheng,Bungoma\n"Nimekagua hati kuhusu ilonyesha anwani ya uongo katika Nakuru East, Nakuru, tarehe 11 August 2026. leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-30,Kiswahili,Nakuru\n"Kwa usambazaji wa mafuta katika Likoni, Mombasa, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Tukio linahusu usambazaji wa mafuta yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Procurement Fraud,realistic-procurement-fraud-24,Kiswahili,Mombasa\n"On 3 January 2026 pale Kisumu Central, Kisumu, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-040.",Extortion,realistic-extortion-40,Mixed,Kisumu\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Likoni, Mombasa. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Conflict of Interest,realistic-conflict-of-interest-09,Kiswahili,Mombasa\n"Kwa zabuni ya vifaa vya hospitali katika Machakos Central, Machakos, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Procurement Fraud,realistic-procurement-fraud-42,Kiswahili,Machakos\n"Kuna doo za chakula na blanketi zilipotea pale Kisumu Central, Kisumu after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Embezzlement,realistic-embezzlement-48,Sheng,Kisumu\n"Nimekagua hati kuhusu kilitiwa sahihi katika Embakasi East, Nairobi, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-49,Kiswahili,Nairobi\n"Tender ya zabuni ya ukarabati wa barabara pale Kisumu Central, Kisumu ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 5000, nimehifadhi screenshot na receipt. Ref KW-041.",Procurement Fraud,realistic-procurement-fraud-41,Sheng,Kisumu\n"Pale Garissa Township, Garissa on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-017.",Other,realistic-other-17,Sheng,Garissa\n"On 11 August 2026 pale Likoni, Mombasa, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-043.",Extortion,realistic-extortion-43,Sheng,Mombasa\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Machakos Central, Machakos, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Embezzlement,realistic-embezzlement-19,Kiswahili,Machakos\n"Tarehe 6 May 2026, nilienda ofisi ya umma katika Garissa Township, Garissa, kwa ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Bribery,realistic-bribery-53,Kiswahili,Garissa\n"On 6 May 2026 in Bungoma Central, Bungoma, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 20,000. They and demanded an immediate private payment. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Extortion,realistic-extortion-12,English,Bungoma\n"I am reporting irregular use of KSh 5,000 in Kisumu Central, Kisumu, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 2,500 entered receipts that could not be matched to payments. I kept the receipt and the message thread. Case note KW-058.",Embezzlement,realistic-embezzlement-58,English,Kisumu\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa kaunti wengine wakaambiwa wangojee; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Abuse of Office,realistic-abuse-of-office-48,Sheng,Eldoret\n"On 9 March 2026 in Embakasi East, Nairobi, the county askari took the goods and demanded KSh 1,000, about KSh 500. They said they would be returned only after a payment. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Extortion,realistic-extortion-26,English,Nairobi\n"Juzi 3 January 2026 nikaingia pale office ya Garissa Township, Garissa, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 1000, nimehifadhi screenshot na receipt. Ref KW-033.",Bribery,realistic-bribery-33,Sheng,Garissa\n"I checked a delivery invoice relating to the procurement file in Imenti Central, Meru, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-02,English,Meru\n"Juzi 14 June 2026 nikaingia pale office ya Kilifi North, Kilifi, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 500, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Bribery,realistic-bribery-50,Sheng,Kilifi\n"Katika Nakuru East, Nakuru, tarehe 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Nepotism/Favoritism,realistic-nepotism-favoritism-24,Kiswahili,Nakuru\n"On 14 June 2026 in Imenti Central, Meru, the police officer held my identification card and demanded KSh 2,500, about KSh 300. They threatening arrest unless I paid in cash. I kept the receipt and the message thread. Case note KW-049.",Extortion,realistic-extortion-49,English,Meru\n"I checked a business permit relating to the revenue office in Machakos Central, Machakos, on 11 August 2026. The document showed a false address and a permit number not in the official register. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-54,English,Machakos\n"On 28 July 2026 pale Kapseret, Eldoret, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Conflict of Interest,realistic-conflict-of-interest-20,Sheng,Eldoret\n"Pale Machakos Central, Machakos, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-018.",Nepotism/Favoritism,realistic-nepotism-favoritism-18,Mixed,Machakos\n"Pale Embakasi East, Nairobi, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-033.",Nepotism/Favoritism,realistic-nepotism-favoritism-33,Mixed,Nairobi\n"Pale Machakos Central, Machakos, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-058.",Nepotism/Favoritism,realistic-nepotism-favoritism-58,Mixed,Machakos\n"Tender ya ukodishaji wa magari pale Imenti Central, Meru ilikuwa imepangwa. kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Procurement Fraud,realistic-procurement-fraud-30,Sheng,Meru\n"On 9 March 2026, I went to the public office in Garissa Township, Garissa, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-043.",Bribery,realistic-bribery-43,English,Garissa\n"On 14 June 2026 pale Machakos Central, Machakos, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-041.",Extortion,realistic-extortion-41,Sheng,Machakos\n"Kuna doo za fedha za chakula zilipotea pale Machakos Central, Machakos after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 300, nimehifadhi screenshot na receipt. Ref KW-019.",Embezzlement,realistic-embezzlement-19,Sheng,Machakos\n"On 17 February 2026 in Machakos Central, Machakos, the school principal directed a teacher to use school equipment for a private function during school time. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Abuse of Office,realistic-abuse-of-office-15,English,Machakos\n"Pale Nakuru East, Nakuru on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Other,realistic-other-26,Sheng,Nakuru\n"Pale Imenti Central, Meru on 14 June 2026, bodi ya CDF bila kura ya wazi; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Nepotism/Favoritism,realistic-nepotism-favoritism-46,Sheng,Meru\n"I am reporting irregular use of KSh 20,000 in Imenti Central, Meru, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 5,000 was withdrawn but the borehole was never completed. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Embezzlement,realistic-embezzlement-17,English,Meru\n"Pale Machakos Central, Machakos on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no action taken, nimeweka details ya time na office number. Case/rekodi KW-040.",Other,realistic-other-40,Mixed,Machakos\n"On 9 March 2026, I dealt with the water department in Bungoma Central, Bungoma. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Other,realistic-other-11,English,Bungoma\n"Pale Kisumu Central, Kisumu, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-027.",Nepotism/Favoritism,realistic-nepotism-favoritism-27,Mixed,Kisumu\n"Nimekagua hati kuhusu kilitiwa sahihi katika Kisumu Central, Kisumu, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-33,Kiswahili,Kisumu\n"On 3 January 2026, I dealt with the public office in Kisumu Central, Kisumu. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-049.",Other,realistic-other-49,English,Kisumu\n"On 28 July 2026 pale Bungoma Central, Bungoma, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-028.",Conflict of Interest,realistic-conflict-of-interest-28,Mixed,Bungoma\n"Kuna money misuse kwa fedha za wanafunzi pale Bungoma Central, Bungoma, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 2,500, nimeweka details ya time na office number. Case/rekodi KW-010.",Embezzlement,realistic-embezzlement-10,Mixed,Bungoma\n"Katika Machakos Central, Machakos, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Nepotism/Favoritism,realistic-nepotism-favoritism-28,Kiswahili,Machakos\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Garissa Township, Garissa. bila taarifa wala haikutangaza wakati huduma ingerejea. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Other,realistic-other-57,Kiswahili,Garissa\n"I am reporting irregular use of KSh 2,500 in Nakuru East, Nakuru, after 11 August 2026. Records show that the county market cashier handled weekly market fee collections, and the funds worth KSh 300 were missing from the ledger and bank deposit. I kept the receipt and the message thread. Case note KW-055.",Embezzlement,realistic-embezzlement-55,English,Nakuru\n"I am reporting irregular use of KSh 5,000 in Kilifi North, Kilifi, after 14 June 2026. Records show that the public works foreman handled road-repair funds, and the funds worth KSh 5,000 were spent while the damaged section remained unfinished. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Embezzlement,realistic-embezzlement-53,English,Kilifi\n"On 28 July 2026 pale Embakasi East, Nairobi, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-006.",Extortion,realistic-extortion-06,Mixed,Nairobi\n"On 14 June 2026 pale Machakos Central, Machakos, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-001.",Extortion,realistic-extortion-01,Sheng,Machakos\n"On 14 June 2026, I went to the public office in Nakuru East, Nakuru, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Bribery,realistic-bribery-42,English,Nakuru\n"On 6 May 2026, I went to the public office in Kisumu Central, Kisumu, for permission to continue trading. The the county askari offered to overlook the stall issue for a private payment; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Bribery,realistic-bribery-05,English,Kisumu\n"On 6 May 2026 in Embakasi East, Nairobi, the records officer refused access to my own file despite an approved request and threatened to remove the request record. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Abuse of Office,realistic-abuse-of-office-30,English,Nairobi\n"I checked a business permit relating to the revenue office in Kilifi North, Kilifi, on 11 August 2026. The document showed a false address and a permit number not in the official register. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-38,English,Kilifi\n"Pale Imenti Central, Meru, on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-016.",Nepotism/Favoritism,realistic-nepotism-favoritism-16,Mixed,Meru\n"Tarehe 17 February 2026, nilishughulika na dirisha la hospitali katika Imenti Central, Meru. kwa nini foleni ilisitishwa bila kutoa njia ya kulalamika. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Other,realistic-other-38,Kiswahili,Meru\n"Tarehe 14 June 2026 katika Kilifi North, Kilifi, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Abuse of Office,realistic-abuse-of-office-19,Kiswahili,Kilifi\n"Pale Likoni, Mombasa on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no action taken, nimeweka details ya time na office number. Case/rekodi KW-032.",Other,realistic-other-32,Mixed,Mombasa\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Machakos Central, Machakos, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Embezzlement,realistic-embezzlement-49,Kiswahili,Machakos\n"On 9 March 2026 pale Kisumu Central, Kisumu, afisa wa uteuzi wa umma joined the decision on alikuwa mwenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-016.",Conflict of Interest,realistic-conflict-of-interest-16,Mixed,Kisumu\n"Nilienda pale office ya Kisumu Central, Kisumu, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 300, na screenshot na receipt bado. Case/rekodi KW-025.",Bribery,realistic-bribery-25,Mixed,Kisumu\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Nakuru East, Nakuru, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Bribery,realistic-bribery-12,Kiswahili,Nakuru\n"Pale Kilifi North, Kilifi on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Other,realistic-other-34,Sheng,Kilifi\n"Tarehe 6 May 2026, nilishughulika na ofisi ya barabara katika Kisumu Central, Kisumu. kwenye barabara yetu lakini hakuna kazi iliyoanza kabla ya muda. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Other,realistic-other-29,Kiswahili,Kisumu\n"On 6 May 2026 in Garissa Township, Garissa, the records officer refused access to my own file despite an approved request and threatened to remove the request record. This was not a routine decision. I kept the receipt and the message thread. Case note KW-022.",Abuse of Office,realistic-abuse-of-office-22,English,Garissa\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Garissa Township, Garissa, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-009.",Procurement Fraud,realistic-procurement-fraud-09,Mixed,Garissa\n"Katika Machakos Central, Machakos, tarehe 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Nepotism/Favoritism,realistic-nepotism-favoritism-08,Kiswahili,Machakos\n"Kuna doo za fedha za ukarabati wa barabara zilipotea pale Kilifi North, Kilifi after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 5000, nimehifadhi screenshot na receipt. Ref KW-053.",Embezzlement,realistic-embezzlement-53,Sheng,Kilifi\n"Nili-check document ya ilibadilishwa pale Kisumu Central, Kisumu, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-003.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-03,Sheng,Kisumu\n"Nili-check document ya kilikuwa na jina lililobadilishwa pale Kisumu Central, Kisumu, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, nimehifadhi screenshot na receipt. Ref KW-053.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-53,Sheng,Kisumu\n"On 3 January 2026 pale Embakasi East, Nairobi, mjumbe wa bodi ya shule joined the decision on aliwahi kuajiri; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-022.",Conflict of Interest,realistic-conflict-of-interest-22,Mixed,Nairobi\n"On 17 February 2026 pale Machakos Central, Machakos, mkuu wa shule wakati wa shule; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-015.",Abuse of Office,realistic-abuse-of-office-15,Sheng,Machakos\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkaguzi akadai malipo ya faragha mara moja, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Extortion,realistic-extortion-12,Sheng,Bungoma\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Conflict of Interest,realistic-conflict-of-interest-18,Sheng,Bungoma\n"Katika Garissa Township, Garissa, tarehe 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Nepotism/Favoritism,realistic-nepotism-favoritism-15,Kiswahili,Garissa\n"Tender ya huduma za ulinzi pale Bungoma Central, Bungoma ilikuwa imepangwa. taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri, around 5000, nimehifadhi screenshot na receipt. Ref KW-023.",Procurement Fraud,realistic-procurement-fraud-23,Sheng,Bungoma\n"On 14 June 2026 pale Nakuru East, Nakuru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-011.",Abuse of Office,realistic-abuse-of-office-11,Sheng,Nakuru\n"Pale Garissa Township, Garissa on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no action taken, na screenshot na receipt bado. Case/rekodi KW-037.",Other,realistic-other-37,Mixed,Garissa\n"On 11 August 2026 pale Nakuru East, Nakuru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-021.",Abuse of Office,realistic-abuse-of-office-21,Sheng,Nakuru\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Embakasi East, Nairobi, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Bribery,realistic-bribery-51,Kiswahili,Nairobi\n"On 14 June 2026 pale Likoni, Mombasa, mwenyekiti wa jopo la ruzuku aliingia decision ya alikuwa mshirika; was a partner, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-039.",Conflict of Interest,realistic-conflict-of-interest-39,Sheng,Mombasa\n"On 22 April 2026, the county finance officer took part in a decision about a lease decision in Nakuru East, Nakuru. The officer personally owned the land being leased to the public entity. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Conflict of Interest,realistic-conflict-of-interest-53,English,Nakuru\n"On 14 June 2026, I went to the public office in Kilifi North, Kilifi, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Bribery,realistic-bribery-50,English,Kilifi\n"Ninawasilisha tuhuma za matumizi mabaya ya mapato ya ada za soko katika Nakuru East, Nakuru, tangu 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Embezzlement,realistic-embezzlement-15,Kiswahili,Nakuru\n"Kuna money misuse kwa pesa ya mradi wa vijana pale Bungoma Central, Bungoma, after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 20,000, nimeweka details ya time na office number. Case/rekodi KW-030.",Embezzlement,realistic-embezzlement-30,Mixed,Bungoma\n"Juzi 14 June 2026 nikaingia pale office ya Machakos Central, Machakos, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 500, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Bribery,realistic-bribery-26,Sheng,Machakos\n"Nilikagua document for ilikuwa na sahihi iliyorudishwa pale Kisumu Central, Kisumu, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, na screenshot na receipt bado. Case/rekodi KW-023.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-23,Mixed,Kisumu\n"Pale Nakuru East, Nakuru, on 14 June 2026, bodi ya CDF bila kura ya wazi, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-014.",Nepotism/Favoritism,realistic-nepotism-favoritism-14,Mixed,Nakuru\n"Nilikagua document for kilikuwa na jina lililobadilishwa pale Kisumu Central, Kisumu, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, na screenshot na receipt bado. Case/rekodi KW-053.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-53,Mixed,Kisumu\n"On 9 March 2026, I went to the public office in Embakasi East, Nairobi, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Bribery,realistic-bribery-51,English,Nairobi\n"Tender ya zabuni ya vifaa vya hospitali pale Machakos Central, Machakos ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Procurement Fraud,realistic-procurement-fraud-02,Sheng,Machakos\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Nakuru East, Nakuru. kwa fomu ya kawaida bila risiti rasmi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Other,realistic-other-36,Kiswahili,Nakuru\n"Nilienda pale office ya Imenti Central, Meru, on 17 February 2026 for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni, 20,000, nimeweka details ya time na office number. Case/rekodi KW-054.",Bribery,realistic-bribery-54,Mixed,Meru\n"On 22 April 2026 in Nakuru East, Nakuru, the gate security supervisor blocked my lorry and demanded KSh 1,500, about KSh 300. They threatening to call a taxing inspection. I kept the receipt and the message thread. Case note KW-007.",Extortion,realistic-extortion-07,English,Nakuru\n"Nili-check document ya ilikuwa na sahihi iliyorudishwa pale Garissa Township, Garissa, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, nimehifadhi screenshot na receipt. Ref KW-031.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-31,Sheng,Garissa\n"Nilikagua document for ilikuwa na sahihi iliyorudishwa pale Garissa Township, Garissa, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, na screenshot na receipt bado. Case/rekodi KW-031.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-31,Mixed,Garissa\n"On 22 April 2026 pale Likoni, Mombasa, afisa wa fedha wa kaunti aliingia decision ya alikuwa mmiliki binafsi; personally owned, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-029.",Conflict of Interest,realistic-conflict-of-interest-29,Sheng,Mombasa\n"Nimekagua hati kuhusu ilikuwa na sahihi iliyorudishwa katika Kisumu Central, Kisumu, tarehe 6 May 2026. vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-23,Kiswahili,Kisumu\n"Nimekagua hati kuhusu ilonyesha anwani ya uongo katika Likoni, Mombasa, tarehe 11 August 2026. leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-06,Kiswahili,Mombasa\n"I am reporting irregular use of KSh 5,000 in Kapseret, Eldoret, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 500 entered receipts that could not be matched to payments. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Embezzlement,realistic-embezzlement-02,English,Eldoret\n"Tarehe 17 February 2026 katika Machakos Central, Machakos, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Extortion,realistic-extortion-21,Kiswahili,Machakos\n"For tender SCH-204 in Garissa Township, Garissa, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Procurement Fraud,realistic-procurement-fraud-59,English,Garissa\n"Tender ya usambazaji wa mafuta pale Nakuru East, Nakuru ilikuwa imepangwa. katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Procurement Fraud,realistic-procurement-fraud-48,Sheng,Nakuru\n"In Kapseret, Eldoret, on 3 January 2026, the appointment board promoted an officer\'s spouse to a senior post despite a lower interview score. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Nepotism/Favoritism,realistic-nepotism-favoritism-21,English,Eldoret\n"Tarehe 9 March 2026 katika Kisumu Central, Kisumu, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Extortion,realistic-extortion-50,Kiswahili,Kisumu\n"Nilienda pale office ya Nakuru East, Nakuru, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 2,500, nimeweka details ya time na office number. Case/rekodi KW-052.",Bribery,realistic-bribery-52,Mixed,Nakuru\n"Kwa ukodishaji wa magari katika Kilifi North, Kilifi, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Tukio linahusu ukodishaji wa magari yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Procurement Fraud,realistic-procurement-fraud-46,Kiswahili,Kilifi\n"Pale Kisumu Central, Kisumu on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-019.",Other,realistic-other-19,Mixed,Kisumu\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Nakuru East, Nakuru. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Conflict of Interest,realistic-conflict-of-interest-33,English,Nakuru\n"On 11 August 2026, I went to the public office in Kilifi North, Kilifi, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Bribery,realistic-bribery-60,English,Kilifi\n"Pale Machakos Central, Machakos on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Other,realistic-other-60,Sheng,Machakos\n"Pale Kisumu Central, Kisumu on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-049.",Other,realistic-other-49,Sheng,Kisumu\n"Pale Garissa Township, Garissa on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-015.",Nepotism/Favoritism,realistic-nepotism-favoritism-15,Sheng,Garissa\n"On 17 February 2026 pale Machakos Central, Machakos, mkuu wa shule wakati wa shule; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-015.",Abuse of Office,realistic-abuse-of-office-15,Mixed,Machakos\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Embakasi East, Nairobi, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Embezzlement,realistic-embezzlement-44,Kiswahili,Nairobi\n"On 14 June 2026 in Kilifi North, Kilifi, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I kept the receipt and the message thread. Case note KW-019.",Abuse of Office,realistic-abuse-of-office-19,English,Kilifi\n"I checked a meeting minute relating to the committee record in Kilifi North, Kilifi, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-028.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-28,English,Kilifi\n"Tarehe 14 June 2026 katika Imenti Central, Meru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Abuse of Office,realistic-abuse-of-office-03,Kiswahili,Meru\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Garissa Township, Garissa, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-049.",Procurement Fraud,realistic-procurement-fraud-49,Mixed,Garissa\n"Tender ya zabuni ya vifaa vya hospitali pale Likoni, Mombasa ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Procurement Fraud,realistic-procurement-fraud-34,Sheng,Mombasa\n"Pale Embakasi East, Nairobi, on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-053.",Nepotism/Favoritism,realistic-nepotism-favoritism-53,Mixed,Nairobi\n"On 11 August 2026 pale Likoni, Mombasa, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-037.",Abuse of Office,realistic-abuse-of-office-37,Mixed,Mombasa\n"On 6 May 2026 pale Kisumu Central, Kisumu, mkaguzi akadai malipo ya faragha mara moja, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Extortion,realistic-extortion-20,Sheng,Kisumu\n"I am reporting irregular use of KSh 20,000 in Machakos Central, Machakos, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 5,000 were used to buy items not delivered to the school. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Embezzlement,realistic-embezzlement-59,English,Machakos\n"On 9 March 2026 in Garissa Township, Garissa, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I kept the receipt and the message thread. Case note KW-052.",Abuse of Office,realistic-abuse-of-office-52,English,Garissa\n"Kuna doo za fedha za chakula zilipotea pale Likoni, Mombasa after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 1000, nimehifadhi screenshot na receipt. Ref KW-051.",Embezzlement,realistic-embezzlement-51,Sheng,Mombasa\n"Nili-check document ya ilihaririwa pale Machakos Central, Machakos, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-04,Sheng,Machakos\n"Pale Imenti Central, Meru on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-026.",Nepotism/Favoritism,realistic-nepotism-favoritism-26,Sheng,Meru\n"On 9 March 2026 pale Kapseret, Eldoret, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-034.",Extortion,realistic-extortion-34,Mixed,Eldoret\n"For tender VEH-441 in Likoni, Mombasa, the contract committee accepted a bid after the deadline while rejecting timely bidders. The file concerns vehicle leasing and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Procurement Fraud,realistic-procurement-fraud-14,English,Mombasa\n"Nilikagua document for ilibadilishwa pale Embakasi East, Nairobi, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-059.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-59,Mixed,Nairobi\n"On 11 August 2026 in Kilifi North, Kilifi, the department director used an official vehicle for a private family trip and signed the fuel claim through the office. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Abuse of Office,realistic-abuse-of-office-29,English,Kilifi\n"On 6 May 2026, I dealt with the roads office in Kapseret, Eldoret. The office promised resurfacing for our street but no work began before the deadline. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Other,realistic-other-53,English,Eldoret\n"On 6 May 2026 in Kisumu Central, Kisumu, the records officer refused access to my own file despite an approved request and threatened to remove the request record. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Abuse of Office,realistic-abuse-of-office-14,English,Kisumu\n"Tarehe 3 January 2026, mjumbe wa bodi ya shule alishiriki uamuzi kuhusu aliwahi kuajiri katika Garissa Township, Garissa. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Conflict of Interest,realistic-conflict-of-interest-14,Kiswahili,Garissa\n"On 14 June 2026 in Nakuru East, Nakuru, the police officer held my identification card and demanded KSh 2,500, about KSh 5,000. They threatening arrest unless I paid in cash. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Extortion,realistic-extortion-17,English,Nakuru\n"I checked a meeting minute relating to the committee record in Likoni, Mombasa, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-36,English,Mombasa\n"Nilikagua document for ilibadilishwa pale Garissa Township, Garissa, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-011.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-11,Mixed,Garissa\n"Nimekagua hati kuhusu kilikuwa na jina lililobadilishwa katika Garissa Township, Garissa, tarehe 9 March 2026. cheti cha elimu na muhuri ambao shule inasema si halisi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-21,Kiswahili,Garissa\n"Pale Likoni, Mombasa, on 14 June 2026, bodi ya CDF bila kura ya wazi, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-030.",Nepotism/Favoritism,realistic-nepotism-favoritism-30,Mixed,Mombasa\n"Kuna money misuse kwa mapato ya ada za soko pale Nakuru East, Nakuru, after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 300, na screenshot na receipt bado. Case/rekodi KW-055.",Embezzlement,realistic-embezzlement-55,Mixed,Nakuru\n"Nimekagua hati kuhusu kilikuwa na jina lililobadilishwa katika Embakasi East, Nairobi, tarehe 9 March 2026. cheti cha elimu na muhuri ambao shule inasema si halisi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-29,Kiswahili,Nairobi\n"Tender ya zabuni ya ujenzi wa madarasa pale Kapseret, Eldoret ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 5000, nimehifadhi screenshot na receipt. Ref KW-035.",Procurement Fraud,realistic-procurement-fraud-35,Sheng,Eldoret\n"On 28 July 2026, I dealt with the county portal in Bungoma Central, Bungoma. The office removed a public notice without a published reason after residents requested services. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-031.",Other,realistic-other-31,English,Bungoma\n"Nilienda pale office ya Machakos Central, Machakos, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 20,000, nimeweka details ya time na office number. Case/rekodi KW-036.",Bribery,realistic-bribery-36,Mixed,Machakos\n"Tarehe 28 July 2026 katika Bungoma Central, Bungoma, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Extortion,realistic-extortion-22,Kiswahili,Bungoma\n"Tender ya huduma za ulinzi pale Kisumu Central, Kisumu ilikuwa imepangwa. taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri, around 300, nimehifadhi screenshot na receipt. Ref KW-031.",Procurement Fraud,realistic-procurement-fraud-31,Sheng,Kisumu\n"Pale Kilifi North, Kilifi on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no action taken, nimeweka details ya time na office number. Case/rekodi KW-034.",Other,realistic-other-34,Mixed,Kilifi\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Kilifi North, Kilifi, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Bribery,realistic-bribery-60,Kiswahili,Kilifi\n"Pale Kisumu Central, Kisumu on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-057.",Nepotism/Favoritism,realistic-nepotism-favoritism-57,Sheng,Kisumu\n"Kwa ukodishaji wa magari katika Kilifi North, Kilifi, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Tukio linahusu ukodishaji wa magari yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Procurement Fraud,realistic-procurement-fraud-06,Kiswahili,Kilifi\n"Tarehe 3 January 2026 katika Garissa Township, Garissa, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Abuse of Office,realistic-abuse-of-office-02,Kiswahili,Garissa\n"On 11 August 2026 pale Kilifi North, Kilifi, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-041.",Conflict of Interest,realistic-conflict-of-interest-41,Sheng,Kilifi\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Kilifi North, Kilifi. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Conflict of Interest,realistic-conflict-of-interest-41,Kiswahili,Kilifi\n"Kwa mifumo ya kompyuta na mtandao katika Garissa Township, Garissa, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Tukio linahusu mifumo ya kompyuta na mtandao yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Procurement Fraud,realistic-procurement-fraud-29,Kiswahili,Garissa\n"Nilikagua document for iliandika bidhaa pale Kilifi North, Kilifi, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-058.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-58,Mixed,Kilifi\n"Juzi 9 March 2026 nikaingia pale office ya Kapseret, Eldoret, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 5000, nimehifadhi screenshot na receipt. Ref KW-059.",Bribery,realistic-bribery-59,Sheng,Eldoret\n"Kuna doo za chakula na blanketi zilipotea pale Kapseret, Eldoret after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Embezzlement,realistic-embezzlement-32,Sheng,Eldoret\n"On 9 March 2026 pale Kisumu Central, Kisumu, askari wa kaunti akasema zingerudishwa baada ya malipo, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Extortion,realistic-extortion-10,Sheng,Kisumu\n"For tender SEC-55 in Kapseret, Eldoret, the procuring entity gave one supplier confidential bid information. The file concerns security services and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-055.",Procurement Fraud,realistic-procurement-fraud-55,English,Eldoret\n"On 14 June 2026 pale Imenti Central, Meru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-043.",Abuse of Office,realistic-abuse-of-office-43,Mixed,Meru\n"On 11 August 2026 pale Likoni, Mombasa, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-009.",Conflict of Interest,realistic-conflict-of-interest-09,Mixed,Mombasa\n"Tarehe 3 January 2026, mjumbe wa bodi ya shule alishiriki uamuzi kuhusu aliwahi kuajiri katika Embakasi East, Nairobi. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Conflict of Interest,realistic-conflict-of-interest-22,Kiswahili,Nairobi\n"On 9 March 2026, I dealt with the water department in Garissa Township, Garissa. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Other,realistic-other-27,English,Garissa\n"Pale Bungoma Central, Bungoma on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no action taken, na screenshot na receipt bado. Case/rekodi KW-031.",Other,realistic-other-31,Mixed,Bungoma\n"Nilienda pale office ya Likoni, Mombasa, on 22 April 2026 for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada, 20,000, nimeweka details ya time na office number. Case/rekodi KW-048.",Bribery,realistic-bribery-48,Mixed,Mombasa\n"Kuna money misuse kwa pesa ya mradi wa vijana pale Embakasi East, Nairobi, after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 20,000, nimeweka details ya time na office number. Case/rekodi KW-054.",Embezzlement,realistic-embezzlement-54,Mixed,Nairobi\n"Tarehe 14 June 2026 katika Machakos Central, Machakos, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Extortion,realistic-extortion-01,Kiswahili,Machakos\n"Juzi 11 August 2026 nikaingia pale office ya Imenti Central, Meru, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Bribery,realistic-bribery-04,Sheng,Meru\n"For tender HOS-118 in Imenti Central, Meru, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-010.",Procurement Fraud,realistic-procurement-fraud-10,English,Meru\n"On 9 March 2026 pale Kapseret, Eldoret, askari wa kaunti akasema zingerudishwa baada ya malipo, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Extortion,realistic-extortion-34,Sheng,Eldoret\n"On 9 March 2026 in Garissa Township, Garissa, the county askari took the goods and demanded KSh 1,000, about KSh 20,000. They said they would be returned only after a payment. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Extortion,realistic-extortion-18,English,Garissa\n"Tarehe 6 May 2026 katika Bungoma Central, Bungoma, afisa wa rekodi akatishia kuondoa rekodi ya ombi. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Abuse of Office,realistic-abuse-of-office-06,Kiswahili,Bungoma\n"On 9 March 2026 pale Kapseret, Eldoret, afisa wa uteuzi wa umma joined the decision on alikuwa mwenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-040.",Conflict of Interest,realistic-conflict-of-interest-40,Mixed,Eldoret\n"Nilienda pale office ya Kapseret, Eldoret, on 6 May 2026 for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha, 5,000, na screenshot na receipt bado. Case/rekodi KW-029.",Bribery,realistic-bribery-29,Mixed,Eldoret\n"Tarehe 6 May 2026 katika Embakasi East, Nairobi, mkaguzi akadai malipo ya faragha mara moja, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Extortion,realistic-extortion-36,Kiswahili,Nairobi\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Kapseret, Eldoret, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Embezzlement,realistic-embezzlement-02,Kiswahili,Eldoret\n"Kwa zabuni ya ujenzi wa madarasa katika Bungoma Central, Bungoma, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Procurement Fraud,realistic-procurement-fraud-03,Kiswahili,Bungoma\n"For tender SCH-204 in Bungoma Central, Bungoma, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Procurement Fraud,realistic-procurement-fraud-03,English,Bungoma\n"Juzi 28 July 2026 nikaingia pale office ya Bungoma Central, Bungoma, for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida 5000, nimehifadhi screenshot na receipt. Ref KW-047.",Bribery,realistic-bribery-47,Sheng,Bungoma\n"Nilienda pale office ya Garissa Township, Garissa, on 6 May 2026 for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha, 5,000, na screenshot na receipt bado. Case/rekodi KW-053.",Bribery,realistic-bribery-53,Mixed,Garissa\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa ya mradi wa vijana katika Embakasi East, Nairobi, tangu 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Embezzlement,realistic-embezzlement-54,Kiswahili,Nairobi\n"Nilikagua document for kilitiwa sahihi pale Kapseret, Eldoret, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-057.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-57,Mixed,Eldoret\n"In Garissa Township, Garissa, on 9 March 2026, the public internship programme selected a local official\'s child for all available placements even though the position was not advertised. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Nepotism/Favoritism,realistic-nepotism-favoritism-15,English,Garissa\n"Nilikagua document for kilitiwa sahihi pale Bungoma Central, Bungoma, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-025.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-25,Mixed,Bungoma\n"Tarehe 17 February 2026 katika Likoni, Mombasa, mkuu wa shule wakati wa shule. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Abuse of Office,realistic-abuse-of-office-47,Kiswahili,Mombasa\n"I checked a police clearance letter relating to the public application in Imenti Central, Meru, on 17 February 2026. The document used a fabricated stamp to make an unapproved clearance appear valid. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-32,English,Meru\n"On 11 August 2026 pale Kilifi North, Kilifi, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-035.",Extortion,realistic-extortion-35,Mixed,Kilifi\n"Juzi 17 February 2026 nikaingia pale office ya Likoni, Mombasa, for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni 500, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Bribery,realistic-bribery-38,Sheng,Mombasa\n"On 14 June 2026 pale Imenti Central, Meru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 300, na screenshot na receipt bado. Case/rekodi KW-049.",Extortion,realistic-extortion-49,Mixed,Meru\n"Tarehe 22 April 2026 katika Machakos Central, Machakos, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Extortion,realistic-extortion-31,Kiswahili,Machakos\n"On 11 August 2026, I went to the public office in Nakuru East, Nakuru, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-052.",Bribery,realistic-bribery-52,English,Nakuru\n"On 3 January 2026 pale Embakasi East, Nairobi, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-050.",Abuse of Office,realistic-abuse-of-office-50,Mixed,Nairobi\n"Tarehe 9 March 2026 katika Garissa Township, Garissa, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Abuse of Office,realistic-abuse-of-office-52,Kiswahili,Garissa\n"On 6 May 2026 in Kisumu Central, Kisumu, the records officer refused access to my own file despite an approved request and threatened to remove the request record. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Abuse of Office,realistic-abuse-of-office-54,English,Kisumu\n"Kuna doo za mapato ya ada za soko zilipotea pale Nakuru East, Nakuru after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 1000, nimehifadhi screenshot na receipt. Ref KW-015.",Embezzlement,realistic-embezzlement-15,Sheng,Nakuru\n"Kuna money misuse kwa chakula na blanketi pale Kisumu Central, Kisumu, after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 20,000, nimeweka details ya time na office number. Case/rekodi KW-048.",Embezzlement,realistic-embezzlement-48,Mixed,Kisumu\n"Pale Imenti Central, Meru, on 14 June 2026, bodi ya CDF bila kura ya wazi, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-046.",Nepotism/Favoritism,realistic-nepotism-favoritism-46,Mixed,Meru\n"Tarehe 3 January 2026, mjumbe wa bodi ya shule alishiriki uamuzi kuhusu aliwahi kuajiri katika Garissa Township, Garissa. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Conflict of Interest,realistic-conflict-of-interest-54,Kiswahili,Garissa\n"For tender HOS-118 in Nakuru East, Nakuru, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-058.",Procurement Fraud,realistic-procurement-fraud-58,English,Nakuru\n"On 9 March 2026, the public appointment officer took part in a decision about a job assessment in Kapseret, Eldoret. The officer was the spouse of one shortlisted candidate. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-040.",Conflict of Interest,realistic-conflict-of-interest-40,English,Eldoret\n"On 9 March 2026 pale Embakasi East, Nairobi, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Abuse of Office,realistic-abuse-of-office-20,Sheng,Nairobi\n"Nilikagua document for ilonyesha anwani ya uongo pale Likoni, Mombasa, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeweka details ya time na office number. Case/rekodi KW-046.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-46,Mixed,Mombasa\n"I am reporting irregular use of KSh 2,500 in Nakuru East, Nakuru, after 11 August 2026. Records show that the county market cashier handled weekly market fee collections, and the funds worth KSh 1,000 were missing from the ledger and bank deposit. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Embezzlement,realistic-embezzlement-15,English,Nakuru\n"I am reporting irregular use of KSh 20,000 in Embakasi East, Nairobi, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 500 paid a supplier for drugs that are absent from the store. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Embezzlement,realistic-embezzlement-44,English,Nairobi\n"Pale Nakuru East, Nakuru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no action taken, nimeweka details ya time na office number. Case/rekodi KW-016.",Other,realistic-other-16,Mixed,Nakuru\n"Tarehe 22 April 2026 katika Likoni, Mombasa, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Abuse of Office,realistic-abuse-of-office-57,Kiswahili,Mombasa\n"Kuna doo za fedha za ukarabati wa barabara zilipotea pale Machakos Central, Machakos after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 5000, nimehifadhi screenshot na receipt. Ref KW-029.",Embezzlement,realistic-embezzlement-29,Sheng,Machakos\n"Ninawasilisha tuhuma za matumizi mabaya ya mapato ya ada za soko katika Machakos Central, Machakos, tangu 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Embezzlement,realistic-embezzlement-39,Kiswahili,Machakos\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Machakos Central, Machakos. kwa fomu ya kawaida bila risiti rasmi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Other,realistic-other-20,Kiswahili,Machakos\n"Kwa usambazaji wa mafuta katika Nakuru East, Nakuru, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Tukio linahusu usambazaji wa mafuta yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Procurement Fraud,realistic-procurement-fraud-48,Kiswahili,Nakuru\n"In Embakasi East, Nairobi, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Nepotism/Favoritism,realistic-nepotism-favoritism-33,English,Nairobi\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Kapseret, Eldoret, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Embezzlement,realistic-embezzlement-12,Kiswahili,Eldoret\n"In Machakos Central, Machakos, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-058.",Nepotism/Favoritism,realistic-nepotism-favoritism-58,English,Machakos\n"On 22 April 2026 pale Machakos Central, Machakos, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-025.",Abuse of Office,realistic-abuse-of-office-25,Mixed,Machakos\n"Kuna money misuse kwa pesa za dawa pale Garissa Township, Garissa, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 20,000, nimeweka details ya time na office number. Case/rekodi KW-036.",Embezzlement,realistic-embezzlement-36,Mixed,Garissa\n"In Imenti Central, Meru, on 14 June 2026, the CDF board funded the chairman\'s project ahead of equally eligible projects without an open vote. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Nepotism/Favoritism,realistic-nepotism-favoritism-06,English,Meru\n"On 6 May 2026 pale Bungoma Central, Bungoma, afisa wa rekodi akatishia kuondoa rekodi ya ombi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-006.",Abuse of Office,realistic-abuse-of-office-06,Mixed,Bungoma\n"On 14 June 2026, the grant panel chair took part in a decision about a community grant in Nakuru East, Nakuru. The officer was a partner in the organisation applying for funds. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Conflict of Interest,realistic-conflict-of-interest-23,English,Nakuru\n"Nili-check document ya ilikuwa na sahihi iliyorudishwa pale Bungoma Central, Bungoma, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, nimehifadhi screenshot na receipt. Ref KW-055.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-55,Sheng,Bungoma\n"Tarehe 28 July 2026, nilienda ofisi ya umma katika Embakasi East, Nairobi, kwa uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Bribery,realistic-bribery-31,Kiswahili,Nairobi\n"Tarehe 17 February 2026, nilishughulika na dirisha la hospitali katika Kilifi North, Kilifi. kwa nini foleni ilisitishwa bila kutoa njia ya kulalamika. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Other,realistic-other-54,Kiswahili,Kilifi\n"For contract FUEL-33 in Machakos Central, Machakos, the evaluation secretary altered the comparison sheet after bids were submitted. The file concerns fuel supply and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Procurement Fraud,realistic-procurement-fraud-32,English,Machakos\n"Tarehe 11 August 2026 katika Likoni, Mombasa, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Abuse of Office,realistic-abuse-of-office-37,Kiswahili,Mombasa\n"Nilikagua document for kilitiwa sahihi pale Garissa Township, Garissa, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-001.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-01,Mixed,Garissa\n"Pale Kapseret, Eldoret on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-021.",Nepotism/Favoritism,realistic-nepotism-favoritism-21,Sheng,Eldoret\n"I am reporting irregular use of KSh 5,000 in Likoni, Mombasa, after 14 June 2026. Records show that the public works foreman handled road-repair funds, and the funds worth KSh 1,000 were spent while the damaged section remained unfinished. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Embezzlement,realistic-embezzlement-21,English,Mombasa\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Garissa Township, Garissa. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Conflict of Interest,realistic-conflict-of-interest-44,English,Garissa\n"Kuna money misuse kwa fedha za chakula pale Imenti Central, Meru, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 1,000, na screenshot na receipt bado. Case/rekodi KW-027.",Embezzlement,realistic-embezzlement-27,Mixed,Meru\n"I checked a business permit relating to the revenue office in Likoni, Mombasa, on 11 August 2026. The document showed a false address and a permit number not in the official register. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-046.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-46,English,Mombasa\n"On 17 February 2026, I dealt with the hospital desk in Nakuru East, Nakuru. The office refused to explain why the queue was suspended and gave no complaint channel. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Other,realistic-other-06,English,Nakuru\n"Juzi 9 March 2026 nikaingia pale office ya Garissa Township, Garissa, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 300, nimehifadhi screenshot na receipt. Ref KW-043.",Bribery,realistic-bribery-43,Sheng,Garissa\n"I am reporting irregular use of KSh 5,000 in Kisumu Central, Kisumu, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 20,000 entered receipts that could not be matched to payments. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Embezzlement,realistic-embezzlement-18,English,Kisumu\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Kisumu Central, Kisumu. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Conflict of Interest,realistic-conflict-of-interest-26,Kiswahili,Kisumu\n"Pale Garissa Township, Garissa on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-025.",Nepotism/Favoritism,realistic-nepotism-favoritism-25,Sheng,Garissa\n"Tarehe 9 March 2026 katika Garissa Township, Garissa, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Extortion,realistic-extortion-58,Kiswahili,Garissa\n"On 6 May 2026 pale Kapseret, Eldoret, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-010.",Conflict of Interest,realistic-conflict-of-interest-10,Mixed,Eldoret\n"On 3 January 2026 pale Kisumu Central, Kisumu, mjumbe wa bodi ya shule joined the decision on aliwahi kuajiri; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-046.",Conflict of Interest,realistic-conflict-of-interest-46,Mixed,Kisumu\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Machakos Central, Machakos, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-002.",Procurement Fraud,realistic-procurement-fraud-02,Mixed,Machakos\n"Kuna money misuse kwa pesa ya mradi wa vijana pale Kapseret, Eldoret, after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 2,500, nimeweka details ya time na office number. Case/rekodi KW-022.",Embezzlement,realistic-embezzlement-22,Mixed,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za ukarabati wa barabara katika Machakos Central, Machakos, tangu 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Embezzlement,realistic-embezzlement-29,Kiswahili,Machakos\n"Kuna money misuse kwa pesa za dawa pale Kapseret, Eldoret, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 2,500, nimeweka details ya time na office number. Case/rekodi KW-052.",Embezzlement,realistic-embezzlement-52,Mixed,Eldoret\n"Kuna money misuse kwa pesa za dawa pale Kapseret, Eldoret, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 20,000, nimeweka details ya time na office number. Case/rekodi KW-012.",Embezzlement,realistic-embezzlement-12,Mixed,Eldoret\n"Kwa samani za ofisi katika Kilifi North, Kilifi, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Procurement Fraud,realistic-procurement-fraud-36,Kiswahili,Kilifi\n"Nilienda pale office ya Kilifi North, Kilifi, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 500, nimeweka details ya time na office number. Case/rekodi KW-050.",Bribery,realistic-bribery-50,Mixed,Kilifi\n"Tarehe 9 March 2026, afisa wa uteuzi wa umma alishiriki uamuzi kuhusu alikuwa mwenzi katika Embakasi East, Nairobi. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Conflict of Interest,realistic-conflict-of-interest-32,Kiswahili,Nairobi\n"Tarehe 3 January 2026, mjumbe wa bodi ya shule alishiriki uamuzi kuhusu aliwahi kuajiri katika Kisumu Central, Kisumu. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Conflict of Interest,realistic-conflict-of-interest-46,Kiswahili,Kisumu\n"In Imenti Central, Meru, on 11 August 2026, the procurement committee gave repeated contracts to the chairperson\'s close associate while qualified suppliers were not invited. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-016.",Nepotism/Favoritism,realistic-nepotism-favoritism-16,English,Meru\n"Tarehe 22 April 2026 katika Likoni, Mombasa, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Extortion,realistic-extortion-23,Kiswahili,Mombasa\n"Pale Imenti Central, Meru, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-036.",Nepotism/Favoritism,realistic-nepotism-favoritism-36,Mixed,Meru\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Kilifi North, Kilifi. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Conflict of Interest,realistic-conflict-of-interest-51,Kiswahili,Kilifi\n"Pale Embakasi East, Nairobi on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-035.",Other,realistic-other-35,Sheng,Nairobi\n"Kuna money misuse kwa mapato ya ada za soko pale Machakos Central, Machakos, after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 1,000, na screenshot na receipt bado. Case/rekodi KW-039.",Embezzlement,realistic-embezzlement-39,Mixed,Machakos\n"On 6 May 2026, I dealt with the roads office in Garissa Township, Garissa. The office promised resurfacing for our street but no work began before the deadline. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-037.",Other,realistic-other-37,English,Garissa\n"For tender KW-026 in Garissa Township, Garissa, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Procurement Fraud,realistic-procurement-fraud-09,English,Garissa\n"Nilienda pale office ya Embakasi East, Nairobi, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 5,000, na screenshot na receipt bado. Case/rekodi KW-041.",Bribery,realistic-bribery-41,Mixed,Nairobi\n"On 6 May 2026 pale Kisumu Central, Kisumu, mkaguzi akadai malipo ya faragha mara moja, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-020.",Extortion,realistic-extortion-20,Mixed,Kisumu\n"On 11 August 2026 pale Nakuru East, Nakuru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-027.",Extortion,realistic-extortion-27,Sheng,Nakuru\n"Katika Kapseret, Eldoret, tarehe 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Nepotism/Favoritism,realistic-nepotism-favoritism-31,Kiswahili,Eldoret\n"On 11 August 2026 pale Machakos Central, Machakos, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-011.",Extortion,realistic-extortion-11,Mixed,Machakos\n"On 11 August 2026, I went to the public office in Likoni, Mombasa, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-028.",Bribery,realistic-bribery-28,English,Mombasa\n"Pale Kilifi North, Kilifi on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Nepotism/Favoritism,realistic-nepotism-favoritism-32,Sheng,Kilifi\n"On 3 January 2026 pale Embakasi East, Nairobi, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Extortion,realistic-extortion-56,Sheng,Nairobi\n"For tender VEH-441 in Imenti Central, Meru, the contract committee accepted a bid after the deadline while rejecting timely bidders. The file concerns vehicle leasing and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Procurement Fraud,realistic-procurement-fraud-30,English,Meru\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Machakos Central, Machakos. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Conflict of Interest,realistic-conflict-of-interest-27,English,Machakos\n"On 22 April 2026, I dealt with the public help line in Nakuru East, Nakuru. The office did not answer for repeated calls and provided no follow-up reference. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-016.",Other,realistic-other-16,English,Nakuru\n"Pale Kapseret, Eldoret on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-023.",Other,realistic-other-23,Sheng,Eldoret\n"On 22 April 2026, I dealt with the public help line in Nakuru East, Nakuru. The office did not answer for repeated calls and provided no follow-up reference. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Other,realistic-other-56,English,Nakuru\n"Pale Embakasi East, Nairobi on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-023.",Nepotism/Favoritism,realistic-nepotism-favoritism-23,Sheng,Nairobi\n"Pale Kilifi North, Kilifi, on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-032.",Nepotism/Favoritism,realistic-nepotism-favoritism-32,Mixed,Kilifi\n"Nimekagua hati kuhusu ilibadilishwa katika Embakasi East, Nairobi, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-59,Kiswahili,Nairobi\n"On 17 February 2026 pale Nakuru East, Nakuru, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, walidai 300, na screenshot na receipt bado. Case/rekodi KW-037.",Extortion,realistic-extortion-37,Mixed,Nakuru\n"Nili-check document ya ilonyesha anwani ya uongo pale Nakuru East, Nakuru, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-30,Sheng,Nakuru\n"Pale Embakasi East, Nairobi, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-003.",Nepotism/Favoritism,realistic-nepotism-favoritism-03,Mixed,Nairobi\n"Nili-check document ya ilonyesha anwani ya uongo pale Likoni, Mombasa, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-06,Sheng,Mombasa\n"Tarehe 11 August 2026 katika Kilifi North, Kilifi, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Extortion,realistic-extortion-35,Kiswahili,Kilifi\n"I checked a completion certificate relating to the public works file in Kapseret, Eldoret, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-17,English,Eldoret\n"Pale Bungoma Central, Bungoma on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-001.",Other,realistic-other-01,Sheng,Bungoma\n"Kwa zabuni ya ujenzi wa madarasa katika Kisumu Central, Kisumu, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Procurement Fraud,realistic-procurement-fraud-51,Kiswahili,Kisumu\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkaguzi akadai malipo ya faragha mara moja, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-012.",Extortion,realistic-extortion-12,Mixed,Bungoma\n"Pale Kilifi North, Kilifi, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-002.",Nepotism/Favoritism,realistic-nepotism-favoritism-02,Mixed,Kilifi\n"On 22 April 2026 pale Imenti Central, Meru, afisa wa fedha wa kaunti aliingia decision ya alikuwa mmiliki binafsi; personally owned, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-005.",Conflict of Interest,realistic-conflict-of-interest-05,Sheng,Meru\n"For contract SUP-77 in Nakuru East, Nakuru, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-028.",Procurement Fraud,realistic-procurement-fraud-28,English,Nakuru\n"Pale Imenti Central, Meru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Other,realistic-other-48,Sheng,Meru\n"On 3 January 2026 pale Garissa Township, Garissa, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-008.",Extortion,realistic-extortion-08,Mixed,Garissa\n"Kwa usambazaji wa mafuta katika Imenti Central, Meru, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Tukio linahusu usambazaji wa mafuta yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Procurement Fraud,realistic-procurement-fraud-40,Kiswahili,Meru\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Kapseret, Eldoret, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Bribery,realistic-bribery-59,Kiswahili,Eldoret\n"Kuna money misuse kwa fedha za kisima pale Imenti Central, Meru, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 5,000, na screenshot na receipt bado. Case/rekodi KW-017.",Embezzlement,realistic-embezzlement-17,Mixed,Meru\n"Tarehe 6 May 2026, nilienda ofisi ya umma katika Kapseret, Eldoret, kwa ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Bribery,realistic-bribery-29,Kiswahili,Eldoret\n"Katika Garissa Township, Garissa, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Nepotism/Favoritism,realistic-nepotism-favoritism-35,Kiswahili,Garissa\n"Kuna doo za fedha za wanafunzi zilipotea pale Kapseret, Eldoret after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-042.",Embezzlement,realistic-embezzlement-42,Sheng,Eldoret\n"For tender KW-026 in Embakasi East, Nairobi, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Procurement Fraud,realistic-procurement-fraud-17,English,Nairobi\n"For tender VEH-441 in Kilifi North, Kilifi, the contract committee accepted a bid after the deadline while rejecting timely bidders. The file concerns vehicle leasing and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-046.",Procurement Fraud,realistic-procurement-fraud-46,English,Kilifi\n"Nilienda pale office ya Bungoma Central, Bungoma, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 5,000, na screenshot na receipt bado. Case/rekodi KW-017.",Bribery,realistic-bribery-17,Mixed,Bungoma\n"Pale Bungoma Central, Bungoma, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 1,000, na screenshot na receipt bado. Case/rekodi KW-009.",Nepotism/Favoritism,realistic-nepotism-favoritism-09,Mixed,Bungoma\n"I checked a delivery invoice relating to the procurement file in Kilifi North, Kilifi, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-058.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-58,English,Kilifi\n"Kuna money misuse kwa pesa ya mradi wa vijana pale Garissa Township, Garissa, after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 2,500, nimeweka details ya time na office number. Case/rekodi KW-046.",Embezzlement,realistic-embezzlement-46,Mixed,Garissa\n"Katika Embakasi East, Nairobi, tarehe 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Nepotism/Favoritism,realistic-nepotism-favoritism-53,Kiswahili,Nairobi\n"On 6 May 2026 in Kisumu Central, Kisumu, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 500. They and demanded an immediate private payment. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Extortion,realistic-extortion-20,English,Kisumu\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-015.",Extortion,realistic-extortion-15,Mixed,Kilifi\n"Nili-check document ya ilibadilishwa pale Embakasi East, Nairobi, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-059.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-59,Sheng,Nairobi\n"On 14 June 2026 in Likoni, Mombasa, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Abuse of Office,realistic-abuse-of-office-27,English,Mombasa\n"Tender ya zabuni ya ujenzi wa madarasa pale Bungoma Central, Bungoma ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 300, nimehifadhi screenshot na receipt. Ref KW-043.",Procurement Fraud,realistic-procurement-fraud-43,Sheng,Bungoma\n"Tarehe 17 February 2026 katika Machakos Central, Machakos, mkuu wa shule wakati wa shule. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Abuse of Office,realistic-abuse-of-office-55,Kiswahili,Machakos\n"Nilienda pale office ya Garissa Township, Garissa, on 28 July 2026 for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida, 5,000, na screenshot na receipt bado. Case/rekodi KW-023.",Bribery,realistic-bribery-23,Mixed,Garissa\n"Nili-check document ya ilonyesha anwani ya uongo pale Imenti Central, Meru, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-22,Sheng,Meru\n"Tarehe 17 February 2026, katibu wa kamati alishiriki uamuzi kuhusu alilipwa katika Machakos Central, Machakos. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Conflict of Interest,realistic-conflict-of-interest-27,Kiswahili,Machakos\n"On 6 May 2026 pale Embakasi East, Nairobi, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Conflict of Interest,realistic-conflict-of-interest-02,Sheng,Nairobi\n"Tarehe 14 June 2026 katika Imenti Central, Meru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Extortion,realistic-extortion-09,Kiswahili,Meru\n"Tender ya mifumo ya kompyuta na mtandao pale Bungoma Central, Bungoma ilikuwa imepangwa. timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa, around 5000, nimehifadhi screenshot na receipt. Ref KW-053.",Procurement Fraud,realistic-procurement-fraud-53,Sheng,Bungoma\n"Juzi 3 January 2026 nikaingia pale office ya Kapseret, Eldoret, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 1000, nimehifadhi screenshot na receipt. Ref KW-009.",Bribery,realistic-bribery-09,Sheng,Eldoret\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Kisumu Central, Kisumu. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Conflict of Interest,realistic-conflict-of-interest-36,Kiswahili,Kisumu\n"Juzi 6 May 2026 nikaingia pale office ya Kisumu Central, Kisumu, for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha 1000, nimehifadhi screenshot na receipt. Ref KW-045.",Bribery,realistic-bribery-45,Sheng,Kisumu\n"On 11 August 2026, I went to the public office in Imenti Central, Meru, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-004.",Bribery,realistic-bribery-04,English,Meru\n"Kwa tender ya usambazaji wa mafuta pale Imenti Central, Meru, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-040.",Procurement Fraud,realistic-procurement-fraud-40,Mixed,Meru\n"Tender ya usambazaji wa mafuta pale Nakuru East, Nakuru ilikuwa imepangwa. katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Procurement Fraud,realistic-procurement-fraud-08,Sheng,Nakuru\n"On 6 May 2026, I went to the public office in Bungoma Central, Bungoma, for permission to continue trading. The the county askari offered to overlook the stall issue for a private payment; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-037.",Bribery,realistic-bribery-37,English,Bungoma\n"On 28 July 2026 in Garissa Township, Garissa, the station officer refused to release my brother and demanded KSh 2,000, about KSh 500. They while warning that the case file could be damaged. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Extortion,realistic-extortion-38,English,Garissa\n"On 28 July 2026, I went to the public office in Garissa Township, Garissa, for printing an identification document. The the Huduma Centre clerk asked for a personal payment to speed up the ordinary service; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Bribery,realistic-bribery-23,English,Garissa\n"Nilikagua document for ilitumia muhuri wa kughushi pale Kilifi North, Kilifi, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeweka details ya time na office number. Case/rekodi KW-048.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-48,Mixed,Kilifi\n"On 9 March 2026 pale Kisumu Central, Kisumu, askari wa kaunti akasema zingerudishwa baada ya malipo, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-010.",Extortion,realistic-extortion-10,Mixed,Kisumu\n"Tender ya zabuni ya ukarabati wa barabara pale Embakasi East, Nairobi ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 1000, nimehifadhi screenshot na receipt. Ref KW-057.",Procurement Fraud,realistic-procurement-fraud-57,Sheng,Nairobi\n"On 3 January 2026, I went to the public office in Embakasi East, Nairobi, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Bribery,realistic-bribery-41,English,Nairobi\n"Pale Likoni, Mombasa on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Other,realistic-other-12,Sheng,Mombasa\n"Tarehe 6 May 2026, nilishughulika na ofisi ya barabara katika Kapseret, Eldoret. kwenye barabara yetu lakini hakuna kazi iliyoanza kabla ya muda. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Other,realistic-other-53,Kiswahili,Eldoret\n"Tarehe 9 March 2026, afisa wa uteuzi wa umma alishiriki uamuzi kuhusu alikuwa mwenzi katika Kapseret, Eldoret. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Conflict of Interest,realistic-conflict-of-interest-40,Kiswahili,Eldoret\n"On 17 February 2026 pale Nakuru East, Nakuru, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-003.",Conflict of Interest,realistic-conflict-of-interest-03,Sheng,Nakuru\n"Katika Bungoma Central, Bungoma, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Nepotism/Favoritism,realistic-nepotism-favoritism-49,Kiswahili,Bungoma\n"On 14 June 2026, the grant panel chair took part in a decision about a community grant in Machakos Central, Machakos. The officer was a partner in the organisation applying for funds. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-007.",Conflict of Interest,realistic-conflict-of-interest-07,English,Machakos\n"I am reporting irregular use of KSh 20,000 in Likoni, Mombasa, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 5,000 were used to buy items not delivered to the school. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Embezzlement,realistic-embezzlement-11,English,Mombasa\n"Katika Machakos Central, Machakos, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Nepotism/Favoritism,realistic-nepotism-favoritism-18,Kiswahili,Machakos\n"I checked a meeting minute relating to the committee record in Imenti Central, Meru, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-052.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-52,English,Meru\n"Ninawasilisha tuhuma za matumizi mabaya ya chakula na blanketi katika Kisumu Central, Kisumu, tangu 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Embezzlement,realistic-embezzlement-48,Kiswahili,Kisumu\n"Tarehe 9 March 2026 katika Bungoma Central, Bungoma, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Extortion,realistic-extortion-42,Kiswahili,Bungoma\n"Tarehe 14 June 2026 katika Likoni, Mombasa, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Extortion,realistic-extortion-33,Kiswahili,Mombasa\n"On 6 May 2026 in Garissa Township, Garissa, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 2,500. They and demanded an immediate private payment. I kept the receipt and the message thread. Case note KW-028.",Extortion,realistic-extortion-28,English,Garissa\n"Pale Kapseret, Eldoret on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-031.",Nepotism/Favoritism,realistic-nepotism-favoritism-31,Sheng,Eldoret\n"For tender ICT-09 in Bungoma Central, Bungoma, the bid evaluation team used scoring criteria that were not published in the notice. The file concerns computer and network systems and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Procurement Fraud,realistic-procurement-fraud-53,English,Bungoma\n"Tarehe 6 May 2026, nilishughulika na ofisi ya barabara katika Kapseret, Eldoret. kwenye barabara yetu lakini hakuna kazi iliyoanza kabla ya muda. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Other,realistic-other-13,Kiswahili,Eldoret\n"Kwa tender ya mifumo ya kompyuta na mtandao pale Kapseret, Eldoret, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-005.",Procurement Fraud,realistic-procurement-fraud-05,Mixed,Eldoret\n"On 22 April 2026 pale Machakos Central, Machakos, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-025.",Abuse of Office,realistic-abuse-of-office-25,Sheng,Machakos\n"Ninawasilisha tuhuma za matumizi mabaya ya chakula na blanketi katika Embakasi East, Nairobi, tangu 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Embezzlement,realistic-embezzlement-24,Kiswahili,Nairobi\n"Nilikagua document for ilitumia muhuri wa kughushi pale Machakos Central, Machakos, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeweka details ya time na office number. Case/rekodi KW-024.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-24,Mixed,Machakos\n"Kuna doo za pesa ya mradi wa vijana zilipotea pale Embakasi East, Nairobi after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Embezzlement,realistic-embezzlement-54,Sheng,Nairobi\n"On 3 January 2026 pale Kisumu Central, Kisumu, mjumbe wa bodi ya shule aliingia decision ya aliwahi kuajiri; employed, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Conflict of Interest,realistic-conflict-of-interest-06,Sheng,Kisumu\n"I am reporting irregular use of KSh 5,000 in Bungoma Central, Bungoma, after 28 July 2026. Records show that the bursary account clerk handled student bursary funds, and the funds worth KSh 2,500 entered receipts that could not be matched to payments. I kept the receipt and the message thread. Case note KW-010.",Embezzlement,realistic-embezzlement-10,English,Bungoma\n"On 14 June 2026 pale Machakos Central, Machakos, mwenyekiti wa jopo la ruzuku aliingia decision ya alikuwa mshirika; was a partner, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-047.",Conflict of Interest,realistic-conflict-of-interest-47,Sheng,Machakos\n"Tarehe 6 May 2026 katika Garissa Township, Garissa, afisa wa rekodi akatishia kuondoa rekodi ya ombi. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Abuse of Office,realistic-abuse-of-office-22,Kiswahili,Garissa\n"Kwa huduma za ulinzi katika Garissa Township, Garissa, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Tukio linahusu huduma za ulinzi yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Procurement Fraud,realistic-procurement-fraud-39,Kiswahili,Garissa\n"On 6 May 2026 pale Garissa Township, Garissa, afisa wa rekodi akatishia kuondoa rekodi ya ombi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-022.",Abuse of Office,realistic-abuse-of-office-22,Mixed,Garissa\n"Tarehe 6 May 2026 katika Bungoma Central, Bungoma, mkaguzi akadai malipo ya faragha mara moja, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Extortion,realistic-extortion-52,Kiswahili,Bungoma\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Kilifi North, Kilifi. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Conflict of Interest,realistic-conflict-of-interest-01,Kiswahili,Kilifi\n"Kuna doo za mapato ya ada za soko zilipotea pale Imenti Central, Meru after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 300, nimehifadhi screenshot na receipt. Ref KW-007.",Embezzlement,realistic-embezzlement-07,Sheng,Meru\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Kisumu Central, Kisumu, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Embezzlement,realistic-embezzlement-28,Kiswahili,Kisumu\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-015.",Extortion,realistic-extortion-15,Sheng,Kilifi\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Machakos Central, Machakos. kwa fomu ya kawaida bila risiti rasmi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Other,realistic-other-60,Kiswahili,Machakos\n"Tarehe 9 March 2026, afisa wa uteuzi wa umma alishiriki uamuzi kuhusu alikuwa mwenzi katika Bungoma Central, Bungoma. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Conflict of Interest,realistic-conflict-of-interest-08,Kiswahili,Bungoma\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Bungoma Central, Bungoma, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-043.",Procurement Fraud,realistic-procurement-fraud-43,Mixed,Bungoma\n"Pale Bungoma Central, Bungoma on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-051.",Other,realistic-other-51,Sheng,Bungoma\n"Tarehe 28 July 2026, nilienda ofisi ya umma katika Garissa Township, Garissa, kwa uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Bribery,realistic-bribery-23,Kiswahili,Garissa\n"On 17 February 2026 pale Likoni, Mombasa, mkuu wa shule wakati wa shule; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-047.",Abuse of Office,realistic-abuse-of-office-47,Sheng,Mombasa\n"On 28 July 2026 in Bungoma Central, Bungoma, the county officer changed a queue order to serve a personal acquaintance first while others were told to wait. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Abuse of Office,realistic-abuse-of-office-56,English,Bungoma\n"Juzi 28 July 2026 nikaingia pale office ya Kisumu Central, Kisumu, for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida 1000, nimehifadhi screenshot na receipt. Ref KW-015.",Bribery,realistic-bribery-15,Sheng,Kisumu\n"On 11 August 2026 pale Machakos Central, Machakos, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-017.",Conflict of Interest,realistic-conflict-of-interest-17,Sheng,Machakos\n"Tarehe 22 April 2026 katika Kilifi North, Kilifi, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Abuse of Office,realistic-abuse-of-office-09,Kiswahili,Kilifi\n"Nimekagua hati kuhusu kilikuwa na jina lililobadilishwa katika Kisumu Central, Kisumu, tarehe 9 March 2026. cheti cha elimu na muhuri ambao shule inasema si halisi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-13,Kiswahili,Kisumu\n"Katika Kapseret, Eldoret, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-001.",Nepotism/Favoritism,realistic-nepotism-favoritism-01,Kiswahili,Eldoret\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Kisumu Central, Kisumu, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-051.",Procurement Fraud,realistic-procurement-fraud-51,Mixed,Kisumu\n"Tarehe 3 January 2026 katika Embakasi East, Nairobi, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Abuse of Office,realistic-abuse-of-office-50,Kiswahili,Nairobi\n"Tarehe 11 August 2026 katika Imenti Central, Meru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Abuse of Office,realistic-abuse-of-office-13,Kiswahili,Meru\n"Kuna doo za chakula na blanketi zilipotea pale Garissa Township, Garissa after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Embezzlement,realistic-embezzlement-56,Sheng,Garissa\n"Pale Garissa Township, Garissa, on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-055.",Nepotism/Favoritism,realistic-nepotism-favoritism-55,Mixed,Garissa\n"Nili-check document ya ilihaririwa pale Machakos Central, Machakos, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-44,Sheng,Machakos\n"On 11 August 2026 pale Likoni, Mombasa, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 300, na screenshot na receipt bado. Case/rekodi KW-043.",Extortion,realistic-extortion-43,Mixed,Mombasa\n"On 22 April 2026, the county finance officer took part in a decision about a lease decision in Machakos Central, Machakos. The officer personally owned the land being leased to the public entity. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-037.",Conflict of Interest,realistic-conflict-of-interest-37,English,Machakos\n"On 28 July 2026 pale Bungoma Central, Bungoma, afisa wa kaunti wengine wakaambiwa wangojee; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-056.",Abuse of Office,realistic-abuse-of-office-56,Mixed,Bungoma\n"Kuna doo za fedha za chakula zilipotea pale Imenti Central, Meru after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 1000, nimehifadhi screenshot na receipt. Ref KW-027.",Embezzlement,realistic-embezzlement-27,Sheng,Meru\n"Kwa samani za ofisi katika Imenti Central, Meru, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Procurement Fraud,realistic-procurement-fraud-20,Kiswahili,Meru\n"On 17 February 2026 pale Machakos Central, Machakos, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-021.",Extortion,realistic-extortion-21,Sheng,Machakos\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Kapseret, Eldoret, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Embezzlement,realistic-embezzlement-42,Kiswahili,Eldoret\n"Nimekagua hati kuhusu kilikuwa na jina lililobadilishwa katika Bungoma Central, Bungoma, tarehe 9 March 2026. cheti cha elimu na muhuri ambao shule inasema si halisi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-45,Kiswahili,Bungoma\n"Kuna doo za fedha za chakula zilipotea pale Nakuru East, Nakuru after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 5000, nimehifadhi screenshot na receipt. Ref KW-035.",Embezzlement,realistic-embezzlement-35,Sheng,Nakuru\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Kisumu Central, Kisumu, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 300, na screenshot na receipt bado. Case/rekodi KW-001.",Procurement Fraud,realistic-procurement-fraud-01,Mixed,Kisumu\n"Pale Embakasi East, Nairobi on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-035.",Other,realistic-other-35,Mixed,Nairobi\n"On 9 March 2026 pale Garissa Township, Garissa, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Abuse of Office,realistic-abuse-of-office-12,Sheng,Garissa\n"On 17 February 2026 in Machakos Central, Machakos, the school principal directed a teacher to use school equipment for a private function during school time. This was not a routine decision. I kept the receipt and the message thread. Case note KW-055.",Abuse of Office,realistic-abuse-of-office-55,English,Machakos\n"On 11 August 2026 pale Machakos Central, Machakos, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-005.",Abuse of Office,realistic-abuse-of-office-05,Mixed,Machakos\n"On 28 July 2026, I dealt with the county portal in Embakasi East, Nairobi. The office removed a public notice without a published reason after residents requested services. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Other,realistic-other-15,English,Nairobi\n"Tarehe 6 May 2026 katika Kapseret, Eldoret, mkaguzi akadai malipo ya faragha mara moja, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Extortion,realistic-extortion-04,Kiswahili,Eldoret\n"On 14 June 2026, the grant panel chair took part in a decision about a community grant in Kilifi North, Kilifi. The officer was a partner in the organisation applying for funds. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-031.",Conflict of Interest,realistic-conflict-of-interest-31,English,Kilifi\n"Pale Kisumu Central, Kisumu, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-017.",Nepotism/Favoritism,realistic-nepotism-favoritism-17,Mixed,Kisumu\n"For tender HOS-118 in Kilifi North, Kilifi, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Procurement Fraud,realistic-procurement-fraud-26,English,Kilifi\n"For tender VEH-441 in Kilifi North, Kilifi, the contract committee accepted a bid after the deadline while rejecting timely bidders. The file concerns vehicle leasing and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Procurement Fraud,realistic-procurement-fraud-06,English,Kilifi\n"Nimekagua hati kuhusu iliandika bidhaa katika Imenti Central, Meru, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-42,Kiswahili,Meru\n"Kuna money misuse kwa fedha za ukarabati wa barabara pale Likoni, Mombasa, after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 1,000, na screenshot na receipt bado. Case/rekodi KW-021.",Embezzlement,realistic-embezzlement-21,Mixed,Mombasa\n"Pale Machakos Central, Machakos on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Other,realistic-other-40,Sheng,Machakos\n"Kwa tender ya ukodishaji wa magari pale Likoni, Mombasa, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-014.",Procurement Fraud,realistic-procurement-fraud-14,Mixed,Mombasa\n"Tarehe 11 August 2026 katika Machakos Central, Machakos, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Abuse of Office,realistic-abuse-of-office-05,Kiswahili,Machakos\n"Nilikagua document for ilihaririwa pale Machakos Central, Machakos, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-004.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-04,Mixed,Machakos\n"On 22 April 2026, I went to the public office in Likoni, Mombasa, for an approved police abstract. The the police abstract clerk said the document would not be issued without an extra amount; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Bribery,realistic-bribery-08,English,Mombasa\n"Kuna money misuse kwa fedha za ukarabati wa barabara pale Nakuru East, Nakuru, after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 5,000, na screenshot na receipt bado. Case/rekodi KW-005.",Embezzlement,realistic-embezzlement-05,Mixed,Nakuru\n"I am reporting irregular use of KSh 20,000 in Embakasi East, Nairobi, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 2,500 paid a supplier for drugs that are absent from the store. I kept the receipt and the message thread. Case note KW-004.",Embezzlement,realistic-embezzlement-04,English,Nairobi\n"Pale Nakuru East, Nakuru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no action taken, nimeweka details ya time na office number. Case/rekodi KW-056.",Other,realistic-other-56,Mixed,Nakuru\n"On 11 August 2026 pale Imenti Central, Meru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-059.",Extortion,realistic-extortion-59,Mixed,Meru\n"On 17 February 2026 pale Nakuru East, Nakuru, mkuu wa shule wakati wa shule; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-031.",Abuse of Office,realistic-abuse-of-office-31,Mixed,Nakuru\n"On 6 May 2026 in Kapseret, Eldoret, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 500. They and demanded an immediate private payment. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Extortion,realistic-extortion-44,English,Eldoret\n"On 11 August 2026, I went to the public office in Nakuru East, Nakuru, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Bribery,realistic-bribery-12,English,Nakuru\n"Pale Kapseret, Eldoret on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-053.",Other,realistic-other-53,Sheng,Eldoret\n"Pale Kisumu Central, Kisumu, on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-007.",Nepotism/Favoritism,realistic-nepotism-favoritism-07,Mixed,Kisumu\n"Kwa tender ya usambazaji wa mafuta pale Kilifi North, Kilifi, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-016.",Procurement Fraud,realistic-procurement-fraud-16,Mixed,Kilifi\n"Tender ya usambazaji wa mafuta pale Machakos Central, Machakos ilikuwa imepangwa. katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Procurement Fraud,realistic-procurement-fraud-32,Sheng,Machakos\n"Nilikagua document for ilibadilishwa pale Embakasi East, Nairobi, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-019.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-19,Mixed,Nairobi\n"On 14 June 2026 pale Imenti Central, Meru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-003.",Abuse of Office,realistic-abuse-of-office-03,Mixed,Meru\n"Kwa zabuni ya ukarabati wa barabara katika Garissa Township, Garissa, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Procurement Fraud,realistic-procurement-fraud-49,Kiswahili,Garissa\n"On 14 June 2026 pale Imenti Central, Meru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 300; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-049.",Extortion,realistic-extortion-49,Sheng,Meru\n"Pale Likoni, Mombasa, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-010.",Nepotism/Favoritism,realistic-nepotism-favoritism-10,Mixed,Mombasa\n"On 28 July 2026 pale Embakasi East, Nairobi, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Extortion,realistic-extortion-06,Sheng,Nairobi\n"I checked a delivery invoice relating to the procurement file in Machakos Central, Machakos, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-034.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-34,English,Machakos\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Kilifi North, Kilifi. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Conflict of Interest,realistic-conflict-of-interest-11,English,Kilifi\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Embakasi East, Nairobi, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Embezzlement,realistic-embezzlement-34,Kiswahili,Nairobi\n"On 14 June 2026 in Kilifi North, Kilifi, the police officer held my identification card and demanded KSh 2,500, about KSh 300. They threatening arrest unless I paid in cash. I kept the receipt and the message thread. Case note KW-025.",Extortion,realistic-extortion-25,English,Kilifi\n"Kwa mifumo ya kompyuta na mtandao katika Embakasi East, Nairobi, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Tukio linahusu mifumo ya kompyuta na mtandao yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Procurement Fraud,realistic-procurement-fraud-37,Kiswahili,Nairobi\n"Juzi 17 February 2026 nikaingia pale office ya Machakos Central, Machakos, for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Bribery,realistic-bribery-46,Sheng,Machakos\n"Nimekagua hati kuhusu ilibadilishwa katika Kisumu Central, Kisumu, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-43,Kiswahili,Kisumu\n"Tarehe 14 June 2026 katika Kilifi North, Kilifi, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Extortion,realistic-extortion-25,Kiswahili,Kilifi\n"Kuna money misuse kwa chakula na blanketi pale Kisumu Central, Kisumu, after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 500, nimeweka details ya time na office number. Case/rekodi KW-008.",Embezzlement,realistic-embezzlement-08,Mixed,Kisumu\n"In Kilifi North, Kilifi, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Nepotism/Favoritism,realistic-nepotism-favoritism-02,English,Kilifi\n"On 6 May 2026 pale Bungoma Central, Bungoma, afisa wa rekodi akatishia kuondoa rekodi ya ombi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Abuse of Office,realistic-abuse-of-office-46,Sheng,Bungoma\n"Kuna money misuse kwa chakula na blanketi pale Embakasi East, Nairobi, after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 20,000, nimeweka details ya time na office number. Case/rekodi KW-024.",Embezzlement,realistic-embezzlement-24,Mixed,Nairobi\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Kisumu Central, Kisumu, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-011.",Procurement Fraud,realistic-procurement-fraud-11,Mixed,Kisumu\n"Tarehe 11 August 2026, mwenyekiti wa ununuzi alishiriki uamuzi kuhusu alikuwa na hisa katika Imenti Central, Meru. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Conflict of Interest,realistic-conflict-of-interest-25,Kiswahili,Meru\n"Katika Embakasi East, Nairobi, tarehe 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Nepotism/Favoritism,realistic-nepotism-favoritism-13,Kiswahili,Nairobi\n"Pale Kapseret, Eldoret, on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-041.",Nepotism/Favoritism,realistic-nepotism-favoritism-41,Mixed,Eldoret\n"Tarehe 17 February 2026, nilienda ofisi ya umma katika Kilifi North, Kilifi, kwa utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Bribery,realistic-bribery-30,Kiswahili,Kilifi\n"Pale Kilifi North, Kilifi on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Other,realistic-other-14,Sheng,Kilifi\n"I checked a land transfer record relating to the county registry in Kisumu Central, Kisumu, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-043.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-43,English,Kisumu\n"Tarehe 3 January 2026 katika Embakasi East, Nairobi, chifu wakati wa saa za kazi. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Abuse of Office,realistic-abuse-of-office-10,Kiswahili,Nairobi\n"Kwa zabuni ya vifaa vya hospitali katika Nakuru East, Nakuru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Procurement Fraud,realistic-procurement-fraud-58,Kiswahili,Nakuru\n"Nilienda pale office ya Nakuru East, Nakuru, on 17 February 2026 for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni, 2,500, nimeweka details ya time na office number. Case/rekodi KW-022.",Bribery,realistic-bribery-22,Mixed,Nakuru\n"Nili-check document ya ilihaririwa pale Imenti Central, Meru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-52,Sheng,Meru\n"Nilienda pale office ya Nakuru East, Nakuru, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 20,000, nimeweka details ya time na office number. Case/rekodi KW-042.",Bribery,realistic-bribery-42,Mixed,Nakuru\n"Pale Nakuru East, Nakuru on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-016.",Other,realistic-other-16,Sheng,Nakuru\n"Kwa zabuni ya vifaa vya hospitali katika Imenti Central, Meru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Procurement Fraud,realistic-procurement-fraud-10,Kiswahili,Meru\n"Kuna money misuse kwa fedha za ukarabati wa barabara pale Kilifi North, Kilifi, after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 300, na screenshot na receipt bado. Case/rekodi KW-013.",Embezzlement,realistic-embezzlement-13,Mixed,Kilifi\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, walidai 300, na screenshot na receipt bado. Case/rekodi KW-007.",Extortion,realistic-extortion-07,Mixed,Nakuru\n"On 22 April 2026, I dealt with the public help line in Machakos Central, Machakos. The office did not answer for repeated calls and provided no follow-up reference. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-040.",Other,realistic-other-40,English,Machakos\n"Tarehe 11 August 2026 katika Nakuru East, Nakuru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Abuse of Office,realistic-abuse-of-office-21,Kiswahili,Nakuru\n"In Likoni, Mombasa, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Nepotism/Favoritism,realistic-nepotism-favoritism-50,English,Mombasa\n"For tender SEC-55 in Kisumu Central, Kisumu, the procuring entity gave one supplier confidential bid information. The file concerns security services and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-031.",Procurement Fraud,realistic-procurement-fraud-31,English,Kisumu\n"Nilienda pale office ya Bungoma Central, Bungoma, on 6 May 2026 for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha, 300, na screenshot na receipt bado. Case/rekodi KW-037.",Bribery,realistic-bribery-37,Mixed,Bungoma\n"Nimekagua hati kuhusu ilitumia muhuri wa kughushi katika Imenti Central, Meru, tarehe 17 February 2026. barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-32,Kiswahili,Meru\n"Juzi 9 March 2026 nikaingia pale office ya Kisumu Central, Kisumu, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 5000, nimehifadhi screenshot na receipt. Ref KW-035.",Bribery,realistic-bribery-35,Sheng,Kisumu\n"Katika Machakos Central, Machakos, tarehe 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Nepotism/Favoritism,realistic-nepotism-favoritism-48,Kiswahili,Machakos\n"On 14 June 2026 pale Imenti Central, Meru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-009.",Extortion,realistic-extortion-09,Sheng,Meru\n"In Imenti Central, Meru, on 22 April 2026, the market allocation team allocated premium stalls to officials\' friends before the published selection date. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Nepotism/Favoritism,realistic-nepotism-favoritism-36,English,Meru\n"Nilikagua document for ilonyesha anwani ya uongo pale Machakos Central, Machakos, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeweka details ya time na office number. Case/rekodi KW-014.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-14,Mixed,Machakos\n"I am reporting irregular use of KSh 5,000 in Kisumu Central, Kisumu, after 6 May 2026. Records show that the relief distribution committee handled food and blankets, and the funds worth KSh 500 were recorded as delivered but never reached beneficiaries. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Embezzlement,realistic-embezzlement-08,English,Kisumu\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Embakasi East, Nairobi. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-052.",Conflict of Interest,realistic-conflict-of-interest-52,English,Nairobi\n"For tender SCH-204 in Garissa Township, Garissa, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-019.",Procurement Fraud,realistic-procurement-fraud-19,English,Garissa\n"I checked a police clearance letter relating to the public application in Likoni, Mombasa, on 17 February 2026. The document used a fabricated stamp to make an unapproved clearance appear valid. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-016.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-16,English,Mombasa\n"I am reporting irregular use of KSh 20,000 in Embakasi East, Nairobi, after 9 March 2026. Records show that the youth fund administrator handled approved youth project money, and the funds worth KSh 500 was transferred to a personal account. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Embezzlement,realistic-embezzlement-14,English,Nairobi\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-041.",Abuse of Office,realistic-abuse-of-office-41,Sheng,Nakuru\n"On 9 March 2026 in Bungoma Central, Bungoma, the county askari took the goods and demanded KSh 1,000, about KSh 500. They said they would be returned only after a payment. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Extortion,realistic-extortion-02,English,Bungoma\n"On 17 February 2026, the committee secretary took part in a decision about a bursary award in Imenti Central, Meru. The officer was paid by an organization that received the award. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Conflict of Interest,realistic-conflict-of-interest-35,English,Meru\n"In Likoni, Mombasa, on 14 June 2026, the CDF board funded the chairman\'s project ahead of equally eligible projects without an open vote. The estimated benefit or contract value was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Nepotism/Favoritism,realistic-nepotism-favoritism-30,English,Mombasa\n"On 3 January 2026 pale Bungoma Central, Bungoma, mjumbe wa bodi ya shule aliingia decision ya aliwahi kuajiri; employed, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Conflict of Interest,realistic-conflict-of-interest-38,Sheng,Bungoma\n"Nimekagua hati kuhusu ilitumia muhuri wa kughushi katika Kilifi North, Kilifi, tarehe 17 February 2026. barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-48,Kiswahili,Kilifi\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Likoni, Mombasa. kwa fomu ya kawaida bila risiti rasmi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Other,realistic-other-12,Kiswahili,Mombasa\n"Pale Kapseret, Eldoret on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-003.",Other,realistic-other-03,Mixed,Eldoret\n"On 17 February 2026 pale Imenti Central, Meru, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-035.",Conflict of Interest,realistic-conflict-of-interest-35,Sheng,Meru\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Embakasi East, Nairobi. bila taarifa wala haikutangaza wakati huduma ingerejea. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-025.",Other,realistic-other-25,Kiswahili,Nairobi\n"I am reporting irregular use of KSh 20,000 in Kisumu Central, Kisumu, after 9 March 2026. Records show that the youth fund administrator handled approved youth project money, and the funds worth KSh 500 was transferred to a personal account. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Embezzlement,realistic-embezzlement-38,English,Kisumu\n"Nilikagua document for kilikuwa na jina lililobadilishwa pale Kisumu Central, Kisumu, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, na screenshot na receipt bado. Case/rekodi KW-013.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-13,Mixed,Kisumu\n"Nilikagua document for ilitumia muhuri wa kughushi pale Likoni, Mombasa, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeweka details ya time na office number. Case/rekodi KW-016.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-16,Mixed,Mombasa\n"On 17 February 2026 pale Likoni, Mombasa, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-019.",Conflict of Interest,realistic-conflict-of-interest-19,Mixed,Mombasa\n"I checked a police clearance letter relating to the public application in Nakuru East, Nakuru, on 17 February 2026. The document used a fabricated stamp to make an unapproved clearance appear valid. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-040.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-40,English,Nakuru\n"On 22 April 2026 pale Imenti Central, Meru, afisa wa fedha wa kaunti aliingia decision ya alikuwa mmiliki binafsi; personally owned, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-045.",Conflict of Interest,realistic-conflict-of-interest-45,Sheng,Meru\n"On 17 February 2026, I dealt with the hospital desk in Nakuru East, Nakuru. The office refused to explain why the queue was suspended and gave no complaint channel. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-046.",Other,realistic-other-46,English,Nakuru\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Machakos Central, Machakos, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-036.",Bribery,realistic-bribery-36,Kiswahili,Machakos\n"Nilienda pale office ya Machakos Central, Machakos, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 500, nimeweka details ya time na office number. Case/rekodi KW-026.",Bribery,realistic-bribery-26,Mixed,Machakos\n"On 14 June 2026 pale Imenti Central, Meru, mwenyekiti wa jopo la ruzuku aliingia decision ya alikuwa mshirika; was a partner, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-055.",Conflict of Interest,realistic-conflict-of-interest-55,Sheng,Meru\n"In Machakos Central, Machakos, on 11 August 2026, the procurement committee gave repeated contracts to the chairperson\'s close associate while qualified suppliers were not invited. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Nepotism/Favoritism,realistic-nepotism-favoritism-08,English,Machakos\n"For tender ICT-09 in Kisumu Central, Kisumu, the bid evaluation team used scoring criteria that were not published in the notice. The file concerns computer and network systems and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Procurement Fraud,realistic-procurement-fraud-21,English,Kisumu\n"On 11 August 2026 pale Imenti Central, Meru, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-025.",Conflict of Interest,realistic-conflict-of-interest-25,Sheng,Meru\n"Kuna money misuse kwa fedha za kisima pale Kilifi North, Kilifi, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 1,000, na screenshot na receipt bado. Case/rekodi KW-033.",Embezzlement,realistic-embezzlement-33,Mixed,Kilifi\n"Tarehe 17 February 2026 katika Imenti Central, Meru, mkuu wa shule wakati wa shule. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Abuse of Office,realistic-abuse-of-office-23,Kiswahili,Meru\n"Nili-check document ya kilikuwa na jina lililobadilishwa pale Garissa Township, Garissa, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, nimehifadhi screenshot na receipt. Ref KW-021.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-21,Sheng,Garissa\n"Tarehe 28 July 2026, mjumbe wa bodi ya leseni alishiriki uamuzi kuhusu alikuwa mkurugenzi katika Bungoma Central, Bungoma. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Conflict of Interest,realistic-conflict-of-interest-28,Kiswahili,Bungoma\n"Nili-check document ya ilonyesha anwani ya uongo pale Kilifi North, Kilifi, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-38,Sheng,Kilifi\n"Kuna money misuse kwa fedha za ukarabati wa barabara pale Nakuru East, Nakuru, after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 1,000, na screenshot na receipt bado. Case/rekodi KW-045.",Embezzlement,realistic-embezzlement-45,Mixed,Nakuru\n"On 9 March 2026, I went to the public office in Kisumu Central, Kisumu, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Bribery,realistic-bribery-35,English,Kisumu\n"On 14 June 2026, I went to the public office in Likoni, Mombasa, for a business permit. The the revenue officer demanded an unofficial fee to approve the permit; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-058.",Bribery,realistic-bribery-58,English,Mombasa\n"Kuna doo za chakula na blanketi zilipotea pale Kisumu Central, Kisumu after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Embezzlement,realistic-embezzlement-08,Sheng,Kisumu\n"On 14 June 2026, I dealt with the service desk in Machakos Central, Machakos. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-010.",Other,realistic-other-10,English,Machakos\n"I am reporting irregular use of KSh 20,000 in Bungoma Central, Bungoma, after 9 March 2026. Records show that the youth fund administrator handled approved youth project money, and the funds worth KSh 20,000 was transferred to a personal account. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Embezzlement,realistic-embezzlement-30,English,Bungoma\n"Kuna money misuse kwa chakula na blanketi pale Garissa Township, Garissa, after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 2,500, nimeweka details ya time na office number. Case/rekodi KW-016.",Embezzlement,realistic-embezzlement-16,Mixed,Garissa\n"Tarehe 22 April 2026, nilishughulika na simu ya msaada ya umma katika Machakos Central, Machakos. baada ya simu nyingi bila nambari ya kufuatilia. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Other,realistic-other-40,Kiswahili,Machakos\n"Nimekagua hati kuhusu ilikuwa na sahihi iliyorudishwa katika Bungoma Central, Bungoma, tarehe 6 May 2026. vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-055.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-55,Kiswahili,Bungoma\n"Nili-check document ya ilitumia muhuri wa kughushi pale Imenti Central, Meru, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-32,Sheng,Meru\n"Pale Garissa Township, Garissa on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-057.",Other,realistic-other-57,Sheng,Garissa\n"On 3 January 2026, I dealt with the public office in Garissa Township, Garissa. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Other,realistic-other-57,English,Garissa\n"I checked a payment voucher relating to the hospital account in Garissa Township, Garissa, on 6 May 2026. The document carried a duplicate signature for a purchase that has no supplier evidence. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-031.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-31,English,Garissa\n"Pale Kisumu Central, Kisumu on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no action taken, na screenshot na receipt bado. Case/rekodi KW-029.",Other,realistic-other-29,Mixed,Kisumu\n"Pale Kilifi North, Kilifi on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Other,realistic-other-44,Sheng,Kilifi\n"Nimekagua hati kuhusu ilihaririwa katika Machakos Central, Machakos, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-44,Kiswahili,Machakos\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-049.",Abuse of Office,realistic-abuse-of-office-49,Mixed,Kilifi\n"For contract SUP-77 in Kilifi North, Kilifi, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Procurement Fraud,realistic-procurement-fraud-36,English,Kilifi\n"Tarehe 6 May 2026, nilishughulika na ofisi ya barabara katika Bungoma Central, Bungoma. kwenye barabara yetu lakini hakuna kazi iliyoanza kabla ya muda. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Other,realistic-other-21,Kiswahili,Bungoma\n"Juzi 11 August 2026 nikaingia pale office ya Imenti Central, Meru, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Bribery,realistic-bribery-44,Sheng,Meru\n"Juzi 6 May 2026 nikaingia pale office ya Kisumu Central, Kisumu, for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha 5000, nimehifadhi screenshot na receipt. Ref KW-005.",Bribery,realistic-bribery-05,Sheng,Kisumu\n"Kuna doo za fedha za kisima zilipotea pale Kilifi North, Kilifi after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 1000, nimehifadhi screenshot na receipt. Ref KW-033.",Embezzlement,realistic-embezzlement-33,Sheng,Kilifi\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Kilifi North, Kilifi, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-003.",Embezzlement,realistic-embezzlement-03,Kiswahili,Kilifi\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Embakasi East, Nairobi. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Conflict of Interest,realistic-conflict-of-interest-02,Kiswahili,Nairobi\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Garissa Township, Garissa. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-004.",Conflict of Interest,realistic-conflict-of-interest-04,English,Garissa\n"On 17 February 2026 pale Machakos Central, Machakos, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-027.",Conflict of Interest,realistic-conflict-of-interest-27,Mixed,Machakos\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Imenti Central, Meru, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Bribery,realistic-bribery-34,Kiswahili,Meru\n"On 9 March 2026, I dealt with the water department in Embakasi East, Nairobi. The office left the community pump out of service for several months without an explanation. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Other,realistic-other-35,English,Nairobi\n"Tarehe 22 April 2026 katika Kilifi North, Kilifi, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Extortion,realistic-extortion-15,Kiswahili,Kilifi\n"On 11 August 2026 in Machakos Central, Machakos, the department director used an official vehicle for a private family trip and signed the fuel claim through the office. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Abuse of Office,realistic-abuse-of-office-05,English,Machakos\n"Nilikagua document for kilikuwa na jina lililobadilishwa pale Embakasi East, Nairobi, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, na screenshot na receipt bado. Case/rekodi KW-029.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-29,Mixed,Nairobi\n"On 17 February 2026 in Imenti Central, Meru, the school principal directed a teacher to use school equipment for a private function during school time. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Abuse of Office,realistic-abuse-of-office-23,English,Meru\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za kisima katika Likoni, Mombasa, tangu 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Embezzlement,realistic-embezzlement-41,Kiswahili,Mombasa\n"Tender ya zabuni ya vifaa vya hospitali pale Imenti Central, Meru ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Procurement Fraud,realistic-procurement-fraud-50,Sheng,Meru\n"Kwa zabuni ya ujenzi wa madarasa katika Embakasi East, Nairobi, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Procurement Fraud,realistic-procurement-fraud-27,Kiswahili,Nairobi\n"On 9 March 2026, I went to the public office in Garissa Township, Garissa, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Bribery,realistic-bribery-03,English,Garissa\n"Pale Kapseret, Eldoret on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-013.",Other,realistic-other-13,Sheng,Eldoret\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Kapseret, Eldoret, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-035.",Procurement Fraud,realistic-procurement-fraud-35,Mixed,Eldoret\n"Nilienda pale office ya Machakos Central, Machakos, on 17 February 2026 for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni, 2,500, nimeweka details ya time na office number. Case/rekodi KW-046.",Bribery,realistic-bribery-46,Mixed,Machakos\n"On 6 May 2026 pale Kapseret, Eldoret, mkaguzi akadai malipo ya faragha mara moja, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-044.",Extortion,realistic-extortion-44,Mixed,Eldoret\n"On 14 June 2026 pale Imenti Central, Meru, mwenyekiti wa jopo la ruzuku joined the decision on alikuwa mshirika; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-055.",Conflict of Interest,realistic-conflict-of-interest-55,Mixed,Meru\n"For contract FUEL-33 in Nakuru East, Nakuru, the evaluation secretary altered the comparison sheet after bids were submitted. The file concerns fuel supply and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Procurement Fraud,realistic-procurement-fraud-08,English,Nakuru\n"Pale Imenti Central, Meru on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-036.",Nepotism/Favoritism,realistic-nepotism-favoritism-36,Sheng,Meru\n"Tarehe 28 July 2026 katika Kisumu Central, Kisumu, afisa wa kaunti wengine wakaambiwa wangojee. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Abuse of Office,realistic-abuse-of-office-24,Kiswahili,Kisumu\n"On 3 January 2026 pale Embakasi East, Nairobi, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Abuse of Office,realistic-abuse-of-office-10,Sheng,Nairobi\n"Pale Kapseret, Eldoret on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-001.",Nepotism/Favoritism,realistic-nepotism-favoritism-01,Sheng,Eldoret\n"Juzi 11 August 2026 nikaingia pale office ya Kilifi North, Kilifi, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Bribery,realistic-bribery-60,Sheng,Kilifi\n"Nilienda pale office ya Likoni, Mombasa, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 2,500, nimeweka details ya time na office number. Case/rekodi KW-058.",Bribery,realistic-bribery-58,Mixed,Mombasa\n"On 11 August 2026 in Machakos Central, Machakos, the department director used an official vehicle for a private family trip and signed the fuel claim through the office. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Abuse of Office,realistic-abuse-of-office-45,English,Machakos\n"Nili-check document ya ilibadilishwa pale Garissa Township, Garissa, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-051.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-51,Sheng,Garissa\n"On 22 April 2026 in Nakuru East, Nakuru, the gate security supervisor blocked my lorry and demanded KSh 1,500, about KSh 5,000. They threatening to call a taxing inspection. I have a screenshot and the names of two people who witnessed it. Case note KW-047.",Extortion,realistic-extortion-47,English,Nakuru\n"On 11 August 2026, I went to the public office in Kilifi North, Kilifi, for release of a matatu. The the traffic officer said the vehicle would be held unless payment was made; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Bribery,realistic-bribery-20,English,Kilifi\n"Nilikagua document for ilikuwa na sahihi iliyorudishwa pale Bungoma Central, Bungoma, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, na screenshot na receipt bado. Case/rekodi KW-055.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-55,Mixed,Bungoma\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-049.",Abuse of Office,realistic-abuse-of-office-49,Sheng,Kilifi\n"Pale Kilifi North, Kilifi, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-042.",Nepotism/Favoritism,realistic-nepotism-favoritism-42,Mixed,Kilifi\n"I checked a land transfer record relating to the county registry in Bungoma Central, Bungoma, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-35,English,Bungoma\n"Pale Nakuru East, Nakuru on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Nepotism/Favoritism,realistic-nepotism-favoritism-04,Sheng,Nakuru\n"Nilikagua document for iliandika bidhaa pale Likoni, Mombasa, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeweka details ya time na office number. Case/rekodi KW-026.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-26,Mixed,Mombasa\n"On 6 May 2026 pale Garissa Township, Garissa, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Conflict of Interest,realistic-conflict-of-interest-34,Sheng,Garissa\n"On 28 July 2026, I went to the public office in Kisumu Central, Kisumu, for printing an identification document. The the Huduma Centre clerk asked for a personal payment to speed up the ordinary service; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-015.",Bribery,realistic-bribery-15,English,Kisumu\n"Pale Kapseret, Eldoret on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-033.",Other,realistic-other-33,Sheng,Eldoret\n"Pale Kapseret, Eldoret on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-011.",Nepotism/Favoritism,realistic-nepotism-favoritism-11,Sheng,Eldoret\n"In Bungoma Central, Bungoma, on 3 January 2026, the appointment board promoted an officer\'s spouse to a senior post despite a lower interview score. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Nepotism/Favoritism,realistic-nepotism-favoritism-29,English,Bungoma\n"Kuna money misuse kwa fedha za chakula pale Machakos Central, Machakos, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 300, na screenshot na receipt bado. Case/rekodi KW-019.",Embezzlement,realistic-embezzlement-19,Mixed,Machakos\n"Pale Likoni, Mombasa on 14 June 2026, bodi ya CDF bila kura ya wazi; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Nepotism/Favoritism,realistic-nepotism-favoritism-30,Sheng,Mombasa\n"Kwa tender ya mifumo ya kompyuta na mtandao pale Bungoma Central, Bungoma, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-053.",Procurement Fraud,realistic-procurement-fraud-53,Mixed,Bungoma\n"On 22 April 2026 in Likoni, Mombasa, the gate security supervisor blocked my lorry and demanded KSh 1,500, about KSh 5,000. They threatening to call a taxing inspection. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Extortion,realistic-extortion-23,English,Mombasa\n"On 9 March 2026 pale Embakasi East, Nairobi, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-060.",Abuse of Office,realistic-abuse-of-office-60,Mixed,Nairobi\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, walidai 300, na screenshot na receipt bado. Case/rekodi KW-055.",Extortion,realistic-extortion-55,Mixed,Kilifi\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Likoni, Mombasa, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-018.",Bribery,realistic-bribery-18,Kiswahili,Mombasa\n"On 6 May 2026 pale Kapseret, Eldoret, mkaguzi akadai malipo ya faragha mara moja, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Extortion,realistic-extortion-44,Sheng,Eldoret\n"On 22 April 2026, I went to the public office in Likoni, Mombasa, for an approved police abstract. The the police abstract clerk said the document would not be issued without an extra amount; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Bribery,realistic-bribery-48,English,Mombasa\n"Nili-check document ya ilonyesha anwani ya uongo pale Machakos Central, Machakos, on 11 August 2026; leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-54,Sheng,Machakos\n"On 9 March 2026 pale Kisumu Central, Kisumu, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Abuse of Office,realistic-abuse-of-office-04,Sheng,Kisumu\n"Nilikagua document for ilitumia muhuri wa kughushi pale Nakuru East, Nakuru, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeweka details ya time na office number. Case/rekodi KW-040.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-40,Mixed,Nakuru\n"On 6 May 2026, I went to the public office in Garissa Township, Garissa, for permission to continue trading. The the county askari offered to overlook the stall issue for a private payment; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Bribery,realistic-bribery-53,English,Garissa\n"Pale Bungoma Central, Bungoma on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no action taken, na screenshot na receipt bado. Case/rekodi KW-021.",Other,realistic-other-21,Mixed,Bungoma\n"On 17 February 2026, I went to the public office in Machakos Central, Machakos, for release of a passport. The the immigration officer requested money to place the application ahead of the queue; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-006.",Bribery,realistic-bribery-06,English,Machakos\n"Tarehe 17 February 2026, nilienda ofisi ya umma katika Imenti Central, Meru, kwa utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Bribery,realistic-bribery-14,Kiswahili,Meru\n"Pale Imenti Central, Meru, on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-056.",Nepotism/Favoritism,realistic-nepotism-favoritism-56,Mixed,Meru\n"On 6 May 2026, the project director took part in a decision about a construction contract in Kisumu Central, Kisumu. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Conflict of Interest,realistic-conflict-of-interest-26,English,Kisumu\n"On 28 July 2026, I went to the public office in Kisumu Central, Kisumu, for printing an identification document. The the Huduma Centre clerk asked for a personal payment to speed up the ordinary service; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-055.",Bribery,realistic-bribery-55,English,Kisumu\n"Nilienda pale office ya Bungoma Central, Bungoma, on 28 July 2026 for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida, 300, na screenshot na receipt bado. Case/rekodi KW-007.",Bribery,realistic-bribery-07,Mixed,Bungoma\n"On 3 January 2026, I went to the public office in Kapseret, Eldoret, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 300. I kept the receipt and the message thread. Case note KW-049.",Bribery,realistic-bribery-49,English,Eldoret\n"On 3 January 2026, I dealt with the public office in Bungoma Central, Bungoma. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Other,realistic-other-41,English,Bungoma\n"In Kapseret, Eldoret, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Nepotism/Favoritism,realistic-nepotism-favoritism-41,English,Eldoret\n"For tender SCH-204 in Kisumu Central, Kisumu, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-011.",Procurement Fraud,realistic-procurement-fraud-11,English,Kisumu\n"On 17 February 2026 pale Likoni, Mombasa, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-053.",Extortion,realistic-extortion-53,Sheng,Mombasa\n"Nilikagua document for ilihaririwa pale Imenti Central, Meru, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-012.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-12,Mixed,Meru\n"Kuna money misuse kwa fedha za ukarabati wa barabara pale Kilifi North, Kilifi, after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 5,000, na screenshot na receipt bado. Case/rekodi KW-053.",Embezzlement,realistic-embezzlement-53,Mixed,Kilifi\n"Tender ya samani za ofisi pale Imenti Central, Meru ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 500, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Procurement Fraud,realistic-procurement-fraud-20,Sheng,Meru\n"Pale Kisumu Central, Kisumu on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-037.",Nepotism/Favoritism,realistic-nepotism-favoritism-37,Sheng,Kisumu\n"Kwa ukodishaji wa magari katika Nakuru East, Nakuru, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Tukio linahusu ukodishaji wa magari yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Procurement Fraud,realistic-procurement-fraud-38,Kiswahili,Nakuru\n"On 28 July 2026 pale Embakasi East, Nairobi, afisa wa kaunti wengine wakaambiwa wangojee; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Abuse of Office,realistic-abuse-of-office-40,Sheng,Nairobi\n"On 3 January 2026, I went to the public office in Garissa Township, Garissa, for a land-title search. The the lands clerk asked for money before opening the file; the demand was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Bribery,realistic-bribery-33,English,Garissa\n"On 17 February 2026 pale Likoni, Mombasa, mkuu wa shule wakati wa shule; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-007.",Abuse of Office,realistic-abuse-of-office-07,Mixed,Mombasa\n"Pale Garissa Township, Garissa, on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-005.",Nepotism/Favoritism,realistic-nepotism-favoritism-05,Mixed,Garissa\n"Katika Likoni, Mombasa, tarehe 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Nepotism/Favoritism,realistic-nepotism-favoritism-20,Kiswahili,Mombasa\n"On 6 May 2026 pale Garissa Township, Garissa, afisa wa rekodi akatishia kuondoa rekodi ya ombi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Abuse of Office,realistic-abuse-of-office-22,Sheng,Garissa\n"On 28 July 2026 pale Embakasi East, Nairobi, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-012.",Conflict of Interest,realistic-conflict-of-interest-12,Mixed,Nairobi\n"On 17 February 2026, I dealt with the hospital desk in Kilifi North, Kilifi. The office refused to explain why the queue was suspended and gave no complaint channel. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Other,realistic-other-14,English,Kilifi\n"Tender ya zabuni ya ujenzi wa madarasa pale Garissa Township, Garissa ilikuwa imepangwa. jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo, around 5000, nimehifadhi screenshot na receipt. Ref KW-059.",Procurement Fraud,realistic-procurement-fraud-59,Sheng,Garissa\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Extortion,realistic-extortion-54,Sheng,Eldoret\n"I am reporting irregular use of KSh 20,000 in Kilifi North, Kilifi, after 22 April 2026. Records show that the school feeding committee handled lunch programme funds, and the funds worth KSh 300 were used to buy items not delivered to the school. I kept the receipt and the message thread. Case note KW-043.",Embezzlement,realistic-embezzlement-43,English,Kilifi\n"Tender ya samani za ofisi pale Imenti Central, Meru ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Procurement Fraud,realistic-procurement-fraud-60,Sheng,Meru\n"I am reporting irregular use of KSh 5,000 in Embakasi East, Nairobi, after 6 May 2026. Records show that the relief distribution committee handled food and blankets, and the funds worth KSh 20,000 were recorded as delivered but never reached beneficiaries. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Embezzlement,realistic-embezzlement-24,English,Nairobi\n"Juzi 14 June 2026 nikaingia pale office ya Imenti Central, Meru, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Bribery,realistic-bribery-34,Sheng,Meru\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Kapseret, Eldoret. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Conflict of Interest,realistic-conflict-of-interest-60,English,Eldoret\n"On 22 April 2026 pale Imenti Central, Meru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-039.",Extortion,realistic-extortion-39,Mixed,Meru\n"On 3 January 2026 pale Garissa Township, Garissa, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-002.",Abuse of Office,realistic-abuse-of-office-02,Mixed,Garissa\n"I checked a police clearance letter relating to the public application in Machakos Central, Machakos, on 17 February 2026. The document used a fabricated stamp to make an unapproved clearance appear valid. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-24,English,Machakos\n"On 9 March 2026 pale Embakasi East, Nairobi, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Abuse of Office,realistic-abuse-of-office-60,Sheng,Nairobi\n"Ninawasilisha tuhuma za matumizi mabaya ya mapato ya ada za soko katika Likoni, Mombasa, tangu 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-031.",Embezzlement,realistic-embezzlement-31,Kiswahili,Mombasa\n"On 17 February 2026 pale Likoni, Mombasa, mkuu wa shule wakati wa shule; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-047.",Abuse of Office,realistic-abuse-of-office-47,Mixed,Mombasa\n"Tarehe 17 February 2026, nilishughulika na dirisha la hospitali katika Machakos Central, Machakos. kwa nini foleni ilisitishwa bila kutoa njia ya kulalamika. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Other,realistic-other-30,Kiswahili,Machakos\n"I am reporting irregular use of KSh 20,000 in Machakos Central, Machakos, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 1,000 was withdrawn but the borehole was never completed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Embezzlement,realistic-embezzlement-09,English,Machakos\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Imenti Central, Meru, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Bribery,realistic-bribery-44,Kiswahili,Meru\n"On 11 August 2026, I dealt with the public counter in Likoni, Mombasa. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Other,realistic-other-12,English,Mombasa\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Kisumu Central, Kisumu, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Bribery,realistic-bribery-35,Kiswahili,Kisumu\n"On 6 May 2026, I dealt with the roads office in Kapseret, Eldoret. The office promised resurfacing for our street but no work began before the deadline. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-013.",Other,realistic-other-13,English,Eldoret\n"On 22 April 2026 in Imenti Central, Meru, the ward administrator blocked my valid application without recorded reasons because I challenged a previous decision. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-033.",Abuse of Office,realistic-abuse-of-office-33,English,Meru\n"On 28 July 2026 pale Embakasi East, Nairobi, mjumbe wa bodi ya leseni aliingia decision ya alikuwa mkurugenzi; was a director, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Conflict of Interest,realistic-conflict-of-interest-12,Sheng,Nairobi\n"Katika Nakuru East, Nakuru, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Nepotism/Favoritism,realistic-nepotism-favoritism-34,Kiswahili,Nakuru\n"Nili-check document ya kilitiwa sahihi pale Kapseret, Eldoret, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-017.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-17,Sheng,Eldoret\n"On 28 July 2026 in Embakasi East, Nairobi, the county officer changed a queue order to serve a personal acquaintance first while others were told to wait. This was not a routine decision. I kept the receipt and the message thread. Case note KW-040.",Abuse of Office,realistic-abuse-of-office-40,English,Nairobi\n"I checked a completion certificate relating to the public works file in Bungoma Central, Bungoma, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-025.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-25,English,Bungoma\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Kapseret, Eldoret. bila taarifa wala haikutangaza wakati huduma ingerejea. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Other,realistic-other-33,Kiswahili,Eldoret\n"Kwa usambazaji wa mafuta katika Machakos Central, Machakos, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Tukio linahusu usambazaji wa mafuta yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-032.",Procurement Fraud,realistic-procurement-fraud-32,Kiswahili,Machakos\n"Katika Nakuru East, Nakuru, tarehe 14 June 2026, bodi ya CDF bila kura ya wazi. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Nepotism/Favoritism,realistic-nepotism-favoritism-14,Kiswahili,Nakuru\n"Tarehe 6 May 2026, nilienda ofisi ya umma katika Bungoma Central, Bungoma, kwa ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha; Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Bribery,realistic-bribery-37,Kiswahili,Bungoma\n"Tarehe 11 August 2026 katika Imenti Central, Meru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-019.",Extortion,realistic-extortion-19,Kiswahili,Meru\n"Kuna doo za pesa za dawa zilipotea pale Kapseret, Eldoret after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Embezzlement,realistic-embezzlement-12,Sheng,Eldoret\n"On 6 May 2026 in Embakasi East, Nairobi, the inspector threatened to close my kiosk and demanded KSh 3,000, about KSh 20,000. They and demanded an immediate private payment. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Extortion,realistic-extortion-36,English,Nairobi\n"For tender SEC-55 in Bungoma Central, Bungoma, the procuring entity gave one supplier confidential bid information. The file concerns security services and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-023.",Procurement Fraud,realistic-procurement-fraud-23,English,Bungoma\n"Nilienda pale office ya Garissa Township, Garissa, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 300, na screenshot na receipt bado. Case/rekodi KW-043.",Bribery,realistic-bribery-43,Mixed,Garissa\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa ya mradi wa vijana katika Bungoma Central, Bungoma, tangu 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-030.",Embezzlement,realistic-embezzlement-30,Kiswahili,Bungoma\n"On 11 August 2026 pale Machakos Central, Machakos, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-051.",Extortion,realistic-extortion-51,Sheng,Machakos\n"Tarehe 11 August 2026 katika Machakos Central, Machakos, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Extortion,realistic-extortion-11,Kiswahili,Machakos\n"Tarehe 9 March 2026 katika Kapseret, Eldoret, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-028.",Abuse of Office,realistic-abuse-of-office-28,Kiswahili,Eldoret\n"On 3 January 2026 pale Garissa Township, Garissa, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-002.",Abuse of Office,realistic-abuse-of-office-02,Sheng,Garissa\n"Nimekagua hati kuhusu ilibadilishwa katika Bungoma Central, Bungoma, tarehe 3 January 2026. rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-035.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-35,Kiswahili,Bungoma\n"In Kisumu Central, Kisumu, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Nepotism/Favoritism,realistic-nepotism-favoritism-17,English,Kisumu\n"Tarehe 17 February 2026, nilishughulika na dirisha la hospitali katika Kilifi North, Kilifi. kwa nini foleni ilisitishwa bila kutoa njia ya kulalamika. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-014.",Other,realistic-other-14,Kiswahili,Kilifi\n"On 3 January 2026, the school board member took part in a decision about a tender for uniforms in Kapseret, Eldoret. The officer employed the bidder as a private consultant. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Conflict of Interest,realistic-conflict-of-interest-30,English,Eldoret\n"Kwa tender ya huduma za ulinzi pale Bungoma Central, Bungoma, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-023.",Procurement Fraud,realistic-procurement-fraud-23,Mixed,Bungoma\n"In Garissa Township, Garissa, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-025.",Nepotism/Favoritism,realistic-nepotism-favoritism-25,English,Garissa\n"Pale Machakos Central, Machakos on 14 June 2026, bodi ya CDF bila kura ya wazi; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Nepotism/Favoritism,realistic-nepotism-favoritism-38,Sheng,Machakos\n"On 11 August 2026 pale Nakuru East, Nakuru, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-033.",Conflict of Interest,realistic-conflict-of-interest-33,Sheng,Nakuru\n"Kwa tender ya mifumo ya kompyuta na mtandao pale Garissa Township, Garissa, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Value ilikuwa around 5,000, na screenshot na receipt bado. Case/rekodi KW-029.",Procurement Fraud,realistic-procurement-fraud-29,Mixed,Garissa\n"In Bungoma Central, Bungoma, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-019.",Nepotism/Favoritism,realistic-nepotism-favoritism-19,English,Bungoma\n"Kwa tender ya samani za ofisi pale Machakos Central, Machakos, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-012.",Procurement Fraud,realistic-procurement-fraud-12,Mixed,Machakos\n"On 3 January 2026 pale Kapseret, Eldoret, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-058.",Abuse of Office,realistic-abuse-of-office-58,Mixed,Eldoret\n"Tarehe 6 May 2026, nilishughulika na ofisi ya barabara katika Garissa Township, Garissa. kwenye barabara yetu lakini hakuna kazi iliyoanza kabla ya muda. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-037.",Other,realistic-other-37,Kiswahili,Garissa\n"On 22 April 2026 pale Kilifi North, Kilifi, afisa wa fedha wa kaunti joined the decision on alikuwa mmiliki binafsi; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-021.",Conflict of Interest,realistic-conflict-of-interest-21,Mixed,Kilifi\n"Kuna doo za pesa za dawa zilipotea pale Bungoma Central, Bungoma after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-060.",Embezzlement,realistic-embezzlement-60,Sheng,Bungoma\n"I checked a meeting minute relating to the committee record in Nakuru East, Nakuru, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-20,English,Nakuru\n"Kuna money misuse kwa pesa ya mradi wa vijana pale Embakasi East, Nairobi, after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 500, nimeweka details ya time na office number. Case/rekodi KW-014.",Embezzlement,realistic-embezzlement-14,Mixed,Nairobi\n"Tarehe 11 August 2026 katika Imenti Central, Meru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Extortion,realistic-extortion-59,Kiswahili,Meru\n"Kuna doo za fedha za wanafunzi zilipotea pale Bungoma Central, Bungoma after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Embezzlement,realistic-embezzlement-50,Sheng,Bungoma\n"Kuna doo za fedha za wanafunzi zilipotea pale Bungoma Central, Bungoma after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Embezzlement,realistic-embezzlement-10,Sheng,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za wanafunzi katika Garissa Township, Garissa, tangu 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Embezzlement,realistic-embezzlement-26,Kiswahili,Garissa\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za ukarabati wa barabara katika Kilifi North, Kilifi, tangu 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-013.",Embezzlement,realistic-embezzlement-13,Kiswahili,Kilifi\n"Tender ya samani za ofisi pale Nakuru East, Nakuru ilikuwa imepangwa. ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Procurement Fraud,realistic-procurement-fraud-28,Sheng,Nakuru\n"Tarehe 11 August 2026, nilishughulika na dirisha la umma katika Kilifi North, Kilifi. kwa fomu ya kawaida bila risiti rasmi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Other,realistic-other-04,Kiswahili,Kilifi\n"Tarehe 3 January 2026, mjumbe wa bodi ya shule alishiriki uamuzi kuhusu aliwahi kuajiri katika Bungoma Central, Bungoma. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Conflict of Interest,realistic-conflict-of-interest-38,Kiswahili,Bungoma\n"Kwa zabuni ya ujenzi wa madarasa katika Garissa Township, Garissa, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Procurement Fraud,realistic-procurement-fraud-59,Kiswahili,Garissa\n"Kwa tender ya zabuni ya vifaa vya hospitali pale Nakuru East, Nakuru, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-018.",Procurement Fraud,realistic-procurement-fraud-18,Mixed,Nakuru\n"Pale Kisumu Central, Kisumu on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-009.",Other,realistic-other-09,Sheng,Kisumu\n"For contract SUP-77 in Machakos Central, Machakos, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 2,500. I kept the receipt and the message thread. Case note KW-052.",Procurement Fraud,realistic-procurement-fraud-52,English,Machakos\n"Pale Nakuru East, Nakuru on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Nepotism/Favoritism,realistic-nepotism-favoritism-44,Sheng,Nakuru\n"Nilienda pale office ya Likoni, Mombasa, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 2,500, nimeweka details ya time na office number. Case/rekodi KW-028.",Bribery,realistic-bribery-28,Mixed,Mombasa\n"Kwa tender ya samani za ofisi pale Likoni, Mombasa, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-004.",Procurement Fraud,realistic-procurement-fraud-04,Mixed,Mombasa\n"On 11 August 2026 pale Nakuru East, Nakuru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-027.",Extortion,realistic-extortion-27,Mixed,Nakuru\n"Pale Embakasi East, Nairobi on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-015.",Other,realistic-other-15,Sheng,Nairobi\n"Pale Bungoma Central, Bungoma on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-009.",Nepotism/Favoritism,realistic-nepotism-favoritism-09,Sheng,Bungoma\n"Pale Kapseret, Eldoret on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no action taken, na screenshot na receipt bado. Case/rekodi KW-043.",Other,realistic-other-43,Mixed,Eldoret\n"Tarehe 28 July 2026 katika Embakasi East, Nairobi, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-006.",Extortion,realistic-extortion-06,Kiswahili,Nairobi\n"I am reporting irregular use of KSh 20,000 in Garissa Township, Garissa, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 20,000 paid a supplier for drugs that are absent from the store. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Embezzlement,realistic-embezzlement-36,English,Garissa\n"Katika Imenti Central, Meru, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Nepotism/Favoritism,realistic-nepotism-favoritism-26,Kiswahili,Meru\n"Kwa tender ya ukodishaji wa magari pale Machakos Central, Machakos, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-022.",Procurement Fraud,realistic-procurement-fraud-22,Mixed,Machakos\n"Pale Bungoma Central, Bungoma on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-059.",Nepotism/Favoritism,realistic-nepotism-favoritism-59,Sheng,Bungoma\n"Kwa zabuni ya ukarabati wa barabara katika Kisumu Central, Kisumu, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Tukio linahusu zabuni ya ukarabati wa barabara yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-041.",Procurement Fraud,realistic-procurement-fraud-41,Kiswahili,Kisumu\n"On 6 May 2026 in Bungoma Central, Bungoma, the records officer refused access to my own file despite an approved request and threatened to remove the request record. This was not a routine decision. I kept the receipt and the message thread. Case note KW-046.",Abuse of Office,realistic-abuse-of-office-46,English,Bungoma\n"On 17 February 2026 pale Kilifi North, Kilifi, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-011.",Conflict of Interest,realistic-conflict-of-interest-11,Sheng,Kilifi\n"On 11 August 2026 pale Imenti Central, Meru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-053.",Abuse of Office,realistic-abuse-of-office-53,Sheng,Meru\n"On 6 May 2026, the project director took part in a decision about a construction contract in Kapseret, Eldoret. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Conflict of Interest,realistic-conflict-of-interest-50,English,Eldoret\n"Tender ya ukodishaji wa magari pale Kilifi North, Kilifi ilikuwa imepangwa. kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Procurement Fraud,realistic-procurement-fraud-06,Sheng,Kilifi\n"Kuna doo za mapato ya ada za soko zilipotea pale Likoni, Mombasa after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 300, nimehifadhi screenshot na receipt. Ref KW-031.",Embezzlement,realistic-embezzlement-31,Sheng,Mombasa\n"Nilienda pale office ya Imenti Central, Meru, on 22 April 2026 for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada, 20,000, nimeweka details ya time na office number. Case/rekodi KW-024.",Bribery,realistic-bribery-24,Mixed,Meru\n"Katika Imenti Central, Meru, tarehe 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Nepotism/Favoritism,realistic-nepotism-favoritism-16,Kiswahili,Meru\n"Tarehe 14 June 2026, nilienda ofisi ya umma katika Kilifi North, Kilifi, kwa leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Bribery,realistic-bribery-10,Kiswahili,Kilifi\n"On 28 July 2026 pale Kisumu Central, Kisumu, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-030.",Extortion,realistic-extortion-30,Mixed,Kisumu\n"Tarehe 3 January 2026 katika Embakasi East, Nairobi, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Extortion,realistic-extortion-56,Kiswahili,Nairobi\n"Kwa mifumo ya kompyuta na mtandao katika Kisumu Central, Kisumu, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Tukio linahusu mifumo ya kompyuta na mtandao yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Procurement Fraud,realistic-procurement-fraud-21,Kiswahili,Kisumu\n"On 11 August 2026 in Nakuru East, Nakuru, the department director used an official vehicle for a private family trip and signed the fuel claim through the office. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Abuse of Office,realistic-abuse-of-office-21,English,Nakuru\n"Kwa tender ya samani za ofisi pale Kilifi North, Kilifi, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-036.",Procurement Fraud,realistic-procurement-fraud-36,Mixed,Kilifi\n"Pale Kilifi North, Kilifi on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-052.",Nepotism/Favoritism,realistic-nepotism-favoritism-52,Sheng,Kilifi\n"Nimekagua hati kuhusu ilikuwa na sahihi iliyorudishwa katika Kapseret, Eldoret, tarehe 6 May 2026. vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-047.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-47,Kiswahili,Eldoret\n"Nimekagua hati kuhusu kilitiwa sahihi katika Embakasi East, Nairobi, tarehe 28 July 2026. cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-009.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-09,Kiswahili,Nairobi\n"Tarehe 28 July 2026 katika Bungoma Central, Bungoma, afisa wa kaunti wengine wakaambiwa wangojee. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Abuse of Office,realistic-abuse-of-office-16,Kiswahili,Bungoma\n"On 3 January 2026 pale Kisumu Central, Kisumu, mjumbe wa bodi ya shule aliingia decision ya aliwahi kuajiri; employed, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Conflict of Interest,realistic-conflict-of-interest-46,Sheng,Kisumu\n"Pale Kapseret, Eldoret on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-041.",Nepotism/Favoritism,realistic-nepotism-favoritism-41,Sheng,Eldoret\n"Pale Kilifi North, Kilifi on 14 June 2026, bodi ya CDF bila kura ya wazi; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-022.",Nepotism/Favoritism,realistic-nepotism-favoritism-22,Sheng,Kilifi\n"In Kisumu Central, Kisumu, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Nepotism/Favoritism,realistic-nepotism-favoritism-27,English,Kisumu\n"Tarehe 11 August 2026 katika Likoni, Mombasa, afisa wa kizuizi akitishia kurekodi dosari ya uongo, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Extortion,realistic-extortion-43,Kiswahili,Mombasa\n"Tarehe 22 April 2026 katika Imenti Central, Meru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Extortion,realistic-extortion-39,Kiswahili,Meru\n"On 17 February 2026 pale Imenti Central, Meru, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-035.",Conflict of Interest,realistic-conflict-of-interest-35,Mixed,Meru\n"Pale Likoni, Mombasa on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Nepotism/Favoritism,realistic-nepotism-favoritism-40,Sheng,Mombasa\n"Kuna money misuse kwa fedha za kisima pale Machakos Central, Machakos, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 1,000, na screenshot na receipt bado. Case/rekodi KW-009.",Embezzlement,realistic-embezzlement-09,Mixed,Machakos\n"Ninawasilisha tuhuma za matumizi mabaya ya chakula na blanketi katika Kisumu Central, Kisumu, tangu 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Embezzlement,realistic-embezzlement-08,Kiswahili,Kisumu\n"On 11 August 2026 pale Machakos Central, Machakos, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-017.",Conflict of Interest,realistic-conflict-of-interest-17,Mixed,Machakos\n"On 9 March 2026 pale Bungoma Central, Bungoma, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-036.",Abuse of Office,realistic-abuse-of-office-36,Mixed,Bungoma\n"On 28 July 2026 pale Kapseret, Eldoret, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-060.",Conflict of Interest,realistic-conflict-of-interest-60,Mixed,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Machakos Central, Machakos, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-059.",Embezzlement,realistic-embezzlement-59,Kiswahili,Machakos\n"Nilikagua document for kilikuwa na jina lililobadilishwa pale Bungoma Central, Bungoma, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, na screenshot na receipt bado. Case/rekodi KW-005.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-05,Mixed,Bungoma\n"On 14 June 2026, the grant panel chair took part in a decision about a community grant in Likoni, Mombasa. The officer was a partner in the organisation applying for funds. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Conflict of Interest,realistic-conflict-of-interest-39,English,Mombasa\n"Pale Bungoma Central, Bungoma, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-019.",Nepotism/Favoritism,realistic-nepotism-favoritism-19,Mixed,Bungoma\n"Nilienda pale office ya Garissa Township, Garissa, on 6 May 2026 for ruhusa ya kuuza sokoni. askari wa kaunti aliahidi kuacha tatizo la kibanda kwa malipo ya faragha, 300, na screenshot na receipt bado. Case/rekodi KW-013.",Bribery,realistic-bribery-13,Mixed,Garissa\n"On 6 May 2026 pale Kisumu Central, Kisumu, afisa wa rekodi akatishia kuondoa rekodi ya ombi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-054.",Abuse of Office,realistic-abuse-of-office-54,Sheng,Kisumu\n"Pale Embakasi East, Nairobi on 6 May 2026, ofisi ya barabara lakini hakuna kazi iliyoanza kabla ya muda; no action taken, na screenshot na receipt bado. Case/rekodi KW-005.",Other,realistic-other-05,Mixed,Nairobi\n"Kwa tender ya zabuni ya ukarabati wa barabara pale Embakasi East, Nairobi, kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-057.",Procurement Fraud,realistic-procurement-fraud-57,Mixed,Nairobi\n"Nili-check document ya kilikuwa na jina lililobadilishwa pale Kapseret, Eldoret, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, nimehifadhi screenshot na receipt. Ref KW-037.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-37,Sheng,Eldoret\n"On 22 April 2026, I went to the public office in Kilifi North, Kilifi, for an approved police abstract. The the police abstract clerk said the document would not be issued without an extra amount; the demand was KSh 2,500. I kept the receipt and the message thread. Case note KW-040.",Bribery,realistic-bribery-40,English,Kilifi\n"On 14 June 2026 pale Nakuru East, Nakuru, mwenyekiti wa jopo la ruzuku joined the decision on alikuwa mshirika; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-023.",Conflict of Interest,realistic-conflict-of-interest-23,Mixed,Nakuru\n"Nili-check document ya ilibadilishwa pale Kisumu Central, Kisumu, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-043.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-43,Sheng,Kisumu\n"On 9 March 2026 pale Kapseret, Eldoret, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-028.",Abuse of Office,realistic-abuse-of-office-28,Mixed,Eldoret\n"Nili-check document ya kilitiwa sahihi pale Kisumu Central, Kisumu, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, nimehifadhi screenshot na receipt. Ref KW-033.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-33,Sheng,Kisumu\n"Nilienda pale office ya Embakasi East, Nairobi, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 5,000, na screenshot na receipt bado. Case/rekodi KW-011.",Bribery,realistic-bribery-11,Mixed,Nairobi\n"Pale Likoni, Mombasa on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-052.",Other,realistic-other-52,Mixed,Mombasa\n"I checked a business permit relating to the revenue office in Machakos Central, Machakos, on 11 August 2026. The document showed a false address and a permit number not in the official register. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-14,English,Machakos\n"On 6 May 2026 pale Garissa Township, Garissa, mkaguzi akadai malipo ya faragha mara moja, walidai 2,500, nimeweka details ya time na office number. Case/rekodi KW-028.",Extortion,realistic-extortion-28,Mixed,Garissa\n"I am reporting irregular use of KSh 2,500 in Likoni, Mombasa, after 11 August 2026. Records show that the county market cashier handled weekly market fee collections, and the funds worth KSh 300 were missing from the ledger and bank deposit. I kept the receipt and the message thread. Case note KW-031.",Embezzlement,realistic-embezzlement-31,English,Mombasa\n"Tender ya mifumo ya kompyuta na mtandao pale Garissa Township, Garissa ilikuwa imepangwa. timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa, around 5000, nimehifadhi screenshot na receipt. Ref KW-029.",Procurement Fraud,realistic-procurement-fraud-29,Sheng,Garissa\n"Kuna doo za fedha za ukarabati wa barabara zilipotea pale Kilifi North, Kilifi after 14 June 2026. msimamizi wa kazi za umma zilitumika huku sehemu iliyoharibika bado haijarekebishwa, 300, nimehifadhi screenshot na receipt. Ref KW-013.",Embezzlement,realistic-embezzlement-13,Sheng,Kilifi\n"Pale Kisumu Central, Kisumu, on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili, benefit ilikuwa 300, na screenshot na receipt bado. Case/rekodi KW-037.",Nepotism/Favoritism,realistic-nepotism-favoritism-37,Mixed,Kisumu\n"Pale Kilifi North, Kilifi on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no action taken, nimeweka details ya time na office number. Case/rekodi KW-054.",Other,realistic-other-54,Mixed,Kilifi\n"Kwa huduma za ulinzi katika Kapseret, Eldoret, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Tukio linahusu huduma za ulinzi yenye thamani ya Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Procurement Fraud,realistic-procurement-fraud-15,Kiswahili,Eldoret\n"In Garissa Township, Garissa, on 3 January 2026, the appointment board promoted an officer\'s spouse to a senior post despite a lower interview score. The estimated benefit or contract value was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Nepotism/Favoritism,realistic-nepotism-favoritism-05,English,Garissa\n"On 17 February 2026, I dealt with the hospital desk in Machakos Central, Machakos. The office refused to explain why the queue was suspended and gave no complaint channel. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Other,realistic-other-30,English,Machakos\n"On 28 July 2026 pale Kisumu Central, Kisumu, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-036.",Conflict of Interest,realistic-conflict-of-interest-36,Mixed,Kisumu\n"For contract SUP-77 in Likoni, Mombasa, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Procurement Fraud,realistic-procurement-fraud-44,English,Mombasa\n"I am reporting irregular use of KSh 5,000 in Nakuru East, Nakuru, after 14 June 2026. Records show that the public works foreman handled road-repair funds, and the funds worth KSh 1,000 were spent while the damaged section remained unfinished. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Embezzlement,realistic-embezzlement-45,English,Nakuru\n"Kuna doo za mapato ya ada za soko zilipotea pale Machakos Central, Machakos after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 1000, nimehifadhi screenshot na receipt. Ref KW-039.",Embezzlement,realistic-embezzlement-39,Sheng,Machakos\n"Nilienda pale office ya Nakuru East, Nakuru, on 22 April 2026 for abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada, 500, nimeweka details ya time na office number. Case/rekodi KW-032.",Bribery,realistic-bribery-32,Mixed,Nakuru\n"For contract SUP-77 in Imenti Central, Meru, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-060.",Procurement Fraud,realistic-procurement-fraud-60,English,Meru\n"Kwa usambazaji wa mafuta katika Nakuru East, Nakuru, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Tukio linahusu usambazaji wa mafuta yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Procurement Fraud,realistic-procurement-fraud-08,Kiswahili,Nakuru\n"Juzi 3 January 2026 nikaingia pale office ya Bungoma Central, Bungoma, for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili 1000, nimehifadhi screenshot na receipt. Ref KW-057.",Bribery,realistic-bribery-57,Sheng,Bungoma\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Embakasi East, Nairobi, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-004.",Embezzlement,realistic-embezzlement-04,Kiswahili,Nairobi\n"On 9 March 2026, I went to the public office in Kapseret, Eldoret, for a food-premises certificate. The the public health inspector asked for a gift before signing the certificate; the demand was KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Bribery,realistic-bribery-59,English,Eldoret\n"Nilienda pale office ya Kapseret, Eldoret, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 1,000, na screenshot na receipt bado. Case/rekodi KW-009.",Bribery,realistic-bribery-09,Mixed,Eldoret\n"On 6 May 2026 pale Embakasi East, Nairobi, mkurugenzi wa mradi joined the decision on pia alikuwa mwenye; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-002.",Conflict of Interest,realistic-conflict-of-interest-02,Mixed,Nairobi\n"On 17 February 2026 pale Nakuru East, Nakuru, katibu wa kamati joined the decision on alilipwa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-043.",Conflict of Interest,realistic-conflict-of-interest-43,Mixed,Nakuru\n"On 17 February 2026 pale Likoni, Mombasa, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-059.",Conflict of Interest,realistic-conflict-of-interest-59,Sheng,Mombasa\n"On 22 April 2026, I dealt with the public help line in Imenti Central, Meru. The office did not answer for repeated calls and provided no follow-up reference. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-008.",Other,realistic-other-08,English,Meru\n"On 3 January 2026 pale Bungoma Central, Bungoma, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, walidai 500, nimeweka details ya time na office number. Case/rekodi KW-032.",Extortion,realistic-extortion-32,Mixed,Bungoma\n"On 14 June 2026 in Kilifi North, Kilifi, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-059.",Abuse of Office,realistic-abuse-of-office-59,English,Kilifi\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Garissa Township, Garissa. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-034.",Conflict of Interest,realistic-conflict-of-interest-34,Kiswahili,Garissa\n"Nilienda pale office ya Kilifi North, Kilifi, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 20,000, nimeweka details ya time na office number. Case/rekodi KW-060.",Bribery,realistic-bribery-60,Mixed,Kilifi\n"Nilikagua document for ilikuwa na sahihi iliyorudishwa pale Kapseret, Eldoret, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, na screenshot na receipt bado. Case/rekodi KW-007.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-07,Mixed,Eldoret\n"On 11 August 2026, I dealt with the public counter in Kilifi North, Kilifi. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-004.",Other,realistic-other-04,English,Kilifi\n"Juzi 11 August 2026 nikaingia pale office ya Nakuru East, Nakuru, for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-012.",Bribery,realistic-bribery-12,Sheng,Nakuru\n"Kwa samani za ofisi katika Machakos Central, Machakos, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Procurement Fraud,realistic-procurement-fraud-12,Kiswahili,Machakos\n"On 17 February 2026 pale Imenti Central, Meru, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-029.",Extortion,realistic-extortion-29,Mixed,Meru\n"Katika Kapseret, Eldoret, tarehe 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-021.",Nepotism/Favoritism,realistic-nepotism-favoritism-21,Kiswahili,Eldoret\n"Nimekagua hati kuhusu ilitumia muhuri wa kughushi katika Machakos Central, Machakos, tarehe 17 February 2026. barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-024.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-24,Kiswahili,Machakos\n"Nilikagua document for ilihaririwa pale Kilifi North, Kilifi, on 14 June 2026; kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali, nimeweka details ya time na office number. Case/rekodi KW-028.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-28,Mixed,Kilifi\n"Tarehe 28 July 2026, nilishughulika na tovuti ya kaunti katika Kisumu Central, Kisumu. bila sababu iliyochapishwa baada ya wakazi kuomba huduma. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Other,realistic-other-39,Kiswahili,Kisumu\n"Kuna money misuse kwa chakula na blanketi pale Garissa Township, Garissa, after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 500, nimeweka details ya time na office number. Case/rekodi KW-056.",Embezzlement,realistic-embezzlement-56,Mixed,Garissa\n"On 14 June 2026 in Machakos Central, Machakos, the police officer held my identification card and demanded KSh 2,500, about KSh 5,000. They threatening arrest unless I paid in cash. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Extortion,realistic-extortion-41,English,Machakos\n"Pale Bungoma Central, Bungoma on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-039.",Nepotism/Favoritism,realistic-nepotism-favoritism-39,Sheng,Bungoma\n"I am reporting irregular use of KSh 20,000 in Kapseret, Eldoret, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 2,500 paid a supplier for drugs that are absent from the store. I kept the receipt and the message thread. Case note KW-052.",Embezzlement,realistic-embezzlement-52,English,Eldoret\n"On 11 August 2026 pale Kilifi North, Kilifi, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-001.",Conflict of Interest,realistic-conflict-of-interest-01,Mixed,Kilifi\n"On 9 March 2026 in Bungoma Central, Bungoma, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-036.",Abuse of Office,realistic-abuse-of-office-36,English,Bungoma\n"On 9 March 2026 pale Kisumu Central, Kisumu, afisa wa uteuzi wa umma joined the decision on alikuwa mwenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-056.",Conflict of Interest,realistic-conflict-of-interest-56,Mixed,Kisumu\n"Katika Embakasi East, Nairobi, tarehe 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Nepotism/Favoritism,realistic-nepotism-favoritism-23,Kiswahili,Nairobi\n"Kuna doo za fedha za wanafunzi zilipotea pale Kisumu Central, Kisumu after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Embezzlement,realistic-embezzlement-18,Sheng,Kisumu\n"On 6 May 2026 pale Kapseret, Eldoret, mkaguzi akadai malipo ya faragha mara moja, 2500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Extortion,realistic-extortion-04,Sheng,Eldoret\n"Tarehe 22 April 2026, nilienda ofisi ya umma katika Likoni, Mombasa, kwa abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada; Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-048.",Bribery,realistic-bribery-48,Kiswahili,Mombasa\n"Pale Garissa Township, Garissa on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-055.",Nepotism/Favoritism,realistic-nepotism-favoritism-55,Sheng,Garissa\n"Pale Likoni, Mombasa on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-012.",Other,realistic-other-12,Mixed,Mombasa\n"Tarehe 28 July 2026, nilienda ofisi ya umma katika Kapseret, Eldoret, kwa uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Bribery,realistic-bribery-39,Kiswahili,Eldoret\n"Katika Embakasi East, Nairobi, tarehe 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Nepotism/Favoritism,realistic-nepotism-favoritism-33,Kiswahili,Nairobi\n"On 3 January 2026, the school board member took part in a decision about a tender for uniforms in Garissa Township, Garissa. The officer employed the bidder as a private consultant. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-014.",Conflict of Interest,realistic-conflict-of-interest-14,English,Garissa\n"Kuna money misuse kwa pesa za dawa pale Bungoma Central, Bungoma, after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 20,000, nimeweka details ya time na office number. Case/rekodi KW-060.",Embezzlement,realistic-embezzlement-60,Mixed,Bungoma\n"Tarehe 28 July 2026 katika Kapseret, Eldoret, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Extortion,realistic-extortion-54,Kiswahili,Eldoret\n"Kuna doo za mapato ya ada za soko zilipotea pale Nakuru East, Nakuru after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 300, nimehifadhi screenshot na receipt. Ref KW-055.",Embezzlement,realistic-embezzlement-55,Sheng,Nakuru\n"Nimekagua hati kuhusu kilikuwa na jina lililobadilishwa katika Bungoma Central, Bungoma, tarehe 9 March 2026. cheti cha elimu na muhuri ambao shule inasema si halisi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-05,Kiswahili,Bungoma\n"Tarehe 17 February 2026 katika Kilifi North, Kilifi, mkuu wa shule wakati wa shule. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Abuse of Office,realistic-abuse-of-office-39,Kiswahili,Kilifi\n"On 3 January 2026, I dealt with the public office in Kisumu Central, Kisumu. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Other,realistic-other-09,English,Kisumu\n"Juzi 17 February 2026 nikaingia pale office ya Machakos Central, Machakos, for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-006.",Bribery,realistic-bribery-06,Sheng,Machakos\n"I am reporting irregular use of KSh 20,000 in Likoni, Mombasa, after 17 February 2026. Records show that the CDF committee handled borehole money, and the funds worth KSh 300 was withdrawn but the borehole was never completed. I kept the receipt and the message thread. Case note KW-001.",Embezzlement,realistic-embezzlement-01,English,Mombasa\n"On 9 March 2026 in Garissa Township, Garissa, the public programme head reserved a vacant opportunity for a person outside the published process and ordered staff to alter the applicant list. This was not a routine decision. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Abuse of Office,realistic-abuse-of-office-12,English,Garissa\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Bungoma Central, Bungoma, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-057.",Bribery,realistic-bribery-57,Kiswahili,Bungoma\n"In Bungoma Central, Bungoma, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-009.",Nepotism/Favoritism,realistic-nepotism-favoritism-09,English,Bungoma\n"Nilikagua document for ilibadilishwa pale Kisumu Central, Kisumu, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, na screenshot na receipt bado. Case/rekodi KW-003.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-03,Mixed,Kisumu\n"Tender ya zabuni ya vifaa vya hospitali pale Nakuru East, Nakuru ilikuwa imepangwa. afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani, around 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Procurement Fraud,realistic-procurement-fraud-18,Sheng,Nakuru\n"Pale Bungoma Central, Bungoma on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-019.",Nepotism/Favoritism,realistic-nepotism-favoritism-19,Sheng,Bungoma\n"Tarehe 3 January 2026 katika Embakasi East, Nairobi, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Extortion,realistic-extortion-16,Kiswahili,Nairobi\n"For tender ICT-09 in Kapseret, Eldoret, the bid evaluation team used scoring criteria that were not published in the notice. The file concerns computer and network systems and an estimated value of KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Procurement Fraud,realistic-procurement-fraud-45,English,Eldoret\n"On 3 January 2026 in Bungoma Central, Bungoma, the chief used public staff to collect campaign materials during official working hours. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-026.",Abuse of Office,realistic-abuse-of-office-26,English,Bungoma\n"On 11 August 2026 pale Imenti Central, Meru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, walidai 300, na screenshot na receipt bado. Case/rekodi KW-019.",Extortion,realistic-extortion-19,Mixed,Meru\n"For tender HOS-118 in Machakos Central, Machakos, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Procurement Fraud,realistic-procurement-fraud-42,English,Machakos\n"Nilikagua document for ilitumia muhuri wa kughushi pale Kilifi North, Kilifi, on 17 February 2026; barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali, nimeweka details ya time na office number. Case/rekodi KW-008.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-08,Mixed,Kilifi\n"On 22 April 2026 in Imenti Central, Meru, the gate security supervisor blocked my lorry and demanded KSh 1,500, about KSh 1,000. They threatening to call a taxing inspection. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-039.",Extortion,realistic-extortion-39,English,Meru\n"For tender SCH-204 in Kapseret, Eldoret, the tender panel awarded the bid to a company owned by a panel member. The file concerns classroom construction tender and an estimated value of KSh 5,000. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Procurement Fraud,realistic-procurement-fraud-35,English,Eldoret\n"Kwa tender ya zabuni ya ujenzi wa madarasa pale Embakasi East, Nairobi, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-027.",Procurement Fraud,realistic-procurement-fraud-27,Mixed,Nairobi\n"On 14 June 2026 pale Nakuru East, Nakuru, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-051.",Abuse of Office,realistic-abuse-of-office-51,Sheng,Nakuru\n"I checked a police clearance letter relating to the public application in Kilifi North, Kilifi, on 17 February 2026. The document used a fabricated stamp to make an unapproved clearance appear valid. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-48,English,Kilifi\n"Kwa zabuni ya ujenzi wa madarasa katika Bungoma Central, Bungoma, jopo la zabuni ilitenga zabuni kwa kampuni inayomilikiwa na mjumbe wa jopo. Tukio linahusu zabuni ya ujenzi wa madarasa yenye thamani ya Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Procurement Fraud,realistic-procurement-fraud-43,Kiswahili,Bungoma\n"Nilikagua document for kilikuwa na jina lililobadilishwa pale Bungoma Central, Bungoma, on 9 March 2026; cheti cha elimu na muhuri ambao shule inasema si halisi, na screenshot na receipt bado. Case/rekodi KW-045.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-45,Mixed,Bungoma\n"On 14 June 2026 pale Machakos Central, Machakos, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-041.",Extortion,realistic-extortion-41,Mixed,Machakos\n"Kwa tender ya samani za ofisi pale Likoni, Mombasa, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-044.",Procurement Fraud,realistic-procurement-fraud-44,Mixed,Mombasa\n"On 17 February 2026 in Kilifi North, Kilifi, the public officer threatened to blacklist my supplier account and demanded KSh 5,000, about KSh 1,000. They unless I paid from my personal funds. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Extortion,realistic-extortion-45,English,Kilifi\n"For tender KW-026 in Kapseret, Eldoret, the evaluation committee changed requirements after bids were opened. The file concerns road-maintenance tender and an estimated value of KSh 300. I kept the receipt and the message thread. Case note KW-025.",Procurement Fraud,realistic-procurement-fraud-25,English,Eldoret\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Nakuru East, Nakuru, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-052.",Bribery,realistic-bribery-52,Kiswahili,Nakuru\n"Tarehe 9 March 2026 katika Embakasi East, Nairobi, askari wa kaunti akasema zingerudishwa baada ya malipo, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-026.",Extortion,realistic-extortion-26,Kiswahili,Nairobi\n"On 11 August 2026 pale Imenti Central, Meru, afisa wa kizuizi akitishia kurekodi dosari ya uongo, 5000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-059.",Extortion,realistic-extortion-59,Sheng,Meru\n"Kuna money misuse kwa fedha za kisima pale Machakos Central, Machakos, after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 300, na screenshot na receipt bado. Case/rekodi KW-049.",Embezzlement,realistic-embezzlement-49,Mixed,Machakos\n"Nili-check document ya ilikuwa na sahihi iliyorudishwa pale Kisumu Central, Kisumu, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, nimehifadhi screenshot na receipt. Ref KW-023.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-23,Sheng,Kisumu\n"Tarehe 17 February 2026, nilienda ofisi ya umma katika Nakuru East, Nakuru, kwa utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni; Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-022.",Bribery,realistic-bribery-22,Kiswahili,Nakuru\n"Tarehe 22 April 2026, nilienda ofisi ya umma katika Machakos Central, Machakos, kwa abstract ya polisi. karani wa polisi alisema hati haitaandikwa bila kiasi cha ziada; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Bribery,realistic-bribery-56,Kiswahili,Machakos\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa ya mradi wa vijana katika Garissa Township, Garissa, tangu 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi. Nimeona rekodi zinazoashiria kiasi cha Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Embezzlement,realistic-embezzlement-46,Kiswahili,Garissa\n"On 9 March 2026 pale Embakasi East, Nairobi, afisa wa uteuzi wa umma aliingia decision ya alikuwa mwenzi; was the spouse, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-032.",Conflict of Interest,realistic-conflict-of-interest-32,Sheng,Nairobi\n"For tender VEH-441 in Likoni, Mombasa, the contract committee accepted a bid after the deadline while rejecting timely bidders. The file concerns vehicle leasing and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Procurement Fraud,realistic-procurement-fraud-54,English,Mombasa\n"Pale Likoni, Mombasa on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-010.",Nepotism/Favoritism,realistic-nepotism-favoritism-10,Sheng,Mombasa\n"On 9 March 2026 pale Bungoma Central, Bungoma, afisa wa uteuzi wa umma aliingia decision ya alikuwa mwenzi; was the spouse, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Conflict of Interest,realistic-conflict-of-interest-48,Sheng,Bungoma\n"Nilienda pale office ya Nakuru East, Nakuru, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 500, nimeweka details ya time na office number. Case/rekodi KW-002.",Bribery,realistic-bribery-02,Mixed,Nakuru\n"Tarehe 6 May 2026 katika Bungoma Central, Bungoma, mkaguzi akadai malipo ya faragha mara moja, Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Extortion,realistic-extortion-12,Kiswahili,Bungoma\n"Tarehe 3 January 2026, nilishughulika na ofisi ya umma katika Garissa Township, Garissa. bila taarifa wala haikutangaza wakati huduma ingerejea. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Other,realistic-other-17,Kiswahili,Garissa\n"On 22 April 2026 pale Machakos Central, Machakos, afisa wa fedha wa kaunti aliingia decision ya alikuwa mmiliki binafsi; personally owned, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-037.",Conflict of Interest,realistic-conflict-of-interest-37,Sheng,Machakos\n"On 14 June 2026, I dealt with the service desk in Likoni, Mombasa. The office sent me between offices for the same document and nobody took responsibility. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Other,realistic-other-02,English,Mombasa\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Kilifi North, Kilifi. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-041.",Conflict of Interest,realistic-conflict-of-interest-41,English,Kilifi\n"Kuna money misuse kwa fedha za chakula pale Machakos Central, Machakos, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 5,000, na screenshot na receipt bado. Case/rekodi KW-059.",Embezzlement,realistic-embezzlement-59,Mixed,Machakos\n"On 22 April 2026 pale Imenti Central, Meru, afisa wa fedha wa kaunti joined the decision on alikuwa mmiliki binafsi; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-005.",Conflict of Interest,realistic-conflict-of-interest-05,Mixed,Meru\n"Nili-check document ya ilikuwa na sahihi iliyorudishwa pale Embakasi East, Nairobi, on 6 May 2026; vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji, nimehifadhi screenshot na receipt. Ref KW-039.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-39,Sheng,Nairobi\n"Tarehe 28 July 2026, nilishughulika na tovuti ya kaunti katika Garissa Township, Garissa. bila sababu iliyochapishwa baada ya wakazi kuomba huduma. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Other,realistic-other-07,Kiswahili,Garissa\n"Pale Kisumu Central, Kisumu, on 9 March 2026, programu ya mafunzo ya umma ingawa nafasi haikutangazwa, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-047.",Nepotism/Favoritism,realistic-nepotism-favoritism-47,Mixed,Kisumu\n"Pale Machakos Central, Machakos on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa; benefit/value ilikuwa 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-048.",Nepotism/Favoritism,realistic-nepotism-favoritism-48,Sheng,Machakos\n"Pale Embakasi East, Nairobi on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-003.",Nepotism/Favoritism,realistic-nepotism-favoritism-03,Sheng,Nairobi\n"Kuna doo za fedha za kisima zilipotea pale Imenti Central, Meru after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 1000, nimehifadhi screenshot na receipt. Ref KW-057.",Embezzlement,realistic-embezzlement-57,Sheng,Meru\n"On 9 March 2026, the public appointment officer took part in a decision about a job assessment in Garissa Township, Garissa. The officer was the spouse of one shortlisted candidate. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Conflict of Interest,realistic-conflict-of-interest-24,English,Garissa\n"Katika Garissa Township, Garissa, tarehe 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili. Thamani ya manufaa au mkataba ilikuwa Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Nepotism/Favoritism,realistic-nepotism-favoritism-05,Kiswahili,Garissa\n"On 22 April 2026, I went to the public office in Imenti Central, Meru, for an approved police abstract. The the police abstract clerk said the document would not be issued without an extra amount; the demand was KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-024.",Bribery,realistic-bribery-24,English,Meru\n"Kuna doo za mapato ya ada za soko zilipotea pale Kilifi North, Kilifi after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 5000, nimehifadhi screenshot na receipt. Ref KW-023.",Embezzlement,realistic-embezzlement-23,Sheng,Kilifi\n"Pale Kisumu Central, Kisumu on 3 January 2026, ofisi ya umma wala haikutangaza wakati huduma ingerejea; no action taken, na screenshot na receipt bado. Case/rekodi KW-009.",Other,realistic-other-09,Mixed,Kisumu\n"Pale Imenti Central, Meru, on 14 June 2026, bodi ya CDF bila kura ya wazi, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-006.",Nepotism/Favoritism,realistic-nepotism-favoritism-06,Mixed,Meru\n"Tarehe 22 April 2026, nilishughulika na simu ya msaada ya umma katika Imenti Central, Meru. baada ya simu nyingi bila nambari ya kufuatilia. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Other,realistic-other-08,Kiswahili,Meru\n"On 11 August 2026 pale Machakos Central, Machakos, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-057.",Conflict of Interest,realistic-conflict-of-interest-57,Mixed,Machakos\n"Nimekagua hati kuhusu ilitumia muhuri wa kughushi katika Likoni, Mombasa, tarehe 17 February 2026. barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-016.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-16,Kiswahili,Mombasa\n"Kwa tender ya ukodishaji wa magari pale Imenti Central, Meru, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Value ilikuwa around 20,000, nimeweka details ya time na office number. Case/rekodi KW-030.",Procurement Fraud,realistic-procurement-fraud-30,Mixed,Meru\n"On 28 July 2026 pale Bungoma Central, Bungoma, afisa wa kaunti wengine wakaambiwa wangojee; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Abuse of Office,realistic-abuse-of-office-56,Sheng,Bungoma\n"On 11 August 2026, I dealt with the public counter in Kilifi North, Kilifi. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Other,realistic-other-44,English,Kilifi\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-009.",Abuse of Office,realistic-abuse-of-office-09,Mixed,Kilifi\n"I checked a delivery invoice relating to the procurement file in Imenti Central, Meru, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-042.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-42,English,Meru\n"Nilienda pale office ya Kilifi North, Kilifi, on 17 February 2026 for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni, 20,000, nimeweka details ya time na office number. Case/rekodi KW-030.",Bribery,realistic-bribery-30,Mixed,Kilifi\n"Pale Likoni, Mombasa, on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-040.",Nepotism/Favoritism,realistic-nepotism-favoritism-40,Mixed,Mombasa\n"On 14 June 2026 pale Kilifi North, Kilifi, meneja wa leseni bila kumpa mmiliki utaratibu wa kukata rufaa; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-019.",Abuse of Office,realistic-abuse-of-office-19,Sheng,Kilifi\n"On 3 January 2026 pale Kapseret, Eldoret, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Abuse of Office,realistic-abuse-of-office-18,Sheng,Eldoret\n"On 22 April 2026 pale Likoni, Mombasa, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-057.",Abuse of Office,realistic-abuse-of-office-57,Sheng,Mombasa\n"Pale Machakos Central, Machakos on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-020.",Other,realistic-other-20,Mixed,Machakos\n"Pale Machakos Central, Machakos on 11 August 2026, dirisha la umma bila risiti rasmi; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-020.",Other,realistic-other-20,Sheng,Machakos\n"On 14 June 2026 pale Kilifi North, Kilifi, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 300, na screenshot na receipt bado. Case/rekodi KW-025.",Extortion,realistic-extortion-25,Mixed,Kilifi\n"On 11 August 2026 pale Imenti Central, Meru, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-025.",Conflict of Interest,realistic-conflict-of-interest-25,Mixed,Meru\n"Kwa samani za ofisi katika Imenti Central, Meru, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Tukio linahusu samani za ofisi yenye thamani ya Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Procurement Fraud,realistic-procurement-fraud-60,Kiswahili,Meru\n"Katika Nakuru East, Nakuru, tarehe 14 June 2026, bodi ya CDF bila kura ya wazi. Thamani ya manufaa au mkataba ilikuwa Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Nepotism/Favoritism,realistic-nepotism-favoritism-54,Kiswahili,Nakuru\n"On 6 May 2026 pale Kisumu Central, Kisumu, afisa wa rekodi akatishia kuondoa rekodi ya ombi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-054.",Abuse of Office,realistic-abuse-of-office-54,Mixed,Kisumu\n"Pale Imenti Central, Meru, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-026.",Nepotism/Favoritism,realistic-nepotism-favoritism-26,Mixed,Meru\n"Kuna doo za chakula na blanketi zilipotea pale Bungoma Central, Bungoma after 6 May 2026. kamati ya usambazaji wa msaada ziliandikwa kuwa zimewasilishwa lakini hazikufika kwa walengwa, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Embezzlement,realistic-embezzlement-40,Sheng,Bungoma\n"Pale Kilifi North, Kilifi on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-004.",Other,realistic-other-04,Mixed,Kilifi\n"I checked a land transfer record relating to the county registry in Kapseret, Eldoret, on 3 January 2026. The document was altered to show a transfer that had not been approved. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-027.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-27,English,Eldoret\n"Nilienda pale office ya Garissa Township, Garissa, on 9 March 2026 for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti, 1,000, na screenshot na receipt bado. Case/rekodi KW-003.",Bribery,realistic-bribery-03,Mixed,Garissa\n"I am reporting irregular use of KSh 5,000 in Kisumu Central, Kisumu, after 6 May 2026. Records show that the relief distribution committee handled food and blankets, and the funds worth KSh 20,000 were recorded as delivered but never reached beneficiaries. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-048.",Embezzlement,realistic-embezzlement-48,English,Kisumu\n"Tarehe 22 April 2026, afisa wa fedha wa kaunti alishiriki uamuzi kuhusu alikuwa mmiliki binafsi katika Nakuru East, Nakuru. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Conflict of Interest,realistic-conflict-of-interest-53,Kiswahili,Nakuru\n"Pale Machakos Central, Machakos, on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-008.",Nepotism/Favoritism,realistic-nepotism-favoritism-08,Mixed,Machakos\n"Katika Kilifi North, Kilifi, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Nepotism/Favoritism,realistic-nepotism-favoritism-02,Kiswahili,Kilifi\n"Kwa tender ya samani za ofisi pale Imenti Central, Meru, ofisi ya vifaa ya kaunti ililipa vitengo 40 lakini ikapokea 12. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-020.",Procurement Fraud,realistic-procurement-fraud-20,Mixed,Meru\n"On 9 March 2026 pale Embakasi East, Nairobi, afisa wa uteuzi wa umma joined the decision on alikuwa mwenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-032.",Conflict of Interest,realistic-conflict-of-interest-32,Mixed,Nairobi\n"Kwa tender ya ukodishaji wa magari pale Kilifi North, Kilifi, kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati. Value ilikuwa around 2,500, nimeweka details ya time na office number. Case/rekodi KW-046.",Procurement Fraud,realistic-procurement-fraud-46,Mixed,Kilifi\n"Nilienda pale office ya Imenti Central, Meru, on 11 August 2026 for kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa, 2,500, nimeweka details ya time na office number. Case/rekodi KW-004.",Bribery,realistic-bribery-04,Mixed,Meru\n"Kwa tender ya usambazaji wa mafuta pale Machakos Central, Machakos, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Value ilikuwa around 500, nimeweka details ya time na office number. Case/rekodi KW-032.",Procurement Fraud,realistic-procurement-fraud-32,Mixed,Machakos\n"On 3 January 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya shule aliingia decision ya aliwahi kuajiri; employed, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-014.",Conflict of Interest,realistic-conflict-of-interest-14,Sheng,Garissa\n"Nimekagua hati kuhusu ilitumia muhuri wa kughushi katika Nakuru East, Nakuru, tarehe 17 February 2026. barua ya kibali cha polisi kuifanya idhini isiyoidhinishwa ionekane halali. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-40,Kiswahili,Nakuru\n"Ninawasilisha tuhuma za matumizi mabaya ya fedha za chakula katika Kilifi North, Kilifi, tangu 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni. Nimeona rekodi zinazoashiria kiasi cha Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Embezzlement,realistic-embezzlement-43,Kiswahili,Kilifi\n"Nilienda pale office ya Embakasi East, Nairobi, on 28 July 2026 for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida, 300, na screenshot na receipt bado. Case/rekodi KW-031.",Bribery,realistic-bribery-31,Mixed,Nairobi\n"Nilienda pale office ya Kapseret, Eldoret, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 300, na screenshot na receipt bado. Case/rekodi KW-049.",Bribery,realistic-bribery-49,Mixed,Eldoret\n"Tarehe 22 April 2026 katika Imenti Central, Meru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-033.",Abuse of Office,realistic-abuse-of-office-33,Kiswahili,Meru\n"On 28 July 2026, I dealt with the county portal in Garissa Township, Garissa. The office removed a public notice without a published reason after residents requested services. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-007.",Other,realistic-other-07,English,Garissa\n"Juzi 9 March 2026 nikaingia pale office ya Bungoma Central, Bungoma, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 1000, nimehifadhi screenshot na receipt. Ref KW-027.",Bribery,realistic-bribery-27,Sheng,Bungoma\n"On 3 January 2026, I dealt with the public office in Garissa Township, Garissa. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-017.",Other,realistic-other-17,English,Garissa\n"Pale Kapseret, Eldoret on 28 July 2026, tovuti ya kaunti baada ya wakazi kuomba huduma; no action taken, na screenshot na receipt bado. Case/rekodi KW-023.",Other,realistic-other-23,Mixed,Eldoret\n"Tarehe 9 March 2026, nilienda ofisi ya umma katika Bungoma Central, Bungoma, kwa cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti; Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-027.",Bribery,realistic-bribery-27,Kiswahili,Bungoma\n"Tender ya zabuni ya ukarabati wa barabara pale Bungoma Central, Bungoma ilikuwa imepangwa. kamati ya tathmini ilibadilisha masharti baada ya zabuni kufunguliwa, around 1000, nimehifadhi screenshot na receipt. Ref KW-033.",Procurement Fraud,realistic-procurement-fraud-33,Sheng,Bungoma\n"On 14 June 2026 pale Nakuru East, Nakuru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, 1000; kuni-threaten kama sikuli ipa, nimehifadhi screenshot na receipt. Ref KW-057.",Extortion,realistic-extortion-57,Sheng,Nakuru\n"Nilienda pale office ya Imenti Central, Meru, on 17 February 2026 for utoaji wa pasipoti. afisa wa uhamiaji aliomba pesa kuingiza ombi mbele ya foleni, 500, nimeweka details ya time na office number. Case/rekodi KW-014.",Bribery,realistic-bribery-14,Mixed,Meru\n"Nilienda pale office ya Kisumu Central, Kisumu, on 28 July 2026 for uchapishaji wa kitambulisho. karani wa Huduma Centre aliomba malipo ya binafsi kuharakisha huduma ya kawaida, 1,000, na screenshot na receipt bado. Case/rekodi KW-015.",Bribery,realistic-bribery-15,Mixed,Kisumu\n"Tarehe 28 July 2026 katika Garissa Township, Garissa, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Extortion,realistic-extortion-38,Kiswahili,Garissa\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Bungoma Central, Bungoma. Maslahi ya kifedha hayakutangazwa hadharani. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Conflict of Interest,realistic-conflict-of-interest-58,Kiswahili,Bungoma\n"Nimekagua hati kuhusu ilihaririwa katika Nakuru East, Nakuru, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-60,Kiswahili,Nakuru\n"On 3 January 2026 pale Garissa Township, Garissa, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-008.",Extortion,realistic-extortion-08,Sheng,Garissa\n"On 22 April 2026 pale Nakuru East, Nakuru, afisa wa fedha wa kaunti aliingia decision ya alikuwa mmiliki binafsi; personally owned, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-053.",Conflict of Interest,realistic-conflict-of-interest-53,Sheng,Nakuru\n"I am reporting irregular use of KSh 20,000 in Kapseret, Eldoret, after 3 January 2026. Records show that the county hospital accountant handled medicine purchase money, and the funds worth KSh 20,000 paid a supplier for drugs that are absent from the store. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Embezzlement,realistic-embezzlement-12,English,Eldoret\n"Tarehe 22 April 2026, nilishughulika na simu ya msaada ya umma katika Nakuru East, Nakuru. baada ya simu nyingi bila nambari ya kufuatilia. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Other,realistic-other-56,Kiswahili,Nakuru\n"Pale Machakos Central, Machakos, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-028.",Nepotism/Favoritism,realistic-nepotism-favoritism-28,Mixed,Machakos\n"In Kapseret, Eldoret, on 28 July 2026, the bursary committee gave the bursary to the committee secretary\'s child while several eligible applicants received nothing. The estimated benefit or contract value was KSh 1,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-051.",Nepotism/Favoritism,realistic-nepotism-favoritism-51,English,Eldoret\n"On 22 April 2026 pale Likoni, Mombasa, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-017.",Abuse of Office,realistic-abuse-of-office-17,Sheng,Mombasa\n"Tarehe 11 August 2026, nilienda ofisi ya umma katika Kilifi North, Kilifi, kwa kurejeshwa kwa matatu. afisa wa trafiki alisema magari yangeshikwa kama pesa hazitolipwa; Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Bribery,realistic-bribery-20,Kiswahili,Kilifi\n"On 14 June 2026 pale Imenti Central, Meru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-009.",Extortion,realistic-extortion-09,Mixed,Meru\n"On 11 August 2026 in Likoni, Mombasa, the roadblock officer stopped my vehicle and demanded KSh 500, about KSh 1,000. They threatened to make a false inspection fault. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-003.",Extortion,realistic-extortion-03,English,Mombasa\n"Pale Bungoma Central, Bungoma on 6 May 2026, jopo la ajira la kaunti ingawa binamu hakuwa ameomba; benefit/value ilikuwa 300, nimehifadhi screenshot na receipt. Ref KW-049.",Nepotism/Favoritism,realistic-nepotism-favoritism-49,Sheng,Bungoma\n"Pale Kilifi North, Kilifi on 22 April 2026, simu ya msaada ya umma bila nambari ya kufuatilia; no action taken, nimeweka details ya time na office number. Case/rekodi KW-024.",Other,realistic-other-24,Mixed,Kilifi\n"On 9 March 2026 pale Kapseret, Eldoret, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Abuse of Office,realistic-abuse-of-office-28,Sheng,Eldoret\n"Pale Nakuru East, Nakuru on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-036.",Other,realistic-other-36,Mixed,Nakuru\n"On 28 July 2026 pale Garissa Township, Garissa, mjumbe wa bodi ya leseni joined the decision on alikuwa mkurugenzi; vested interest haikudeclear, nimeweka details ya time na office number. Case/rekodi KW-004.",Conflict of Interest,realistic-conflict-of-interest-04,Mixed,Garissa\n"For tender HOS-118 in Machakos Central, Machakos, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-002.",Procurement Fraud,realistic-procurement-fraud-02,English,Machakos\n"On 3 January 2026 pale Kisumu Central, Kisumu, chifu wakati wa saa za kazi; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-034.",Abuse of Office,realistic-abuse-of-office-34,Mixed,Kisumu\n"Katika Likoni, Mombasa, tarehe 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-040.",Nepotism/Favoritism,realistic-nepotism-favoritism-40,Kiswahili,Mombasa\n"Pale Garissa Township, Garissa, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-035.",Nepotism/Favoritism,realistic-nepotism-favoritism-35,Mixed,Garissa\n"Kuna money misuse kwa mapato ya ada za soko pale Imenti Central, Meru, after 11 August 2026. mhasibu wa soko la kaunti hayakuonekana kwenye daftari wala amana ya benki, 5,000, na screenshot na receipt bado. Case/rekodi KW-047.",Embezzlement,realistic-embezzlement-47,Mixed,Meru\n"Juzi 9 March 2026 nikaingia pale office ya Kapseret, Eldoret, for cheti cha eneo la chakula. mkaguzi wa afya aliomba zawadi kabla ya kutia sahihi kwenye cheti 300, nimehifadhi screenshot na receipt. Ref KW-019.",Bribery,realistic-bribery-19,Sheng,Eldoret\n"On 22 April 2026, the county finance officer took part in a decision about a lease decision in Imenti Central, Meru. The officer personally owned the land being leased to the public entity. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-045.",Conflict of Interest,realistic-conflict-of-interest-45,English,Meru\n"In Likoni, Mombasa, on 17 February 2026, the school committee awarded a supply contract to a relative\'s company after other bidders were excluded. The estimated benefit or contract value was KSh 2,500. I kept the receipt and the message thread. Case note KW-010.",Nepotism/Favoritism,realistic-nepotism-favoritism-10,English,Mombasa\n"On 28 July 2026 pale Garissa Township, Garissa, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, 500; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-038.",Extortion,realistic-extortion-38,Sheng,Garissa\n"Pale Kapseret, Eldoret, on 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa, benefit ilikuwa 5,000, na screenshot na receipt bado. Case/rekodi KW-011.",Nepotism/Favoritism,realistic-nepotism-favoritism-11,Mixed,Eldoret\n"Tarehe 14 June 2026, nilishughulika na dirisha la huduma katika Likoni, Mombasa. kwa hati ileile bila mtu yeyote kuchukua jukumu. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Other,realistic-other-02,Kiswahili,Mombasa\n"Nilienda pale office ya Likoni, Mombasa, on 14 June 2026 for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni, 20,000, nimeweka details ya time na office number. Case/rekodi KW-018.",Bribery,realistic-bribery-18,Mixed,Mombasa\n"Katika Embakasi East, Nairobi, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-043.",Nepotism/Favoritism,realistic-nepotism-favoritism-43,Kiswahili,Nairobi\n"On 6 May 2026, the project director took part in a decision about a construction contract in Bungoma Central, Bungoma. The officer also owned the company selected for the work. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Conflict of Interest,realistic-conflict-of-interest-18,English,Bungoma\n"On 14 June 2026 pale Imenti Central, Meru, mwenyekiti wa jopo la ruzuku aliingia decision ya alikuwa mshirika; was a partner, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-015.",Conflict of Interest,realistic-conflict-of-interest-15,Sheng,Meru\n"Kuna money misuse kwa fedha za chakula pale Likoni, Mombasa, after 22 April 2026. kamati ya chakula cha shule zilitumika kununua bidhaa ambazo hazikupelekwa shuleni, 5,000, na screenshot na receipt bado. Case/rekodi KW-011.",Embezzlement,realistic-embezzlement-11,Mixed,Mombasa\n"On 28 July 2026 in Bungoma Central, Bungoma, the station officer refused to release my brother and demanded KSh 2,000, about KSh 2,500. They while warning that the case file could be damaged. I kept the receipt and the message thread. Case note KW-022.",Extortion,realistic-extortion-22,English,Bungoma\n"Tarehe 28 July 2026, nilishughulika na tovuti ya kaunti katika Embakasi East, Nairobi. bila sababu iliyochapishwa baada ya wakazi kuomba huduma. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-015.",Other,realistic-other-15,Kiswahili,Nairobi\n"Tarehe 28 July 2026 katika Bungoma Central, Bungoma, afisa wa kaunti wengine wakaambiwa wangojee. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Abuse of Office,realistic-abuse-of-office-56,Kiswahili,Bungoma\n"Nilienda pale office ya Bungoma Central, Bungoma, on 3 January 2026 for upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili, 1,000, na screenshot na receipt bado. Case/rekodi KW-057.",Bribery,realistic-bribery-57,Mixed,Bungoma\n"Nimekagua hati kuhusu iliandika bidhaa katika Kilifi North, Kilifi, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-058.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-58,Kiswahili,Kilifi\n"For tender HOS-118 in Imenti Central, Meru, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-050.",Procurement Fraud,realistic-procurement-fraud-50,English,Meru\n"On 14 June 2026 pale Nakuru East, Nakuru, afisa wa polisi akitishia kunikamata nisipoleta pesa taslimu, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-057.",Extortion,realistic-extortion-57,Mixed,Nakuru\n"On 14 June 2026 in Imenti Central, Meru, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I kept the receipt and the message thread. Case note KW-043.",Abuse of Office,realistic-abuse-of-office-43,English,Meru\n"On 3 January 2026, the school board member took part in a decision about a tender for uniforms in Garissa Township, Garissa. The officer employed the bidder as a private consultant. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-054.",Conflict of Interest,realistic-conflict-of-interest-54,English,Garissa\n"On 22 April 2026 pale Nakuru East, Nakuru, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; hii ilikuwa abuse ya ofisi, na screenshot na receipt bado. Case/rekodi KW-041.",Abuse of Office,realistic-abuse-of-office-41,Mixed,Nakuru\n"Katika Likoni, Mombasa, tarehe 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-010.",Nepotism/Favoritism,realistic-nepotism-favoritism-10,Kiswahili,Mombasa\n"Nimekagua hati kuhusu ilikuwa na sahihi iliyorudishwa katika Embakasi East, Nairobi, tarehe 6 May 2026. vocha ya malipo kwa ununuzi usio na ushahidi wa msambazaji. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-039.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-39,Kiswahili,Nairobi\n"Kwa zabuni ya vifaa vya hospitali katika Machakos Central, Machakos, afisa wa ununuzi aligawanya mkataba mmoja kuwa ununuzi mdogo ili kuepuka ushindani. Tukio linahusu zabuni ya vifaa vya hospitali yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-002.",Procurement Fraud,realistic-procurement-fraud-02,Kiswahili,Machakos\n"I am reporting irregular use of KSh 5,000 in Machakos Central, Machakos, after 14 June 2026. Records show that the public works foreman handled road-repair funds, and the funds worth KSh 5,000 were spent while the damaged section remained unfinished. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Embezzlement,realistic-embezzlement-29,English,Machakos\n"Pale Imenti Central, Meru on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa; benefit/value ilikuwa 500, nimeandika time, office n marafiki wawili waliona. Ref KW-056.",Nepotism/Favoritism,realistic-nepotism-favoritism-56,Sheng,Meru\n"Tarehe 22 April 2026 katika Kilifi North, Kilifi, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali. Huu si uamuzi wa kawaida wa kazi. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-049.",Abuse of Office,realistic-abuse-of-office-49,Kiswahili,Kilifi\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Bungoma Central, Bungoma, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-020.",Embezzlement,realistic-embezzlement-20,Kiswahili,Bungoma\n"Pale Machakos Central, Machakos, on 11 August 2026, kamati ya ununuzi huku wasambazaji wenye sifa hawakualikwa, benefit ilikuwa 20,000, nimeweka details ya time na office number. Case/rekodi KW-048.",Nepotism/Favoritism,realistic-nepotism-favoritism-48,Mixed,Machakos\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Kapseret, Eldoret. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-020.",Conflict of Interest,realistic-conflict-of-interest-20,English,Eldoret\n"Ninawasilisha tuhuma za matumizi mabaya ya pesa za dawa katika Bungoma Central, Bungoma, tangu 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani. Nimeona rekodi zinazoashiria kiasi cha Sh20000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-060.",Embezzlement,realistic-embezzlement-60,Kiswahili,Bungoma\n"I am reporting irregular use of KSh 20,000 in Kapseret, Eldoret, after 9 March 2026. Records show that the youth fund administrator handled approved youth project money, and the funds worth KSh 2,500 was transferred to a personal account. I kept the receipt and the message thread. Case note KW-022.",Embezzlement,realistic-embezzlement-22,English,Eldoret\n"Tarehe 17 February 2026 katika Imenti Central, Meru, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-029.",Extortion,realistic-extortion-29,Kiswahili,Meru\n"Tarehe 9 March 2026, nilishughulika na idara ya maji katika Bungoma Central, Bungoma. kutofanya kazi kwa miezi kadhaa bila maelezo. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-011.",Other,realistic-other-11,Kiswahili,Bungoma\n"Kwa tender ya mifumo ya kompyuta na mtandao pale Kisumu Central, Kisumu, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Value ilikuwa around 1,000, na screenshot na receipt bado. Case/rekodi KW-021.",Procurement Fraud,realistic-procurement-fraud-21,Mixed,Kisumu\n"Tarehe 28 July 2026 katika Kapseret, Eldoret, afisa wa kaunti wengine wakaambiwa wangojee. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-008.",Abuse of Office,realistic-abuse-of-office-08,Kiswahili,Eldoret\n"Tender ya usambazaji wa mafuta pale Imenti Central, Meru ilikuwa imepangwa. katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-040.",Procurement Fraud,realistic-procurement-fraud-40,Sheng,Meru\n"I checked an academic certificate relating to the appointment file in Kisumu Central, Kisumu, on 9 March 2026. The document contained a changed name and a stamp that the issuing school says is not genuine. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-053.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-53,English,Kisumu\n"For contract FUEL-33 in Kilifi North, Kilifi, the evaluation secretary altered the comparison sheet after bids were submitted. The file concerns fuel supply and an estimated value of KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-056.",Procurement Fraud,realistic-procurement-fraud-56,English,Kilifi\n"Pale Machakos Central, Machakos on 11 August 2026, dirisha la umma bila risiti rasmi; no action taken, nimeweka details ya time na office number. Case/rekodi KW-060.",Other,realistic-other-60,Mixed,Machakos\n"On 6 May 2026 pale Bungoma Central, Bungoma, mkurugenzi wa mradi aliingia decision ya pia alikuwa mwenye; also owned, lakini haikudeclare, nimeandika time, office n marafiki wawili waliona. Ref KW-058.",Conflict of Interest,realistic-conflict-of-interest-58,Sheng,Bungoma\n"On 14 June 2026 in Machakos Central, Machakos, the licensing manager cancelled a lawful business licence after a personal dispute and did not give the owner a review procedure. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-035.",Abuse of Office,realistic-abuse-of-office-35,English,Machakos\n"Katika Kapseret, Eldoret, tarehe 28 July 2026, kamati ya basari huku waombaji wengine waliostahili wakinyimwa. Thamani ya manufaa au mkataba ilikuwa Sh1000. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-051.",Nepotism/Favoritism,realistic-nepotism-favoritism-51,Kiswahili,Eldoret\n"On 6 May 2026, I dealt with the roads office in Bungoma Central, Bungoma. The office promised resurfacing for our street but no work began before the deadline. I asked for a response but received none. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-021.",Other,realistic-other-21,English,Bungoma\n"On 28 July 2026 pale Kapseret, Eldoret, afisa wa kaunti wengine wakaambiwa wangojee; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-008.",Abuse of Office,realistic-abuse-of-office-08,Mixed,Eldoret\n"I checked a business permit relating to the revenue office in Nakuru East, Nakuru, on 11 August 2026. The document showed a false address and a permit number not in the official register. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-030.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-30,English,Nakuru\n"Kwa mifumo ya kompyuta na mtandao katika Kapseret, Eldoret, timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa. Tukio linahusu mifumo ya kompyuta na mtandao yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Procurement Fraud,realistic-procurement-fraud-05,Kiswahili,Eldoret\n"In Bungoma Central, Bungoma, on 6 May 2026, the county recruitment panel appointed the chairperson\'s cousin a training office job although the cousin had not applied. The estimated benefit or contract value was KSh 300. I kept the receipt and the message thread. Case note KW-049.",Nepotism/Favoritism,realistic-nepotism-favoritism-49,English,Bungoma\n"Kuna money misuse kwa pesa ya mradi wa vijana pale Kisumu Central, Kisumu, after 9 March 2026. msimamizi wa mfuko wa vijana ilihamishiwa akaunti ya binafsi, 500, nimeweka details ya time na office number. Case/rekodi KW-038.",Embezzlement,realistic-embezzlement-38,Mixed,Kisumu\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Embakasi East, Nairobi. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Conflict of Interest,realistic-conflict-of-interest-12,English,Nairobi\n"Katika Imenti Central, Meru, tarehe 14 June 2026, bodi ya CDF bila kura ya wazi. Thamani ya manufaa au mkataba ilikuwa Sh2500. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-046.",Nepotism/Favoritism,realistic-nepotism-favoritism-46,Kiswahili,Meru\n"Kwa huduma za ulinzi katika Bungoma Central, Bungoma, taasisi ya ununuzi ilampa msambazaji mmoja taarifa za zabuni kwa siri. Tukio linahusu huduma za ulinzi yenye thamani ya Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-023.",Procurement Fraud,realistic-procurement-fraud-23,Kiswahili,Bungoma\n"Tender ya ukodishaji wa magari pale Kilifi North, Kilifi ilikuwa imepangwa. kamati ya mikataba ilikubali zabuni baada ya muda na kukataa zile zilizowasilishwa kwa wakati, around 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-046.",Procurement Fraud,realistic-procurement-fraud-46,Sheng,Kilifi\n"On 11 August 2026, the procurement chairperson took part in a decision about a supplier bid in Imenti Central, Meru. The officer owned shares in the bidding company but did not declare it. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-025.",Conflict of Interest,realistic-conflict-of-interest-25,English,Meru\n"Tarehe 6 May 2026 katika Kapseret, Eldoret, afisa wa rekodi akatishia kuondoa rekodi ya ombi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-038.",Abuse of Office,realistic-abuse-of-office-38,Kiswahili,Eldoret\n"Nimekagua hati kuhusu iliandika bidhaa katika Nakuru East, Nakuru, tarehe 22 April 2026. ankara ya uwasilishaji ambazo hazikupokelewa. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-50,Kiswahili,Nakuru\n"Tarehe 3 January 2026, nilienda ofisi ya umma katika Bungoma Central, Bungoma, kwa upekuzi wa hati ya ardhi. karani wa ardhi aliomba pesa kabla ya kufungua faili; Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-017.",Bribery,realistic-bribery-17,Kiswahili,Bungoma\n"On 22 April 2026, the county finance officer took part in a decision about a lease decision in Imenti Central, Meru. The officer personally owned the land being leased to the public entity. The financial interest was not publicly disclosed. I have a screenshot and the names of two people who witnessed it. Case note KW-005.",Conflict of Interest,realistic-conflict-of-interest-05,English,Meru\n"I checked a delivery invoice relating to the procurement file in Kilifi North, Kilifi, on 22 April 2026. The document listed goods that were never received. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-18,English,Kilifi\n"Nimekagua hati kuhusu ilihaririwa katika Imenti Central, Meru, tarehe 14 June 2026. kumbukumbu ya mkutano kuweka azimio ambalo wajumbe hawalikubali. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-012.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-12,Kiswahili,Meru\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Kapseret, Eldoret. Maslahi ya kifedha hayakutangazwa hadharani. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-050.",Conflict of Interest,realistic-conflict-of-interest-50,Kiswahili,Eldoret\n"For tender HOS-118 in Nakuru East, Nakuru, the procurement officer split one contract into smaller purchases to avoid open competition. The file concerns hospital equipment tender and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-018.",Procurement Fraud,realistic-procurement-fraud-18,English,Nakuru\n"I checked a completion certificate relating to the public works file in Kapseret, Eldoret, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-057.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-57,English,Eldoret\n"On 28 July 2026 pale Kisumu Central, Kisumu, afisa wa stesheni akionya kuwa faili ya kesi ingeweza kuharibiwa, 20000; kuni-threaten kama sikuli ipa, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Extortion,realistic-extortion-30,Sheng,Kisumu\n"On 3 January 2026 pale Garissa Township, Garissa, mkaguzi wa leseni akasema angemripoti kwa uvunjaji wa sheria nikikataa, walidai 20,000, nimeweka details ya time na office number. Case/rekodi KW-048.",Extortion,realistic-extortion-48,Mixed,Garissa\n"Kuna doo za fedha za kisima zilipotea pale Nakuru East, Nakuru after 17 February 2026. kamati ya CDF zilitozwa lakini kisima hakikujengwa, 300, nimehifadhi screenshot na receipt. Ref KW-025.",Embezzlement,realistic-embezzlement-25,Sheng,Nakuru\n"Tarehe 6 May 2026, mkurugenzi wa mradi alishiriki uamuzi kuhusu pia alikuwa mwenye katika Embakasi East, Nairobi. Maslahi ya kifedha hayakutangazwa hadharani. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-042.",Conflict of Interest,realistic-conflict-of-interest-42,Kiswahili,Nairobi\n"Tarehe 9 March 2026 katika Kisumu Central, Kisumu, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-044.",Abuse of Office,realistic-abuse-of-office-44,Kiswahili,Kisumu\n"On 14 June 2026 pale Imenti Central, Meru, mwenyekiti wa jopo la ruzuku joined the decision on alikuwa mshirika; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-015.",Conflict of Interest,realistic-conflict-of-interest-15,Mixed,Meru\n"For contract SUP-77 in Machakos Central, Machakos, the county supply office paid for 40 units but received only 12. The file concerns office furniture and an estimated value of KSh 20,000. I recorded the time and wrote down the office number; no identity of a witness is included here. Case note KW-012.",Procurement Fraud,realistic-procurement-fraud-12,English,Machakos\n"On 22 April 2026 pale Kilifi North, Kilifi, msimamizi wa wadi kwa sababu nilipinga uamuzi wa awali; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-009.",Abuse of Office,realistic-abuse-of-office-09,Sheng,Kilifi\n"In Kilifi North, Kilifi, on 11 August 2026, the procurement committee gave repeated contracts to the chairperson\'s close associate while qualified suppliers were not invited. The estimated benefit or contract value was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-032.",Nepotism/Favoritism,realistic-nepotism-favoritism-32,English,Kilifi\n"I checked a meeting minute relating to the committee record in Machakos Central, Machakos, on 14 June 2026. The document was edited to include a resolution that members never passed. The apparent difference is worth reviewing. I have a screenshot and the names of two people who witnessed it. Case note KW-044.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-44,English,Machakos\n"Tarehe 17 February 2026 katika Kilifi North, Kilifi, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, Sh5000. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-005.",Extortion,realistic-extortion-05,Kiswahili,Kilifi\n"On 17 February 2026 pale Likoni, Mombasa, mkuu wa shule wakati wa shule; alitumia cheo vibaya, nimehifadhi screenshot na receipt. Ref KW-007.",Abuse of Office,realistic-abuse-of-office-07,Sheng,Mombasa\n"On 9 March 2026 pale Garissa Township, Garissa, mkuu wa programu ya umma akaagiza wafanyakazi kubadilisha orodha; hii ilikuwa abuse ya ofisi, nimeweka details ya time na office number. Case/rekodi KW-012.",Abuse of Office,realistic-abuse-of-office-12,Mixed,Garissa\n"On 11 August 2026 pale Likoni, Mombasa, mwenyekiti wa ununuzi aliingia decision ya alikuwa na hisa; owned shares, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-049.",Conflict of Interest,realistic-conflict-of-interest-49,Sheng,Mombasa\n"On 17 February 2026 pale Likoni, Mombasa, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, walidai 5,000, na screenshot na receipt bado. Case/rekodi KW-053.",Extortion,realistic-extortion-53,Mixed,Mombasa\n"On 11 August 2026 pale Kilifi North, Kilifi, mwenyekiti wa ununuzi joined the decision on alikuwa na hisa; vested interest haikudeclear, na screenshot na receipt bado. Case/rekodi KW-041.",Conflict of Interest,realistic-conflict-of-interest-41,Mixed,Kilifi\n"Kuna doo za pesa za dawa zilipotea pale Embakasi East, Nairobi after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 500, nimeandika time, office n marafiki wawili waliona. Ref KW-044.",Embezzlement,realistic-embezzlement-44,Sheng,Nairobi\n"Kuna money misuse kwa fedha za wanafunzi pale Kapseret, Eldoret, after 28 July 2026. karani wa akaunti ya basari aliingiza risiti zisizofanana na malipo, 500, nimeweka details ya time na office number. Case/rekodi KW-002.",Embezzlement,realistic-embezzlement-02,Mixed,Eldoret\n"Pale Bungoma Central, Bungoma on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili; benefit/value ilikuwa 5000, nimehifadhi screenshot na receipt. Ref KW-029.",Nepotism/Favoritism,realistic-nepotism-favoritism-29,Sheng,Bungoma\n"Nimekagua hati kuhusu ilonyesha anwani ya uongo katika Machakos Central, Machakos, tarehe 11 August 2026. leseni ya biashara na nambari ya kibali isiyopo kwenye rejista rasmi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-054.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-54,Kiswahili,Machakos\n"On 6 May 2026 in Kapseret, Eldoret, the records officer refused access to my own file despite an approved request and threatened to remove the request record. This was not a routine decision. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Abuse of Office,realistic-abuse-of-office-38,English,Eldoret\n"On 6 May 2026, I dealt with the roads office in Kisumu Central, Kisumu. The office promised resurfacing for our street but no work began before the deadline. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-029.",Other,realistic-other-29,English,Kisumu\n"Tarehe 11 August 2026 katika Machakos Central, Machakos, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi. Huu si uamuzi wa kawaida wa kazi. Nimeandikia saa na nambari ya ofisi bila kujumuisha utambulisho wa shahidi. Rejea KW-045.",Abuse of Office,realistic-abuse-of-office-45,Kiswahili,Machakos\n"Pale Garissa Township, Garissa on 3 January 2026, bodi ya uteuzi licha ya kupata alama ndogo kwenye usaili; benefit/value ilikuwa 1000, nimehifadhi screenshot na receipt. Ref KW-045.",Nepotism/Favoritism,realistic-nepotism-favoritism-45,Sheng,Garissa\n"On 3 January 2026 pale Embakasi East, Nairobi, chifu wakati wa saa za kazi; alitumia cheo vibaya, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Abuse of Office,realistic-abuse-of-office-50,Sheng,Nairobi\n"On 17 February 2026 pale Machakos Central, Machakos, katibu wa kamati aliingia decision ya alilipwa; was paid, lakini haikudeclare, nimehifadhi screenshot na receipt. Ref KW-027.",Conflict of Interest,realistic-conflict-of-interest-27,Sheng,Machakos\n"Nili-check document ya iliandika bidhaa pale Kilifi North, Kilifi, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-18,Sheng,Kilifi\n"On 11 August 2026, I dealt with the public counter in Likoni, Mombasa. The office charged more than the published price for a standard form and gave no official receipt. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-052.",Other,realistic-other-52,English,Mombasa\n"Nilikagua document for kilitiwa sahihi pale Garissa Township, Garissa, on 28 July 2026; cheti cha kukamilisha kazi na mtu ambaye muhuri wake ulikuwashwa, na screenshot na receipt bado. Case/rekodi KW-041.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-41,Mixed,Garissa\n"Tender ya mifumo ya kompyuta na mtandao pale Kisumu Central, Kisumu ilikuwa imepangwa. timu ya tathmini ya zabuni ilitumia vigezo vya alama ambavyo havikutangazwa, around 1000, nimehifadhi screenshot na receipt. Ref KW-021.",Procurement Fraud,realistic-procurement-fraud-21,Sheng,Kisumu\n"On 17 February 2026 pale Kilifi North, Kilifi, afisa wa umma nisipolipa kutoka kwa pesa yangu binafsi, walidai 1,000, na screenshot na receipt bado. Case/rekodi KW-045.",Extortion,realistic-extortion-45,Mixed,Kilifi\n"Pale Likoni, Mombasa, on 17 February 2026, kamati ya shule baada ya wazabuni wengine kutengwa, benefit ilikuwa 500, nimeweka details ya time na office number. Case/rekodi KW-050.",Nepotism/Favoritism,realistic-nepotism-favoritism-50,Mixed,Mombasa\n"Pale Kapseret, Eldoret on 9 March 2026, idara ya maji kwa miezi kadhaa bila maelezo; no answer kuandaliwa, nimehifadhi screenshot na receipt. Ref KW-003.",Other,realistic-other-03,Sheng,Eldoret\n"Pale Machakos Central, Machakos on 14 June 2026, dirisha la huduma bila mtu yeyote kuchukua jukumu; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-050.",Other,realistic-other-50,Sheng,Machakos\n"Pale Machakos Central, Machakos on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa; benefit/value ilikuwa 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-028.",Nepotism/Favoritism,realistic-nepotism-favoritism-28,Sheng,Machakos\n"I checked a completion certificate relating to the public works file in Embakasi East, Nairobi, on 28 July 2026. The document was signed by a person whose office stamp was copied. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-049.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-49,English,Nairobi\n"Tarehe 22 April 2026 katika Nakuru East, Nakuru, msimamizi wa usalama langoni akitishia kupiga ukaguzi wa kodi, Sh300. Nimehifadhi risiti na ujumbe wa mazungumzo. Rejea KW-007.",Extortion,realistic-extortion-07,Kiswahili,Nakuru\n"Nili-check document ya ilibadilishwa pale Garissa Township, Garissa, on 3 January 2026; rekodi ya uhamisho wa ardhi kuonyesha uhamisho ambao haukuidhinishwa, nimehifadhi screenshot na receipt. Ref KW-011.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-11,Sheng,Garissa\n"Tarehe 11 August 2026 katika Imenti Central, Meru, mkurugenzi wa idara akatia sahihi madai ya mafuta kupitia ofisi. Huu si uamuzi wa kawaida wa kazi. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-053.",Abuse of Office,realistic-abuse-of-office-53,Kiswahili,Meru\n"On 3 January 2026, I dealt with the public office in Embakasi East, Nairobi. The office closed early with no notice and did not publish when services would resume. I asked for a response but received none. I kept the receipt and the message thread. Case note KW-025.",Other,realistic-other-25,English,Nairobi\n"Pale Machakos Central, Machakos on 17 February 2026, dirisha la hospitali bila kutoa njia ya kulalamika; no answer kuandaliwa, nimeandika time, office n marafiki wawili waliona. Ref KW-030.",Other,realistic-other-30,Sheng,Machakos\n"On 28 July 2026, the licensing board member took part in a decision about a permit application in Bungoma Central, Bungoma. The officer was a director of the applicant company. The financial interest was not publicly disclosed. I kept the receipt and the message thread. Case note KW-028.",Conflict of Interest,realistic-conflict-of-interest-28,English,Bungoma\n"Kwa usambazaji wa mafuta katika Kilifi North, Kilifi, katibu wa tathmini alibadilisha jedwali la kulinganisha baada ya zabuni kuwasilishwa. Tukio linahusu usambazaji wa mafuta yenye thamani ya Sh500. Nina picha ya skrini na majina ya mashahidi wawili. Rejea KW-056.",Procurement Fraud,realistic-procurement-fraud-56,Kiswahili,Kilifi\n"Juzi 14 June 2026 nikaingia pale office ya Likoni, Mombasa, for leseni ya biashara. afisa wa mapato alidai ada isiyo rasmi ili kuidhinisha leseni 20000, nimeandika time, office n marafiki wawili waliona. Ref KW-018.",Bribery,realistic-bribery-18,Sheng,Mombasa\n"I checked an academic certificate relating to the appointment file in Kapseret, Eldoret, on 9 March 2026. The document contained a changed name and a stamp that the issuing school says is not genuine. The apparent difference is worth reviewing. I kept the receipt and the message thread. Case note KW-037.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-37,English,Eldoret\n"Kuna doo za pesa za dawa zilipotea pale Embakasi East, Nairobi after 3 January 2026. mhasibu wa hospitali ya kaunti alilipa msambazaji wa dawa ambazo hazipo ghalani, 2500, nimeandika time, office n marafiki wawili waliona. Ref KW-004.",Embezzlement,realistic-embezzlement-04,Sheng,Nairobi\n"On 17 February 2026, I went to the public office in Likoni, Mombasa, for release of a passport. The the immigration officer requested money to place the application ahead of the queue; the demand was KSh 500. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Bribery,realistic-bribery-38,English,Mombasa\n"Nili-check document ya iliandika bidhaa pale Machakos Central, Machakos, on 22 April 2026; ankara ya uwasilishaji ambazo hazikupokelewa, nimeandika time, office n marafiki wawili waliona. Ref KW-034.",Fraud/Forgery of Documents,realistic-fraud-forgery-of-documents-34,Sheng,Machakos\n"Pale Nakuru East, Nakuru, on 22 April 2026, timu ya mgao wa soko kabla ya tarehe ya uteuzi iliyotangazwa, benefit ilikuwa 2,500, nimeweka details ya time na office number. Case/rekodi KW-004.",Nepotism/Favoritism,realistic-nepotism-favoritism-04,Mixed,Nakuru\n"On 17 February 2026, I dealt with the hospital desk in Imenti Central, Meru. The office refused to explain why the queue was suspended and gave no complaint channel. I asked for a response but received none. I have a screenshot and the names of two people who witnessed it. Case note KW-038.",Other,realistic-other-38,English,Meru'   # repr() keeps quoting/commas safe

df = pd.read_csv(io.StringIO(CSV_DATA), dtype=str, keep_default_na=False)
df = df.apply(lambda col: col.str.strip())

# --- Dataset audit gates (mirrors ml/src/dataset.py::validate_for_training) ---
counts = Counter(df["label"])
groups = {c: df[df["label"] == c]["group_id"].nunique() for c in CATEGORIES}
duplicates = len(df) - len({t.strip().casefold() for t in df["text"]})

assert set(counts) == set(CATEGORIES), f"missing categories: {set(CATEGORIES) - set(counts)}"
assert duplicates == 0, f"exact duplicate text rows: {duplicates}"
assert all(groups[c] >= 10 for c in CATEGORIES), "each category needs >=10 incident groups"

print(f"Rows               : {len(df)}")
print(f"Distinct incidents : {df['group_id'].nunique()}")
print(f"Rows per category  : {dict(sorted(counts.items()))}")
print(f"Duplicate texts    : {duplicates}")


## 2.3 Exploratory Data Analysis


In [ ]:
df["word_count"] = df["text"].str.split().str.len()

print("Shape:", df.shape)
print("\nClass distribution:")
print(df["label"].value_counts().to_string())
print("\nLanguage mix:")
print(df["language"].value_counts().to_string())
print("\nNarrative length in words:")
print(df["word_count"].describe().round(1).to_string())
print("\nIncident groups per category:")
print(pd.Series(groups).to_string())


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (a) Class balance
counts_sorted = df["label"].value_counts().sort_values()
axes[0, 0].barh(counts_sorted.index, counts_sorted.values, color="#0d9488")
axes[0, 0].set_title("(a) Rows per category")
axes[0, 0].set_xlabel("rows")

# (b) Narratives per incident -> shows the grouping structure
per_group = df.groupby("group_id").size()
axes[0, 1].hist(per_group.values, bins=range(1, per_group.max() + 2), color="#0f766e",
                edgecolor="white", align="left")
axes[0, 1].set_title("(b) Variants per incident group")
axes[0, 1].set_xlabel("rows describing the same incident")

# (c) Narrative length by language
for lang, colour in (("English", "#0f766e"), ("Kiswahili", "#f59e0b")):
    axes[1, 0].hist(df[df["language"] == lang]["word_count"], bins=12, alpha=0.75,
                    label=lang, color=colour)
axes[1, 0].set_title("(c) Narrative length by language")
axes[1, 0].set_xlabel("words per report")
axes[1, 0].legend()

# (d) Vocabulary overlap: shared vs unique words per category
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(lowercase=True)
vocab_by_cat = {}
for cat in CATEGORIES:
    vec.fit(df[df["label"] == cat]["text"])
    vocab_by_cat[cat] = set(vec.vocabulary_)
shared = set.intersection(*vocab_by_cat.values())
axes[1, 1].barh(list(vocab_by_cat), [len(vocab_by_cat[c] - shared) for c in vocab_by_cat],
                color="#0d9488")
axes[1, 1].set_title(f"(d) Category-distinctive words (shared vocabulary = {len(shared)})")
axes[1, 1].set_xlabel("unique terms")

plt.tight_layout()
plt.show()


### Insights that shaped the modelling decisions

1. **Balanced classes** (60 rows per category) — a property of the *designed* corpus, not of real
   reports. Real corruption reporting will be heavily imbalanced, which is why the pipeline uses
   `class_weight="balanced"` and reports **macro F1** rather than plain accuracy.
2. **Narratives are not independent**: each incident appears as 4 paraphrases. Treating rows as
   independent would let the model memorise an incident in training and be scored on a paraphrase
   of it — textbook leakage. Validation therefore **groups by incident**.
3. **Short narratives** (~20–30 words) — favour sparse TF-IDF over long-context models.
4. **Two languages** with different morphology — Kiswahili is agglutinative, so word tokens
   fragment; character n-grams recover much of that signal.
5. **Distinctive vs shared vocabulary** (panel d) — categories do have discriminative vocabulary,
   but share a large common core (words like *report*, *officer*, *payment*), which is exactly why a
   simple keyword classifier alone cannot separate them.

## 2.4 Preprocessing and feature engineering

Preprocessing is embedded **inside** the model pipeline rather than applied to the whole dataset up
front. This is deliberate: vectorisers are then fitted on training folds only, so no information
from the held-out fold leaks into the features.

| Step | Setting / purpose |
|---|---|
| Case folding + Unicode accent stripping | Normalise surface variation |
| Whitespace collapse | Remove formatting noise |
| **Word TF-IDF, n-grams (1–2), sublinear TF** | Captures phrases like *"bid rigging"*; sublinear TF dampens long repeated narratives |
| **Character TF-IDF, `char_wb`, n-grams (3–5)** | Robust to inflection, compounding, misspelling, Kiswahili morphology |
| `min_df=2` | Drops singleton terms (noise on small corpora) |
| `max_features=50 000` per vectoriser | Bounds dimensionality |
| `class_weight="balanced"` | Prepares for class imbalance in real data |
| **No stop-word removal** | Words like *not*, *without*, *before* carry meaning in allegations ("paid **without** receipt") |


---
# 3. Methodology and Model Selection
## 3.1 Model candidates and justification

Two **pre-declared** candidates are compared — not an open-ended hyperparameter search. On a
synthetic corpus an exhaustive search would inflate results without adding evidence about real data.

| Candidate | Why it is included | Known weakness |
|---|---|---|
| **Word TF-IDF (1–2) + Logistic Regression** (`balanced`) | Strong, interpretable text baseline; coefficient signs show which words drive a label | Ignores morphology; brittle to misspelling |
| **Word + char TF-IDF → Linear SVM** (`balanced`) | Character n-grams absorb spelling and Kiswahili agglutination; linear SVM handles very high-dimensional sparse input efficiently | Decision scores are less interpretable than probabilities |
| *(considered, rejected)* fine-tuned transformer (e.g. mBERT) | Potentially strongest on real multilingual text | No real adjudicated labelled data to fine-tune on; synthetic expansion would demonstrate pipeline behaviour, not real fit; too heavy for the deployment target |

**Why linear models on TF-IDF are the right choice here:** the input is short, domain-specific
narrative text; the corpus is small; interpretability matters for a governance-sensitive platform;
and the model must be cheap enough to run on submission without noticeable latency.

## 3.2 Validation design — the decisive methodological choice

```
OUTER : 5-fold StratifiedGroupKFold   → estimates end-to-end generalisation
INNER : 3-fold stratified GROUP CV    → selects between the two candidates
METRIC: macro F1                      → every category weighs equally
GROUP : incident group_id             → paraphrases of one incident never straddle a fold
```

Three properties make this defensible:

1. **Grouped folds** — prevents paraphrase leakage (§2.3 insight 2).
2. **Nested selection** — model choice happens *inside* each outer training fold, so the reported
   score includes the cost of choosing a model. Reporting the score of a model chosen on all the
   data would be optimistically biased.
3. **Trivial baseline alongside** — a fold-local majority-class predictor is scored on the same
   folds, so every result is interpreted relative to what a do-nothing strategy achieves.

## 3.3 Hyperparameters (fixed a priori)

| Component | Setting | Rationale |
|---|---|---|
| Word TF-IDF | `ngram_range=(1,2)`, `min_df=2`, `max_features=50000`, `sublinear_tf=True` | Phrases + noise control |
| Char TF-IDF | `analyzer="char_wb"`, `ngram_range=(3,5)`, `min_df=2`, `max_features=50000` | Multilingual morphology |
| Logistic Regression | `class_weight="balanced"`, `max_iter=2000`, `random_state=42` | Imbalance-aware, reproducible |
| Linear SVM | `class_weight="balanced"`, `random_state=42` | High-dimensional sparse strength |
| Outer / inner folds | 5 / 3, shuffled, seeds 42 / 100+fold | Stable, pre-declared |


## 3.4 Implementation — model construction


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

MODEL_CANDIDATES = ("word_logistic", "word_char_linear_svc")


def _word_features():
    return TfidfVectorizer(
        analyzer="word", lowercase=True, strip_accents="unicode",
        ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True,
    )


def _char_features():
    return TfidfVectorizer(
        analyzer="char_wb", lowercase=True, strip_accents="unicode",
        ngram_range=(3, 5), min_df=2, max_features=50000, sublinear_tf=True,
    )


def build_model(name):
    if name == "word_logistic":
        return Pipeline([
            ("features", _word_features()),
            ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000,
                                              random_state=42)),
        ])
    if name == "word_char_linear_svc":
        return Pipeline([
            ("features", FeatureUnion([("word", _word_features()), ("char", _char_features())])),
            ("classifier", LinearSVC(class_weight="balanced", random_state=42)),
        ])
    raise ValueError(f"unknown candidate: {name}")


# Sanity check: both pipelines build and fit on a stratified sample covering all 9 classes.
_sample = df.groupby("label").head(6)   # 54 rows, every category represented
for _name in MODEL_CANDIDATES:
    _model = build_model(_name).fit(_sample["text"], _sample["label"])
    _accuracy = (_model.predict(_sample["text"]) == _sample["label"]).mean()
    print(f"{_name:26s} build+fit OK on {len(_sample)} rows (in-sample accuracy {_accuracy:.2f})")


## 3.5 Implementation — nested grouped cross-validation


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)

TEXTS = df["text"].tolist()
LABELS = df["label"].tolist()
GROUPS = df["group_id"].tolist()


def compute_metrics(y_true, y_pred, labels=CATEGORIES):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_true, y_pred)), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=labels, average="macro",
                                         zero_division=0)), 4),
        "per_category": {
            c: {"precision": round(float(p[i]), 4), "recall": round(float(r[i]), 4),
                "f1": round(float(f[i]), 4), "support": int(s[i])}
            for i, c in enumerate(labels)
        },
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def cross_validate_candidate(train_idx, name, n_splits, seed):
    """Grouped CV inside an outer training fold; returns out-of-fold predictions + macro F1."""
    t = [TEXTS[i] for i in train_idx]
    y = [LABELS[i] for i in train_idx]
    g = [GROUPS[i] for i in train_idx]
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    preds = [None] * len(train_idx)
    for tr, te in splitter.split(t, y, g):
        assert not (set(g[i] for i in tr) & set(g[i] for i in te)), "group leakage"
        model = build_model(name).fit([t[i] for i in tr], [y[i] for i in tr])
        for i, pred in zip(te, model.predict([t[i] for i in te])):
            preds[i] = str(pred)
    return preds, compute_metrics(y, preds)["macro_f1"]


def select_model(train_idx, n_splits=3, seed=42):
    scores = {name: cross_validate_candidate(train_idx, name, n_splits, seed)[1]
              for name in MODEL_CANDIDATES}
    return max(scores, key=scores.get), scores


print("Helpers ready.")


In [ ]:
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = [None] * len(df)
baseline_preds = [None] * len(df)
fold_records = []

for fold, (tr, te) in enumerate(outer_splitter.split(TEXTS, LABELS, GROUPS), start=1):
    # Leakage guard: no incident on both sides of the split.
    assert not (set(GROUPS[i] for i in tr) & set(GROUPS[i] for i in te)), "group leakage"

    chosen, inner_scores = select_model(tr, n_splits=3, seed=100 + fold)

    model = build_model(chosen).fit([TEXTS[i] for i in tr], [LABELS[i] for i in tr])
    fold_pred = model.predict([TEXTS[i] for i in te])
    for i, pred in zip(te, fold_pred):
        oof_preds[i] = str(pred)

    majority = Counter(LABELS[i] for i in tr).most_common(1)[0][0]
    for i in te:
        baseline_preds[i] = majority

    fold_records.append({
        "fold": fold,
        "selected": chosen,
        "test_rows": len(te),
        "test_incidents": len({GROUPS[i] for i in te}),
        "macro_f1": round(float(f1_score([LABELS[i] for i in te], fold_pred,
                                         labels=CATEGORIES, average="macro",
                                         zero_division=0)), 4),
        **{f"inner_{k}": v for k, v in inner_scores.items()},
    })

folds_df = pd.DataFrame(fold_records)
assert all(p is not None for p in oof_preds), "every row needs an out-of-fold prediction"
print(folds_df.to_string(index=False))


---
# 4. Model Evaluation
## 4.1 Metrics and why each is used

| Metric | What it means | Why it is here |
|---|---|---|
| **Accuracy** | % of reports assigned the correct category | Intuitive headline number |
| **Balanced accuracy** | Mean recall across categories | Insensitive to class sizes |
| **Macro F1** | Mean of per-category F1 scores | **Primary metric** — a rare category counts as much as a common one |
| **Per-category precision / recall / F1** | Which categories are reliable, which are confused | Operational: a missed *Extortion* case matters more than a missed *Other* |
| **Confusion matrix** | Where mistakes go | Reveals semantic boundaries the model confuses |


## 4.2 Headline results vs. trivial baseline


In [ ]:
headline = pd.DataFrame([
    {"model": "Fold-local majority baseline",
     "accuracy": compute_metrics(LABELS, baseline_preds)["accuracy"],
     "balanced_accuracy": compute_metrics(LABELS, baseline_preds)["balanced_accuracy"],
     "macro_f1": compute_metrics(LABELS, baseline_preds)["macro_f1"]},
    {"model": "Nested grouped CV (selected model)",
     **{k: v for k, v in compute_metrics(LABELS, oof_preds).items()
        if k in ("accuracy", "balanced_accuracy", "macro_f1")}},
])

print(headline.to_string(index=False))
print()
print(f"Out-of-fold macro F1 across outer folds: "
      f"mean {folds_df.macro_f1.mean():.4f}  (spread {folds_df.macro_f1.std(ddof=0):.4f})")
print(f"Models chosen per fold: {folds_df.selected.value_counts().to_dict()}")


## 4.3 Comparative analysis

Three comparisons are reported, which is what "benchmarking against baseline models or other
approaches" requires:

1. **Model vs trivial baseline** (table above). The majority classifier scores ≈1/9 ≈ 0.11 by
   chance on a balanced 9-class problem; the trained model must clear that by a wide margin to be
   worth deploying.
2. **Candidate vs candidate** — the per-fold inner scores (`inner_word_logistic` vs
   `inner_word_char_linear_svc`) show *which* model won during selection and by how much,
   rather than hiding the comparison inside a single final number.
3. **Category vs category** — per-category F1 (§4.5) shows whether errors concentrate in one
   semantic boundary (e.g. *Bribery* vs *Extortion*), which is the most actionable comparison for
   the review team.


## 4.4 Visual representation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

headline.set_index("model")[["accuracy", "balanced_accuracy", "macro_f1"]].plot.bar(
    ax=axes[0], color=["#94a3b8", "#0d9488", "#0f766e"], rot=10)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("score")
axes[0].set_title("(a) Trained model vs trivial baseline")
axes[0].legend(loc="lower right")

cm = compute_metrics(LABELS, oof_preds)["confusion_matrix"]
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks(range(len(CATEGORIES)))
axes[1].set_yticks(range(len(CATEGORIES)))
axes[1].set_xticklabels([c[:12] for c in CATEGORIES], rotation=60, ha="right", fontsize=8)
axes[1].set_yticklabels([c[:16] for c in CATEGORIES], fontsize=8)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_title("(b) Confusion matrix (out-of-fold)")
vmax = max(max(row) for row in cm)
for i in range(len(CATEGORIES)):
    for j in range(len(CATEGORIES)):
        if cm[i][j]:
            axes[1].text(j, i, cm[i][j], ha="center", va="center", fontsize=7,
                         color="white" if cm[i][j] > vmax / 2 else "black")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()


## 4.5 Per-category performance and its operational meaning


In [ ]:
per_cat = pd.DataFrame(compute_metrics(LABELS, oof_preds)["per_category"]).T
per_cat.columns = ["precision", "recall", "F1", "support"]
per_cat = per_cat.sort_values("F1")
print(per_cat.to_string())

per_cat["F1"].plot(kind="barh", figsize=(9, 4.2), color="#0d9488",
                   title="Per-category F1 (out-of-fold predictions)")
plt.xlabel("F1")
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()


**Operational reading:** in the live platform a low per-category F1 is not a failure — it is a
reason to set a **confidence floor**. Below the floor the suggestion is shown as "unclassified —
human decides", and the double-blind review decides the label. The model never auto-labels a report.


## 4.6 Language subgroup analysis (an exploratory fairness check)

Kenya Watch is multilingual by design, so metrics are computed per language. This is an
**exploratory** look, not a fairness guarantee — and the Kiswahili in this corpus is draft text that
has not been reviewed by a native speaker, so a gap here is a signal to obtain proper review, not a
conclusion.


In [ ]:
subgroup_rows = []
for language in sorted(df["language"].unique()):
    idx = df.index[df["language"] == language].tolist()
    present = tuple(c for c in CATEGORIES if c in {LABELS[i] for i in idx})
    m = compute_metrics([LABELS[i] for i in idx], [oof_preds[i] for i in idx], present)
    subgroup_rows.append({"language": language, "rows": len(idx),
                          "accuracy": m["accuracy"], "macro_f1": m["macro_f1"]})
print(pd.DataFrame(subgroup_rows).to_string(index=False))


## 4.7 What these numbers do **not** prove

Stated plainly, because this matters more than any score:

1. **The data is synthetic.** It was written to be learnable: templatic phrasing and balanced classes
   inflate scores. These numbers say nothing about real Kenyan corruption reports.
2. **No temporal holdout.** Real deployment needs a later-in-time, incident-separated test set to
   detect drift as language and reporting patterns change.
3. **No native-speaker validation** of Kiswahili, and no code-switching (English/Swahili mixed within
   one narrative), which is very common in real reports.
4. **Synthetic-only calibration.** The runtime artifact stores a temperature fitted on grouped
   out-of-fold scores and the live UI shows only a high/low cue; this does not establish
   real-world confidence.
5. **Fairness is unexplored** beyond a language split; proper evaluation needs stakeholder-defined
   acceptable error rates per category.

> **The defensible claim concerns the pipeline, not the score:** "I built and validated a
> leakage-resistant, nested, multilingual text-classification pipeline, and I can explain precisely
> why its current numbers cannot be presented as real-world accuracy."


---
# 5. Deployment
## 5.1 Where this sits in the live system

```
Citizen submits ──► POST /api/reports ──► report saved ──► response returned immediately
                                                │
                                                └─► setImmediate(...)   [asynchronous — never blocks]
                                                        │
                                              classifyInBackground()
                                                        │
                             ┌──────────────────────────┴─────────────────────────┐
                             │                                                    │
              backend/utils/classifier.js  (LIVE TODAY)          ml/artifacts/category_model.joblib
              weighted keyword triage → category + urgency        experimental model suggestion
                                                                  (reviewer-only, after assessment)
```

**The live app uses both layers for different purposes.** Three safeguards apply:

1. **Independent judgement first.** A reviewer must record their own category assessment before the
   experimental model suggestion is returned.
2. **Synthetic-data warning.** The suggestion is labelled experimental and is never presented as a
   probability, finding or automated decision.
3. **Robustness.** If the runtime artifact is missing or invalid, reporting and review continue
   normally because keyword triage remains available.

## 5.2 Rollout plan (governance-gated)

| Step | Requirement | Status |
|---|---|---|
| 1 | Collect adjudicated labels via double-blind review (`categoryReview`) | **Live workflow exists** |
| 2 | Export approved dataset (`ml/scripts/export-reviewed-dataset.js`, allowlist + approval) | **Implemented** |
| 3 | De-identify and verify narratives manually | Documented process |
| 4 | Re-run this pipeline on real data | **This notebook / `ml/src/train.py`** |
| 5 | Freeze a later-time, incident-separated holdout | Future |
| 6 | **Shadow mode** — model predicts silently, compared with human decisions | Future |
| 7 | **Assist-only** deployment with a confidence floor | Future |

## 5.3 Serving method used

The offline training pipeline exports a compact JSON runtime artifact rather than shipping Python or
joblib to production. `ml/src/runtime_export.py` writes the TF-IDF vocabularies, inverse document
frequencies, classifier weights, preprocessing rules and calibration metadata. The Node.js API
loads and validates that artifact in `backend/utils/experimentalCategoryModel.js`, reproduces the
same vector arithmetic in JavaScript and exposes the suggestion through
`GET /api/reports/:id/ml-category-suggestion` only after the reviewer's independent assessment.

This keeps the public submission path independent of the model: keyword triage remains available,
and reporting and review continue if the optional artifact is missing or invalid.

## 5.4 Demonstration (in-notebook)

Below: fit the final model on all data, then classify **unseen** sentences — including one Kiswahili
example and one deliberately ambiguous report — to show what the reviewer-facing suggestion looks
like, and to show that the model's margin is low when the text is ambiguous.


In [ ]:
final_model_name, final_scores = select_model(list(range(len(df))), n_splits=5, seed=42)
final_model = build_model(final_model_name).fit(TEXTS, LABELS)

print(f"Selected candidate : {final_model_name}")
print(f"Inner macro F1     : { {k: round(v, 4) for k, v in final_scores.items()} }")
print()

demonstration_samples = [
    # Clear example: an unofficial payment for a permit
    "An officer at the county offices told me to pay 10,000 shillings in cash before he would "
    "sign my business permit.",
    # Clear example: funds unaccounted for
    "The money collected from market stalls every week does not appear in the ward development "
    "account and nobody can explain where it went.",
    # Clear example: tender manipulation
    "The same company won three tenders in one month and all of them were evaluated by the same "
    "committee member.",
    # Kiswahili
    "Afisa aliniomba kitu kidogo kabla ya kunipatia kibali cha biashara yangu.",
    # Ambiguous / insufficient information
    "I have concerns about how things are handled at the office but I do not have details.",
]

def suggest(text, model, top_k=2):
    """Ranked (category, score) suggestions - how a reviewer would see it."""
    if hasattr(model, "decision_function"):
        scores = model.decision_function([text])[0]
        order = scores.argsort()[::-1][:top_k]
        return [(model.classes_[i], float(scores[i])) for i in order]
    return [(model.predict([text])[0], float("nan"))]


print(f"{'SUGGESTION':34s} {'RUNNER-UP':28s} SAMPLE")
print("-" * 118)
for sample in demonstration_samples:
    ranked = suggest(sample, final_model)
    first = f"{ranked[0][0]} ({ranked[0][1]:+.2f})"
    second = f"{ranked[1][0]} ({ranked[1][1]:+.2f})" if len(ranked) > 1 else ""
    print(f"{first:34s} {second:28s} {sample[:58]}")

print()
print("Interpretation")
print("-------------")
print("The suggestion list is advisory. Because the model is a LinearSVC, raw scores are")
print("ranked margins, not probabilities - so the app would show the top suggestion")
print("together with a calibrated confidence once calibration has been measured on real data.")
print("Where the top two scores are close, the case is genuinely ambiguous and the")
print("reviewer's independent judgment decides.")


### How the same triage concept appears in the running application

Walk through these three touchpoints during the live demo:

1. **Reviewer Queue** — each case displays an urgency badge (`Low/Medium/High/Critical`) and the
   suggested category, visible to reviewers only. The queue is sorted urgency-first, then by
   classification age.
2. **Admin overview → Escalations** — flags `High`/`Critical` cases still unclaimed past the
   threshold, timed from *classification completion*, not from submission.
3. **Public statistics** — a category × urgency heatmap (aggregate only) so the public can see where
   triage pressure is concentrated, without exposing any individual case.

### Persisting the artifact


In [ ]:
import joblib

# In Colab this writes to /content/artifacts; in the repository it mirrors ml/artifacts/.
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, artifact_dir / "category_model.joblib")
(artifact_dir / "model_metadata.json").write_text(json.dumps({
    "model": final_model_name,
    "categories": list(CATEGORIES),
    "training_rows": len(df),
    "training_group_count": len(set(GROUPS)),
    "data_status": "synthetic_only_not_for_real_world_validation_or_production_training",
    "real_world_performance_claim": False,
    "note": "Synthetic-only notebook demonstration; the production runtime uses the JSON export from ml/src/train.py.",
}, indent=2), encoding="utf-8")

print("Artifact written to:", artifact_dir.resolve())
print("Files:", sorted(p.name for p in artifact_dir.glob("*.joblib")))
print("NOTE: this synthetic-only artifact is experimental and not validated for real reports.")


---
# Summary

The notebook covers the problem and why ML is assistive; a 2,916-row, 9-category,
incident-grouped synthetic dataset; two pre-declared candidates with nested grouped validation;
accuracy, macro-F1, per-category metrics and visualisations; and a fingerprinted runtime export
with reviewer-only gating and governance limits.

### Final honest statement

> The contribution here is a **rigorous, leakage-resistant, multilingual text-classification
> pipeline** with a documented evaluation and deployment plan, together with a live platform in
> which the ML layer is deliberately constrained to be assistive and auditable. The synthetic
> scores are a *pipeline sanity check* — not a claim about real-world accuracy.
